# Performance of Linear Regression Model in Predicting Fuel Consumption for the Government Fleets
**Guimaras State University — College of Science and Technology**

This notebook reproduces the model-development workflow of the study (Chapter III) in Google Colab:

1. Merge the local trip records (Monthly Report of Official Travel) with the vehicle records by plate number,
   and add the supplementary dataset with the same variables
2. Data cleaning: drop missing values and zero distance; IQR outliers of fuel per 100 km within each engine-power group
3. Encoding (fuel type: 1 = Gasoline, 0 = Diesel) and the interaction term Distance × Engine Power
4. Multicollinearity check with the Variance Inflation Factor (VIF above 5 = high collinearity), with distance and
   engine power mean-centered before the interaction term is formed
5. Train-test split (80/20) and Linear Regression training:
   **ŷ = β0 + β1(Distance) + β2(Engine Power) + β3(Fuel Type) + β4(Distance × Engine Power)**
6. Evaluation with R², MAE and RMSE, and 5-fold cross-validation
7. Residual plot, residual histogram and Durbin-Watson test on the local trips arranged by date of travel
8. Saving the trained model with Joblib for the Flask web application

The code is the same preprocessing and diagnostics code used by the web application
(`model/preprocess.py`, `model/diagnostics.py`, `model/train_model.py`).

In [ ]:
# Colab already includes pandas, numpy, scikit-learn, matplotlib and seaborn.
!pip -q install statsmodels joblib

In [ ]:
import json, os, warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
sns.set_theme(style="whitegrid")
RANDOM_STATE, CV_FOLDS = 42, 5

## 1. Data collection — the datasets (built into this notebook)
This version already contains the data exported from the web application (Model Performance → Training Details):
**trip_records.csv** (Monthly Report of Official Travel), **vehicle_records.csv** (plate number, model, fuel type,
engine power) and **supplementary_data.csv** (6,000 real-world trips from the open online dataset of Fan & Song, 2022,
Mendeley Data, https://doi.org/10.17632/jzjxcdn6k4.2, CC BY 4.0).
The next cell writes the three files, so **Runtime → Run all** trains the model with no file upload.

In [ ]:
# Data exported from the web application. For newer records, download the three CSV files again from
# Model Performance and use fuel_consumption_linear_regression.ipynb (the upload version).
TRIPS_PATH, VEHICLES_PATH, SUPPLEMENTARY_PATH = "trip_records.csv", "vehicle_records.csv", "supplementary_data.csv"
TRIPS_CSV = "trip_date,plate_number,distance_km,actual_fuel_consumption\n2026-01-05,SFA 4714,192.0,19.2\n2026-01-06,SFA 4714,128.0,12.8\n2026-01-07,SFA 4714,192.0,19.2\n2026-01-08,SFA 4714,208.0,20.8\n2026-01-12,SFA 4714,64.0,6.4\n2026-01-13,SFA 4714,100.0,10.0\n2026-01-15,SFA 4714,182.0,18.2\n2026-01-16,SFA 4714,93.0,9.3\n2026-01-17,SFA 4714,64.0,6.4\n2026-01-19,SFA 4714,89.0,8.9\n2026-01-20,SFA 4714,262.0,26.2\n2026-01-21,SFA 4714,332.0,33.2\n2026-01-22,SFA 4714,184.0,18.4\n2026-01-23,SFA 4714,278.0,27.8\n2026-01-28,SFA 4714,196.0,19.6\n2026-01-29,SFA 4714,67.0,6.7\n2026-01-30,SFA 4714,154.0,15.4\n2026-02-03,SFA 4714,32.0,3.2\n2026-02-10,SFA 4714,229.0,22.9\n2026-02-11,SFA 4714,208.0,20.8\n2026-02-13,SFA 4714,64.0,6.4\n2026-02-14,SFA 4714,168.0,16.8\n2026-02-16,SFA 4714,283.0,28.3\n2026-02-18,SFA 4714,202.0,20.2\n2026-02-19,SFA 4714,269.0,26.9\n2026-02-20,SFA 4714,188.0,18.8\n2026-02-21,SFA 4714,262.0,26.2\n2026-02-22,SFA 4714,208.0,20.8\n2026-02-23,SFA 4714,215.0,21.5\n2026-02-24,SFA 4714,233.0,23.3\n2026-02-25,SFA 4714,220.0,22.0\n2026-02-27,SFA 4714,132.0,13.2\n2026-03-02,SFA 4714,256.0,25.6\n2026-03-03,SFA 4714,128.0,12.8\n2026-03-04,SFA 4714,248.0,24.8\n2026-03-05,SFA 4714,270.0,27.0\n2026-03-06,SFA 4714,131.0,13.1\n2026-03-10,SFA 4714,112.0,11.2\n2026-03-11,SFA 4714,151.0,15.1\n2026-03-12,SFA 4714,106.0,10.6\n2026-03-19,SFA 4714,169.0,16.9\n2026-03-20,SFA 4714,362.0,36.2\n2026-03-23,SFA 4714,128.0,12.8\n2026-03-25,SFA 4714,209.0,20.9\n2026-03-30,SFA 4714,134.0,13.4\n2026-04-06,SFA 4714,222.0,22.2\n2026-04-07,SFA 4714,156.0,15.6\n2026-04-14,SFA 4714,67.0,6.7\n2026-04-15,SFA 4714,131.0,13.1\n2026-04-17,SFA 4714,64.0,6.4\n2026-04-20,SFA 4714,136.0,13.6\n2026-04-21,SFA 4714,200.0,20.0\n2026-04-22,SFA 4714,364.0,36.4\n2026-04-27,SFA 4714,134.0,13.4\n2026-04-28,SFA 4714,188.0,18.8\n2026-04-29,SFA 4714,138.0,13.8\n2026-05-04,SFA 4714,210.0,21.0\n2026-05-05,SFA 4714,257.0,25.7\n2026-05-06,SFA 4714,154.0,15.4\n2026-05-11,SFA 4714,146.0,14.6\n2026-05-12,SFA 4714,92.0,9.2\n2026-05-13,SFA 4714,184.0,18.4\n2026-05-14,SFA 4714,196.0,19.6\n2026-05-18,SFA 4714,122.0,12.2\n2026-05-20,SFA 4714,232.0,23.2\n2026-05-21,SFA 4714,126.0,12.6\n2026-05-22,SFA 4714,90.0,9.0\n2026-05-23,SFA 4714,64.0,6.4\n2026-05-24,SFA 4714,76.0,7.6\n2026-05-25,SFA 4714,147.0,14.7\n2026-05-26,SFA 4714,84.0,8.4\n2026-05-30,SFA 4714,148.0,14.8\n2026-06-01,SFA 4714,158.0,15.8\n2026-06-02,SFA 4714,372.0,37.2\n2026-06-04,SFA 4714,202.0,20.2\n2026-06-05,SFA 4714,244.0,24.4\n2026-06-08,SFA 4714,104.0,10.4\n2026-06-09,SFA 4714,163.0,16.3\n2026-06-10,SFA 4714,176.0,17.6\n2026-06-11,SFA 4714,257.0,25.7\n2026-06-13,SFA 4714,173.0,17.3\n2026-06-15,SFA 4714,204.0,20.4\n2026-06-16,SFA 4714,338.0,33.8\n2026-06-18,SFA 4714,315.0,31.5\n2026-06-20,SFA 4714,116.0,11.6\n2026-06-22,SFA 4714,218.0,21.8\n2026-06-24,SFA 4714,158.0,15.8\n2026-06-25,SFA 4714,240.0,24.0\n2026-06-26,SFA 4714,67.0,6.7\n2026-06-27,SFA 4714,131.0,13.1\n2026-06-28,SFA 4714,64.0,6.4\n2026-06-29,SFA 4714,190.0,19.0\n2026-06-30,SFA 4714,248.0,24.8\n"
VEHICLES_CSV = "plate_number,brand_model,fuel_type,engine_power_kw\nSEF 535,L300,Diesel,51.0\nSFA 1172,Toyota - Grandia,Diesel,130.0\nSFA 1173,Toyota Innova,Diesel,128.0\nSFA 1741,Hino Bus,Diesel,177.0\nSFA 4712,Nissan Van,Diesel,95.0\nSFA 4713,Nissan Van,Diesel,95.0\nSFA 4714,Nissan Van,Diesel,95.0\nSGF 810,Revo,Diesel,63.0\nSJC 955,Crosswind,Diesel,62.0\n"
SUPPLEMENTARY_CSV = "brand_model,engine_code,distance_km,engine_power_kw,fuel_type,actual_fuel_consumption\nAudi A6L,0230320,215.48,132,Gasoline,16.0883\nAudi A6L,0230320,40.48,132,Gasoline,3.5847\nAudi A6L,0230320,26.48,132,Gasoline,2.2697\nAudi A6L,0230320,1.76,132,Gasoline,0.2609\nAudi A6L,0230320,11.51,132,Gasoline,1.5043\nAudi A6L,0230320,365.59,132,Gasoline,27.7548\nAudi A6L,0230320,247.57,132,Gasoline,20.2132\nAudi A6L,0230320,190.45,132,Gasoline,15.3557\nAudi A6L,0230320,107.03,132,Gasoline,8.8785\nAudi A6L,0230320,3.22,132,Gasoline,0.3864\nAudi A6L,0230320,32.44,132,Gasoline,2.7163\nAudi A6L,0230320,9.51,132,Gasoline,0.9584\nAudi A6L,0230320,16.78,132,Gasoline,1.847\nAudi A6L,0230320,44.35,132,Gasoline,3.7785\nAudi A6L,0230320,21.86,132,Gasoline,2.1835\nAudi A6L,0230320,3.9,132,Gasoline,0.457\nAudi A6L,0230320,38.12,132,Gasoline,2.4397\nAudi A6L,0230320,2.02,132,Gasoline,0.1775\nAudi A6L,0230320,4.05,132,Gasoline,0.5339\nAudi A6L,0230320,39.66,132,Gasoline,3.3925\nAudi A6L,0230320,54.18,132,Gasoline,4.2552\nAudi A6L,0230320,30.56,132,Gasoline,2.9804\nAudi A6L,0230320,1.79,132,Gasoline,0.2229\nAudi A6L,0230320,1.87,132,Gasoline,0.3021\nAudi A6L,0230320,22.19,132,Gasoline,2.5329\nAudi A6L,0230320,19.32,132,Gasoline,1.7322\nAudi A6L,0230320,64.15,132,Gasoline,5.2771\nAudi A6L,0230320,1.66,132,Gasoline,0.2613\nAudi A6L,0230320,29.89,132,Gasoline,2.4037\nAudi A6L,0230320,48.64,132,Gasoline,3.84\nAudi A6L,0230320,111.94,132,Gasoline,7.7716\nAudi A6L,0230320,25.11,132,Gasoline,3.2213\nAudi A6L,0230320,16.53,132,Gasoline,1.5772\nAudi A6L,0230320,109.67,132,Gasoline,8.069\nAudi A6L,0230320,14.01,132,Gasoline,1.3735\nAudi A6L,0230320,11.39,132,Gasoline,0.7677\nAudi A6L,0230320,241.96,132,Gasoline,15.6353\nAudi A6L,0230320,2.84,132,Gasoline,0.316\nAudi A6L,0230320,145.41,132,Gasoline,10.6009\nAudi A6L,0230320,29.21,132,Gasoline,2.687\nAudi A6L,0230320,78.39,132,Gasoline,5.6223\nAudi A6L,0230320,129.26,132,Gasoline,12.1256\nAudi A6L,0230320,12.82,132,Gasoline,1.117\nAudi A6L,0230320,74.63,132,Gasoline,5.068\nAudi A6L,0230320,17.39,132,Gasoline,1.5978\nAudi A6L,0230320,15.74,132,Gasoline,1.2103\nAudi A6L,0230320,3.1,132,Gasoline,0.4657\nAudi A6L,0230320,13.62,132,Gasoline,1.5662\nAudi A6L,0230320,53.76,132,Gasoline,3.5291\nAudi A6L,0230320,11.37,132,Gasoline,0.8677\nAudi A6L,0230320,224.27,132,Gasoline,18.6642\nAudi A6L,0230320,120.97,132,Gasoline,7.7497\nAudi A6L,0230320,33.77,132,Gasoline,2.6521\nAudi A6L,0230320,123.77,132,Gasoline,9.1391\nAudi A6L,0230320,176.85,132,Gasoline,13.8691\nAudi A6L,0230320,108.06,132,Gasoline,8.7847\nAudi A6L,0230320,178.4,132,Gasoline,9.7168\nAudi A6L,0230320,18.36,132,Gasoline,1.5851\nAudi A6L,0230320,101.58,132,Gasoline,8.0194\nAudi A6L,0230320,11.33,132,Gasoline,1.1549\nAudi A6L,0230320,130.41,132,Gasoline,11.9789\nAudi A6L,0230320,187.03,132,Gasoline,14.2775\nAudi A6L,0230320,14.61,132,Gasoline,1.2908\nAudi A6L,0230320,1.56,132,Gasoline,0.2491\nAudi A6L,0230320,15.03,132,Gasoline,1.3071\nAudi A6L,0230320,224.48,132,Gasoline,16.6309\nAudi A6L,0230320,36.76,132,Gasoline,2.2316\nAudi A6L,0230320,2.72,132,Gasoline,0.5366\nAudi A6L,0230320,34.69,132,Gasoline,2.7991\nAudi A6L,0230320,3.53,132,Gasoline,0.427\nAudi A6L,0230320,1.57,132,Gasoline,0.3124\nAudi A6L,0230320,262.92,132,Gasoline,20.433\nAudi A6L,0230320,2.35,132,Gasoline,0.2913\nAudi A6L,0230320,60.81,132,Gasoline,4.9964\nAudi A6L,0230320,10.46,132,Gasoline,1.107\nAudi A6L,0230320,65.41,132,Gasoline,5.2389\nAudi A6L,0230320,180.78,132,Gasoline,15.8678\nAudi A6L,0230320,40.6,132,Gasoline,2.7386\nAudi A6L,0230320,14.1,132,Gasoline,1.2683\nAudi A6L,0230320,4.16,132,Gasoline,0.4995\nAudi A6L,0230320,1.66,132,Gasoline,0.2257\nAudi A6L,0230320,10.17,132,Gasoline,0.8963\nAudi A6L,0230320,113.05,132,Gasoline,7.9507\nAudi A6L,0230320,4.83,132,Gasoline,0.5886\nAudi A6L,0230320,31.27,132,Gasoline,2.2226\nAudi A6L,0230320,18.58,132,Gasoline,2.1159\nAudi A6L,0230320,16.47,132,Gasoline,1.4051\nAudi A6L,0230320,2.16,132,Gasoline,0.2541\nAudi A6L,0230320,48.47,132,Gasoline,3.7628\nAudi A6L,0230320,41.74,132,Gasoline,2.8347\nAudi A6L,0230320,12.12,132,Gasoline,1.0585\nAudi A6L,0230320,7.03,132,Gasoline,0.7141\nAudi A6L,0230320,107.53,132,Gasoline,7.1811\nAudi A6L,0230320,33.65,132,Gasoline,2.624\nAudi A6L,0230320,38.47,132,Gasoline,3.3452\nAudi A6L,0230320,24.47,132,Gasoline,2.6662\nAudi A6L,0230320,8.34,132,Gasoline,0.9545\nAudi A6L,0230320,3.97,132,Gasoline,0.4315\nAudi A6L,0230320,139.08,132,Gasoline,10.1337\nAudi A6L,0230320,30.4,132,Gasoline,2.6953\nAudi A6L,0230320,27.75,132,Gasoline,2.0094\nAudi A6L,0230320,5.79,132,Gasoline,0.5075\nAudi A6L,0230320,213.5,132,Gasoline,16.5538\nAudi A6L,0230320,9.48,132,Gasoline,1.5891\nAudi A6L,0230320,110.2,132,Gasoline,8.6012\nAudi A6L,0230320,35.92,132,Gasoline,2.6773\nAudi A6L,0230320,130.01,132,Gasoline,7.9117\nAudi A6L,0230320,37.28,132,Gasoline,3.9644\nAudi A6L,0230320,4.84,132,Gasoline,0.5299\nAudi A6L,0230320,3.24,132,Gasoline,0.4708\nAudi A6L,0230320,148.48,132,Gasoline,11.0496\nAudi A6L,0230320,18.03,132,Gasoline,2.0502\nAudi A6L,0230320,6.93,132,Gasoline,0.9642\nAudi A6L,0230320,310.66,132,Gasoline,27.0892\nAudi A6L,0230320,116.53,132,Gasoline,7.1921\nAudi A6L,0230320,76.04,132,Gasoline,5.6172\nAudi A6L,0230320,5.68,132,Gasoline,0.9818\nAudi A6L,0230320,50.8,132,Gasoline,4.4675\nAudi A6L,0230320,291.67,132,Gasoline,22.1184\nAudi A6L,0230320,8.19,132,Gasoline,0.6438\nAudi A6L,0230320,111.91,132,Gasoline,8.4068\nAudi A6L,0230320,236.5,132,Gasoline,19.94\nAudi A6L,0230320,34.9,132,Gasoline,3.7289\nAudi A6L,0230320,125.24,132,Gasoline,10.4103\nAudi A6L,0230320,12.42,132,Gasoline,1.2801\nAudi A6L,0230320,12.58,132,Gasoline,1.6393\nAudi A6L,0230320,9.83,132,Gasoline,1.3175\nAudi A6L,0230320,3.7,132,Gasoline,0.5073\nAudi A6L,0230320,56.74,132,Gasoline,4.4455\nAudi A6L,0230320,35.53,132,Gasoline,3.1047\nAudi A6L,0230320,9.74,132,Gasoline,1.0661\nAudi A6L,0230320,11.2,132,Gasoline,1.3557\nAudi A6L,0230320,104.2,132,Gasoline,8.624\nAudi A6L,0230320,38.57,132,Gasoline,2.7532\nAudi A6L,0230320,45.82,132,Gasoline,3.0239\nAudi A6L,0230320,111.33,132,Gasoline,7.0721\nAudi A6L,0230320,123.28,132,Gasoline,8.2513\nAudi A6L,0230320,35.77,132,Gasoline,2.475\nAudi A6L,0230320,10.18,132,Gasoline,1.0\nAudi A6L,0230320,268.99,132,Gasoline,21.1727\nAudi A6L,0230320,34.53,132,Gasoline,2.7078\nAudi A6L,0230320,7.34,132,Gasoline,0.7113\nAudi A6L,0230320,245.18,132,Gasoline,18.378\nAudi A6L,0230320,4.55,132,Gasoline,0.453\nAudi A6L,0230320,45.79,132,Gasoline,4.3665\nAudi A6L,0230320,13.94,132,Gasoline,1.0691\nAudi A6L,0230320,7.77,132,Gasoline,0.73\nAudi A6L,0230320,13.84,132,Gasoline,1.1089\nAudi A6L,0230320,5.5,132,Gasoline,0.7176\nAudi A6L,0230320,26.49,132,Gasoline,2.0863\nAudi A6L,0230320,18.57,132,Gasoline,1.5704\nAudi A6L,0230320,118.9,132,Gasoline,8.5196\nAudi A6L,0230320,173.3,132,Gasoline,11.6554\nAudi A6L,0230320,171.79,132,Gasoline,12.2762\nAudi A6L,0230320,14.77,132,Gasoline,1.3773\nAudi A6L,0230320,14.01,132,Gasoline,1.5948\nAudi A6L,0230320,64.55,132,Gasoline,4.593\nAudi A6L,0230320,40.7,132,Gasoline,3.4677\nAudi A6L,0230320,14.09,132,Gasoline,1.2103\nAudi A6L,0230320,8.7,132,Gasoline,0.7054\nAudi A6L,0230320,117.03,132,Gasoline,8.6118\nAudi A6L,0230320,107.42,132,Gasoline,10.8329\nAudi A6L,0230320,228.12,132,Gasoline,20.1162\nAudi A6L,0230320,16.0,132,Gasoline,1.229\nAudi A6L,0230320,67.76,132,Gasoline,7.3445\nAudi A6L,0230320,131.31,132,Gasoline,10.5113\nAudi A6L,0230320,72.88,132,Gasoline,5.719\nAudi A6L,0230320,14.38,132,Gasoline,1.0647\nAudi A6L,0230320,3.66,132,Gasoline,0.5916\nAudi A6L,0230320,9.71,132,Gasoline,1.4472\nAudi A6L,0230320,7.37,132,Gasoline,0.7618\nAudi A6L,0230320,20.2,132,Gasoline,3.6218\nAudi A6L,0230320,192.22,132,Gasoline,14.1293\nAudi A6L,0230320,25.48,132,Gasoline,1.9923\nAudi A6L,0230320,21.23,132,Gasoline,1.8974\nAudi A6L,0230320,101.08,132,Gasoline,6.375\nAudi A6L,0230320,1.75,132,Gasoline,0.1914\nAudi A6L,0230320,114.61,132,Gasoline,9.8444\nAudi A6L,0230320,1.63,132,Gasoline,0.1714\nAudi A6L,0230320,18.87,132,Gasoline,1.7714\nAudi A6L,0230320,53.17,132,Gasoline,4.262\nAudi A6L,0230320,139.45,132,Gasoline,10.3853\nAudi A6L,0230320,33.82,132,Gasoline,2.3666\nAudi A6L,0230320,132.62,132,Gasoline,9.0778\nAudi A6L,0230320,3.11,132,Gasoline,0.3203\nAudi A6L,0230320,126.88,132,Gasoline,9.609\nAudi A6L,0230320,206.2,132,Gasoline,14.0531\nAudi A6L,0230320,236.58,132,Gasoline,20.3705\nAudi A6L,0230320,26.11,132,Gasoline,2.0573\nAudi A6L,0230320,107.83,132,Gasoline,10.2177\nAudi A6L,0230320,6.7,132,Gasoline,0.8193\nAudi A6L,0230320,116.81,132,Gasoline,8.8965\nAudi A6L,0230320,40.26,132,Gasoline,4.5143\nAudi A6L,0230320,108.29,132,Gasoline,8.0304\nAudi A6L,0230320,63.18,132,Gasoline,4.7048\nAudi A6L,0230320,39.28,132,Gasoline,2.5892\nAudi A6L,0230320,1.95,132,Gasoline,0.3132\nAudi A6L,0230320,176.2,132,Gasoline,10.7678\nAudi A6L,0230320,125.8,132,Gasoline,9.2207\nAudi A6L,0230320,135.11,132,Gasoline,10.2032\nAudi A6L,0230320,180.44,132,Gasoline,12.908\nAudi A6L,0230320,40.26,132,Gasoline,3.0448\nAudi A6L,0230320,51.02,132,Gasoline,2.9633\nAudi A6L,0230320,71.71,132,Gasoline,4.4702\nAudi A6L,0230320,192.97,132,Gasoline,12.4272\nAudi A6L,0230320,133.74,132,Gasoline,10.1712\nAudi A6L,0230320,60.5,132,Gasoline,3.977\nAudi A6L,0230320,179.52,132,Gasoline,14.3018\nAudi A6L,0230320,3.72,132,Gasoline,0.4887\nAudi A6L,0230320,14.54,132,Gasoline,1.3609\nAudi A6L,0230320,56.37,132,Gasoline,4.1154\nAudi A6L,0230320,28.52,132,Gasoline,1.869\nAudi A6L,0230320,2.62,132,Gasoline,0.2709\nAudi A6L,0230320,109.78,132,Gasoline,8.1106\nAudi A6L,0230320,38.67,132,Gasoline,2.9346\nAudi A6L,0230320,21.68,132,Gasoline,2.3623\nAudi A6L,0230320,8.27,132,Gasoline,0.6579\nAudi A6L,0230320,20.26,132,Gasoline,1.981\nAudi A6L,0230320,33.33,132,Gasoline,3.2525\nAudi A6L,0230320,44.0,132,Gasoline,3.5871\nAudi A6L,0230320,115.74,132,Gasoline,8.4691\nAudi A6L,0230320,72.48,132,Gasoline,6.2348\nAudi A6L,0230320,7.1,132,Gasoline,0.716\nAudi A6L,0230320,27.08,132,Gasoline,2.3751\nAudi A6L,0230320,4.16,132,Gasoline,0.3305\nAudi A6L,0230320,9.35,132,Gasoline,1.1529\nAudi A6L,0230320,173.95,132,Gasoline,13.9229\nAudi A6L,0230320,25.57,132,Gasoline,2.323\nAudi A6L,0230320,132.12,132,Gasoline,10.0345\nAudi A6L,0230320,46.83,132,Gasoline,4.4998\nAudi A6L,0230320,11.55,132,Gasoline,1.3823\nAudi A6L,0230320,108.55,132,Gasoline,7.8673\nAudi A6L,0230320,102.66,132,Gasoline,7.3959\nAudi A6L,0230320,11.4,132,Gasoline,1.3623\nAudi A6L,0230320,10.03,132,Gasoline,1.8521\nAudi A6L,0230320,100.7,132,Gasoline,6.1142\nAudi A6L,0230320,38.01,132,Gasoline,3.017\nAudi A6L,0230320,5.66,132,Gasoline,0.6358\nAudi A6L,0230320,13.6,132,Gasoline,1.4658\nAudi A6L,0230320,53.74,132,Gasoline,3.6898\nAudi A6L,0230320,63.77,132,Gasoline,3.9333\nAudi A6L,0230320,26.73,132,Gasoline,2.3538\nAudi A6L,0230320,3.39,132,Gasoline,0.4882\nAudi A6L,0230320,14.71,132,Gasoline,1.9567\nAudi A6L,0230320,115.54,132,Gasoline,8.5386\nAudi A6L,0230320,120.9,132,Gasoline,9.6646\nAudi A6L,0230320,11.26,132,Gasoline,1.1441\nAudi A6L,0230320,2.88,132,Gasoline,0.3909\nAudi A6L,0230320,13.9,132,Gasoline,1.3107\nAudi A6L,0230320,2.09,132,Gasoline,0.4946\nAudi A6L,0230320,22.68,132,Gasoline,1.281\nAudi A6L,0230320,15.76,132,Gasoline,1.5477\nAudi A6L,0230320,14.64,132,Gasoline,1.4685\nAudi A6L,0230320,247.58,132,Gasoline,16.2676\nAudi A6L,0230320,4.85,132,Gasoline,0.4833\nAudi A6L,0230320,51.2,132,Gasoline,3.9707\nAudi A6L,0230320,11.78,132,Gasoline,1.2744\nAudi A6L,0230320,15.08,132,Gasoline,2.1501\nAudi A6L,0230320,13.81,132,Gasoline,1.4074\nAudi A6L,0230320,7.8,132,Gasoline,0.9612\nAudi A6L,0230320,3.72,132,Gasoline,0.4413\nAudi A6L,0230320,4.72,132,Gasoline,0.8199\nAudi A6L,0230320,1.53,132,Gasoline,0.1923\nAudi A6L,0230320,32.29,132,Gasoline,2.1023\nAudi A6L,0230320,12.8,132,Gasoline,2.1154\nAudi A6L,0230320,19.4,132,Gasoline,2.2957\nAudi A6L,0230320,3.31,132,Gasoline,0.3848\nAudi A6L,0230320,11.64,132,Gasoline,1.1026\nAudi A6L,0230320,5.7,132,Gasoline,0.553\nAudi A6L,0230320,52.54,132,Gasoline,3.445\nAudi A6L,0230320,57.61,132,Gasoline,3.9151\nAudi A6L,0230320,21.84,132,Gasoline,1.5382\nAudi A6L,0230320,3.1,132,Gasoline,0.3203\nAudi A6L,0230320,2.9,132,Gasoline,0.4789\nAudi A6L,0230320,220.8,132,Gasoline,20.9861\nAudi A6L,0230320,29.55,132,Gasoline,3.0092\nAudi A6L,0230320,102.32,132,Gasoline,8.5006\nAudi A6L,0230320,2.29,132,Gasoline,0.3074\nAudi A6L,0230320,5.3,132,Gasoline,0.9445\nAudi A6L,0230320,54.18,132,Gasoline,4.6178\nAudi A6L,0230320,18.8,132,Gasoline,1.8567\nAudi A6L,0230320,316.3,132,Gasoline,23.9566\nAudi A6L,0230320,29.17,132,Gasoline,2.2288\nAudi A6L,0230320,107.65,132,Gasoline,8.6522\nAudi A6L,0230320,109.14,132,Gasoline,6.3804\nAudi A6L,0230320,2.76,132,Gasoline,0.4158\nAudi A6L,0230320,3.69,132,Gasoline,0.4057\nAudi A6L,0230320,1.9,132,Gasoline,0.2822\nAudi A6L,0230320,45.68,132,Gasoline,2.9552\nAudi A6L,0230320,51.28,132,Gasoline,4.3882\nAudi A6L,0230320,15.3,132,Gasoline,1.4257\nAudi A6L,0230320,36.28,132,Gasoline,2.9148\nAudi A6L,0230320,22.47,132,Gasoline,2.23\nAudi A6L,0230320,143.85,132,Gasoline,11.0219\nAudi A6L,0230320,3.11,132,Gasoline,0.2956\nAudi A6L,0230320,3.24,132,Gasoline,0.2725\nAudi A6L,0230320,46.28,132,Gasoline,4.0697\nAudi A6L,0230320,12.17,132,Gasoline,1.1873\nAudi A6L,0230320,300.7,132,Gasoline,19.3971\nAudi A6L,0230320,27.12,132,Gasoline,3.1641\nAudi A6L,0230320,25.44,132,Gasoline,2.7205\nAudi A6L,0230320,109.37,132,Gasoline,8.6245\nAudi A6L,0230320,134.23,132,Gasoline,9.4628\nAudi A6L,0230320,3.22,132,Gasoline,0.3299\nAudi A6L,0230320,14.49,132,Gasoline,2.2192\nAudi A6L,0230320,272.18,132,Gasoline,19.0231\nAudi A6L,0230320,189.74,132,Gasoline,16.8495\nAudi A6L,0230320,6.41,132,Gasoline,0.6172\nAudi A6L,0230320,30.5,132,Gasoline,2.892\nAudi A6L,0230320,348.32,132,Gasoline,28.2983\nAudi A6L,0230320,106.84,132,Gasoline,8.8672\nAudi A6L,0230320,2.88,132,Gasoline,0.5467\nAudi A6L,0230320,3.15,132,Gasoline,0.4052\nAudi A6L,0230320,7.97,132,Gasoline,0.8404\nAudi A6L,0230320,248.7,132,Gasoline,17.1284\nAudi A6L,0230320,132.51,132,Gasoline,7.8378\nAudi A6L,0230320,131.55,132,Gasoline,8.6656\nAudi A6L,0230320,39.92,132,Gasoline,3.4628\nAudi A6L,0230320,217.16,132,Gasoline,13.9794\nAudi A6L,0230320,436.12,132,Gasoline,39.223\nAudi A6L,0230320,2.72,132,Gasoline,0.2077\nAudi A6L,0230320,1.76,132,Gasoline,0.288\nAudi A6L,0230320,115.93,132,Gasoline,8.3081\nAudi A6L,0230320,43.87,132,Gasoline,3.146\nAudi A6L,0230320,88.47,132,Gasoline,7.6298\nAudi A6L,0230320,368.06,132,Gasoline,29.4592\nAudi A6L,0230320,12.76,132,Gasoline,1.3307\nAudi A6L,0230320,1.85,132,Gasoline,0.3069\nAudi A6L,0230320,3.46,132,Gasoline,0.5199\nAudi A6L,0230320,100.44,132,Gasoline,7.392\nAudi A6L,0230320,2.56,132,Gasoline,0.343\nAudi A6L,0230320,35.04,132,Gasoline,2.906\nAudi A6L,0230320,37.92,132,Gasoline,3.2537\nAudi A6L,0230320,6.74,132,Gasoline,0.9846\nAudi A6L,0230320,11.15,132,Gasoline,1.1161\nAudi A6L,0230320,1.94,132,Gasoline,0.3104\nAudi A6L,0230320,37.69,132,Gasoline,3.9224\nAudi A6L,0230320,5.16,132,Gasoline,0.5998\nAudi A6L,0230320,3.06,132,Gasoline,0.4577\nAudi A6L,0230320,5.28,132,Gasoline,0.9769\nAudi A6L,0230320,27.99,132,Gasoline,2.6935\nAudi A6L,0230320,101.84,132,Gasoline,8.0598\nAudi A6L,0230320,82.18,132,Gasoline,6.6167\nAudi A6L,0230320,115.06,132,Gasoline,6.8374\nAudi A6L,0230320,16.34,132,Gasoline,1.2201\nAudi A6L,0230320,15.85,132,Gasoline,1.742\nAudi A6L,0230320,77.45,132,Gasoline,8.3384\nAudi A6L,0230320,3.67,132,Gasoline,0.3783\nAudi A6L,0230320,34.14,132,Gasoline,2.6273\nAudi A6L,0230320,51.56,132,Gasoline,4.5136\nAudi A6L,0230320,11.6,132,Gasoline,0.8774\nAudi A6L,0230320,63.21,132,Gasoline,6.2665\nAudi A6L,0230320,8.54,132,Gasoline,0.8703\nAudi A6L,0230320,7.63,132,Gasoline,0.966\nAudi A6L,0230320,3.49,132,Gasoline,0.3962\nAudi A6L,0230320,11.02,132,Gasoline,1.1251\nAudi A6L,0230320,12.26,132,Gasoline,1.3877\nAudi A6L,0230320,38.31,132,Gasoline,2.4162\nAudi A6L,0230320,35.38,132,Gasoline,3.3516\nAudi A6L,0230320,57.89,132,Gasoline,6.6367\nAudi A6L,0230320,104.22,132,Gasoline,8.9912\nAudi A6L,0230320,41.43,132,Gasoline,3.2512\nAudi A6L,0230320,185.41,132,Gasoline,11.8851\nAudi A6L,0230320,10.09,132,Gasoline,1.3963\nAudi A6L,0230320,49.95,132,Gasoline,3.9009\nAudi A6L,0230320,44.35,132,Gasoline,3.6481\nAudi A6L,0230320,181.44,132,Gasoline,12.8012\nAudi A6L,0230320,112.46,132,Gasoline,7.7936\nAudi A6L,0230320,5.31,132,Gasoline,0.7344\nAudi A6L,0230320,112.18,132,Gasoline,8.2846\nAudi A6L,0230320,21.81,132,Gasoline,2.9564\nAudi A6L,0230320,117.77,132,Gasoline,9.382\nAudi A6L,0230320,31.8,132,Gasoline,2.565\nAudi A6L,0230320,39.43,132,Gasoline,3.2216\nAudi A6L,0230320,114.46,132,Gasoline,10.078\nAudi A6L,0230320,83.46,132,Gasoline,6.5011\nAudi A6L,0230320,38.29,132,Gasoline,2.5789\nAudi A6L,0230320,218.28,132,Gasoline,19.4136\nAudi A6L,0230320,19.7,132,Gasoline,1.881\nAudi A6L,0230320,214.51,132,Gasoline,19.1199\nAudi A6L,0230320,8.2,132,Gasoline,0.766\nAudi A6L,0230320,106.68,132,Gasoline,7.3155\nAudi A6L,0230320,78.35,132,Gasoline,5.0278\nAudi A6L,0230320,46.86,132,Gasoline,3.179\nAudi A6L,0230320,192.72,132,Gasoline,11.8529\nAudi A6L,0230320,111.66,132,Gasoline,9.0867\nAudi A6L,0230320,36.77,132,Gasoline,3.7966\nAudi A6L,0230320,73.67,132,Gasoline,7.7337\nAudi A6L,0230320,27.66,132,Gasoline,2.6146\nAudi A6L,0230320,36.4,132,Gasoline,2.4063\nAudi A6L,0230320,38.69,132,Gasoline,3.2153\nAudi A6L,0230320,9.01,132,Gasoline,0.9538\nAudi A6L,0230320,38.19,132,Gasoline,2.4118\nAudi A6L,0230320,48.55,132,Gasoline,3.3733\nAudi A6L,0230320,120.81,132,Gasoline,10.7917\nAudi A6L,0230320,38.47,132,Gasoline,3.255\nAudi A6L,0230320,9.14,132,Gasoline,0.756\nAudi A6L,0230320,31.69,132,Gasoline,3.0481\nAudi A6L,0230320,9.22,132,Gasoline,0.82\nAudi A6L,0230320,1.62,132,Gasoline,0.2187\nAudi A6L,0230320,51.59,132,Gasoline,4.4359\nAudi A6L,0230320,10.69,132,Gasoline,0.811\nAudi A6L,0230320,172.1,132,Gasoline,12.9876\nAudi A6L,0230320,53.93,132,Gasoline,4.0613\nAudi A6L,0230320,8.48,132,Gasoline,1.1132\nAudi A6L,0230320,106.13,132,Gasoline,6.6247\nAudi A6L,0230320,65.89,132,Gasoline,6.0436\nAudi A6L,0230320,203.51,132,Gasoline,17.7436\nAudi A6L,0230320,12.2,132,Gasoline,1.0568\nAudi A6L,0230320,5.93,132,Gasoline,0.8834\nAudi A6L,0230320,57.33,132,Gasoline,3.5716\nAudi A6L,0230320,2.64,132,Gasoline,0.3694\nAudi A6L,0230320,24.3,132,Gasoline,2.2863\nAudi A6L,0230320,21.75,132,Gasoline,2.0833\nAudi A6L,0230320,16.52,132,Gasoline,1.6085\nAudi A6L,0230320,4.61,132,Gasoline,0.43\nAudi A6L,0230320,39.19,132,Gasoline,2.5581\nAudi A6L,0230320,37.15,132,Gasoline,2.5563\nAudi A6L,0230320,1.8,132,Gasoline,0.3066\nAudi A6L,0230320,14.62,132,Gasoline,1.5482\nAudi A6L,0230320,11.28,132,Gasoline,1.4191\nAudi A6L,0230320,30.05,132,Gasoline,1.7631\nAudi A6L,0230320,10.83,132,Gasoline,1.2294\nAudi A6L,0230320,70.92,132,Gasoline,5.2745\nAudi A6L,0230320,12.2,132,Gasoline,2.31\nAudi A6L,0230320,1.95,132,Gasoline,0.3195\nAudi A6L,0230320,24.07,132,Gasoline,1.9433\nAudi A6L,0230320,3.96,132,Gasoline,0.4283\nAudi A6L,0230320,16.09,132,Gasoline,2.1265\nAudi A6L,0230320,15.57,132,Gasoline,1.4013\nAudi A6L,0230320,18.78,132,Gasoline,1.7511\nAudi A6L,0230320,30.02,132,Gasoline,1.9299\nAudi A6L,0230320,2.9,132,Gasoline,0.3366\nAudi A6L,0230320,105.08,132,Gasoline,6.6875\nAudi A6L,0230320,14.93,132,Gasoline,1.1678\nAudi A6L,0230320,12.8,132,Gasoline,1.4845\nAudi A6L,0230320,5.67,132,Gasoline,0.8137\nAudi A6L,0230320,34.27,132,Gasoline,3.1533\nAudi A6L,0230320,59.15,132,Gasoline,6.1616\nAudi A6L,0230320,80.69,132,Gasoline,5.9152\nAudi A6L,0230320,2.51,132,Gasoline,0.3597\nAudi A6L,0230320,41.84,132,Gasoline,3.8418\nAudi A6L,0230320,137.45,132,Gasoline,9.7069\nAudi A6L,0230320,34.65,132,Gasoline,2.7641\nAudi A6L,0230320,1.85,132,Gasoline,0.2509\nAudi A6L,0230320,50.08,132,Gasoline,3.4452\nAudi A6L,0230320,21.1,132,Gasoline,2.1327\nAudi A6L,0230320,2.05,132,Gasoline,0.4034\nAudi A6L,0230320,43.76,132,Gasoline,3.3217\nAudi A6L,0230320,3.02,132,Gasoline,0.4575\nAudi A6L,0230320,30.04,132,Gasoline,1.7355\nAudi A6L,0230320,12.95,132,Gasoline,1.1632\nAudi A6L,0230320,4.45,132,Gasoline,0.9413\nAudi A6L,0230320,19.14,132,Gasoline,1.6608\nAudi A6L,0230320,109.49,132,Gasoline,7.7395\nAudi A6L,0230320,70.58,132,Gasoline,5.2323\nAudi A6L,0230320,12.12,132,Gasoline,1.3823\nAudi A6L,0230320,209.8,132,Gasoline,15.5746\nAudi A6L,0230320,105.56,132,Gasoline,6.5057\nAudi A6L,0230320,157.44,132,Gasoline,11.2472\nAudi A6L,0230320,688.5,132,Gasoline,52.5943\nAudi A6L,0230320,116.39,132,Gasoline,8.919\nAudi A6L,0230320,227.34,132,Gasoline,16.555\nAudi A6L,0230320,5.33,132,Gasoline,0.5805\nAudi A6L,0230320,34.65,132,Gasoline,2.8072\nAudi A6L,0230320,58.93,132,Gasoline,4.5212\nAudi A6L,0230320,32.7,132,Gasoline,2.6604\nAudi A6L,0230320,11.89,132,Gasoline,1.1423\nAudi A6L,0230320,117.84,132,Gasoline,8.8975\nAudi A6L,0230320,36.48,132,Gasoline,2.4482\nAudi A6L,0230320,15.78,132,Gasoline,1.3893\nAudi A6L,0230320,8.02,132,Gasoline,0.6241\nAudi A6L,0230320,134.9,132,Gasoline,11.7876\nAudi A6L,0230320,150.63,132,Gasoline,11.56\nAudi A6L,0230320,28.99,132,Gasoline,2.8687\nAudi A6L,0230320,8.72,132,Gasoline,0.9234\nAudi A6L,0230320,38.27,132,Gasoline,2.9671\nAudi A6L,0230320,6.3,132,Gasoline,0.9715\nAudi A6L,0230320,19.84,132,Gasoline,2.1133\nAudi A6L,0230320,249.56,132,Gasoline,19.1281\nAudi A6L,0230320,24.34,132,Gasoline,2.6748\nAudi A6L,0230320,6.28,132,Gasoline,0.8705\nAudi A6L,0230320,111.52,132,Gasoline,7.7422\nAudi A6L,0230320,240.03,132,Gasoline,17.7071\nAudi A6L,0230320,43.29,132,Gasoline,4.4971\nAudi A6L,0230320,7.48,132,Gasoline,0.6048\nAudi A6L,0230320,31.75,132,Gasoline,2.7867\nAudi A6L,0230320,11.84,132,Gasoline,1.2596\nAudi A6L,0230320,16.96,132,Gasoline,2.4749\nAudi A6L,0230320,32.19,132,Gasoline,2.8324\nAudi A6L,0230320,3.12,132,Gasoline,0.4498\nAudi A6L,0230320,282.26,132,Gasoline,24.5944\nAudi A6L,0230320,179.88,132,Gasoline,15.9647\nAudi A6L,0230320,2.39,132,Gasoline,0.4255\nAudi A6L,0230320,3.52,132,Gasoline,0.3081\nAudi A6L,0230320,4.88,132,Gasoline,0.3973\nAudi A6L,0230320,21.87,132,Gasoline,2.547\nAudi A6L,0230320,16.26,132,Gasoline,1.6089\nAudi A6L,0230320,48.49,132,Gasoline,3.3045\nAudi A6L,0230320,406.93,132,Gasoline,34.3304\nAudi A6L,0230320,36.99,132,Gasoline,2.4066\nAudi A6L,0230320,9.95,132,Gasoline,1.0429\nAudi A6L,0230320,1.97,132,Gasoline,0.2563\nAudi A6L,0230320,108.29,132,Gasoline,8.989\nAudi A6L,0230320,82.22,132,Gasoline,4.7207\nAudi A6L,0230320,38.19,132,Gasoline,3.9237\nAudi A6L,0230320,11.01,132,Gasoline,0.9516\nAudi A6L,0230320,14.14,132,Gasoline,1.1029\nAudi A6L,0230320,1.65,132,Gasoline,0.2146\nAudi A6L,0230320,66.02,132,Gasoline,6.8009\nAudi A6L,0230320,3.8,132,Gasoline,0.3915\nAudi A6L,0230320,5.52,132,Gasoline,0.8386\nAudi A6L,0230320,14.37,132,Gasoline,1.3496\nAudi A6L,0230320,137.63,132,Gasoline,12.7663\nAudi A6L,0230320,39.37,132,Gasoline,4.9225\nAudi A6L,0230320,106.98,132,Gasoline,7.4942\nAudi A6L,0230320,14.84,132,Gasoline,1.2853\nAudi A6L,0230320,124.73,132,Gasoline,9.8281\nAudi A6L,0230320,1.96,132,Gasoline,0.2903\nAudi A6L,0230320,12.67,132,Gasoline,1.2545\nAudi A6L,0230320,70.28,132,Gasoline,5.4759\nAudi A6L,0230320,34.39,132,Gasoline,3.2519\nAudi A6L,0230320,16.53,132,Gasoline,2.1946\nAudi A6L,0230320,110.83,132,Gasoline,7.5094\nAudi A6L,0230320,1.65,132,Gasoline,0.2311\nAudi A6L,0230320,6.44,132,Gasoline,0.8208\nAudi A6L,0230320,74.79,132,Gasoline,4.5513\nAudi A6L,0230320,35.83,132,Gasoline,3.7367\nAudi A6L,0230320,61.81,132,Gasoline,4.7685\nAudi A6L,0230320,3.25,132,Gasoline,0.2993\nAudi A6L,0230320,167.41,132,Gasoline,11.9213\nAudi A6L,0230320,8.24,132,Gasoline,0.8941\nAudi A6L,0230320,24.7,132,Gasoline,2.4554\nAudi A6L,0230320,39.97,132,Gasoline,2.5589\nAudi A6L,0230320,8.68,132,Gasoline,1.1932\nAudi A6L,0230320,2.29,132,Gasoline,0.2824\nAudi A6L,0230320,39.61,132,Gasoline,3.4225\nAudi A6L,0230320,23.42,132,Gasoline,2.1838\nAudi A6L,0230320,35.58,132,Gasoline,2.6459\nAudi A6L,0230320,51.83,132,Gasoline,3.7581\nAudi A6L,0230320,167.97,132,Gasoline,11.3649\nAudi A6L,0230320,17.1,132,Gasoline,1.176\nAudi A6L,0230320,19.26,132,Gasoline,1.2893\nAudi A6L,0230320,15.25,132,Gasoline,1.5534\nAudi A6L,0230320,2.22,132,Gasoline,0.2398\nAudi A6L,0230320,5.67,132,Gasoline,0.6421\nAudi A6L,0230320,57.45,132,Gasoline,3.347\nAudi A6L,0230320,17.39,132,Gasoline,1.7296\nAudi A6L,0230320,13.49,132,Gasoline,0.8977\nAudi A6L,0230320,2.82,132,Gasoline,0.3617\nAudi A6L,0230320,2.54,132,Gasoline,0.3318\nAudi A6L,0230320,65.58,132,Gasoline,5.6098\nAudi A6L,0230320,6.93,132,Gasoline,0.5998\nAudi A6L,0230320,19.66,132,Gasoline,1.7047\nAudi A6L,0230320,5.72,132,Gasoline,0.5852\nAudi A6L,0230320,37.09,132,Gasoline,2.9986\nAudi A6L,0230320,2.85,132,Gasoline,0.3664\nAudi A6L,0230320,3.87,132,Gasoline,0.4543\nAudi A6L,0230320,87.4,132,Gasoline,7.547\nAudi A6L,0230320,17.91,132,Gasoline,1.7502\nAudi A6L,0230320,71.12,132,Gasoline,5.8845\nAudi A6L,0230320,120.22,132,Gasoline,9.3538\nAudi A6L,0230320,10.85,132,Gasoline,1.0188\nAudi A6L,0230320,19.36,132,Gasoline,2.0177\nAudi A6L,0230320,20.27,132,Gasoline,1.8864\nAudi A6L,0230320,13.7,132,Gasoline,1.186\nAudi A6L,0230320,29.86,132,Gasoline,2.339\nAudi A6L,0230320,22.72,132,Gasoline,2.0908\nAudi A6L,0230320,3.87,132,Gasoline,0.4315\nAudi A6L,0230320,6.6,132,Gasoline,0.938\nAudi A6L,0230320,21.86,132,Gasoline,2.2223\nAudi A6L,0230320,101.97,132,Gasoline,7.8444\nAudi A6L,0230320,6.28,132,Gasoline,0.7994\nAudi A6L,0230320,29.96,132,Gasoline,2.2743\nAudi A6L,0230320,18.19,132,Gasoline,1.5144\nAudi A6L,0230320,207.73,132,Gasoline,14.3691\nAudi A6L,0230320,5.61,132,Gasoline,0.8353\nAudi A6L,0230320,41.16,132,Gasoline,4.1724\nAudi A6L,0230320,21.34,132,Gasoline,2.2066\nAudi A6L,0230320,113.83,132,Gasoline,7.7057\nAudi A6L,0230320,30.84,132,Gasoline,2.124\nAudi A6L,0230320,67.39,132,Gasoline,6.1728\nAudi A6L,0230320,105.37,132,Gasoline,6.9805\nAudi A6L,0230320,64.93,132,Gasoline,4.2989\nAudi A6L,0230320,5.81,132,Gasoline,0.5844\nAudi A6L,0230320,8.46,132,Gasoline,0.9015\nAudi A6L,0230320,112.28,132,Gasoline,8.5624\nAudi A6L,0230320,101.98,132,Gasoline,7.7148\nAudi A6L,0230320,35.49,132,Gasoline,3.298\nAudi A6L,0230320,13.59,132,Gasoline,1.3572\nAudi A6L,0230320,7.05,132,Gasoline,0.9102\nAudi A6L,0230320,10.05,132,Gasoline,1.4385\nAudi A6L,0230320,112.05,132,Gasoline,8.5982\nAudi A6L,0230320,17.87,132,Gasoline,1.8764\nAudi A6L,0230320,168.95,132,Gasoline,13.9068\nAudi A6L,0230320,134.26,132,Gasoline,11.216\nAudi A6L,0230320,2.84,132,Gasoline,0.3876\nAudi A6L,0230320,116.82,132,Gasoline,8.4575\nAudi A6L,0230320,2.34,132,Gasoline,0.3958\nAudi A6L,0230320,17.33,132,Gasoline,1.6484\nBuick GL8,0B2020W,32.03,123,Gasoline,2.8621\nBuick GL8,0B2020W,3.28,123,Gasoline,0.6342\nBuick GL8,0B2020W,61.66,123,Gasoline,5.5259\nBuick GL8,0B2020W,59.82,123,Gasoline,5.6814\nBuick GL8,0B2021Y,8.1,147,Gasoline,1.1003\nBuick GL8,0B2020W,3.48,123,Gasoline,0.5445\nBuick GL8,0B2021Y,103.66,147,Gasoline,11.5496\nBuick GL8,0B2021Y,4.92,147,Gasoline,0.5473\nBuick GL8,0B2020W,1.54,123,Gasoline,0.1856\nBuick GL8,0B2020W,1.73,123,Gasoline,0.3877\nBuick GL8,0B2020W,65.58,123,Gasoline,6.8058\nBuick GL8,0B2020W,42.64,123,Gasoline,5.908\nBuick GL8,0B2020W,12.59,123,Gasoline,1.5907\nBuick GL8,0B2020W,32.52,123,Gasoline,2.85\nBuick GL8,0B2020W,78.22,123,Gasoline,8.1535\nBuick GL8,0B2020W,5.66,123,Gasoline,0.5749\nBuick GL8,0B2021Y,189.04,147,Gasoline,15.2896\nBuick GL8,0B2020W,170.14,123,Gasoline,17.2718\nBuick GL8,0B2020W,19.91,123,Gasoline,2.5226\nBuick GL8,0B2020W,100.44,123,Gasoline,13.6065\nBuick GL8,0B2020W,15.82,123,Gasoline,1.8115\nBuick GL8,0B2020W,6.8,123,Gasoline,0.9099\nBuick GL8,0B2020W,16.43,123,Gasoline,2.3731\nBuick GL8,0B2021Y,2.8,147,Gasoline,0.4813\nBuick GL8,0B2021Y,4.43,147,Gasoline,0.5208\nBuick GL8,0B2020W,3.68,123,Gasoline,0.6225\nBuick GL8,0B2020W,7.68,123,Gasoline,1.0144\nBuick GL8,0B2020W,32.5,123,Gasoline,3.984\nBuick GL8,0B2020W,9.1,123,Gasoline,1.4152\nBuick GL8,0B2020W,11.91,123,Gasoline,1.2387\nBuick GL8,0B2021Y,16.38,147,Gasoline,1.5768\nBuick GL8,0B2020W,13.09,123,Gasoline,1.7538\nBuick GL8,0B2020W,1.82,123,Gasoline,0.4336\nBuick GL8,0B2020W,17.56,123,Gasoline,3.2227\nBuick GL8,0B2020W,101.74,123,Gasoline,9.685\nBuick GL8,0B2021Y,48.78,147,Gasoline,5.0196\nBuick GL8,0B2021Y,34.4,147,Gasoline,3.4112\nBuick GL8,0B2020W,51.12,123,Gasoline,6.3248\nBuick GL8,0B2021Y,11.16,147,Gasoline,1.5322\nBuick GL8,0B2021Y,41.3,147,Gasoline,4.1334\nBuick GL8,0B2020W,51.21,123,Gasoline,5.9031\nBuick GL8,0B2020W,22.28,123,Gasoline,2.646\nBuick GL8,0B2021Y,133.96,147,Gasoline,12.4539\nBuick GL8,0B2020W,21.47,123,Gasoline,1.9311\nBuick GL8,0B2020W,20.75,123,Gasoline,2.7851\nBuick GL8,0B2021Y,52.92,147,Gasoline,4.5203\nBuick GL8,0B2020W,41.77,123,Gasoline,4.8844\nBuick GL8,0B2021Y,225.29,147,Gasoline,23.6756\nBuick GL8,0B2020W,5.52,123,Gasoline,0.7838\nBuick GL8,0B2021Y,1.56,147,Gasoline,0.2683\nBuick GL8,0B2021Y,131.84,147,Gasoline,9.8366\nBuick GL8,0B2020W,5.02,123,Gasoline,0.8153\nBuick GL8,0B2021Y,188.44,147,Gasoline,16.6265\nBuick GL8,0B2020W,70.46,123,Gasoline,6.9574\nBuick GL8,0B2021Y,50.6,147,Gasoline,5.6772\nBuick GL8,0B2021Y,111.86,147,Gasoline,10.0301\nBuick GL8,0B2021Y,100.1,147,Gasoline,9.475\nBuick GL8,0B2020W,106.82,123,Gasoline,10.5875\nBuick GL8,0B2020W,199.26,123,Gasoline,17.2246\nBuick GL8,0B2020W,30.78,123,Gasoline,3.771\nBuick GL8,0B2021Y,83.34,147,Gasoline,6.8582\nBuick GL8,0B2020W,195.18,123,Gasoline,17.8397\nBuick GL8,0B2021Y,26.68,147,Gasoline,3.3397\nBuick GL8,0B2021Y,64.84,147,Gasoline,5.414\nBuick GL8,0B2021Y,155.34,147,Gasoline,20.6463\nBuick GL8,0B2021Y,166.35,147,Gasoline,22.847\nBuick GL8,0B2021Y,3.55,147,Gasoline,0.5103\nBuick GL8,0B2020W,100.44,123,Gasoline,11.9268\nBuick GL8,0B2020W,75.48,123,Gasoline,8.3765\nBuick GL8,0B2020W,29.85,123,Gasoline,3.3705\nBuick GL8,0B2020W,24.57,123,Gasoline,2.3666\nBuick GL8,0B2020W,8.53,123,Gasoline,0.9812\nBuick GL8,0B2020W,107.63,123,Gasoline,12.05\nBuick GL8,0B2020W,43.49,123,Gasoline,4.2879\nBuick GL8,0B2020W,106.1,123,Gasoline,9.765\nBuick GL8,0B2020W,3.91,123,Gasoline,0.6697\nBuick GL8,0B2020W,46.36,123,Gasoline,4.3505\nBuick GL8,0B2021Y,11.84,147,Gasoline,1.1332\nBuick GL8,0B2021Y,70.19,147,Gasoline,7.0728\nBuick GL8,0B2021Y,6.74,147,Gasoline,1.0529\nBuick GL8,0B2020W,2.54,123,Gasoline,0.3825\nBuick GL8,0B2021Y,322.79,147,Gasoline,37.4627\nBuick GL8,0B2021Y,21.5,147,Gasoline,1.8315\nBuick GL8,0B2021Y,189.3,147,Gasoline,19.8654\nBuick GL8,0B2021Y,125.26,147,Gasoline,10.2955\nBuick GL8,0B2021Y,5.32,147,Gasoline,0.6958\nBuick GL8,0B2021Y,99.69,147,Gasoline,8.2512\nBuick GL8,0B2020W,30.98,123,Gasoline,2.5041\nBuick GL8,0B2020W,104.07,123,Gasoline,10.1994\nBuick GL8,0B2020W,141.23,123,Gasoline,13.3416\nBuick GL8,0B2020W,22.26,123,Gasoline,2.8598\nBuick GL8,0B2020W,67.82,123,Gasoline,6.4243\nBuick GL8,0B2021Y,18.02,147,Gasoline,2.5237\nBuick GL8,0B2020W,10.37,123,Gasoline,1.328\nBuick GL8,0B2021Y,37.44,147,Gasoline,2.9517\nBuick GL8,0B2020W,115.44,123,Gasoline,11.8472\nBuick GL8,0B2021Y,44.79,147,Gasoline,3.8577\nBuick GL8,0B2021Y,46.42,147,Gasoline,3.7332\nBuick GL8,0B2021Y,24.63,147,Gasoline,2.7582\nBuick GL8,0B2021Y,50.0,147,Gasoline,3.9213\nBuick GL8,0B2020W,13.69,123,Gasoline,1.9997\nBuick GL8,0B2021Y,4.55,147,Gasoline,0.6037\nBuick GL8,0B2021Y,3.14,147,Gasoline,0.3778\nBuick GL8,0B2020W,17.93,123,Gasoline,1.8871\nBuick GL8,0B2020W,5.7,123,Gasoline,0.791\nBuick GL8,0B2020W,13.06,123,Gasoline,1.3647\nBuick GL8,0B2021Y,60.62,147,Gasoline,5.4672\nBuick GL8,0B2020W,38.45,123,Gasoline,4.0734\nBuick GL8,0B2020W,14.72,123,Gasoline,1.6998\nBuick GL8,0B2020W,11.6,123,Gasoline,1.436\nBuick GL8,0B2021Y,1.91,147,Gasoline,0.3143\nBuick GL8,0B2020W,10.03,123,Gasoline,1.2039\nBuick GL8,0B2021Y,21.03,147,Gasoline,4.495\nBuick GL8,0B2020W,138.42,123,Gasoline,10.4512\nBuick GL8,0B2020W,116.7,123,Gasoline,9.9751\nBuick GL8,0B2021Y,33.56,147,Gasoline,4.2311\nBuick GL8,0B2020W,21.92,123,Gasoline,2.412\nBuick GL8,0B2020W,127.95,123,Gasoline,13.1827\nBuick GL8,0B2021Y,3.08,147,Gasoline,0.4646\nBuick GL8,0B2020W,25.45,123,Gasoline,2.574\nBuick GL8,0B2020W,12.85,123,Gasoline,1.4102\nBuick GL8,0B2021Y,148.17,147,Gasoline,16.7204\nBuick GL8,0B2020W,91.05,123,Gasoline,13.0069\nBuick GL8,0B2021Y,9.42,147,Gasoline,1.0923\nBuick GL8,0B2020W,113.82,123,Gasoline,9.3879\nBuick GL8,0B2021Y,10.84,147,Gasoline,1.2548\nBuick GL8,0B2021Y,5.93,147,Gasoline,1.0018\nBuick GL8,0B2020W,4.56,123,Gasoline,0.8029\nBuick GL8,0B2021Y,10.09,147,Gasoline,1.2454\nBuick GL8,0B2021Y,37.37,147,Gasoline,3.0294\nBuick GL8,0B2020W,1.86,123,Gasoline,0.2388\nBuick GL8,0B2021Y,49.77,147,Gasoline,4.1163\nBuick GL8,0B2021Y,175.31,147,Gasoline,14.6262\nBuick GL8,0B2021Y,3.44,147,Gasoline,0.5498\nBuick GL8,0B2021Y,204.79,147,Gasoline,15.6764\nBuick GL8,0B2021Y,41.46,147,Gasoline,3.4331\nBuick GL8,0B2021Y,2.31,147,Gasoline,0.298\nBuick GL8,0B2020W,35.13,123,Gasoline,3.6602\nBuick GL8,0B2021Y,1.77,147,Gasoline,0.2676\nBuick GL8,0B2020W,20.52,123,Gasoline,2.0562\nBuick GL8,0B2020W,5.26,123,Gasoline,0.7233\nBuick GL8,0B2020W,31.22,123,Gasoline,3.0199\nBuick GL8,0B2020W,380.85,123,Gasoline,33.3197\nBuick GL8,0B2020W,119.74,123,Gasoline,9.3943\nBuick GL8,0B2021Y,3.86,147,Gasoline,0.6665\nBuick GL8,0B2020W,14.59,123,Gasoline,2.9305\nBuick GL8,0B2020W,5.46,123,Gasoline,0.5489\nBuick GL8,0B2021Y,1.68,147,Gasoline,0.2421\nBuick GL8,0B2021Y,4.39,147,Gasoline,0.7118\nBuick GL8,0B2021Y,8.82,147,Gasoline,0.842\nBuick GL8,0B2020W,260.58,123,Gasoline,21.0177\nBuick GL8,0B2020W,2.44,123,Gasoline,0.3929\nBuick GL8,0B2020W,107.65,123,Gasoline,9.2457\nBuick GL8,0B2021Y,17.08,147,Gasoline,1.9332\nBuick GL8,0B2021Y,33.24,147,Gasoline,3.4099\nBuick GL8,0B2020W,72.82,123,Gasoline,6.6083\nBuick GL8,0B2020W,24.28,123,Gasoline,2.5928\nBuick GL8,0B2021Y,16.96,147,Gasoline,1.7697\nBuick GL8,0B2021Y,4.23,147,Gasoline,0.5264\nBuick GL8,0B2020W,12.24,123,Gasoline,1.6262\nBuick GL8,0B2021Y,57.14,147,Gasoline,5.5523\nBuick GL8,0B2020W,132.11,123,Gasoline,14.4132\nBuick GL8,0B2021Y,41.31,147,Gasoline,4.2506\nBuick GL8,0B2021Y,37.86,147,Gasoline,4.8409\nBuick GL8,0B2020W,98.82,123,Gasoline,10.5994\nBuick GL8,0B2021Y,3.74,147,Gasoline,0.5993\nBuick GL8,0B2020W,107.75,123,Gasoline,11.2488\nBuick GL8,0B2020W,217.61,123,Gasoline,18.9063\nBuick GL8,0B2020W,132.68,123,Gasoline,11.576\nBuick GL8,0B2021Y,35.84,147,Gasoline,3.9753\nBuick GL8,0B2021Y,6.17,147,Gasoline,0.8579\nBuick GL8,0B2020W,2.2,123,Gasoline,0.3811\nBuick GL8,0B2020W,20.13,123,Gasoline,2.1981\nBuick GL8,0B2020W,195.77,123,Gasoline,18.1264\nBuick GL8,0B2020W,30.38,123,Gasoline,2.6451\nBuick GL8,0B2021Y,39.73,147,Gasoline,3.8986\nBuick GL8,0B2020W,111.54,123,Gasoline,9.9106\nBuick GL8,0B2020W,14.03,123,Gasoline,1.5726\nBuick GL8,0B2020W,7.15,123,Gasoline,0.9078\nBuick GL8,0B2021Y,15.7,147,Gasoline,1.8747\nBuick GL8,0B2021Y,38.28,147,Gasoline,3.2118\nBuick GL8,0B2021Y,2.18,147,Gasoline,0.2401\nBuick GL8,0B2021Y,1.86,147,Gasoline,0.324\nBuick GL8,0B2020W,116.73,123,Gasoline,15.4418\nBuick GL8,0B2021Y,40.58,147,Gasoline,3.1775\nBuick GL8,0B2021Y,36.16,147,Gasoline,3.1633\nBuick GL8,0B2021Y,3.6,147,Gasoline,0.5282\nBuick GL8,0B2020W,22.57,123,Gasoline,3.1621\nBuick GL8,0B2020W,22.27,123,Gasoline,2.7286\nBuick GL8,0B2020W,8.96,123,Gasoline,1.2552\nBuick GL8,0B2021Y,21.14,147,Gasoline,1.9834\nBuick GL8,0B2020W,106.22,123,Gasoline,10.0855\nBuick GL8,0B2021Y,133.13,147,Gasoline,12.2205\nBuick GL8,0B2021Y,64.13,147,Gasoline,4.706\nBuick GL8,0B2020W,10.44,123,Gasoline,1.3621\nBuick GL8,0B2021Y,23.82,147,Gasoline,1.8361\nBuick GL8,0B2020W,21.9,123,Gasoline,2.6751\nBuick GL8,0B2021Y,214.12,147,Gasoline,15.2199\nBuick GL8,0B2020W,155.08,123,Gasoline,15.2779\nBuick GL8,0B2020W,1.57,123,Gasoline,0.2655\nBuick GL8,0B2021Y,5.15,147,Gasoline,0.563\nBuick GL8,0B2020W,18.52,123,Gasoline,2.2625\nBuick GL8,0B2020W,7.3,123,Gasoline,1.1806\nBuick GL8,0B2020W,22.04,123,Gasoline,2.9569\nBuick GL8,0B2021Y,58.96,147,Gasoline,6.0183\nBuick GL8,0B2021Y,210.74,147,Gasoline,25.1073\nBuick GL8,0B2021Y,266.2,147,Gasoline,32.3257\nBuick GL8,0B2020W,16.99,123,Gasoline,1.9442\nBuick GL8,0B2020W,12.6,123,Gasoline,1.3305\nBuick GL8,0B2021Y,1.97,147,Gasoline,0.2714\nBuick GL8,0B2020W,153.21,123,Gasoline,15.9668\nBuick GL8,0B2020W,21.66,123,Gasoline,2.5744\nBuick GL8,0B2021Y,12.39,147,Gasoline,1.5171\nBuick GL8,0B2021Y,26.92,147,Gasoline,1.6857\nBuick GL8,0B2020W,13.53,123,Gasoline,2.1327\nBuick GL8,0B2020W,33.5,123,Gasoline,3.62\nBuick GL8,0B2020W,31.51,123,Gasoline,4.5181\nBuick GL8,0B2020W,119.45,123,Gasoline,10.5076\nBuick GL8,0B2021Y,29.7,147,Gasoline,3.1852\nBuick GL8,0B2021Y,30.47,147,Gasoline,2.8734\nBuick GL8,0B2020W,26.78,123,Gasoline,2.5301\nBuick GL8,0B2020W,90.7,123,Gasoline,10.3015\nBuick GL8,0B2021Y,118.38,147,Gasoline,9.5183\nBuick GL8,0B2020W,13.94,123,Gasoline,1.4837\nBuick GL8,0B2020W,157.39,123,Gasoline,14.7349\nBuick GL8,0B2021Y,3.96,147,Gasoline,0.6995\nBuick GL8,0B2021Y,166.18,147,Gasoline,12.9602\nBuick GL8,0B2020W,71.7,123,Gasoline,7.5882\nBuick GL8,0B2021Y,20.81,147,Gasoline,2.0132\nBuick GL8,0B2021Y,23.24,147,Gasoline,2.8962\nBuick GL8,0B2020W,2.98,123,Gasoline,0.5036\nBuick GL8,0B2020W,111.68,123,Gasoline,10.6012\nBuick GL8,0B2020W,8.9,123,Gasoline,1.5288\nBuick GL8,0B2020W,3.21,123,Gasoline,0.6666\nBuick GL8,0B2020W,27.56,123,Gasoline,3.3941\nBuick GL8,0B2020W,24.86,123,Gasoline,3.7289\nBuick GL8,0B2020W,14.16,123,Gasoline,1.4735\nBuick GL8,0B2021Y,30.23,147,Gasoline,4.3987\nBuick GL8,0B2021Y,3.14,147,Gasoline,0.3605\nBuick GL8,0B2020W,11.55,123,Gasoline,1.8158\nBuick GL8,0B2020W,125.12,123,Gasoline,16.7153\nBuick GL8,0B2020W,157.66,123,Gasoline,12.1326\nBuick GL8,0B2020W,31.01,123,Gasoline,4.6104\nBuick GL8,0B2020W,10.31,123,Gasoline,1.4972\nBuick GL8,0B2020W,107.89,123,Gasoline,10.5108\nBuick GL8,0B2020W,74.09,123,Gasoline,8.1476\nBuick GL8,0B2021Y,1.71,147,Gasoline,0.2863\nBuick GL8,0B2021Y,13.7,147,Gasoline,2.0995\nBuick GL8,0B2020W,47.28,123,Gasoline,6.3834\nBuick GL8,0B2021Y,32.75,147,Gasoline,6.4456\nBuick GL8,0B2020W,45.25,123,Gasoline,4.3415\nBuick GL8,0B2020W,39.94,123,Gasoline,4.5683\nBuick GL8,0B2020W,115.73,123,Gasoline,16.0197\nBuick GL8,0B2021Y,7.09,147,Gasoline,0.8169\nBuick GL8,0B2021Y,39.12,147,Gasoline,3.3118\nBuick GL8,0B2020W,5.34,123,Gasoline,1.1647\nBuick GL8,0B2021Y,137.02,147,Gasoline,14.9525\nBuick GL8,0B2021Y,109.06,147,Gasoline,10.363\nBuick GL8,0B2021Y,148.15,147,Gasoline,14.4418\nBuick GL8,0B2021Y,46.06,147,Gasoline,4.6817\nBuick GL8,0B2020W,13.94,123,Gasoline,2.0033\nBuick GL8,0B2021Y,127.36,147,Gasoline,10.9606\nBuick GL8,0B2021Y,77.01,147,Gasoline,6.8807\nBuick GL8,0B2021Y,24.8,147,Gasoline,1.6845\nBuick GL8,0B2020W,10.5,123,Gasoline,1.2459\nBuick GL8,0B2020W,3.68,123,Gasoline,0.726\nBuick GL8,0B2020W,1.96,123,Gasoline,0.3517\nBuick GL8,0B2021Y,45.22,147,Gasoline,5.5701\nBuick GL8,0B2020W,47.88,123,Gasoline,4.6983\nBuick GL8,0B2020W,133.6,123,Gasoline,17.276\nBuick GL8,0B2021Y,7.52,147,Gasoline,1.6862\nBuick GL8,0B2021Y,159.4,147,Gasoline,14.9768\nBuick GL8,0B2021Y,28.42,147,Gasoline,3.1575\nBuick GL8,0B2021Y,27.16,147,Gasoline,2.8582\nBuick GL8,0B2021Y,14.01,147,Gasoline,1.6411\nBuick GL8,0B2021Y,73.96,147,Gasoline,7.6635\nBuick GL8,0B2021Y,8.92,147,Gasoline,0.9259\nBuick GL8,0B2020W,30.13,123,Gasoline,3.2396\nBuick GL8,0B2021Y,12.83,147,Gasoline,0.9235\nBuick GL8,0B2021Y,16.98,147,Gasoline,2.5899\nBuick GL8,0B2021Y,50.78,147,Gasoline,4.3304\nBuick GL8,0B2021Y,192.36,147,Gasoline,15.3455\nBuick GL8,0B2020W,110.09,123,Gasoline,14.5\nBuick GL8,0B2020W,329.13,123,Gasoline,34.8571\nBuick GL8,0B2021Y,52.66,147,Gasoline,7.1166\nBuick GL8,0B2020W,23.91,123,Gasoline,2.5773\nBuick GL8,0B2021Y,16.28,147,Gasoline,3.0673\nBuick GL8,0B2020W,11.27,123,Gasoline,1.82\nBuick GL8,0B2021Y,128.82,147,Gasoline,9.6531\nBuick GL8,0B2021Y,59.34,147,Gasoline,6.2547\nBuick GL8,0B2020W,119.62,123,Gasoline,11.4877\nBuick GL8,0B2021Y,52.06,147,Gasoline,4.9537\nBuick GL8,0B2020W,6.72,123,Gasoline,0.7651\nBuick GL8,0B2021Y,110.1,147,Gasoline,12.8429\nBuick GL8,0B2020W,185.49,123,Gasoline,16.4408\nBuick GL8,0B2020W,35.29,123,Gasoline,3.3783\nBuick GL8,0B2020W,32.45,123,Gasoline,4.6203\nBuick GL8,0B2020W,11.3,123,Gasoline,1.821\nBuick GL8,0B2021Y,147.62,147,Gasoline,9.5274\nBuick GL8,0B2021Y,146.93,147,Gasoline,11.7595\nBuick GL8,0B2021Y,39.38,147,Gasoline,3.2912\nBuick GL8,0B2020W,24.72,123,Gasoline,4.1131\nBuick GL8,0B2021Y,23.79,147,Gasoline,2.6323\nBuick GL8,0B2020W,6.72,123,Gasoline,1.1282\nBuick GL8,0B2021Y,2.12,147,Gasoline,0.3924\nBuick GL8,0B2021Y,13.85,147,Gasoline,1.5039\nBuick GL8,0B2020W,9.44,123,Gasoline,1.2187\nBuick GL8,0B2021Y,6.66,147,Gasoline,1.2597\nBuick GL8,0B2020W,4.6,123,Gasoline,1.1556\nBuick GL8,0B2021Y,1.62,147,Gasoline,0.2183\nBuick GL8,0B2020W,4.83,123,Gasoline,0.8243\nBuick GL8,0B2020W,161.14,123,Gasoline,16.0872\nBuick GL8,0B2021Y,5.53,147,Gasoline,0.6767\nBuick GL8,0B2020W,33.24,123,Gasoline,4.1619\nBuick GL8,0B2021Y,3.34,147,Gasoline,0.4545\nBuick GL8,0B2020W,21.26,123,Gasoline,3.967\nBuick GL8,0B2021Y,3.16,147,Gasoline,0.6266\nBuick GL8,0B2021Y,61.97,147,Gasoline,6.6172\nBuick GL8,0B2021Y,89.5,147,Gasoline,6.8834\nBuick GL8,0B2021Y,9.84,147,Gasoline,1.0789\nBuick GL8,0B2021Y,25.55,147,Gasoline,3.3027\nBuick GL8,0B2021Y,110.33,147,Gasoline,8.1589\nBuick GL8,0B2021Y,2.56,147,Gasoline,0.3728\nBuick GL8,0B2021Y,5.72,147,Gasoline,1.012\nBuick GL8,0B2021Y,351.53,147,Gasoline,30.6194\nBuick GL8,0B2020W,83.43,123,Gasoline,9.6112\nBuick GL8,0B2020W,120.62,123,Gasoline,14.5073\nBuick GL8,0B2021Y,13.9,147,Gasoline,1.5524\nBuick GL8,0B2021Y,38.14,147,Gasoline,3.9761\nBuick GL8,0B2020W,221.72,123,Gasoline,20.8831\nBuick GL8,0B2021Y,149.62,147,Gasoline,12.6716\nBuick GL8,0B2020W,10.14,123,Gasoline,2.3169\nBuick GL8,0B2020W,52.16,123,Gasoline,5.636\nBuick GL8,0B2021Y,84.82,147,Gasoline,14.7463\nBuick GL8,0B2021Y,24.62,147,Gasoline,5.5604\nBuick GL8,0B2021Y,12.5,147,Gasoline,1.2935\nBuick GL8,0B2021Y,40.31,147,Gasoline,3.8087\nBuick GL8,0B2021Y,177.9,147,Gasoline,16.0466\nBuick GL8,0B2021Y,121.38,147,Gasoline,13.6399\nBuick GL8,0B2020W,7.51,123,Gasoline,1.8528\nBuick GL8,0B2021Y,155.74,147,Gasoline,14.1131\nBuick GL8,0B2020W,3.52,123,Gasoline,0.6679\nBuick GL8,0B2021Y,125.33,147,Gasoline,9.8632\nBuick GL8,0B2020W,17.85,123,Gasoline,1.798\nBuick GL8,0B2020W,182.45,123,Gasoline,16.1925\nBuick GL8,0B2020W,3.44,123,Gasoline,0.4378\nBuick GL8,0B2020W,37.66,123,Gasoline,9.4218\nBuick GL8,0B2021Y,156.38,147,Gasoline,16.5919\nBuick GL8,0B2021Y,150.97,147,Gasoline,12.6195\nBuick GL8,0B2020W,42.03,123,Gasoline,4.3637\nBuick GL8,0B2021Y,49.96,147,Gasoline,6.7482\nBuick GL8,0B2021Y,3.83,147,Gasoline,0.7422\nBuick GL8,0B2020W,32.15,123,Gasoline,5.9276\nBuick GL8,0B2021Y,95.08,147,Gasoline,9.0969\nBuick GL8,0B2020W,60.26,123,Gasoline,6.7604\nBuick GL8,0B2021Y,215.29,147,Gasoline,18.589\nBuick GL8,0B2021Y,12.25,147,Gasoline,1.5288\nBuick GL8,0B2021Y,1.61,147,Gasoline,0.1701\nBuick GL8,0B2021Y,4.16,147,Gasoline,0.6129\nBuick GL8,0B2020W,23.7,123,Gasoline,2.3839\nBuick GL8,0B2021Y,6.03,147,Gasoline,0.7405\nBuick GL8,0B2021Y,83.03,147,Gasoline,6.3043\nBuick GL8,0B2021Y,11.06,147,Gasoline,1.1921\nBuick GL8,0B2020W,59.85,123,Gasoline,6.1602\nBuick GL8,0B2021Y,28.1,147,Gasoline,2.7185\nBuick GL8,0B2021Y,117.15,147,Gasoline,11.3563\nBuick GL8,0B2020W,58.59,123,Gasoline,7.8417\nBuick GL8,0B2020W,31.06,123,Gasoline,3.5473\nBuick GL8,0B2021Y,47.93,147,Gasoline,5.8616\nBuick GL8,0B2021Y,106.55,147,Gasoline,11.6342\nBuick GL8,0B2020W,1.74,123,Gasoline,0.2617\nBuick GL8,0B2021Y,6.78,147,Gasoline,1.4877\nBuick GL8,0B2021Y,3.28,147,Gasoline,0.3878\nBuick GL8,0B2021Y,25.99,147,Gasoline,3.002\nBuick GL8,0B2021Y,5.81,147,Gasoline,0.6315\nBuick GL8,0B2021Y,18.38,147,Gasoline,2.5482\nBuick GL8,0B2020W,3.82,123,Gasoline,0.5366\nBuick GL8,0B2020W,230.85,123,Gasoline,23.0162\nBuick GL8,0B2020W,392.32,123,Gasoline,39.9268\nBuick GL8,0B2020W,201.2,123,Gasoline,18.0992\nBuick GL8,0B2021Y,131.42,147,Gasoline,12.0202\nBuick GL8,0B2020W,7.93,123,Gasoline,0.9802\nBuick GL8,0B2021Y,87.96,147,Gasoline,9.6522\nBuick GL8,0B2021Y,108.21,147,Gasoline,9.1663\nBuick GL8,0B2020W,14.06,123,Gasoline,1.5941\nBuick GL8,0B2021Y,191.14,147,Gasoline,14.7748\nBuick GL8,0B2020W,3.58,123,Gasoline,0.4511\nBuick GL8,0B2020W,241.92,123,Gasoline,23.7302\nBuick GL8,0B2021Y,119.21,147,Gasoline,8.9343\nBuick GL8,0B2020W,1.63,123,Gasoline,0.3324\nBuick GL8,0B2021Y,167.77,147,Gasoline,15.9666\nBuick GL8,0B2021Y,109.34,147,Gasoline,10.5588\nBuick GL8,0B2021Y,54.49,147,Gasoline,5.3582\nBuick GL8,0B2020W,38.75,123,Gasoline,6.8513\nBuick GL8,0B2020W,6.44,123,Gasoline,0.7995\nBuick GL8,0B2021Y,2.94,147,Gasoline,0.5044\nBuick GL8,0B2021Y,4.34,147,Gasoline,0.9267\nBuick GL8,0B2020W,116.42,123,Gasoline,10.0411\nBuick GL8,0B2021Y,160.36,147,Gasoline,11.8301\nBuick GL8,0B2021Y,112.12,147,Gasoline,12.1699\nBuick GL8,0B2021Y,21.32,147,Gasoline,3.1504\nBuick GL8,0B2020W,4.94,123,Gasoline,1.0498\nBuick GL8,0B2021Y,200.85,147,Gasoline,18.6536\nBuick GL8,0B2021Y,185.21,147,Gasoline,15.2448\nBuick GL8,0B2020W,6.18,123,Gasoline,0.9804\nBuick GL8,0B2020W,1.94,123,Gasoline,0.3036\nBuick GL8,0B2020W,44.03,123,Gasoline,4.5824\nBuick GL8,0B2021Y,4.04,147,Gasoline,0.6041\nBuick GL8,0B2021Y,11.36,147,Gasoline,1.0955\nBuick GL8,0B2020W,119.07,123,Gasoline,10.9639\nBuick GL8,0B2021Y,29.22,147,Gasoline,2.3487\nBuick GL8,0B2020W,190.92,123,Gasoline,17.8352\nBuick GL8,0B2021Y,40.84,147,Gasoline,3.6595\nBuick GL8,0B2021Y,6.09,147,Gasoline,1.4307\nBuick GL8,0B2020W,22.27,123,Gasoline,3.2374\nBuick GL8,0B2021Y,26.73,147,Gasoline,3.1714\nBuick GL8,0B2021Y,4.17,147,Gasoline,0.4953\nBuick GL8,0B2021Y,16.79,147,Gasoline,1.3011\nBuick GL8,0B2020W,39.38,123,Gasoline,3.6149\nBuick GL8,0B2021Y,7.3,147,Gasoline,0.8539\nBuick GL8,0B2020W,11.71,123,Gasoline,1.775\nBuick GL8,0B2020W,69.86,123,Gasoline,6.11\nBuick GL8,0B2020W,16.66,123,Gasoline,1.9932\nBuick GL8,0B2021Y,132.66,147,Gasoline,13.8929\nBuick GL8,0B2021Y,2.7,147,Gasoline,0.4147\nBuick GL8,0B2021Y,40.4,147,Gasoline,3.084\nBuick GL8,0B2020W,18.33,123,Gasoline,1.6217\nBuick GL8,0B2020W,12.46,123,Gasoline,2.0218\nBuick GL8,0B2021Y,10.67,147,Gasoline,1.3992\nBuick GL8,0B2020W,101.39,123,Gasoline,10.0487\nBuick GL8,0B2020W,17.42,123,Gasoline,2.8031\nBuick GL8,0B2021Y,10.74,147,Gasoline,1.0783\nBuick GL8,0B2021Y,10.94,147,Gasoline,1.083\nBuick GL8,0B2021Y,16.02,147,Gasoline,2.6894\nBuick GL8,0B2020W,132.03,123,Gasoline,12.1678\nBuick GL8,0B2020W,14.62,123,Gasoline,1.3401\nBuick GL8,0B2020W,112.48,123,Gasoline,10.2247\nBuick GL8,0B2020W,3.5,123,Gasoline,0.5362\nBuick GL8,0B2021Y,3.99,147,Gasoline,0.5382\nBuick GL8,0B2021Y,3.13,147,Gasoline,0.8665\nBuick GL8,0B2021Y,18.99,147,Gasoline,3.5379\nBuick GL8,0B2020W,7.28,123,Gasoline,0.937\nBuick GL8,0B2020W,9.63,123,Gasoline,1.285\nBuick GL8,0B2020W,47.56,123,Gasoline,6.6905\nBuick GL8,0B2020W,3.63,123,Gasoline,0.4891\nBuick GL8,0B2020W,127.18,123,Gasoline,11.8517\nBuick GL8,0B2020W,12.1,123,Gasoline,1.3438\nBuick GL8,0B2021Y,17.03,147,Gasoline,1.6988\nBuick GL8,0B2021Y,32.52,147,Gasoline,3.5723\nBuick GL8,0B2021Y,217.82,147,Gasoline,17.8105\nBuick GL8,0B2020W,296.6,123,Gasoline,28.5127\nBuick GL8,0B2021Y,39.52,147,Gasoline,3.9455\nBuick GL8,0B2021Y,11.36,147,Gasoline,1.6081\nBuick GL8,0B2021Y,3.63,147,Gasoline,0.495\nBuick GL8,0B2021Y,124.11,147,Gasoline,12.6536\nBuick GL8,0B2021Y,169.04,147,Gasoline,15.1286\nBuick GL8,0B2021Y,183.86,147,Gasoline,13.8595\nBuick GL8,0B2021Y,16.48,147,Gasoline,1.43\nBuick GL8,0B2020W,13.4,123,Gasoline,1.5562\nBuick GL8,0B2020W,176.94,123,Gasoline,16.3844\nBuick GL8,0B2020W,61.08,123,Gasoline,5.6601\nBuick GL8,0B2021Y,175.38,147,Gasoline,14.4917\nBuick GL8,0B2020W,52.91,123,Gasoline,4.5585\nBuick GL8,0B2020W,49.99,123,Gasoline,5.0662\nBuick GL8,0B2021Y,10.99,147,Gasoline,1.4374\nBuick GL8,0B2020W,193.72,123,Gasoline,19.5438\nBuick GL8,0B2021Y,114.43,147,Gasoline,8.9926\nBuick GL8,0B2021Y,7.69,147,Gasoline,1.074\nBuick GL8,0B2021Y,190.09,147,Gasoline,16.202\nBuick GL8,0B2021Y,17.54,147,Gasoline,2.0964\nBuick GL8,0B2021Y,80.96,147,Gasoline,6.2055\nBuick GL8,0B2021Y,115.0,147,Gasoline,9.8514\nBuick GL8,0B2020W,6.86,123,Gasoline,0.9103\nBuick GL8,0B2021Y,6.55,147,Gasoline,0.7824\nBuick GL8,0B2020W,166.41,123,Gasoline,13.6403\nBuick GL8,0B2020W,78.81,123,Gasoline,6.963\nBuick GL8,0B2020W,2.66,123,Gasoline,0.3904\nBuick GL8,0B2021Y,43.01,147,Gasoline,3.8007\nBuick GL8,0B2020W,103.92,123,Gasoline,10.628\nBuick GL8,0B2020W,46.74,123,Gasoline,5.12\nBuick GL8,0B2021Y,302.69,147,Gasoline,28.9497\nBuick GL8,0B2021Y,28.99,147,Gasoline,3.3332\nBuick GL8,0B2020W,35.3,123,Gasoline,3.3858\nBuick GL8,0B2021Y,15.46,147,Gasoline,1.6237\nBuick GL8,0B2020W,30.88,123,Gasoline,3.9864\nBuick GL8,0B2020W,8.15,123,Gasoline,0.8995\nBuick GL8,0B2021Y,103.44,147,Gasoline,8.5873\nBuick GL8,0B2020W,7.05,123,Gasoline,0.8648\nBuick GL8,0B2020W,41.68,123,Gasoline,4.2582\nBuick GL8,0B2021Y,164.13,147,Gasoline,15.7428\nBuick GL8,0B2021Y,58.11,147,Gasoline,5.4889\nBuick GL8,0B2020W,2.88,123,Gasoline,0.5226\nBuick GL8,0B2020W,27.78,123,Gasoline,3.1536\nBuick GL8,0B2021Y,12.83,147,Gasoline,1.4847\nBuick GL8,0B2020W,64.44,123,Gasoline,9.2853\nBuick GL8,0B2020W,17.09,123,Gasoline,1.9701\nBuick GL8,0B2020W,105.16,123,Gasoline,9.7343\nBuick GL8,0B2020W,49.3,123,Gasoline,4.492\nBuick GL8,0B2020W,48.05,123,Gasoline,5.2815\nBuick GL8,0B2020W,199.73,123,Gasoline,21.2806\nBuick GL8,0B2020W,148.92,123,Gasoline,14.6767\nBuick GL8,0B2020W,33.2,123,Gasoline,3.1247\nBuick GL8,0B2021Y,51.42,147,Gasoline,4.2178\nBuick GL8,0B2021Y,33.2,147,Gasoline,3.707\nBuick GL8,0B2020W,10.6,123,Gasoline,1.2829\nBuick GL8,0B2021Y,5.25,147,Gasoline,0.5256\nBuick GL8,0B2020W,2.06,123,Gasoline,0.4307\nBuick GL8,0B2021Y,119.58,147,Gasoline,8.9499\nBuick GL8,0B2021Y,4.74,147,Gasoline,0.5758\nBuick GL8,0B2021Y,101.51,147,Gasoline,8.1147\nBuick GL8,0B2020W,161.06,123,Gasoline,15.1646\nBuick GL8,0B2021Y,49.84,147,Gasoline,4.7007\nBuick GL8,0B2020W,1.78,123,Gasoline,0.4352\nBuick GL8,0B2020W,26.65,123,Gasoline,2.5729\nBuick GL8,0B2020W,110.72,123,Gasoline,8.0267\nBuick GL8,0B2020W,66.31,123,Gasoline,9.9328\nBuick GL8,0B2021Y,15.46,147,Gasoline,1.5196\nBuick GL8,0B2020W,104.96,123,Gasoline,10.514\nBuick GL8,0B2020W,119.14,123,Gasoline,14.7159\nBuick GL8,0B2020W,4.04,123,Gasoline,0.5584\nBuick GL8,0B2020W,143.42,123,Gasoline,13.9952\nBuick GL8,0B2021Y,3.58,147,Gasoline,0.3864\nBuick GL8,0B2020W,72.66,123,Gasoline,9.9622\nBuick GL8,0B2021Y,33.68,147,Gasoline,2.8669\nBuick GL8,0B2020W,35.67,123,Gasoline,4.2838\nBuick GL8,0B2021Y,22.0,147,Gasoline,1.9362\nBuick GL8,0B2020W,42.52,123,Gasoline,4.0376\nBuick GL8,0B2021Y,42.8,147,Gasoline,4.1358\nBuick GL8,0B2021Y,178.97,147,Gasoline,13.0918\nBuick GL8,0B2021Y,53.42,147,Gasoline,4.7179\nBuick GL8,0B2020W,2.45,123,Gasoline,0.3641\nBuick GL8,0B2021Y,4.25,147,Gasoline,0.792\nBuick GL8,0B2021Y,11.02,147,Gasoline,1.1354\nBuick GL8,0B2020W,37.48,123,Gasoline,4.9226\nBuick GL8,0B2020W,31.05,123,Gasoline,4.7206\nBuick GL8,0B2021Y,22.21,147,Gasoline,3.5017\nBuick GL8,0B2020W,6.18,123,Gasoline,0.7945\nBuick GL8,0B2020W,8.7,123,Gasoline,1.1524\nBuick GL8,0B2020W,114.01,123,Gasoline,13.4385\nBuick GL8,0B2021Y,23.68,147,Gasoline,2.5109\nBuick GL8,0B2021Y,24.58,147,Gasoline,2.6137\nBuick GL8,0B2020W,3.58,123,Gasoline,0.5282\nBuick GL8,0B2021Y,323.52,147,Gasoline,21.7584\nBuick GL8,0B2020W,6.13,123,Gasoline,0.8245\nBuick GL8,0B2021Y,13.13,147,Gasoline,1.3532\nBuick GL8,0B2020W,28.01,123,Gasoline,2.912\nBuick GL8,0B2021Y,14.8,147,Gasoline,1.7433\nBuick GL8,0B2021Y,90.35,147,Gasoline,6.4998\nBuick GL8,0B2021Y,30.17,147,Gasoline,3.7095\nBuick GL8,0B2021Y,110.51,147,Gasoline,7.6787\nBuick GL8,0B2021Y,32.15,147,Gasoline,2.8579\nBuick GL8,0B2020W,37.99,123,Gasoline,3.2903\nBuick GL8,0B2021Y,20.97,147,Gasoline,1.7356\nBuick GL8,0B2021Y,4.78,147,Gasoline,0.7355\nBuick GL8,0B2020W,18.52,123,Gasoline,3.0752\nBuick GL8,0B2020W,10.96,123,Gasoline,1.398\nBuick GL8,0B2021Y,17.89,147,Gasoline,1.5795\nBuick GL8,0B2020W,115.08,123,Gasoline,10.1271\nBuick GL8,0B2020W,51.95,123,Gasoline,6.5228\nBuick GL8,0B2020W,154.15,123,Gasoline,16.3858\nBuick GL8,0B2021Y,141.95,147,Gasoline,14.2803\nBuick GL8,0B2020W,16.26,123,Gasoline,2.5954\nBuick GL8,0B2020W,10.49,123,Gasoline,1.2489\nBuick GL8,0B2020W,91.87,123,Gasoline,8.7891\nBuick GL8,0B2020W,9.43,123,Gasoline,1.2579\nBuick GL8,0B2020W,2.43,123,Gasoline,0.4827\nBuick GL8,0B2021Y,27.38,147,Gasoline,3.5591\nBuick GL8,0B2021Y,30.44,147,Gasoline,3.1892\nBuick GL8,0B2020W,10.32,123,Gasoline,1.8039\nBuick GL8,0B2021Y,2.57,147,Gasoline,0.4197\nBuick GL8,0B2020W,210.7,123,Gasoline,20.8631\nBuick GL8,0B2020W,3.06,123,Gasoline,0.3484\nBuick GL8,0B2020W,9.89,123,Gasoline,1.0887\nBuick GL8,0B2020W,110.78,123,Gasoline,11.7315\nBuick GL8,0B2020W,38.99,123,Gasoline,4.8086\nBuick GL8,0B2020W,9.27,123,Gasoline,1.7611\nBuick GL8,0B2020W,6.72,123,Gasoline,1.0156\nBuick GL8,0B2021Y,4.3,147,Gasoline,1.0155\nBuick GL8,0B2021Y,18.81,147,Gasoline,2.0244\nBuick GL8,0B2021Y,2.72,147,Gasoline,0.3975\nBuick GL8,0B2021Y,263.7,147,Gasoline,15.4713\nBuick GL8,0B2021Y,108.75,147,Gasoline,8.5298\nBuick GL8,0B2020W,6.08,123,Gasoline,0.8301\nBuick GL8,0B2020W,5.84,123,Gasoline,0.8204\nBuick GL8,0B2020W,8.03,123,Gasoline,0.7566\nBuick GL8,0B2021Y,2.44,147,Gasoline,0.4633\nBuick GL8,0B2020W,12.23,123,Gasoline,1.4924\nBuick GL8,0B2020W,22.15,123,Gasoline,1.9688\nBuick GL8,0B2020W,60.28,123,Gasoline,6.7709\nBuick GL8,0B2021Y,32.83,147,Gasoline,2.8632\nBuick GL8,0B2021Y,3.76,147,Gasoline,0.6115\nBuick GL8,0B2021Y,108.12,147,Gasoline,9.4507\nBuick GL8,0B2020W,38.84,123,Gasoline,4.7121\nBuick GL8,0B2020W,9.49,123,Gasoline,1.821\nBuick GL8,0B2020W,21.51,123,Gasoline,2.0882\nBuick GL8,0B2020W,46.92,123,Gasoline,4.6808\nBuick GL8,0B2020W,16.37,123,Gasoline,1.5653\nBuick GL8,0B2021Y,28.11,147,Gasoline,2.9827\nBuick GL8,0B2020W,65.8,123,Gasoline,6.7434\nBuick GL8,0B2020W,3.42,123,Gasoline,0.5902\nChevrolet Captiva,3K10409,16.7,123,Gasoline,1.7574\nChevrolet Captiva,3K10409,87.17,123,Gasoline,8.1683\nChevrolet Captiva,3K10409,3.25,123,Gasoline,0.8101\nChevrolet Captiva,3K10409,4.39,123,Gasoline,0.7346\nChevrolet Captiva,3K10409,13.74,123,Gasoline,1.8568\nChevrolet Captiva,3K10409,3.98,123,Gasoline,0.7316\nChevrolet Captiva,3K10409,2.72,123,Gasoline,0.4248\nChevrolet Captiva,3K10409,197.69,123,Gasoline,21.7249\nChevrolet Captiva,3K10409,15.76,123,Gasoline,1.7545\nChevrolet Captiva,3K10409,184.42,123,Gasoline,20.1791\nChevrolet Captiva,3K10409,5.9,123,Gasoline,0.9728\nChevrolet Captiva,3K10409,2.82,123,Gasoline,0.3906\nChevrolet Captiva,3K10409,13.86,123,Gasoline,1.3675\nChevrolet Captiva,3K10409,4.98,123,Gasoline,0.9078\nChevrolet Captiva,3K10409,16.71,123,Gasoline,1.9354\nChevrolet Captiva,3K10409,17.59,123,Gasoline,1.8372\nChevrolet Captiva,3K10409,6.86,123,Gasoline,1.2393\nChevrolet Captiva,3K10409,5.65,123,Gasoline,0.8124\nChevrolet Captiva,3K10409,14.21,123,Gasoline,2.4304\nChevrolet Captiva,3K10409,11.17,123,Gasoline,1.4088\nChevrolet Captiva,3K10409,36.5,123,Gasoline,4.6156\nChevrolet Captiva,3K10409,20.04,123,Gasoline,2.856\nChevrolet Captiva,3K10409,57.18,123,Gasoline,6.4709\nChevrolet Captiva,3K10409,4.08,123,Gasoline,0.7332\nChevrolet Captiva,3K10409,53.11,123,Gasoline,5.9097\nChevrolet Captiva,3K10409,4.52,123,Gasoline,0.568\nChevrolet Captiva,3K10409,32.7,123,Gasoline,3.1705\nChevrolet Captiva,3K10409,45.22,123,Gasoline,4.0776\nChevrolet Captiva,3K10409,10.0,123,Gasoline,1.304\nChevrolet Captiva,3K10409,12.32,123,Gasoline,1.7522\nChevrolet Captiva,3K10409,17.2,123,Gasoline,1.9833\nChevrolet Captiva,3K10409,5.53,123,Gasoline,1.0252\nChevrolet Captiva,3K10409,2.7,123,Gasoline,0.5631\nChevrolet Captiva,3K10409,101.59,123,Gasoline,9.5314\nChevrolet Captiva,3K10409,54.57,123,Gasoline,5.7989\nChevrolet Captiva,3K10409,102.36,123,Gasoline,9.1788\nChevrolet Captiva,3K10409,83.21,123,Gasoline,8.8694\nChevrolet Captiva,3K10409,4.58,123,Gasoline,0.7396\nChevrolet Captiva,3K10409,6.16,123,Gasoline,0.8319\nChevrolet Captiva,3K10409,7.83,123,Gasoline,1.0066\nChevrolet Captiva,3K10409,21.66,123,Gasoline,3.2525\nChevrolet Captiva,3K10409,56.77,123,Gasoline,6.0744\nChevrolet Captiva,3K10409,8.36,123,Gasoline,1.2329\nChevrolet Captiva,3K10409,13.97,123,Gasoline,1.8611\nChevrolet Captiva,3K10409,51.41,123,Gasoline,6.0557\nChevrolet Captiva,3K10409,94.26,123,Gasoline,9.4303\nChevrolet Captiva,3K10409,17.45,123,Gasoline,2.0028\nChevrolet Captiva,3K10409,5.28,123,Gasoline,0.9424\nChevrolet Captiva,3K10409,60.92,123,Gasoline,5.5932\nChevrolet Captiva,3K10409,2.53,123,Gasoline,0.4831\nChevrolet Captiva,3K10409,27.22,123,Gasoline,2.9508\nChevrolet Captiva,3K10409,503.81,123,Gasoline,46.6664\nChevrolet Captiva,3K10409,9.53,123,Gasoline,1.2544\nChevrolet Captiva,3K10409,112.94,123,Gasoline,10.4417\nChevrolet Captiva,3K10409,32.91,123,Gasoline,3.2557\nChevrolet Captiva,3K10409,30.9,123,Gasoline,3.3893\nChevrolet Captiva,3K10409,67.28,123,Gasoline,9.283\nChevrolet Captiva,3K10409,36.47,123,Gasoline,4.3558\nChevrolet Captiva,3K10409,79.59,123,Gasoline,9.2591\nChevrolet Captiva,3K10409,100.81,123,Gasoline,12.6724\nChevrolet Captiva,3K10409,16.84,123,Gasoline,1.8468\nChevrolet Captiva,3K10409,50.48,123,Gasoline,5.6997\nChevrolet Captiva,3K10409,183.0,123,Gasoline,16.399\nChevrolet Captiva,3K10409,6.39,123,Gasoline,1.0857\nChevrolet Captiva,3K10409,223.66,123,Gasoline,23.804\nChevrolet Captiva,3K10409,2.95,123,Gasoline,0.51\nChevrolet Captiva,3K10409,5.67,123,Gasoline,0.8731\nChevrolet Captiva,3K10409,253.24,123,Gasoline,26.9195\nChevrolet Captiva,3K10409,235.1,123,Gasoline,23.9095\nChevrolet Captiva,3K10409,173.32,123,Gasoline,16.8545\nChevrolet Captiva,3K10409,3.94,123,Gasoline,0.7852\nChevrolet Captiva,3K10409,192.01,123,Gasoline,19.926\nChevrolet Captiva,3K10409,12.86,123,Gasoline,1.4154\nChevrolet Captiva,3K10409,4.12,123,Gasoline,0.6014\nChevrolet Captiva,3K10409,10.93,123,Gasoline,1.5797\nChevrolet Captiva,3K10409,74.87,123,Gasoline,7.0307\nChevrolet Captiva,3K10409,21.45,123,Gasoline,3.9895\nChevrolet Captiva,3K10409,37.34,123,Gasoline,4.2527\nChevrolet Captiva,3K10409,217.57,123,Gasoline,20.7313\nChevrolet Captiva,3K10409,2.55,123,Gasoline,0.5574\nChevrolet Captiva,3K10409,148.73,123,Gasoline,15.6379\nChevrolet Captiva,3K10409,114.4,123,Gasoline,11.7508\nChevrolet Captiva,3K10409,348.8,123,Gasoline,31.8371\nChevrolet Captiva,3K10409,116.52,123,Gasoline,10.7386\nChevrolet Captiva,3K10409,6.93,123,Gasoline,1.0228\nChevrolet Captiva,3K10409,113.75,123,Gasoline,10.6058\nChevrolet Captiva,3K10409,484.54,123,Gasoline,45.5469\nChevrolet Captiva,3K10409,8.22,123,Gasoline,1.1798\nChevrolet Captiva,3K10409,367.0,123,Gasoline,37.7565\nChevrolet Captiva,3K10409,1.88,123,Gasoline,0.3607\nChevrolet Captiva,3K10409,32.84,123,Gasoline,3.6277\nChevrolet Captiva,3K10409,182.99,123,Gasoline,16.8455\nChevrolet Captiva,3K10409,18.8,123,Gasoline,2.1345\nChevrolet Captiva,3K10409,5.9,123,Gasoline,0.8642\nChevrolet Captiva,3K10409,31.78,123,Gasoline,3.383\nChevrolet Captiva,3K10409,32.45,123,Gasoline,3.3408\nChevrolet Captiva,3K10409,1.87,123,Gasoline,0.4751\nChevrolet Captiva,3K10409,41.49,123,Gasoline,4.2372\nChevrolet Captiva,3K10409,54.32,123,Gasoline,5.7568\nChevrolet Captiva,3K10409,5.07,123,Gasoline,0.5204\nChevrolet Captiva,3K10409,2.96,123,Gasoline,0.4804\nChevrolet Captiva,3K10409,53.78,123,Gasoline,4.805\nChevrolet Captiva,3K10409,14.43,123,Gasoline,1.5915\nChevrolet Captiva,3K10409,4.3,123,Gasoline,0.5568\nChevrolet Captiva,3K10409,3.61,123,Gasoline,0.5583\nChevrolet Captiva,3K10409,147.75,123,Gasoline,13.1742\nChevrolet Captiva,3K10409,14.25,123,Gasoline,1.68\nChevrolet Captiva,3K10409,3.1,123,Gasoline,0.6256\nChevrolet Captiva,3K10409,12.93,123,Gasoline,1.5491\nChevrolet Captiva,3K10409,12.54,123,Gasoline,1.4296\nChevrolet Captiva,3K10409,80.96,123,Gasoline,9.6864\nChevrolet Captiva,3K10409,6.8,123,Gasoline,0.946\nChevrolet Captiva,3K10409,103.1,123,Gasoline,9.1019\nChevrolet Captiva,3K10409,4.49,123,Gasoline,0.4916\nChevrolet Captiva,3K10409,17.54,123,Gasoline,1.845\nChevrolet Captiva,3K10409,16.42,123,Gasoline,1.5438\nChevrolet Captiva,3K10409,19.29,123,Gasoline,2.0456\nChevrolet Captiva,3K10409,60.35,123,Gasoline,5.3868\nChevrolet Captiva,3K10409,44.61,123,Gasoline,4.3604\nChevrolet Captiva,3K10409,6.98,123,Gasoline,0.9606\nChevrolet Captiva,3K10409,140.39,123,Gasoline,14.0952\nChevrolet Captiva,3K10409,82.47,123,Gasoline,7.6256\nChevrolet Captiva,3K10409,103.28,123,Gasoline,10.4551\nChevrolet Captiva,3K10409,109.43,123,Gasoline,10.5348\nChevrolet Captiva,3K10409,34.76,123,Gasoline,3.274\nChevrolet Captiva,3K10409,7.28,123,Gasoline,1.1792\nChevrolet Captiva,3K10409,34.78,123,Gasoline,2.8257\nChevrolet Captiva,3K10409,49.63,123,Gasoline,4.3269\nChevrolet Captiva,3K10409,14.34,123,Gasoline,1.697\nChevrolet Captiva,3K10409,45.82,123,Gasoline,4.8421\nChevrolet Captiva,3K10409,5.33,123,Gasoline,0.8994\nChevrolet Captiva,3K10409,50.07,123,Gasoline,4.6914\nChevrolet Captiva,3K10409,216.2,123,Gasoline,16.982\nChevrolet Captiva,3K10409,5.19,123,Gasoline,0.7867\nChevrolet Captiva,3K10409,124.26,123,Gasoline,11.2669\nChevrolet Captiva,3K10409,63.64,123,Gasoline,6.5264\nChevrolet Captiva,3K10409,3.33,123,Gasoline,0.6193\nChevrolet Captiva,3K10409,172.42,123,Gasoline,14.7318\nChevrolet Captiva,3K10409,25.97,123,Gasoline,2.8369\nChevrolet Captiva,3K10409,50.52,123,Gasoline,4.7339\nChevrolet Captiva,3K10409,43.88,123,Gasoline,4.7201\nChevrolet Captiva,3K10409,3.78,123,Gasoline,0.4064\nChevrolet Captiva,3K10409,17.64,123,Gasoline,2.2943\nChevrolet Captiva,3K10409,123.84,123,Gasoline,11.3632\nChevrolet Captiva,3K10409,2.72,123,Gasoline,0.6069\nChevrolet Captiva,3K10409,10.65,123,Gasoline,1.2526\nChevrolet Captiva,3K10409,1.63,123,Gasoline,0.3112\nChevrolet Captiva,3K10409,13.07,123,Gasoline,1.9215\nChevrolet Captiva,3K10409,117.09,123,Gasoline,12.1576\nChevrolet Captiva,3K10409,78.29,123,Gasoline,8.2745\nChevrolet Captiva,3K10409,9.66,123,Gasoline,1.4837\nChevrolet Captiva,3K10409,11.47,123,Gasoline,1.9157\nChevrolet Captiva,3K10409,125.24,123,Gasoline,11.1949\nChevrolet Captiva,3K10409,20.93,123,Gasoline,2.423\nChevrolet Captiva,3K10409,11.8,123,Gasoline,1.3549\nChevrolet Captiva,3K10409,10.48,123,Gasoline,1.242\nChevrolet Captiva,3K10409,108.86,123,Gasoline,8.0699\nChevrolet Captiva,3K10409,13.83,123,Gasoline,1.7436\nChevrolet Captiva,3K10409,51.61,123,Gasoline,4.6723\nChevrolet Captiva,3K10409,14.26,123,Gasoline,2.0875\nChevrolet Captiva,3K10409,27.1,123,Gasoline,3.0071\nChevrolet Captiva,3K10409,14.71,123,Gasoline,1.8922\nChevrolet Captiva,3K10409,45.37,123,Gasoline,4.1051\nChevrolet Captiva,3K10409,18.0,123,Gasoline,2.1809\nChevrolet Captiva,3K10409,8.54,123,Gasoline,1.1466\nChevrolet Captiva,3K10409,61.66,123,Gasoline,5.8796\nChevrolet Captiva,3K10409,132.38,123,Gasoline,12.8864\nChevrolet Captiva,3K10409,18.03,123,Gasoline,1.8646\nChevrolet Captiva,3K10409,33.11,123,Gasoline,3.0852\nChevrolet Captiva,3K10409,26.05,123,Gasoline,3.3349\nChevrolet Captiva,3K10409,4.16,123,Gasoline,0.7954\nChevrolet Captiva,3K10409,105.3,123,Gasoline,9.0941\nChevrolet Captiva,3K10409,96.86,123,Gasoline,9.6541\nChevrolet Captiva,3K10409,1.58,123,Gasoline,0.3489\nChevrolet Captiva,3K10409,1.76,123,Gasoline,0.3905\nChevrolet Captiva,3K10409,48.65,123,Gasoline,4.7235\nChevrolet Captiva,3K10409,112.8,123,Gasoline,10.4764\nChevrolet Captiva,3K10409,74.76,123,Gasoline,7.4129\nChevrolet Captiva,3K10409,12.58,123,Gasoline,1.7061\nChevrolet Captiva,3K10409,131.25,123,Gasoline,11.5673\nChevrolet Captiva,3K10409,14.62,123,Gasoline,1.8356\nChevrolet Captiva,3K10409,16.23,123,Gasoline,2.144\nChevrolet Captiva,3K10409,10.04,123,Gasoline,1.1462\nChevrolet Captiva,3K10409,6.91,123,Gasoline,0.7727\nChevrolet Captiva,3K10409,11.59,123,Gasoline,1.6978\nChevrolet Captiva,3K10409,104.24,123,Gasoline,9.6788\nChevrolet Captiva,3K10409,109.16,123,Gasoline,9.6463\nChevrolet Captiva,3K10409,16.65,123,Gasoline,1.7388\nChevrolet Captiva,3K10409,18.97,123,Gasoline,1.9784\nChevrolet Captiva,3K10409,50.22,123,Gasoline,4.9096\nChevrolet Captiva,3K10409,2.14,123,Gasoline,0.4148\nChevrolet Captiva,3K10409,79.3,123,Gasoline,7.5142\nChevrolet Captiva,3K10409,12.21,123,Gasoline,1.294\nChevrolet Captiva,3K10409,32.63,123,Gasoline,3.3167\nChevrolet Captiva,3K10409,29.34,123,Gasoline,3.4031\nChevrolet Captiva,3K10409,5.11,123,Gasoline,0.7589\nChevrolet Captiva,3K10409,33.83,123,Gasoline,3.0769\nChevrolet Captiva,3K10409,10.53,123,Gasoline,1.167\nChevrolet Captiva,3K10409,35.24,123,Gasoline,3.5812\nChevrolet Captiva,3K10409,126.82,123,Gasoline,11.3456\nChevrolet Captiva,3K10409,118.59,123,Gasoline,11.0728\nChevrolet Captiva,3K10409,46.47,123,Gasoline,4.497\nChevrolet Captiva,3K10409,10.37,123,Gasoline,1.5964\nChevrolet Captiva,3K10409,21.88,123,Gasoline,2.4916\nChevrolet Captiva,3K10409,3.57,123,Gasoline,0.4959\nChevrolet Captiva,3K10409,13.61,123,Gasoline,1.8876\nChevrolet Captiva,3K10409,49.95,123,Gasoline,4.3762\nChevrolet Captiva,3K10409,2.98,123,Gasoline,0.4006\nChevrolet Captiva,3K10409,104.27,123,Gasoline,9.6254\nChevrolet Captiva,3K10409,126.47,123,Gasoline,12.3434\nChevrolet Captiva,3K10409,153.29,123,Gasoline,14.8152\nChevrolet Captiva,3K10409,204.91,123,Gasoline,20.5497\nChevrolet Captiva,3K10409,141.79,123,Gasoline,15.0893\nChevrolet Captiva,3K10409,40.62,123,Gasoline,4.7925\nChevrolet Captiva,3K10409,117.86,123,Gasoline,10.9753\nChevrolet Captiva,3K10409,65.02,123,Gasoline,5.8614\nChevrolet Captiva,3K10409,9.51,123,Gasoline,1.2233\nChevrolet Captiva,3K10409,99.58,123,Gasoline,9.3326\nChevrolet Captiva,3K10409,310.22,123,Gasoline,31.312\nChevrolet Captiva,3K10409,126.3,123,Gasoline,13.8058\nChevrolet Captiva,3K10409,8.58,123,Gasoline,0.953\nChevrolet Captiva,3K10409,2.07,123,Gasoline,0.3336\nChevrolet Captiva,3K10409,13.07,123,Gasoline,1.5502\nChevrolet Captiva,3K10409,88.98,123,Gasoline,8.1466\nChevrolet Captiva,3K10409,109.62,123,Gasoline,9.9258\nChevrolet Captiva,3K10409,45.67,123,Gasoline,4.4955\nChevrolet Captiva,3K10409,174.48,123,Gasoline,18.3212\nChevrolet Captiva,3K10409,222.28,123,Gasoline,18.4166\nChevrolet Captiva,3K10409,48.57,123,Gasoline,5.3986\nChevrolet Captiva,3K10409,56.7,123,Gasoline,6.563\nChevrolet Captiva,3K10409,9.88,123,Gasoline,1.2471\nChevrolet Captiva,3K10409,2.06,123,Gasoline,0.3463\nChevrolet Captiva,3K10409,43.54,123,Gasoline,3.6894\nChevrolet Captiva,3K10409,148.62,123,Gasoline,11.665\nChevrolet Captiva,3K10409,75.59,123,Gasoline,6.373\nChevrolet Captiva,3K10409,18.56,123,Gasoline,2.3798\nChevrolet Captiva,3K10409,132.81,123,Gasoline,9.5083\nChevrolet Captiva,3K10409,5.8,123,Gasoline,0.9492\nChevrolet Captiva,3K10409,27.44,123,Gasoline,2.6271\nChevrolet Captiva,3K10409,114.73,123,Gasoline,9.6557\nChevrolet Captiva,3K10409,8.43,123,Gasoline,1.1348\nChevrolet Captiva,3K10409,2.85,123,Gasoline,0.5273\nChevrolet Captiva,3K10409,139.07,123,Gasoline,12.203\nChevrolet Captiva,3K10409,23.3,123,Gasoline,2.9753\nChevrolet Captiva,3K10409,4.56,123,Gasoline,0.5503\nChevrolet Captiva,3K10409,34.1,123,Gasoline,4.8543\nChevrolet Captiva,3K10409,31.68,123,Gasoline,2.8878\nChevrolet Captiva,3K10409,18.42,123,Gasoline,2.6863\nChevrolet Captiva,3K10409,13.78,123,Gasoline,1.4194\nChevrolet Captiva,3K10409,203.22,123,Gasoline,21.4404\nChevrolet Captiva,3K10409,25.82,123,Gasoline,2.9534\nChevrolet Captiva,3K10409,46.24,123,Gasoline,4.838\nChevrolet Captiva,3K10409,203.29,123,Gasoline,21.291\nChevrolet Captiva,3K10409,11.84,123,Gasoline,1.472\nChevrolet Captiva,3K10409,141.05,123,Gasoline,12.0361\nChevrolet Captiva,3K10409,10.42,123,Gasoline,1.0965\nChevrolet Captiva,3K10409,47.85,123,Gasoline,4.8997\nChevrolet Captiva,3K10409,31.39,123,Gasoline,3.108\nChevrolet Captiva,3K10409,40.57,123,Gasoline,3.5347\nChevrolet Captiva,3K10409,127.1,123,Gasoline,10.5851\nChevrolet Captiva,3K10409,182.48,123,Gasoline,15.6344\nChevrolet Captiva,3K10409,72.34,123,Gasoline,8.2996\nChevrolet Captiva,3K10409,2.78,123,Gasoline,0.5043\nChevrolet Captiva,3K10409,49.18,123,Gasoline,4.0029\nChevrolet Captiva,3K10409,4.52,123,Gasoline,0.7583\nChevrolet Captiva,3K10409,35.85,123,Gasoline,4.6243\nChevrolet Captiva,3K10409,16.67,123,Gasoline,1.4583\nChevrolet Captiva,3K10409,13.58,123,Gasoline,2.014\nChevrolet Captiva,3K10409,107.52,123,Gasoline,12.029\nChevrolet Captiva,3K10409,57.47,123,Gasoline,6.0792\nChevrolet Captiva,3K10409,38.22,123,Gasoline,6.1807\nChevrolet Captiva,3K10409,4.27,123,Gasoline,0.7485\nChevrolet Captiva,3K10409,16.62,123,Gasoline,2.0468\nChevrolet Captiva,3K10409,206.62,123,Gasoline,18.3862\nChevrolet Captiva,3K10409,48.32,123,Gasoline,4.8097\nChevrolet Captiva,3K10409,121.56,123,Gasoline,11.5913\nChevrolet Captiva,3K10409,28.08,123,Gasoline,3.7612\nChevrolet Captiva,3K10409,191.66,123,Gasoline,15.4623\nChevrolet Captiva,3K10409,109.22,123,Gasoline,9.4687\nChevrolet Captiva,3K10409,229.46,123,Gasoline,23.0337\nChevrolet Captiva,3K10409,313.11,123,Gasoline,29.5031\nChevrolet Captiva,3K10409,115.27,123,Gasoline,9.3383\nChevrolet Captiva,3K10409,3.48,123,Gasoline,0.681\nChevrolet Captiva,3K10409,133.14,123,Gasoline,14.3916\nChevrolet Captiva,3K10409,1.6,123,Gasoline,0.2673\nChevrolet Captiva,3K10409,8.29,123,Gasoline,1.3142\nChevrolet Captiva,3K10409,18.46,123,Gasoline,2.5015\nChevrolet Captiva,3K10409,94.92,123,Gasoline,7.5054\nChevrolet Captiva,3K10409,10.87,123,Gasoline,1.5242\nChevrolet Captiva,3K10409,3.92,123,Gasoline,0.4362\nChevrolet Captiva,3K10409,39.93,123,Gasoline,4.6263\nChevrolet Captiva,3K10409,13.7,123,Gasoline,1.5235\nChevrolet Captiva,3K10409,13.79,123,Gasoline,1.5826\nChevrolet Captiva,3K10409,9.98,123,Gasoline,1.1487\nChevrolet Captiva,3K10409,31.66,123,Gasoline,3.046\nChevrolet Captiva,3K10409,10.53,123,Gasoline,2.0292\nChevrolet Captiva,3K10409,103.22,123,Gasoline,10.7418\nChevrolet Captiva,3K10409,48.2,123,Gasoline,3.7008\nChevrolet Captiva,3K10409,12.28,123,Gasoline,1.856\nChevrolet Captiva,3K10409,5.49,123,Gasoline,0.8177\nChevrolet Captiva,3K10409,7.59,123,Gasoline,1.3428\nChevrolet Captiva,3K10409,16.3,123,Gasoline,1.9683\nChevrolet Captiva,3K10409,33.35,123,Gasoline,3.4155\nChevrolet Captiva,3K10409,8.63,123,Gasoline,1.3248\nChevrolet Captiva,3K10409,3.97,123,Gasoline,0.6322\nChevrolet Captiva,3K10409,8.28,123,Gasoline,1.6349\nChevrolet Captiva,3K10409,92.02,123,Gasoline,8.5947\nChevrolet Captiva,3K10409,41.63,123,Gasoline,5.5083\nChevrolet Captiva,3K10409,106.27,123,Gasoline,11.0893\nChevrolet Captiva,3K10409,44.26,123,Gasoline,4.2621\nChevrolet Captiva,3K10409,307.53,123,Gasoline,29.8627\nChevrolet Captiva,3K10409,10.01,123,Gasoline,1.1945\nChevrolet Captiva,3K10409,17.53,123,Gasoline,2.0917\nChevrolet Captiva,3K10409,5.31,123,Gasoline,0.8781\nChevrolet Captiva,3K10409,3.63,123,Gasoline,0.5363\nChevrolet Captiva,3K10409,37.98,123,Gasoline,4.7822\nChevrolet Captiva,3K10409,1.73,123,Gasoline,0.2461\nChevrolet Captiva,3K10409,16.62,123,Gasoline,1.6276\nChevrolet Captiva,3K10409,10.18,123,Gasoline,1.2062\nChevrolet Captiva,3K10409,184.43,123,Gasoline,19.2917\nChevrolet Captiva,3K10409,137.65,123,Gasoline,13.9736\nChevrolet Captiva,3K10409,34.95,123,Gasoline,4.5049\nChevrolet Captiva,3K10409,4.86,123,Gasoline,1.1859\nChevrolet Captiva,3K10409,40.94,123,Gasoline,4.5576\nChevrolet Captiva,3K10409,32.57,123,Gasoline,3.8486\nChevrolet Captiva,3K10409,134.44,123,Gasoline,14.533\nChevrolet Captiva,3K10409,107.3,123,Gasoline,9.347\nChevrolet Captiva,3K10409,4.38,123,Gasoline,0.7762\nChevrolet Captiva,3K10409,3.96,123,Gasoline,0.7622\nChevrolet Captiva,3K10409,2.59,123,Gasoline,0.481\nChevrolet Captiva,3K10409,16.68,123,Gasoline,1.5738\nChevrolet Captiva,3K10409,133.48,123,Gasoline,13.8852\nChevrolet Captiva,3K10409,24.03,123,Gasoline,4.5013\nChevrolet Captiva,3K10409,1.55,123,Gasoline,0.2401\nChevrolet Captiva,3K10409,31.99,123,Gasoline,4.228\nChevrolet Captiva,3K10409,31.12,123,Gasoline,2.995\nChevrolet Captiva,3K10409,54.76,123,Gasoline,7.5433\nChevrolet Captiva,3K10409,2.01,123,Gasoline,0.3857\nChevrolet Captiva,3K10409,22.14,123,Gasoline,2.6453\nChevrolet Captiva,3K10409,9.53,123,Gasoline,1.4995\nChevrolet Captiva,3K10409,28.78,123,Gasoline,3.6104\nChevrolet Captiva,3K10409,71.39,123,Gasoline,7.813\nChevrolet Captiva,3K10409,71.87,123,Gasoline,7.2432\nChevrolet Captiva,3K10409,33.98,123,Gasoline,4.3505\nChevrolet Captiva,3K10409,137.58,123,Gasoline,11.953\nChevrolet Captiva,3K10409,1.88,123,Gasoline,0.237\nChevrolet Captiva,3K10409,3.18,123,Gasoline,0.4979\nChevrolet Captiva,3K10409,8.63,123,Gasoline,1.1363\nChevrolet Captiva,3K10409,20.16,123,Gasoline,2.7056\nChevrolet Captiva,3K10409,76.1,123,Gasoline,7.0459\nChevrolet Captiva,3K10409,125.73,123,Gasoline,10.3489\nChevrolet Captiva,3K10409,234.0,123,Gasoline,27.0745\nChevrolet Captiva,3K10409,275.73,123,Gasoline,30.3717\nChevrolet Captiva,3K10409,16.01,123,Gasoline,2.091\nChevrolet Captiva,3K10409,21.98,123,Gasoline,2.2402\nChevrolet Captiva,3K10409,13.47,123,Gasoline,2.1084\nChevrolet Captiva,3K10409,41.98,123,Gasoline,4.0878\nChevrolet Captiva,3K10409,109.84,123,Gasoline,12.1433\nChevrolet Captiva,3K10409,51.53,123,Gasoline,5.2492\nChevrolet Captiva,3K10409,110.12,123,Gasoline,10.9617\nChevrolet Captiva,3K10409,110.48,123,Gasoline,11.1039\nChevrolet Captiva,3K10409,28.19,123,Gasoline,3.5948\nChevrolet Captiva,3K10409,30.5,123,Gasoline,3.3523\nChevrolet Captiva,3K10409,27.89,123,Gasoline,2.869\nChevrolet Captiva,3K10409,10.9,123,Gasoline,1.5615\nChevrolet Captiva,3K10409,141.96,123,Gasoline,12.5704\nChevrolet Captiva,3K10409,3.32,123,Gasoline,0.7104\nChevrolet Captiva,3K10409,321.59,123,Gasoline,30.2134\nChevrolet Captiva,3K10409,38.08,123,Gasoline,4.1624\nChevrolet Captiva,3K10409,12.37,123,Gasoline,1.7583\nChevrolet Captiva,3K10409,17.62,123,Gasoline,1.7875\nChevrolet Captiva,3K10409,11.82,123,Gasoline,1.6177\nChevrolet Captiva,3K10409,88.54,123,Gasoline,9.6343\nChevrolet Captiva,3K10409,169.9,123,Gasoline,17.2352\nChevrolet Captiva,3K10409,98.36,123,Gasoline,8.219\nChevrolet Captiva,3K10409,13.9,123,Gasoline,1.5187\nChevrolet Captiva,3K10409,194.86,123,Gasoline,19.47\nChevrolet Captiva,3K10409,2.17,123,Gasoline,0.5605\nChevrolet Captiva,3K10409,8.98,123,Gasoline,2.2205\nChevrolet Captiva,3K10409,46.22,123,Gasoline,4.6857\nChevrolet Captiva,3K10409,123.16,123,Gasoline,17.3892\nChevrolet Captiva,3K10409,49.03,123,Gasoline,5.8532\nChevrolet Captiva,3K10409,7.37,123,Gasoline,1.271\nChevrolet Captiva,3K10409,1.96,123,Gasoline,0.3698\nChevrolet Captiva,3K10409,292.47,123,Gasoline,29.2458\nChevrolet Captiva,3K10409,7.18,123,Gasoline,1.2548\nChevrolet Captiva,3K10409,8.64,123,Gasoline,1.1297\nChevrolet Captiva,3K10409,49.96,123,Gasoline,4.7475\nChevrolet Captiva,3K10409,93.5,123,Gasoline,10.1949\nChevrolet Captiva,3K10409,233.39,123,Gasoline,24.5563\nChevrolet Captiva,3K10409,339.31,123,Gasoline,31.3005\nChevrolet Captiva,3K10409,174.99,123,Gasoline,14.8765\nChevrolet Captiva,3K10409,16.92,123,Gasoline,1.6119\nChevrolet Captiva,3K10409,37.35,123,Gasoline,3.7755\nChevrolet Captiva,3K10409,129.5,123,Gasoline,12.6752\nChevrolet Captiva,3K10409,119.77,123,Gasoline,7.8402\nChevrolet Captiva,3K10409,463.38,123,Gasoline,51.1415\nChevrolet Captiva,3K10409,212.44,123,Gasoline,23.2294\nChevrolet Captiva,3K10409,2.87,123,Gasoline,0.5081\nChevrolet Captiva,3K10409,5.59,123,Gasoline,1.1001\nChevrolet Captiva,3K10409,285.68,123,Gasoline,28.5645\nChevrolet Captiva,3K10409,152.98,123,Gasoline,17.2097\nChevrolet Captiva,3K10409,221.09,123,Gasoline,23.2508\nChevrolet Captiva,3K10409,62.24,123,Gasoline,5.6351\nChevrolet Captiva,3K10409,6.66,123,Gasoline,0.8527\nChevrolet Captiva,3K10409,261.58,123,Gasoline,25.6718\nChevrolet Captiva,3K10409,9.77,123,Gasoline,1.3257\nChevrolet Captiva,3K10409,35.73,123,Gasoline,4.4792\nChevrolet Captiva,3K10409,3.5,123,Gasoline,0.4161\nChevrolet Captiva,3K10409,3.09,123,Gasoline,0.5401\nChevrolet Captiva,3K10409,11.95,123,Gasoline,1.6182\nChevrolet Captiva,3K10409,160.59,123,Gasoline,15.3709\nChevrolet Captiva,3K10409,161.51,123,Gasoline,14.6643\nChevrolet Captiva,3K10409,235.36,123,Gasoline,21.1088\nChevrolet Captiva,3K10409,12.09,123,Gasoline,1.4399\nChevrolet Captiva,3K10409,181.86,123,Gasoline,16.6039\nChevrolet Captiva,3K10409,33.98,123,Gasoline,3.3331\nChevrolet Captiva,3K10409,297.71,123,Gasoline,29.5297\nChevrolet Captiva,3K10409,19.43,123,Gasoline,1.8177\nChevrolet Captiva,3K10409,4.45,123,Gasoline,0.8712\nChevrolet Captiva,3K10409,4.69,123,Gasoline,0.7463\nChevrolet Captiva,3K10409,10.65,123,Gasoline,1.7805\nChevrolet Captiva,3K10409,138.31,123,Gasoline,13.6666\nChevrolet Captiva,3K10409,7.07,123,Gasoline,1.2845\nChevrolet Captiva,3K10409,185.73,123,Gasoline,19.9252\nChevrolet Captiva,3K10409,170.82,123,Gasoline,16.2215\nChevrolet Captiva,3K10409,23.03,123,Gasoline,3.268\nChevrolet Captiva,3K10409,152.86,123,Gasoline,13.1207\nChevrolet Captiva,3K10409,455.13,123,Gasoline,46.5934\nChevrolet Captiva,3K10409,186.34,123,Gasoline,19.9241\nChevrolet Captiva,3K10409,109.77,123,Gasoline,7.7332\nChevrolet Captiva,3K10409,169.88,123,Gasoline,16.6615\nChevrolet Captiva,3K10409,6.64,123,Gasoline,0.8229\nChevrolet Captiva,3K10409,7.37,123,Gasoline,1.8836\nChevrolet Captiva,3K10409,183.3,123,Gasoline,16.8483\nChevrolet Captiva,3K10409,38.16,123,Gasoline,3.5836\nChevrolet Captiva,3K10409,2.74,123,Gasoline,0.4619\nChevrolet Captiva,3K10409,36.32,123,Gasoline,3.901\nChevrolet Captiva,3K10409,192.71,123,Gasoline,19.5495\nChevrolet Captiva,3K10409,87.88,123,Gasoline,7.7645\nChevrolet Captiva,3K10409,17.04,123,Gasoline,2.3503\nChevrolet Captiva,3K10409,12.6,123,Gasoline,2.5835\nChevrolet Captiva,3K10409,338.34,123,Gasoline,33.707\nChevrolet Captiva,3K10409,141.61,123,Gasoline,14.9211\nChevrolet Captiva,3K10409,55.28,123,Gasoline,4.8175\nChevrolet Captiva,3K10409,21.13,123,Gasoline,2.8594\nChevrolet Captiva,3K10409,2.8,123,Gasoline,0.3876\nChevrolet Captiva,3K10409,272.16,123,Gasoline,22.0122\nChevrolet Captiva,3K10409,144.45,123,Gasoline,13.8112\nChevrolet Captiva,3K10409,8.97,123,Gasoline,1.2955\nChevrolet Captiva,3K10409,14.48,123,Gasoline,1.7928\nChevrolet Captiva,3K10409,218.17,123,Gasoline,17.9459\nChevrolet Captiva,3K10409,157.13,123,Gasoline,14.7473\nChevrolet Captiva,3K10409,32.47,123,Gasoline,3.0446\nChevrolet Captiva,3K10409,222.9,123,Gasoline,22.1603\nChevrolet Captiva,3K10409,57.68,123,Gasoline,6.7213\nChevrolet Captiva,3K10409,115.58,123,Gasoline,12.2053\nChevrolet Captiva,3K10409,49.25,123,Gasoline,6.3575\nChevrolet Captiva,3K10409,13.92,123,Gasoline,2.4593\nChevrolet Captiva,3K10409,218.9,123,Gasoline,20.9377\nChevrolet Captiva,3K10409,16.76,123,Gasoline,1.9056\nChevrolet Captiva,3K10409,28.64,123,Gasoline,3.0331\nChevrolet Captiva,3K10409,16.63,123,Gasoline,1.6903\nChevrolet Captiva,3K10409,35.73,123,Gasoline,3.2118\nChevrolet Captiva,3K10409,3.78,123,Gasoline,0.7398\nChevrolet Captiva,3K10409,56.6,123,Gasoline,5.0277\nChevrolet Captiva,3K10409,118.54,123,Gasoline,10.9399\nChevrolet Captiva,3K10409,22.52,123,Gasoline,2.1595\nChevrolet Captiva,3K10409,16.3,123,Gasoline,2.2802\nChevrolet Captiva,3K10409,24.81,123,Gasoline,3.0786\nChevrolet Captiva,3K10409,139.84,123,Gasoline,12.8727\nChevrolet Captiva,3K10409,102.02,123,Gasoline,10.0131\nChevrolet Captiva,3K10409,36.33,123,Gasoline,4.1741\nChevrolet Captiva,3K10409,3.46,123,Gasoline,0.6287\nChevrolet Captiva,3K10409,118.69,123,Gasoline,11.8868\nChevrolet Captiva,3K10409,20.75,123,Gasoline,2.8162\nChevrolet Captiva,3K10409,5.63,123,Gasoline,0.9942\nChevrolet Captiva,3K10409,16.42,123,Gasoline,2.2645\nChevrolet Captiva,3K10409,5.99,123,Gasoline,1.0443\nChevrolet Captiva,3K10409,31.33,123,Gasoline,3.0098\nChevrolet Captiva,3K10409,112.39,123,Gasoline,9.6625\nChevrolet Captiva,3K10409,32.43,123,Gasoline,3.3293\nChevrolet Captiva,3K10409,6.85,123,Gasoline,0.9282\nChevrolet Captiva,3K10409,17.01,123,Gasoline,2.7761\nChevrolet Captiva,3K10409,6.38,123,Gasoline,1.3019\nChevrolet Captiva,3K10409,12.4,123,Gasoline,1.1881\nChevrolet Captiva,3K10409,15.62,123,Gasoline,2.2863\nChevrolet Captiva,3K10409,7.86,123,Gasoline,1.6191\nChevrolet Captiva,3K10409,13.83,123,Gasoline,1.7634\nChevrolet Captiva,3K10409,17.07,123,Gasoline,1.6164\nChevrolet Captiva,3K10409,16.65,123,Gasoline,1.6381\nChevrolet Captiva,3K10409,54.46,123,Gasoline,5.2719\nChevrolet Captiva,3K10409,13.88,123,Gasoline,1.5245\nChevrolet Captiva,3K10409,5.05,123,Gasoline,0.8915\nChevrolet Captiva,3K10409,19.92,123,Gasoline,2.2082\nChevrolet Captiva,3K10409,57.07,123,Gasoline,7.0464\nChevrolet Captiva,3K10409,32.43,123,Gasoline,3.543\nChevrolet Captiva,3K10409,216.51,123,Gasoline,19.5839\nChevrolet Captiva,3K10409,169.83,123,Gasoline,13.3725\nChevrolet Captiva,3K10409,114.85,123,Gasoline,10.6997\nChevrolet Captiva,3K10409,178.79,123,Gasoline,16.3646\nChevrolet Captiva,3K10409,105.37,123,Gasoline,8.557\nChevrolet Captiva,3K10409,6.27,123,Gasoline,0.7092\nChevrolet Captiva,3K10409,215.26,123,Gasoline,21.8513\nChevrolet Captiva,3K10409,34.95,123,Gasoline,3.374\nChevrolet Captiva,3K10409,230.22,123,Gasoline,21.3172\nChevrolet Captiva,3K10409,160.87,123,Gasoline,16.5861\nChevrolet Captiva,3K10409,30.59,123,Gasoline,3.1082\nChevrolet Captiva,3K10409,26.28,123,Gasoline,2.7737\nChevrolet Captiva,3K10409,121.48,123,Gasoline,10.8839\nChevrolet Captiva,3K10409,127.31,123,Gasoline,12.6998\nChevrolet Captiva,3K10409,67.42,123,Gasoline,6.8896\nChevrolet Captiva,3K10409,25.98,123,Gasoline,2.8061\nChevrolet Captiva,3K10409,19.0,123,Gasoline,2.2635\nChevrolet Captiva,3K10409,16.96,123,Gasoline,3.1899\nChevrolet Captiva,3K10409,19.38,123,Gasoline,2.14\nChevrolet Captiva,3K10409,7.58,123,Gasoline,1.2277\nChevrolet Captiva,3K10409,2.39,123,Gasoline,0.4522\nChevrolet Captiva,3K10409,33.25,123,Gasoline,3.6736\nChevrolet Captiva,3K10409,4.03,123,Gasoline,0.5751\nChevrolet Captiva,3K10409,38.13,123,Gasoline,3.9727\nChevrolet Captiva,3K10409,15.67,123,Gasoline,1.6298\nChevrolet Captiva,3K10409,72.07,123,Gasoline,6.475\nChevrolet Captiva,3K10409,111.16,123,Gasoline,10.7814\nChevrolet Captiva,3K10409,4.32,123,Gasoline,0.7715\nChevrolet Captiva,3K10409,8.14,123,Gasoline,1.0884\nChevrolet Captiva,3K10409,4.36,123,Gasoline,0.8559\nChevrolet Captiva,3K10409,19.66,123,Gasoline,3.0743\nChevrolet Captiva,3K10409,33.23,123,Gasoline,3.8917\nChevrolet Captiva,3K10409,31.37,123,Gasoline,3.6909\nChevrolet Captiva,3K10409,3.89,123,Gasoline,0.7854\nChevrolet Captiva,3K10409,3.23,123,Gasoline,0.5032\nChevrolet Captiva,3K10409,4.44,123,Gasoline,0.6319\nChevrolet Captiva,3K10409,7.25,123,Gasoline,1.1598\nChevrolet Captiva,3K10409,50.36,123,Gasoline,5.6192\nChevrolet Captiva,3K10409,38.4,123,Gasoline,3.8285\nChevrolet Captiva,3K10409,145.91,123,Gasoline,15.635\nChevrolet Captiva,3K10409,47.68,123,Gasoline,4.7211\nChevrolet Captiva,3K10409,56.3,123,Gasoline,6.084\nChevrolet Captiva,3K10409,13.35,123,Gasoline,1.6788\nChevrolet Captiva,3K10409,35.89,123,Gasoline,3.6542\nChevrolet Captiva,3K10409,12.54,123,Gasoline,1.7574\nChevrolet Captiva,3K10409,128.87,123,Gasoline,11.4293\nChevrolet Captiva,3K10409,6.99,123,Gasoline,0.8932\nChevrolet Captiva,3K10409,54.1,123,Gasoline,5.2317\nChevrolet Captiva,3K10409,42.87,123,Gasoline,4.337\nChevrolet Captiva,3K10409,21.16,123,Gasoline,2.5112\nChevrolet Captiva,3K10409,64.05,123,Gasoline,6.1953\nChevrolet Captiva,3K10409,134.14,123,Gasoline,12.2511\nChevrolet Captiva,3K10409,11.72,123,Gasoline,1.6162\nChevrolet Captiva,3K10409,50.2,123,Gasoline,5.2302\nChevrolet Captiva,3K10409,21.66,123,Gasoline,2.5479\nChevrolet Captiva,3K10409,4.99,123,Gasoline,0.8889\nChevrolet Captiva,3K10409,9.28,123,Gasoline,1.4302\nChevrolet Captiva,3K10409,39.16,123,Gasoline,3.99\nChevrolet Captiva,3K10409,30.16,123,Gasoline,3.4296\nChevrolet Captiva,3K10409,11.19,123,Gasoline,1.47\nChevrolet Captiva,3K10409,12.6,123,Gasoline,1.6967\nChevrolet Captiva,3K10409,130.24,123,Gasoline,12.3875\nChevrolet Captiva,3K10409,6.81,123,Gasoline,1.1083\nChevrolet Captiva,3K10409,3.82,123,Gasoline,0.7615\nChevrolet Captiva,3K10409,6.56,123,Gasoline,1.3192\nChevrolet Captiva,3K10409,5.11,123,Gasoline,0.5439\nChevrolet Captiva,3K10409,10.76,123,Gasoline,1.4884\nChevrolet Captiva,3K10409,43.14,123,Gasoline,4.2457\nChevrolet Captiva,3K10409,16.41,123,Gasoline,1.8388\nChevrolet Captiva,3K10409,2.56,123,Gasoline,0.4106\nChevrolet Captiva,3K10409,1.56,123,Gasoline,0.2822\nChevrolet Captiva,3K10409,5.64,123,Gasoline,0.8366\nChevrolet Captiva,3K10409,1.95,123,Gasoline,0.2991\nChevrolet Captiva,3K10409,101.6,123,Gasoline,12.2581\nChevrolet Captiva,3K10409,13.64,123,Gasoline,2.1072\nChevrolet Captiva,3K10409,4.9,123,Gasoline,0.7515\nChevrolet Captiva,3K10409,2.88,123,Gasoline,0.5631\nChevrolet Captiva,3K10409,10.59,123,Gasoline,1.2224\nChevrolet Captiva,3K10409,37.95,123,Gasoline,3.939\nChevrolet Captiva,3K10409,30.35,123,Gasoline,2.574\nChevrolet Captiva,3K10409,31.07,123,Gasoline,3.2175\nChevrolet Captiva,3K10409,4.87,123,Gasoline,0.7535\nChevrolet Captiva,3K10409,2.3,123,Gasoline,0.408\nChevrolet Captiva,3K10409,65.81,123,Gasoline,6.9282\nChevrolet Captiva,3K10409,15.14,123,Gasoline,1.9166\nChevrolet Captiva,3K10409,167.24,123,Gasoline,15.1527\nChevrolet Captiva,3K10409,10.19,123,Gasoline,1.3562\nChevrolet Captiva,3K10409,129.28,123,Gasoline,12.6258\nChevrolet Captiva,3K10409,32.22,123,Gasoline,3.9483\nChevrolet Captiva,3K10409,33.52,123,Gasoline,3.4308\nChevrolet Captiva,3K10409,11.18,123,Gasoline,1.9166\nChevrolet Captiva,3K10409,9.78,123,Gasoline,1.3483\nChevrolet Captiva,3K10409,26.49,123,Gasoline,3.1707\nChevrolet Captiva,3K10409,3.48,123,Gasoline,0.6355\nChevrolet Captiva,3K10409,67.91,123,Gasoline,7.6422\nChevrolet Captiva,3K10409,11.44,123,Gasoline,1.4032\nChevrolet Captiva,3K10409,10.02,123,Gasoline,1.1061\nChevrolet Captiva,3K10409,17.67,123,Gasoline,2.2878\nChevrolet Captiva,3K10409,31.54,123,Gasoline,3.6134\nChevrolet Captiva,3K10409,3.11,123,Gasoline,0.8215\nChevrolet Captiva,3K10409,5.52,123,Gasoline,0.8069\nChevrolet Captiva,3K10409,12.27,123,Gasoline,1.858\nChevrolet Captiva,3K10409,20.24,123,Gasoline,3.6563\nHonda Accord,0H3082Y,126.69,114,Gasoline,10.7472\nHonda Accord,0H3082Y,103.0,114,Gasoline,7.3366\nHonda Accord,0H3082Y,8.05,114,Gasoline,0.8176\nHonda Accord,0H3082Y,123.5,114,Gasoline,11.893\nHonda Accord,0H3082Y,6.32,114,Gasoline,0.4745\nHonda Accord,0H3082Y,26.17,114,Gasoline,1.8839\nHonda Accord,0H3082Y,121.41,114,Gasoline,10.2315\nHonda Accord,0H3082Y,47.06,114,Gasoline,3.19\nHonda Accord,0H3082Y,7.51,114,Gasoline,0.5543\nHonda Accord,0H3082Y,4.64,114,Gasoline,0.5153\nHonda Accord,0H3082Y,2.52,114,Gasoline,0.3655\nHonda Accord,0H3082Y,36.14,114,Gasoline,2.7182\nHonda Accord,0H3082Y,43.76,114,Gasoline,3.7482\nHonda Accord,0H3082Y,155.2,114,Gasoline,12.144\nHonda Accord,0H3082Y,206.62,114,Gasoline,18.2041\nHonda Accord,0H3082Y,46.0,114,Gasoline,3.4008\nHonda Accord,0H3082Y,15.03,114,Gasoline,1.1318\nHonda Accord,0H3082Y,303.37,114,Gasoline,19.6292\nHonda Accord,0H3082Y,19.74,114,Gasoline,1.4406\nHonda Accord,0H3082Y,8.79,114,Gasoline,0.9344\nHonda Accord,0H3082Y,2.51,114,Gasoline,0.2911\nHonda Accord,0H3082Y,9.18,114,Gasoline,0.6972\nHonda Accord,0H3082Y,19.29,114,Gasoline,1.2391\nHonda Accord,0H3082Y,32.49,114,Gasoline,2.0679\nHonda Accord,0H3082Y,42.86,114,Gasoline,3.539\nHonda Accord,0H3082Y,2.09,114,Gasoline,0.3131\nHonda Accord,0H3082Y,10.62,114,Gasoline,0.7201\nHonda Accord,0H3082Y,5.51,114,Gasoline,0.4605\nHonda Accord,0H3082Y,121.76,114,Gasoline,10.176\nHonda Accord,0H3082Y,16.23,114,Gasoline,1.254\nHonda Accord,0H3082Y,161.61,114,Gasoline,10.1611\nHonda Accord,0H3082Y,12.13,114,Gasoline,0.9957\nHonda Accord,0H3082Y,326.36,114,Gasoline,21.2522\nHonda Accord,0H3082Y,221.74,114,Gasoline,19.2741\nHonda Accord,0H3082Y,189.59,114,Gasoline,15.2126\nHonda Accord,0H3082Y,24.77,114,Gasoline,2.6784\nHonda Accord,0H3082Y,10.44,114,Gasoline,0.9984\nHonda Accord,0H3082Y,5.96,114,Gasoline,0.6314\nHonda Accord,0H3082Y,21.41,114,Gasoline,2.2923\nHonda Accord,0H3082Y,16.22,114,Gasoline,1.8801\nHonda Accord,0H3082Y,1.95,114,Gasoline,0.3941\nHonda Accord,0H3082Y,4.89,114,Gasoline,0.3985\nHonda Accord,0H3082Y,3.04,114,Gasoline,0.2849\nHonda Accord,0H3082Y,29.07,114,Gasoline,3.1345\nHonda Accord,0H3082Y,2.53,114,Gasoline,0.304\nHonda Accord,0H3082Y,17.91,114,Gasoline,1.4795\nHonda Accord,0H3082Y,165.16,114,Gasoline,9.929\nHonda Accord,0H3082Y,165.28,114,Gasoline,10.6462\nHonda Accord,0H3082Y,26.03,114,Gasoline,1.8605\nHonda Accord,0H3082Y,131.18,114,Gasoline,9.6382\nHonda Accord,0H3082Y,351.24,114,Gasoline,22.634\nHonda Accord,0H3082Y,200.89,114,Gasoline,15.2151\nHonda Accord,0H3082Y,431.1,114,Gasoline,28.6061\nHonda Accord,0H3082Y,1.98,114,Gasoline,0.2788\nHonda Accord,0H3082Y,2.76,114,Gasoline,0.1561\nHonda Accord,0H3082Y,64.44,114,Gasoline,4.8158\nHonda Accord,0H3082Y,309.46,114,Gasoline,17.0907\nHonda Accord,0H3082Y,44.46,114,Gasoline,3.2716\nHonda Accord,0H3082Y,28.81,114,Gasoline,2.0569\nHonda Accord,0H3082Y,173.42,114,Gasoline,9.9978\nHonda Accord,0H3082Y,277.26,114,Gasoline,16.542\nHonda Accord,0H3082Y,2.24,114,Gasoline,0.2378\nHonda Accord,0H3082Y,13.15,114,Gasoline,1.5444\nHonda Accord,0H3082Y,34.62,114,Gasoline,2.286\nHonda Accord,0H3082Y,6.24,114,Gasoline,0.8199\nHonda Accord,0H3082Y,111.73,114,Gasoline,7.0149\nHonda Accord,0H3082Y,32.82,114,Gasoline,2.1394\nHonda Accord,0H3082Y,43.34,114,Gasoline,3.6892\nHonda Accord,0H3082Y,151.89,114,Gasoline,12.2208\nHonda Accord,0H3082Y,35.77,114,Gasoline,2.0686\nHonda Accord,0H3082Y,100.75,114,Gasoline,7.7988\nHonda Accord,0H3082Y,296.18,114,Gasoline,18.112\nHonda Accord,0H3082Y,172.94,114,Gasoline,9.7347\nHonda Accord,0H3082Y,5.74,114,Gasoline,0.6741\nHonda Accord,0H3082Y,147.84,114,Gasoline,12.6136\nHonda Accord,0H3082Y,2.63,114,Gasoline,0.3143\nHonda Accord,0H3082Y,4.04,114,Gasoline,0.3392\nHonda Accord,0H3082Y,3.55,114,Gasoline,0.3503\nHonda Accord,0H3082Y,54.26,114,Gasoline,3.5793\nHonda Accord,0H3082Y,3.51,114,Gasoline,0.4918\nHonda Accord,0H3082Y,156.43,114,Gasoline,11.1993\nHonda Accord,0H3082Y,9.12,114,Gasoline,1.0512\nHonda Accord,0H3082Y,31.47,114,Gasoline,2.1729\nHonda Accord,0H3082Y,16.3,114,Gasoline,1.2289\nHonda Accord,0H3082Y,3.95,114,Gasoline,0.5457\nHonda Accord,0H3082Y,6.35,114,Gasoline,0.4919\nHonda Accord,0H3082Y,130.09,114,Gasoline,5.3556\nHonda Accord,0H3082Y,11.96,114,Gasoline,0.8411\nHonda Accord,0H3082Y,2.55,114,Gasoline,0.3545\nHonda Accord,0H3082Y,114.62,114,Gasoline,11.567\nHonda Accord,0H3082Y,2.12,114,Gasoline,0.242\nHonda Accord,0H3082Y,10.53,114,Gasoline,0.7297\nHonda Accord,0H3082Y,206.32,114,Gasoline,11.9571\nHonda Accord,0H3082Y,8.07,114,Gasoline,0.5766\nHonda Accord,0H3082Y,13.12,114,Gasoline,1.3446\nHonda Accord,0H3082Y,131.05,114,Gasoline,10.1295\nHonda Accord,0H3082Y,12.83,114,Gasoline,1.0923\nHonda Accord,0H3082Y,35.73,114,Gasoline,2.3396\nHonda Accord,0H3082Y,405.89,114,Gasoline,25.3727\nHonda Accord,0H3082Y,31.6,114,Gasoline,1.8185\nHonda Accord,0H3082Y,61.68,114,Gasoline,4.0531\nHonda Accord,0H3082Y,104.55,114,Gasoline,9.2777\nHonda Accord,0H3082Y,5.31,114,Gasoline,0.5071\nHonda Accord,0H3082Y,1.76,114,Gasoline,0.1636\nHonda Accord,0H3082Y,6.9,114,Gasoline,0.735\nHonda Accord,0H3082Y,19.26,114,Gasoline,1.4308\nHonda Accord,0H3082Y,124.96,114,Gasoline,9.3866\nHonda Accord,0H3082Y,4.85,114,Gasoline,0.555\nHonda Accord,0H3082Y,56.1,114,Gasoline,5.2088\nHonda Accord,0H3082Y,2.46,114,Gasoline,0.239\nHonda Accord,0H3082Y,58.39,114,Gasoline,3.4792\nHonda Accord,0H3082Y,22.19,114,Gasoline,2.2659\nHonda Accord,0H3082Y,7.58,114,Gasoline,0.4292\nHonda Accord,0H3082Y,16.2,114,Gasoline,1.3962\nHonda Accord,0H3082Y,148.89,114,Gasoline,10.5443\nHonda Accord,0H3082Y,11.42,114,Gasoline,0.7972\nHonda Accord,0H3082Y,24.66,114,Gasoline,1.8828\nHonda Accord,0H3082Y,68.03,114,Gasoline,4.1892\nHonda Accord,0H3082Y,11.16,114,Gasoline,0.9793\nHonda Accord,0H3082Y,50.09,114,Gasoline,3.2956\nHonda Accord,0H3082Y,37.75,114,Gasoline,3.3712\nHonda Accord,0H3082Y,30.7,114,Gasoline,2.1641\nHonda Accord,0H3082Y,16.29,114,Gasoline,0.9961\nHonda Accord,0H3082Y,25.53,114,Gasoline,1.9849\nHonda Accord,0H3082Y,22.67,114,Gasoline,2.5155\nHonda Accord,0H3082Y,6.65,114,Gasoline,0.4816\nHonda Accord,0H3082Y,8.82,114,Gasoline,0.5693\nHonda Accord,0H3082Y,14.32,114,Gasoline,1.3229\nHonda Accord,0H3082Y,2.31,114,Gasoline,0.2054\nHonda Accord,0H3082Y,239.99,114,Gasoline,17.4576\nHonda Accord,0H3082Y,32.2,114,Gasoline,2.5935\nHonda Accord,0H3082Y,120.31,114,Gasoline,9.5795\nHonda Accord,0H3082Y,2.14,114,Gasoline,0.2647\nHonda Accord,0H3082Y,13.03,114,Gasoline,1.0497\nHonda Accord,0H3082Y,8.44,114,Gasoline,0.6871\nHonda Accord,0H3082Y,12.79,114,Gasoline,1.6917\nHonda Accord,0H3082Y,26.03,114,Gasoline,1.7369\nHonda Accord,0H3082Y,4.8,114,Gasoline,0.5069\nHonda Accord,0H3082Y,6.9,114,Gasoline,0.4072\nHonda Accord,0H3082Y,113.33,114,Gasoline,8.4509\nHonda Accord,0H3082Y,20.12,114,Gasoline,1.7505\nHonda Accord,0H3082Y,37.42,114,Gasoline,2.6365\nHonda Accord,0H3082Y,5.54,114,Gasoline,0.4447\nHonda Accord,0H3082Y,27.45,114,Gasoline,1.7492\nHonda Accord,0H3082Y,40.21,114,Gasoline,3.6999\nHonda Accord,0H3082Y,2.3,114,Gasoline,0.3044\nHonda Accord,0H3082Y,18.43,114,Gasoline,1.6378\nHonda Accord,0H3082Y,18.19,114,Gasoline,1.7344\nHonda Accord,0H3082Y,10.11,114,Gasoline,0.6904\nHonda Accord,0H3082Y,24.28,114,Gasoline,1.5732\nHonda Accord,0H3082Y,2.3,114,Gasoline,0.1633\nHonda Accord,0H3082Y,14.84,114,Gasoline,1.0351\nHonda Accord,0H3082Y,28.65,114,Gasoline,3.5148\nHonda Accord,0H3082Y,114.78,114,Gasoline,8.1806\nHonda Accord,0H3082Y,2.15,114,Gasoline,0.2538\nHonda Accord,0H3082Y,54.25,114,Gasoline,5.3751\nHonda Accord,0H3082Y,88.31,114,Gasoline,5.7854\nHonda Accord,0H3082Y,1.7,114,Gasoline,0.1154\nHonda Accord,0H3082Y,16.21,114,Gasoline,0.985\nHonda Accord,0H3082Y,11.37,114,Gasoline,0.7388\nHonda Accord,0H3082Y,249.96,114,Gasoline,14.7882\nHonda Accord,0H3082Y,144.36,114,Gasoline,9.602\nHonda Accord,0H3082Y,185.08,114,Gasoline,8.6707\nHonda Accord,0H3082Y,5.62,114,Gasoline,0.4222\nHonda Accord,0H3082Y,21.55,114,Gasoline,1.7487\nHonda Accord,0H3082Y,4.17,114,Gasoline,0.3854\nHonda Accord,0H3082Y,6.78,114,Gasoline,0.5105\nHonda Accord,0H3082Y,64.34,114,Gasoline,4.7831\nHonda Accord,0H3082Y,22.1,114,Gasoline,2.131\nHonda Accord,0H3082Y,64.33,114,Gasoline,5.778\nHonda Accord,0H3082Y,17.58,114,Gasoline,0.9962\nHonda Accord,0H3082Y,16.48,114,Gasoline,1.5913\nHonda Accord,0H3082Y,5.47,114,Gasoline,0.3402\nHonda Accord,0H3082Y,51.77,114,Gasoline,5.0938\nHonda Accord,0H3082Y,11.98,114,Gasoline,1.1728\nHonda Accord,0H3082Y,148.97,114,Gasoline,11.3004\nHonda Accord,0H3082Y,11.46,114,Gasoline,0.9701\nHonda Accord,0H3082Y,14.44,114,Gasoline,1.1751\nHonda Accord,0H3082Y,5.08,114,Gasoline,0.6865\nHonda Accord,0H3082Y,2.21,114,Gasoline,0.2403\nHonda Accord,0H3082Y,65.56,114,Gasoline,6.4528\nHonda Accord,0H3082Y,79.17,114,Gasoline,7.4744\nHonda Accord,0H3082Y,102.83,114,Gasoline,8.7417\nHonda Accord,0H3082Y,112.69,114,Gasoline,8.3314\nHonda Accord,0H3082Y,13.34,114,Gasoline,0.9466\nHonda Accord,0H3082Y,63.62,114,Gasoline,5.3957\nHonda Accord,0H3082Y,54.44,114,Gasoline,5.2902\nHonda Accord,0H3082Y,249.0,114,Gasoline,14.8126\nHonda Accord,0H3082Y,125.31,114,Gasoline,10.3225\nHonda Accord,0H3082Y,11.92,114,Gasoline,0.7366\nHonda Accord,0H3082Y,30.57,114,Gasoline,2.126\nHonda Accord,0H3082Y,4.43,114,Gasoline,0.5678\nHonda Accord,0H3082Y,170.46,114,Gasoline,10.0581\nHonda Accord,0H3082Y,28.6,114,Gasoline,1.7114\nHonda Accord,0H3082Y,2.25,114,Gasoline,0.1781\nHonda Accord,0H3082Y,34.63,114,Gasoline,3.1559\nHonda Accord,0H3082Y,84.32,114,Gasoline,7.2177\nHonda Accord,0H3082Y,4.24,114,Gasoline,0.4546\nHonda Accord,0H3082Y,12.96,114,Gasoline,1.3254\nHonda Accord,0H3082Y,19.08,114,Gasoline,1.4914\nHonda Accord,0H3082Y,71.94,114,Gasoline,5.5901\nHonda Accord,0H3082Y,2.15,114,Gasoline,0.2298\nHonda Accord,0H3082Y,1.61,114,Gasoline,0.1398\nHonda Accord,0H3082Y,161.44,114,Gasoline,12.8016\nHonda Accord,0H3082Y,16.51,114,Gasoline,1.1233\nHonda Accord,0H3082Y,84.77,114,Gasoline,6.7946\nHonda Accord,0H3082Y,27.7,114,Gasoline,2.7644\nHonda Accord,0H3082Y,125.12,114,Gasoline,11.124\nHonda Accord,0H3082Y,2.6,114,Gasoline,0.2747\nHonda Accord,0H3082Y,100.26,114,Gasoline,10.0376\nHonda Accord,0H3082Y,25.62,114,Gasoline,1.852\nHonda Accord,0H3082Y,4.3,114,Gasoline,0.2787\nHonda Accord,0H3082Y,76.82,114,Gasoline,7.4389\nHonda Accord,0H3082Y,76.96,114,Gasoline,4.8741\nHonda Accord,0H3082Y,36.54,114,Gasoline,2.5588\nHonda Accord,0H3082Y,12.32,114,Gasoline,0.8693\nHonda Accord,0H3082Y,34.83,114,Gasoline,2.5324\nHonda Accord,0H3082Y,52.84,114,Gasoline,4.6519\nHonda Accord,0H3082Y,29.84,114,Gasoline,1.9457\nHonda Accord,0H3082Y,104.96,114,Gasoline,10.6915\nHonda Accord,0H3082Y,6.22,114,Gasoline,0.4685\nHonda Accord,0H3082Y,2.35,114,Gasoline,0.4132\nHonda Accord,0H3082Y,27.72,114,Gasoline,2.7789\nHonda Accord,0H3082Y,28.12,114,Gasoline,2.1367\nHonda Accord,0H3082Y,5.46,114,Gasoline,0.3904\nHonda Accord,0H3082Y,230.38,114,Gasoline,14.4926\nHonda Accord,0H3082Y,4.56,114,Gasoline,0.5226\nHonda Accord,0H3082Y,118.35,114,Gasoline,9.4794\nHonda Accord,0H3082Y,8.6,114,Gasoline,0.9939\nHonda Accord,0H3082Y,10.72,114,Gasoline,1.5494\nHonda Accord,0H3082Y,8.78,114,Gasoline,0.6412\nHonda Accord,0H3082Y,7.27,114,Gasoline,0.4369\nHonda Accord,0H3082Y,165.96,114,Gasoline,10.183\nHonda Accord,0H3082Y,2.82,114,Gasoline,0.3868\nHonda Accord,0H3082Y,54.62,114,Gasoline,4.0466\nHonda Accord,0H3082Y,62.75,114,Gasoline,3.9244\nHonda Accord,0H3082Y,264.6,114,Gasoline,18.5139\nHonda Accord,0H3082Y,17.16,114,Gasoline,1.3281\nHonda Accord,0H3082Y,21.74,114,Gasoline,1.4738\nHonda Accord,0H3082Y,101.49,114,Gasoline,6.4951\nHonda Accord,0H3082Y,146.87,114,Gasoline,9.6251\nHonda Accord,0H3082Y,318.93,114,Gasoline,20.6948\nHonda Accord,0H3082Y,125.79,114,Gasoline,10.6784\nHonda Accord,0H3082Y,3.08,114,Gasoline,0.2619\nHonda Accord,0H3082Y,27.48,114,Gasoline,1.7337\nHonda Accord,0H3082Y,3.36,114,Gasoline,0.2564\nHonda Accord,0H3082Y,22.06,114,Gasoline,1.959\nHonda Accord,0H3082Y,105.1,114,Gasoline,9.2413\nHonda Accord,0H3082Y,25.35,114,Gasoline,2.0109\nHonda Accord,0H3082Y,35.75,114,Gasoline,3.2385\nHonda Accord,0H3082Y,11.7,114,Gasoline,0.7819\nHonda Accord,0H3082Y,67.08,114,Gasoline,4.8359\nHonda Accord,0H3082Y,3.53,114,Gasoline,0.2243\nHonda Accord,0H3082Y,167.67,114,Gasoline,16.9898\nHonda Accord,0H3082Y,46.29,114,Gasoline,3.6524\nHonda Accord,0H3082Y,31.93,114,Gasoline,2.4874\nHonda Accord,0H3082Y,39.34,114,Gasoline,2.6776\nHonda Accord,0H3082Y,36.64,114,Gasoline,3.6667\nHonda Accord,0H3082Y,9.01,114,Gasoline,0.9564\nHonda Accord,0H3082Y,111.75,114,Gasoline,8.6974\nHonda Accord,0H3082Y,28.73,114,Gasoline,2.07\nHonda Accord,0H3082Y,19.75,114,Gasoline,1.2\nHonda Accord,0H3082Y,4.21,114,Gasoline,0.4008\nHonda Accord,0H3082Y,2.09,114,Gasoline,0.2995\nHonda Accord,0H3082Y,26.5,114,Gasoline,2.4764\nHonda Accord,0H3082Y,5.08,114,Gasoline,0.8317\nHonda Accord,0H3082Y,144.37,114,Gasoline,9.3813\nHonda Accord,0H3082Y,164.26,114,Gasoline,15.2084\nHonda Accord,0H3082Y,57.36,114,Gasoline,3.203\nHonda Accord,0H3082Y,113.56,114,Gasoline,7.3663\nHonda Accord,0H3082Y,108.89,114,Gasoline,7.3971\nHonda Accord,0H3082Y,22.97,114,Gasoline,1.5723\nHonda Accord,0H3082Y,45.01,114,Gasoline,4.276\nHonda Accord,0H3082Y,26.7,114,Gasoline,2.0067\nHonda Accord,0H3082Y,101.33,114,Gasoline,6.2493\nHonda Accord,0H3082Y,5.12,114,Gasoline,0.3612\nHonda Accord,0H3082Y,11.26,114,Gasoline,1.1374\nHonda Accord,0H3082Y,72.17,114,Gasoline,5.4204\nHonda Accord,0H3082Y,42.58,114,Gasoline,2.9234\nHonda Accord,0H3082Y,34.69,114,Gasoline,3.3437\nHonda Accord,0H3082Y,31.41,114,Gasoline,3.4533\nHonda Accord,0H3082Y,74.1,114,Gasoline,5.8426\nHonda Accord,0H3082Y,108.22,114,Gasoline,9.4706\nHonda Accord,0H3082Y,2.51,114,Gasoline,0.327\nHonda Accord,0H3082Y,144.85,114,Gasoline,10.1372\nHonda Accord,0H3082Y,271.02,114,Gasoline,25.069\nHonda Accord,0H3082Y,27.02,114,Gasoline,1.8118\nHonda Accord,0H3082Y,34.78,114,Gasoline,1.8466\nHonda Accord,0H3082Y,18.27,114,Gasoline,2.4115\nHonda Accord,0H3082Y,6.98,114,Gasoline,0.7854\nHonda Accord,0H3082Y,3.36,114,Gasoline,0.2359\nHonda Accord,0H3082Y,3.58,114,Gasoline,0.4481\nHonda Accord,0H3082Y,10.85,114,Gasoline,0.8344\nHonda Accord,0H3082Y,140.84,114,Gasoline,12.4347\nHonda Accord,0H3082Y,21.07,114,Gasoline,1.6075\nHonda Accord,0H3082Y,105.26,114,Gasoline,10.9901\nHonda Accord,0H3082Y,78.96,114,Gasoline,8.7754\nHonda Accord,0H3082Y,189.07,114,Gasoline,12.3251\nHonda Accord,0H3082Y,21.58,114,Gasoline,3.3215\nHonda Accord,0H3082Y,47.62,114,Gasoline,5.7381\nHonda Accord,0H3082Y,27.55,114,Gasoline,2.6225\nHonda Accord,0H3082Y,348.92,114,Gasoline,23.384\nHonda Accord,0H3082Y,102.07,114,Gasoline,6.0364\nHonda Accord,0H3082Y,3.24,114,Gasoline,0.3524\nHonda Accord,0H3082Y,4.9,114,Gasoline,0.3151\nHonda Accord,0H3082Y,2.53,114,Gasoline,0.1753\nHonda Accord,0H3082Y,7.23,114,Gasoline,0.8527\nHonda Accord,0H3082Y,1.56,114,Gasoline,0.1125\nHonda Accord,0H3082Y,30.68,114,Gasoline,1.7166\nHonda Accord,0H3082Y,12.13,114,Gasoline,1.5409\nHonda Accord,0H3082Y,10.73,114,Gasoline,0.8443\nHonda Accord,0H3082Y,19.64,114,Gasoline,1.2626\nHonda Accord,0H3082Y,16.21,114,Gasoline,1.5185\nHonda Accord,0H3082Y,36.53,114,Gasoline,2.9136\nHonda Accord,0H3082Y,114.21,114,Gasoline,9.8122\nHonda Accord,0H3082Y,34.13,114,Gasoline,3.9957\nHonda Accord,0H3082Y,4.52,114,Gasoline,0.5416\nHonda Accord,0H3082Y,28.08,114,Gasoline,2.8182\nHonda Accord,0H3082Y,252.02,114,Gasoline,16.1745\nHonda Accord,0H3082Y,35.4,114,Gasoline,2.5594\nHonda Accord,0H3082Y,155.86,114,Gasoline,14.7055\nHonda Accord,0H3082Y,123.87,114,Gasoline,9.366\nHonda Accord,0H3082Y,3.64,114,Gasoline,0.4059\nHonda Accord,0H3082Y,148.96,114,Gasoline,14.0851\nHonda Accord,0H3082Y,3.06,114,Gasoline,0.3482\nHonda Accord,0H3082Y,2.65,114,Gasoline,0.3476\nHonda Accord,0H3082Y,103.12,114,Gasoline,8.679\nHonda Accord,0H3082Y,91.43,114,Gasoline,6.6491\nHonda Accord,0H3082Y,34.36,114,Gasoline,2.7575\nHonda Accord,0H3082Y,14.44,114,Gasoline,1.2629\nHonda Accord,0H3082Y,27.55,114,Gasoline,2.9593\nHonda Accord,0H3082Y,38.19,114,Gasoline,2.9692\nHonda Accord,0H3082Y,14.0,114,Gasoline,1.0217\nHonda Accord,0H3082Y,24.88,114,Gasoline,1.9972\nHonda Accord,0H3082Y,36.21,114,Gasoline,2.2715\nHonda Accord,0H3082Y,15.2,114,Gasoline,1.4911\nHonda Accord,0H3082Y,7.26,114,Gasoline,0.9703\nHonda Accord,0H3082Y,113.61,114,Gasoline,9.9355\nHonda Accord,0H3082Y,9.94,114,Gasoline,0.7158\nHonda Accord,0H3082Y,101.17,114,Gasoline,10.6519\nHonda Accord,0H3082Y,104.07,114,Gasoline,8.263\nHonda Accord,0H3082Y,14.28,114,Gasoline,1.5742\nHonda Accord,0H3082Y,21.22,114,Gasoline,2.8019\nHonda Accord,0H3082Y,27.2,114,Gasoline,4.1126\nHonda Accord,0H3082Y,39.0,114,Gasoline,5.6375\nHonda Accord,0H3082Y,10.49,114,Gasoline,1.3331\nHonda Accord,0H3082Y,33.8,114,Gasoline,2.4652\nHonda Accord,0H3082Y,1.58,114,Gasoline,0.2595\nHonda Accord,0H3082Y,1.65,114,Gasoline,0.21\nHonda Accord,0H3082Y,1.79,114,Gasoline,0.2232\nHonda Accord,0H3082Y,26.02,114,Gasoline,1.6103\nHonda Accord,0H3082Y,13.85,114,Gasoline,1.1774\nHonda Accord,0H3082Y,2.87,114,Gasoline,0.1699\nHonda Accord,0H3082Y,116.85,114,Gasoline,10.4722\nHonda Accord,0H3082Y,38.38,114,Gasoline,2.5729\nHonda Accord,0H3082Y,115.58,114,Gasoline,12.1391\nHonda Accord,0H3082Y,134.77,114,Gasoline,8.7206\nHonda Accord,0H3082Y,41.46,114,Gasoline,2.6033\nHonda Accord,0H3082Y,7.54,114,Gasoline,0.8566\nHonda Accord,0H3082Y,142.0,114,Gasoline,9.4341\nHonda Accord,0H3082Y,19.84,114,Gasoline,1.488\nHonda Accord,0H3082Y,1.75,114,Gasoline,0.1293\nHonda Accord,0H3082Y,6.14,114,Gasoline,0.7663\nHonda Accord,0H3082Y,5.94,114,Gasoline,0.4181\nHonda Accord,0H3082Y,42.68,114,Gasoline,3.4983\nHonda Accord,0H3082Y,25.56,114,Gasoline,4.0155\nHonda Accord,0H3082Y,103.64,114,Gasoline,7.4207\nHonda Accord,0H3082Y,29.41,114,Gasoline,1.7203\nHonda Accord,0H3082Y,36.5,114,Gasoline,3.9008\nHonda Accord,0H3082Y,273.74,114,Gasoline,20.213\nHonda Accord,0H3082Y,53.12,114,Gasoline,3.357\nHonda Accord,0H3082Y,85.52,114,Gasoline,5.7584\nHonda Accord,0H3082Y,3.16,114,Gasoline,0.367\nHonda Accord,0H3082Y,1.58,114,Gasoline,0.2169\nHonda Accord,0H3082Y,10.54,114,Gasoline,0.9592\nHonda Accord,0H3082Y,6.41,114,Gasoline,0.8967\nHonda Accord,0H3082Y,10.48,114,Gasoline,0.7493\nHonda Accord,0H3082Y,16.82,114,Gasoline,1.1163\nHonda Accord,0H3082Y,128.58,114,Gasoline,10.166\nHonda Accord,0H3082Y,76.35,114,Gasoline,4.1383\nHonda Accord,0H3082Y,1.99,114,Gasoline,0.2758\nHonda Accord,0H3082Y,1.54,114,Gasoline,0.1955\nHonda Accord,0H3082Y,13.29,114,Gasoline,0.8582\nHonda Accord,0H3082Y,32.32,114,Gasoline,2.5121\nHonda Accord,0H3082Y,3.58,114,Gasoline,0.239\nHonda Accord,0H3082Y,2.71,114,Gasoline,0.3083\nHonda Accord,0H3082Y,9.11,114,Gasoline,0.8214\nHonda Accord,0H3082Y,108.57,114,Gasoline,7.3164\nHonda Accord,0H3082Y,123.89,114,Gasoline,7.9915\nHonda Accord,0H3082Y,108.66,114,Gasoline,8.0593\nHonda Accord,0H3082Y,51.99,114,Gasoline,5.5889\nHonda Accord,0H3082Y,109.94,114,Gasoline,6.7082\nHonda Accord,0H3082Y,121.72,114,Gasoline,9.1965\nHonda Accord,0H3082Y,59.7,114,Gasoline,3.927\nHonda Accord,0H3082Y,57.31,114,Gasoline,3.4321\nHonda Accord,0H3082Y,153.66,114,Gasoline,12.1771\nHonda Accord,0H3082Y,50.89,114,Gasoline,6.5853\nHonda Accord,0H3082Y,54.5,114,Gasoline,5.9134\nHonda Accord,0H3082Y,164.31,114,Gasoline,13.9812\nHonda Accord,0H3082Y,130.53,114,Gasoline,9.8177\nHonda Accord,0H3082Y,44.1,114,Gasoline,5.2009\nHonda Accord,0H3082Y,1.72,114,Gasoline,0.1488\nHonda Accord,0H3082Y,141.23,114,Gasoline,12.9263\nHonda Accord,0H3082Y,2.71,114,Gasoline,0.2047\nHonda Accord,0H3082Y,34.77,114,Gasoline,2.7674\nHonda Accord,0H3082Y,18.04,114,Gasoline,1.481\nHonda Accord,0H3082Y,45.83,114,Gasoline,3.5845\nHonda Accord,0H3082Y,19.19,114,Gasoline,1.4086\nHonda Accord,0H3082Y,26.55,114,Gasoline,2.2198\nHonda Accord,0H3082Y,3.66,114,Gasoline,0.3842\nHonda Accord,0H3082Y,27.98,114,Gasoline,1.5659\nHonda Accord,0H3082Y,102.08,114,Gasoline,6.8584\nHonda Accord,0H3082Y,2.97,114,Gasoline,0.1823\nHonda Accord,0H3082Y,110.44,114,Gasoline,7.3407\nHonda Accord,0H3082Y,15.47,114,Gasoline,1.0772\nHonda Accord,0H3082Y,1.51,114,Gasoline,0.1902\nHonda Accord,0H3082Y,14.81,114,Gasoline,1.1425\nHonda Accord,0H3082Y,54.53,114,Gasoline,3.2893\nHonda Accord,0H3082Y,19.76,114,Gasoline,1.13\nHonda Accord,0H3082Y,43.44,114,Gasoline,2.9204\nHonda Accord,0H3082Y,142.02,114,Gasoline,10.0337\nHonda Accord,0H3082Y,43.35,114,Gasoline,4.0246\nHonda Accord,0H3082Y,15.43,114,Gasoline,1.4653\nHonda Accord,0H3082Y,16.28,114,Gasoline,1.442\nHonda Accord,0H3082Y,2.42,114,Gasoline,0.1753\nHonda Accord,0H3082Y,31.84,114,Gasoline,2.0137\nHonda Accord,0H3082Y,40.71,114,Gasoline,2.4153\nHonda Accord,0H3082Y,7.82,114,Gasoline,1.1144\nHonda Accord,0H3082Y,22.55,114,Gasoline,1.3795\nHonda Accord,0H3082Y,105.32,114,Gasoline,8.5542\nHonda Accord,0H3082Y,66.63,114,Gasoline,4.6428\nHonda Accord,0H3082Y,4.6,114,Gasoline,0.4632\nHonda Accord,0H3082Y,191.96,114,Gasoline,11.7049\nHonda Accord,0H3082Y,5.0,114,Gasoline,0.5137\nHonda Accord,0H3082Y,73.55,114,Gasoline,4.694\nHonda Accord,0H3082Y,264.93,114,Gasoline,17.0869\nHonda Accord,0H3082Y,129.92,114,Gasoline,9.7388\nHonda Accord,0H3082Y,149.88,114,Gasoline,8.787\nHonda Accord,0H3082Y,42.92,114,Gasoline,3.2199\nHonda Accord,0H3082Y,274.73,114,Gasoline,15.5051\nHonda Accord,0H3082Y,251.42,114,Gasoline,13.2333\nHonda Accord,0H3082Y,77.28,114,Gasoline,4.9934\nHonda Accord,0H3082Y,5.19,114,Gasoline,0.3508\nHonda Accord,0H3082Y,28.11,114,Gasoline,2.1428\nHonda Accord,0H3082Y,17.48,114,Gasoline,1.2751\nHonda Accord,0H3082Y,2.17,114,Gasoline,0.3652\nHonda Accord,0H3082Y,67.63,114,Gasoline,4.4125\nHonda Accord,0H3082Y,123.19,114,Gasoline,10.21\nHonda Accord,0H3082Y,15.1,114,Gasoline,0.972\nHonda Accord,0H3082Y,10.33,114,Gasoline,1.0841\nHonda Accord,0H3082Y,15.14,114,Gasoline,0.9499\nHonda Accord,0H3082Y,41.67,114,Gasoline,2.9436\nHonda Accord,0H3082Y,1.65,114,Gasoline,0.2547\nHonda Accord,0H3082Y,83.76,114,Gasoline,4.7594\nHonda Accord,0H3082Y,386.22,114,Gasoline,24.4283\nHonda Accord,0H3082Y,46.05,114,Gasoline,3.6159\nHonda Accord,0H3082Y,75.73,114,Gasoline,4.2734\nHonda Accord,0H3082Y,12.79,114,Gasoline,1.1874\nHonda Accord,0H3082Y,12.84,114,Gasoline,1.293\nHonda Accord,0H3082Y,1.72,114,Gasoline,0.1268\nHonda Accord,0H3082Y,19.07,114,Gasoline,1.8778\nHonda Accord,0H3082Y,31.08,114,Gasoline,1.8842\nHonda Accord,0H3082Y,128.53,114,Gasoline,8.4488\nHonda Accord,0H3082Y,32.59,114,Gasoline,2.0427\nHonda Accord,0H3082Y,101.31,114,Gasoline,9.7665\nHonda Accord,0H3082Y,68.09,114,Gasoline,4.8888\nHonda Accord,0H3082Y,136.44,114,Gasoline,12.8628\nHonda Accord,0H3082Y,3.55,114,Gasoline,0.3507\nHonda Accord,0H3082Y,5.43,114,Gasoline,0.3498\nHonda Accord,0H3082Y,55.33,114,Gasoline,4.9158\nHonda Accord,0H3082Y,14.96,114,Gasoline,1.5538\nHonda Accord,0H3082Y,146.17,114,Gasoline,8.8556\nHonda Accord,0H3082Y,22.02,114,Gasoline,1.7878\nHonda Accord,0H3082Y,2.48,114,Gasoline,0.2502\nHonda Accord,0H3082Y,10.14,114,Gasoline,0.9578\nHonda Accord,0H3082Y,33.8,114,Gasoline,2.5756\nHonda Accord,0H3082Y,48.64,114,Gasoline,3.113\nHonda Accord,0H3082Y,12.21,114,Gasoline,1.1347\nHonda Accord,0H3082Y,5.9,114,Gasoline,0.5564\nHonda Accord,0H3082Y,113.33,114,Gasoline,6.7171\nHonda Accord,0H3082Y,35.86,114,Gasoline,2.6331\nHonda Accord,0H3082Y,1.81,114,Gasoline,0.2341\nHonda Accord,0H3082Y,136.54,114,Gasoline,10.8219\nHonda Accord,0H3082Y,114.96,114,Gasoline,9.8732\nHonda Accord,0H3082Y,5.71,114,Gasoline,0.5774\nHonda Accord,0H3082Y,114.69,114,Gasoline,7.5924\nHonda Accord,0H3082Y,21.94,114,Gasoline,1.7898\nHonda Accord,0H3082Y,40.05,114,Gasoline,2.937\nHonda Accord,0H3082Y,10.89,114,Gasoline,0.9368\nHonda Accord,0H3082Y,8.82,114,Gasoline,0.7472\nHonda Accord,0H3082Y,111.43,114,Gasoline,6.2384\nHonda Accord,0H3082Y,4.9,114,Gasoline,0.3055\nHonda Accord,0H3082Y,14.3,114,Gasoline,0.938\nHonda Accord,0H3082Y,4.24,114,Gasoline,0.4321\nHonda Accord,0H3082Y,69.56,114,Gasoline,5.7754\nHonda Accord,0H3082Y,51.14,114,Gasoline,5.7569\nHonda Accord,0H3082Y,34.03,114,Gasoline,2.2006\nHonda Accord,0H3082Y,53.98,114,Gasoline,4.5324\nHonda Accord,0H3082Y,36.06,114,Gasoline,2.2437\nHonda Accord,0H3082Y,80.21,114,Gasoline,5.535\nHonda Accord,0H3082Y,3.2,114,Gasoline,0.2643\nHonda Accord,0H3082Y,147.36,114,Gasoline,8.2835\nHonda Accord,0H3082Y,136.07,114,Gasoline,9.5258\nHonda Accord,0H3082Y,13.69,114,Gasoline,0.7893\nHonda Accord,0H3082Y,73.46,114,Gasoline,5.4006\nHonda Accord,0H3082Y,3.36,114,Gasoline,0.2678\nHonda Accord,0H3082Y,2.28,114,Gasoline,0.2226\nHonda Accord,0H3082Y,70.79,114,Gasoline,5.4676\nHonda Accord,0H3082Y,100.59,114,Gasoline,8.3924\nHonda Accord,0H3082Y,45.99,114,Gasoline,5.3919\nHonda Accord,0H3082Y,2.26,114,Gasoline,0.1687\nHonda Accord,0H3082Y,2.25,114,Gasoline,0.1979\nHonda Accord,0H3082Y,34.9,114,Gasoline,3.0369\nHonda Accord,0H3082Y,79.64,114,Gasoline,5.7446\nHonda Accord,0H3082Y,10.58,114,Gasoline,0.9962\nHonda Accord,0H3082Y,101.66,114,Gasoline,6.447\nHonda Accord,0H3082Y,10.07,114,Gasoline,0.8747\nHonda Accord,0H3082Y,6.44,114,Gasoline,0.5615\nHonda Accord,0H3082Y,42.68,114,Gasoline,3.763\nHonda Accord,0H3082Y,57.42,114,Gasoline,4.279\nHonda Accord,0H3082Y,146.48,114,Gasoline,11.6388\nHonda Accord,0H3082Y,20.01,114,Gasoline,1.2052\nHonda Accord,0H3082Y,230.13,114,Gasoline,13.3249\nHonda Accord,0H3082Y,102.81,114,Gasoline,8.3643\nHonda Accord,0H3082Y,13.14,114,Gasoline,1.0052\nHonda Accord,0H3082Y,26.27,114,Gasoline,2.9345\nHonda Accord,0H3082Y,71.96,114,Gasoline,5.0561\nHonda Accord,0H3082Y,41.25,114,Gasoline,3.7481\nHonda Accord,0H3082Y,4.5,114,Gasoline,0.4438\nHonda Accord,0H3082Y,17.71,114,Gasoline,1.3259\nHonda Accord,0H3082Y,197.74,114,Gasoline,14.429\nHonda Accord,0H3082Y,4.61,114,Gasoline,0.5265\nHonda Accord,0H3082Y,112.54,114,Gasoline,9.4966\nHonda Accord,0H3082Y,153.95,114,Gasoline,11.231\nHonda Accord,0H3082Y,1.79,114,Gasoline,0.2307\nHonda Accord,0H3082Y,7.01,114,Gasoline,0.61\nHonda Accord,0H3082Y,17.41,114,Gasoline,1.9776\nHonda Accord,0H3082Y,35.2,114,Gasoline,3.3741\nHonda Accord,0H3082Y,30.98,114,Gasoline,1.7797\nHonda Accord,0H3082Y,232.45,114,Gasoline,13.8328\nHonda Accord,0H3082Y,102.24,114,Gasoline,7.8598\nHonda Accord,0H3082Y,9.21,114,Gasoline,0.5991\nHonda Accord,0H3082Y,44.03,114,Gasoline,2.63\nHonda Accord,0H3082Y,128.69,114,Gasoline,9.7128\nHonda Accord,0H3082Y,108.12,114,Gasoline,4.5423\nHonda Accord,0H3082Y,45.7,114,Gasoline,3.2797\nHonda Accord,0H3082Y,45.95,114,Gasoline,3.1547\nHonda Accord,0H3082Y,17.24,114,Gasoline,1.6311\nHonda Accord,0H3082Y,31.85,114,Gasoline,3.2171\nHonda Accord,0H3082Y,102.84,114,Gasoline,9.3235\nHonda Accord,0H3082Y,108.56,114,Gasoline,6.9276\nHonda Accord,0H3082Y,19.29,114,Gasoline,2.4384\nHonda Accord,0H3082Y,4.84,114,Gasoline,0.5192\nHonda Accord,0H3082Y,62.26,114,Gasoline,4.2844\nHonda Accord,0H3082Y,2.0,114,Gasoline,0.3245\nHonda Accord,0H3082Y,37.46,114,Gasoline,2.865\nHonda Accord,0H3082Y,9.96,114,Gasoline,0.7733\nHonda Accord,0H3082Y,37.84,114,Gasoline,3.7164\nHonda Accord,0H3082Y,14.92,114,Gasoline,1.9675\nHonda Accord,0H3082Y,10.38,114,Gasoline,1.1089\nHonda Accord,0H3082Y,41.62,114,Gasoline,3.3208\nHonda Accord,0H3082Y,44.99,114,Gasoline,3.5033\nHonda Accord,0H3082Y,1.7,114,Gasoline,0.2809\nHonda Accord,0H3082Y,35.59,114,Gasoline,3.436\nHonda Accord,0H3082Y,105.07,114,Gasoline,9.0047\nHonda Accord,0H3082Y,129.6,114,Gasoline,8.5539\nHonda Accord,0H3082Y,36.43,114,Gasoline,4.4005\nHonda Accord,0H3082Y,101.31,114,Gasoline,9.1514\nHonda Accord,0H3082Y,5.16,114,Gasoline,0.5624\nHonda Accord,0H3082Y,121.34,114,Gasoline,8.5015\nHonda Accord,0H3082Y,15.32,114,Gasoline,1.4291\nHonda Accord,0H3082Y,13.03,114,Gasoline,1.0918\nHonda Accord,0H3082Y,48.27,114,Gasoline,4.0371\nHonda Accord,0H3082Y,13.21,114,Gasoline,1.2198\nHonda Accord,0H3082Y,34.37,114,Gasoline,2.3782\nHonda Accord,0H3082Y,33.76,114,Gasoline,2.241\nHonda Accord,0H3082Y,32.15,114,Gasoline,2.5539\nHonda Accord,0H3082Y,41.57,114,Gasoline,4.7835\nHonda Accord,0H3082Y,3.66,114,Gasoline,0.3648\nHonda Accord,0H3082Y,57.29,114,Gasoline,3.256\nHonda Accord,0H3082Y,62.16,114,Gasoline,6.5702\nHonda Accord,0H3082Y,4.38,114,Gasoline,0.3587\nHonda Accord,0H3082Y,10.16,114,Gasoline,0.7661\nHonda Accord,0H3082Y,82.34,114,Gasoline,6.3253\nHonda Accord,0H3082Y,275.88,114,Gasoline,20.4397\nHonda Accord,0H3082Y,76.53,114,Gasoline,5.3286\nHonda Accord,0H3082Y,11.11,114,Gasoline,0.8762\nHonda Accord,0H3082Y,11.71,114,Gasoline,0.902\nHonda Accord,0H3082Y,22.83,114,Gasoline,1.56\nHonda Accord,0H3082Y,51.85,114,Gasoline,3.166\nHonda Accord,0H3082Y,46.03,114,Gasoline,3.3603\nHonda Accord,0H3082Y,16.0,114,Gasoline,1.5682\nHonda Accord,0H3082Y,6.18,114,Gasoline,0.5528\nHonda Accord,0H3082Y,112.91,114,Gasoline,7.772\nHonda Accord,0H3082Y,163.65,114,Gasoline,10.9854\nHonda Accord,0H3082Y,110.12,114,Gasoline,8.1586\nHonda Accord,0H3082Y,3.3,114,Gasoline,0.3828\nHonda Accord,0H3082Y,112.88,114,Gasoline,11.3511\nHonda Accord,0H3082Y,116.22,114,Gasoline,12.1599\nHonda Accord,0H3082Y,138.18,114,Gasoline,11.2393\nHonda Odyssey,0H3011H,48.91,137,Gasoline,4.2577\nHonda Odyssey,0H3011H,51.59,137,Gasoline,3.8814\nHonda Odyssey,0H3011H,12.58,137,Gasoline,1.1345\nHonda Odyssey,0H3011H,115.1,137,Gasoline,8.7183\nHonda Odyssey,0H3011H,15.81,137,Gasoline,1.7756\nHonda Odyssey,0H3011H,11.28,137,Gasoline,1.4381\nHonda Odyssey,0H3011H,3.41,137,Gasoline,0.4677\nHonda Odyssey,0H3011H,23.58,137,Gasoline,2.3036\nHonda Odyssey,0H3011H,34.17,137,Gasoline,2.8238\nHonda Odyssey,0H3011H,39.71,137,Gasoline,3.8776\nHonda Odyssey,0H3011H,17.46,137,Gasoline,1.4189\nHonda Odyssey,0H3011H,47.53,137,Gasoline,4.2374\nHonda Odyssey,0H3011H,51.46,137,Gasoline,4.5301\nHonda Odyssey,0H3011H,6.0,137,Gasoline,0.6551\nHonda Odyssey,0H3011H,36.39,137,Gasoline,2.5869\nHonda Odyssey,0H3011H,6.22,137,Gasoline,0.5229\nHonda Odyssey,0H3011H,60.2,137,Gasoline,4.9794\nHonda Odyssey,0H3011H,11.88,137,Gasoline,1.0518\nHonda Odyssey,0H3011H,13.55,137,Gasoline,1.4711\nHonda Odyssey,0H3011H,88.3,137,Gasoline,7.0778\nHonda Odyssey,0H3011H,10.49,137,Gasoline,1.2861\nHonda Odyssey,0H3011H,123.69,137,Gasoline,9.8957\nHonda Odyssey,0H3011H,130.04,137,Gasoline,10.3582\nHonda Odyssey,0H3011H,114.52,137,Gasoline,8.713\nHonda Odyssey,0H3011H,177.26,137,Gasoline,11.0755\nHonda Odyssey,0H3011H,19.1,137,Gasoline,2.2288\nHonda Odyssey,0H3011H,35.32,137,Gasoline,2.4602\nHonda Odyssey,0H3011H,32.36,137,Gasoline,2.5351\nHonda Odyssey,0H3011H,43.97,137,Gasoline,3.7978\nHonda Odyssey,0H3011H,14.89,137,Gasoline,1.9137\nHonda Odyssey,0H3011H,24.6,137,Gasoline,2.1703\nHonda Odyssey,0H3011H,32.81,137,Gasoline,3.2096\nHonda Odyssey,0H3011H,50.1,137,Gasoline,3.3945\nHonda Odyssey,0H3011H,12.84,137,Gasoline,1.4362\nHonda Odyssey,0H3011H,4.21,137,Gasoline,0.4241\nHonda Odyssey,0H3011H,46.8,137,Gasoline,3.9078\nHonda Odyssey,0H3011H,4.65,137,Gasoline,0.64\nHonda Odyssey,0H3011H,39.9,137,Gasoline,3.1936\nHonda Odyssey,0H3011H,146.32,137,Gasoline,12.8464\nHonda Odyssey,0H3011H,228.19,137,Gasoline,18.0511\nHonda Odyssey,0H3011H,387.86,137,Gasoline,31.7825\nHonda Odyssey,0H3011H,375.6,137,Gasoline,29.2246\nHonda Odyssey,0H3011H,2.55,137,Gasoline,0.3441\nHonda Odyssey,0H3011H,1.58,137,Gasoline,0.2587\nHonda Odyssey,0H3011H,106.69,137,Gasoline,9.4983\nHonda Odyssey,0H3011H,15.52,137,Gasoline,1.312\nHonda Odyssey,0H3011H,164.95,137,Gasoline,11.015\nHonda Odyssey,0H3011H,39.03,137,Gasoline,4.576\nHonda Odyssey,0H3011H,69.59,137,Gasoline,4.9364\nHonda Odyssey,0H3011H,19.09,137,Gasoline,2.2277\nHonda Odyssey,0H3011H,105.39,137,Gasoline,7.1858\nHonda Odyssey,0H3011H,278.25,137,Gasoline,19.5034\nHonda Odyssey,0H3011H,41.1,137,Gasoline,3.015\nHonda Odyssey,0H3011H,26.93,137,Gasoline,2.0176\nHonda Odyssey,0H3011H,145.62,137,Gasoline,11.7365\nHonda Odyssey,0H3011H,102.21,137,Gasoline,8.304\nHonda Odyssey,0H3011H,127.39,137,Gasoline,9.2439\nHonda Odyssey,0H3011H,63.95,137,Gasoline,4.8904\nHonda Odyssey,0H3011H,21.14,137,Gasoline,1.8045\nHonda Odyssey,0H3011H,5.32,137,Gasoline,0.7971\nHonda Odyssey,0H3011H,22.51,137,Gasoline,1.8497\nHonda Odyssey,0H3011H,17.65,137,Gasoline,1.9897\nHonda Odyssey,0H3011H,3.24,137,Gasoline,0.5189\nHonda Odyssey,0H3011H,27.5,137,Gasoline,2.0988\nHonda Odyssey,0H3011H,18.89,137,Gasoline,1.4778\nHonda Odyssey,0H3011H,3.16,137,Gasoline,0.3355\nHonda Odyssey,0H3011H,147.29,137,Gasoline,11.0569\nHonda Odyssey,0H3011H,23.95,137,Gasoline,1.8317\nHonda Odyssey,0H3011H,110.52,137,Gasoline,7.4563\nHonda Odyssey,0H3011H,54.02,137,Gasoline,4.8191\nHonda Odyssey,0H3011H,2.02,137,Gasoline,0.2389\nHonda Odyssey,0H3011H,102.49,137,Gasoline,8.3056\nHonda Odyssey,0H3011H,144.61,137,Gasoline,10.0997\nHonda Odyssey,0H3011H,16.92,137,Gasoline,1.4613\nHonda Odyssey,0H3011H,198.58,137,Gasoline,14.7885\nHonda Odyssey,0H3011H,56.0,137,Gasoline,4.2483\nHonda Odyssey,0H3011H,401.81,137,Gasoline,31.8096\nHonda Odyssey,0H3011H,4.74,137,Gasoline,0.4409\nHonda Odyssey,0H3011H,7.48,137,Gasoline,1.1359\nHonda Odyssey,0H3011H,471.86,137,Gasoline,39.3466\nHonda Odyssey,0H3011H,249.65,137,Gasoline,18.1018\nHonda Odyssey,0H3011H,510.69,137,Gasoline,37.7557\nHonda Odyssey,0H3011H,151.43,137,Gasoline,10.8698\nHonda Odyssey,0H3011H,24.85,137,Gasoline,1.9934\nHonda Odyssey,0H3011H,12.84,137,Gasoline,1.2294\nHonda Odyssey,0H3011H,12.43,137,Gasoline,1.45\nHonda Odyssey,0H3011H,29.53,137,Gasoline,1.9968\nHonda Odyssey,0H3011H,24.21,137,Gasoline,2.0801\nHonda Odyssey,0H3011H,4.47,137,Gasoline,0.5445\nHonda Odyssey,0H3011H,90.51,137,Gasoline,8.3325\nHonda Odyssey,0H3011H,4.16,137,Gasoline,0.4167\nHonda Odyssey,0H3011H,36.29,137,Gasoline,2.1977\nHonda Odyssey,0H3011H,60.35,137,Gasoline,4.0113\nHonda Odyssey,0H3011H,6.5,137,Gasoline,0.7689\nHonda Odyssey,0H3011H,4.84,137,Gasoline,0.7339\nHonda Odyssey,0H3011H,2.88,137,Gasoline,0.5038\nHonda Odyssey,0H3011H,38.12,137,Gasoline,3.0362\nHonda Odyssey,0H3011H,28.14,137,Gasoline,2.6443\nHonda Odyssey,0H3011H,3.92,137,Gasoline,0.479\nHonda Odyssey,0H3011H,118.95,137,Gasoline,12.4106\nHonda Odyssey,0H3011H,49.75,137,Gasoline,3.7825\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.1406\nHonda Odyssey,0H3011H,5.02,137,Gasoline,0.5607\nHonda Odyssey,0H3011H,36.34,137,Gasoline,2.9417\nHonda Odyssey,0H3011H,5.33,137,Gasoline,0.6385\nHonda Odyssey,0H3011H,48.09,137,Gasoline,5.0461\nHonda Odyssey,0H3011H,5.14,137,Gasoline,0.5977\nHonda Odyssey,0H3011H,15.35,137,Gasoline,1.392\nHonda Odyssey,0H3011H,66.92,137,Gasoline,6.7792\nHonda Odyssey,0H3011H,23.95,137,Gasoline,2.6881\nHonda Odyssey,0H3011H,35.15,137,Gasoline,3.3441\nHonda Odyssey,0H3011H,54.18,137,Gasoline,3.9436\nHonda Odyssey,0H3011H,6.96,137,Gasoline,1.0262\nHonda Odyssey,0H3011H,27.96,137,Gasoline,2.4413\nHonda Odyssey,0H3011H,103.28,137,Gasoline,10.2664\nHonda Odyssey,0H3011H,49.73,137,Gasoline,3.8918\nHonda Odyssey,0H3011H,34.6,137,Gasoline,2.9305\nHonda Odyssey,0H3011H,9.03,137,Gasoline,1.1912\nHonda Odyssey,0H3011H,62.82,137,Gasoline,5.9701\nHonda Odyssey,0H3011H,7.53,137,Gasoline,0.7231\nHonda Odyssey,0H3011H,5.74,137,Gasoline,0.5373\nHonda Odyssey,0H3011H,119.22,137,Gasoline,9.672\nHonda Odyssey,0H3011H,27.2,137,Gasoline,2.4428\nHonda Odyssey,0H3011H,6.7,137,Gasoline,0.5859\nHonda Odyssey,0H3011H,12.25,137,Gasoline,1.0563\nHonda Odyssey,0H3011H,1.64,137,Gasoline,0.2352\nHonda Odyssey,0H3011H,70.22,137,Gasoline,5.9948\nHonda Odyssey,0H3011H,61.73,137,Gasoline,5.334\nHonda Odyssey,0H3011H,29.17,137,Gasoline,2.1345\nHonda Odyssey,0H3011H,347.78,137,Gasoline,26.6494\nHonda Odyssey,0H3011H,75.35,137,Gasoline,5.7664\nHonda Odyssey,0H3011H,4.39,137,Gasoline,0.431\nHonda Odyssey,0H3011H,14.82,137,Gasoline,1.9848\nHonda Odyssey,0H3011H,2.87,137,Gasoline,0.3699\nHonda Odyssey,0H3011H,7.34,137,Gasoline,0.75\nHonda Odyssey,0H3011H,35.82,137,Gasoline,5.2277\nHonda Odyssey,0H3011H,4.33,137,Gasoline,0.4309\nHonda Odyssey,0H3011H,170.61,137,Gasoline,13.4806\nHonda Odyssey,0H3011H,318.99,137,Gasoline,23.5672\nHonda Odyssey,0H3011H,225.17,137,Gasoline,18.3513\nHonda Odyssey,0H3011H,13.08,137,Gasoline,1.2294\nHonda Odyssey,0H3011H,9.25,137,Gasoline,0.9641\nHonda Odyssey,0H3011H,25.13,137,Gasoline,2.13\nHonda Odyssey,0H3011H,32.82,137,Gasoline,3.2998\nHonda Odyssey,0H3011H,30.79,137,Gasoline,2.505\nHonda Odyssey,0H3011H,122.74,137,Gasoline,10.271\nHonda Odyssey,0H3011H,31.35,137,Gasoline,2.8968\nHonda Odyssey,0H3011H,4.06,137,Gasoline,0.5177\nHonda Odyssey,0H3011H,45.0,137,Gasoline,3.6242\nHonda Odyssey,0H3011H,1.66,137,Gasoline,0.2529\nHonda Odyssey,0H3011H,35.02,137,Gasoline,4.253\nHonda Odyssey,0H3011H,7.56,137,Gasoline,0.6306\nHonda Odyssey,0H3011H,44.26,137,Gasoline,3.0822\nHonda Odyssey,0H3011H,170.61,137,Gasoline,10.9203\nHonda Odyssey,0H3011H,18.9,137,Gasoline,2.0051\nHonda Odyssey,0H3011H,62.42,137,Gasoline,4.4755\nHonda Odyssey,0H3011H,117.59,137,Gasoline,7.9632\nHonda Odyssey,0H3011H,202.42,137,Gasoline,14.7986\nHonda Odyssey,0H3011H,48.29,137,Gasoline,3.9295\nHonda Odyssey,0H3011H,49.11,137,Gasoline,4.1633\nHonda Odyssey,0H3011H,33.7,137,Gasoline,2.3605\nHonda Odyssey,0H3011H,4.64,137,Gasoline,0.651\nHonda Odyssey,0H3011H,173.46,137,Gasoline,17.433\nHonda Odyssey,0H3011H,139.19,137,Gasoline,9.019\nHonda Odyssey,0H3011H,132.75,137,Gasoline,9.6542\nHonda Odyssey,0H3011H,10.94,137,Gasoline,1.1132\nHonda Odyssey,0H3011H,217.54,137,Gasoline,18.4449\nHonda Odyssey,0H3011H,206.8,137,Gasoline,14.654\nHonda Odyssey,0H3011H,136.8,137,Gasoline,9.3094\nHonda Odyssey,0H3011H,22.39,137,Gasoline,2.0206\nHonda Odyssey,0H3011H,15.53,137,Gasoline,1.3103\nHonda Odyssey,0H3011H,29.12,137,Gasoline,2.1709\nHonda Odyssey,0H3011H,42.73,137,Gasoline,3.7828\nHonda Odyssey,0H3011H,327.61,137,Gasoline,24.2619\nHonda Odyssey,0H3011H,42.37,137,Gasoline,3.9356\nHonda Odyssey,0H3011H,25.71,137,Gasoline,1.9331\nHonda Odyssey,0H3011H,57.51,137,Gasoline,5.7616\nHonda Odyssey,0H3011H,49.25,137,Gasoline,5.8316\nHonda Odyssey,0H3011H,22.51,137,Gasoline,2.2931\nHonda Odyssey,0H3011H,34.23,137,Gasoline,3.1891\nHonda Odyssey,0H3011H,23.05,137,Gasoline,2.338\nHonda Odyssey,0H3011H,8.5,137,Gasoline,1.1413\nHonda Odyssey,0H3011H,14.38,137,Gasoline,1.6369\nHonda Odyssey,0H3011H,5.41,137,Gasoline,0.5073\nHonda Odyssey,0H3011H,1.51,137,Gasoline,0.2285\nHonda Odyssey,0H3011H,54.71,137,Gasoline,6.573\nHonda Odyssey,0H3011H,15.77,137,Gasoline,1.4553\nHonda Odyssey,0H3011H,6.21,137,Gasoline,0.5109\nHonda Odyssey,0H3011H,8.27,137,Gasoline,1.172\nHonda Odyssey,0H3011H,4.47,137,Gasoline,0.4736\nHonda Odyssey,0H3011H,8.05,137,Gasoline,0.6031\nHonda Odyssey,0H3011H,189.87,137,Gasoline,15.5509\nHonda Odyssey,0H3011H,3.92,137,Gasoline,0.5579\nHonda Odyssey,0H3011H,7.33,137,Gasoline,0.4898\nHonda Odyssey,0H3011H,11.98,137,Gasoline,1.4939\nHonda Odyssey,0H3011H,4.7,137,Gasoline,0.372\nHonda Odyssey,0H3011H,46.01,137,Gasoline,3.6358\nHonda Odyssey,0H3011H,19.22,137,Gasoline,1.5783\nHonda Odyssey,0H3011H,35.97,137,Gasoline,2.7211\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.195\nHonda Odyssey,0H3011H,9.12,137,Gasoline,1.2936\nHonda Odyssey,0H3011H,6.11,137,Gasoline,1.0558\nHonda Odyssey,0H3011H,3.8,137,Gasoline,0.5419\nHonda Odyssey,0H3011H,35.23,137,Gasoline,3.5275\nHonda Odyssey,0H3011H,1.66,137,Gasoline,0.2108\nHonda Odyssey,0H3011H,6.22,137,Gasoline,0.7047\nHonda Odyssey,0H3011H,11.96,137,Gasoline,1.221\nHonda Odyssey,0H3011H,21.52,137,Gasoline,2.0698\nHonda Odyssey,0H3011H,4.1,137,Gasoline,0.5385\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.1786\nHonda Odyssey,0H3011H,51.05,137,Gasoline,3.6299\nHonda Odyssey,0H3011H,9.47,137,Gasoline,1.2253\nHonda Odyssey,0H3011H,56.16,137,Gasoline,5.0786\nHonda Odyssey,0H3011H,64.94,137,Gasoline,4.7441\nHonda Odyssey,0H3011H,21.6,137,Gasoline,2.0979\nHonda Odyssey,0H3011H,7.11,137,Gasoline,0.623\nHonda Odyssey,0H3011H,2.87,137,Gasoline,0.4226\nHonda Odyssey,0H3011H,37.48,137,Gasoline,2.8286\nHonda Odyssey,0H3011H,2.76,137,Gasoline,0.3344\nHonda Odyssey,0H3011H,92.18,137,Gasoline,10.9088\nHonda Odyssey,0H3011H,3.79,137,Gasoline,0.4541\nHonda Odyssey,0H3011H,29.94,137,Gasoline,3.2481\nHonda Odyssey,0H3011H,131.1,137,Gasoline,11.6027\nHonda Odyssey,0H3011H,4.41,137,Gasoline,0.4425\nHonda Odyssey,0H3011H,19.93,137,Gasoline,1.7668\nHonda Odyssey,0H3011H,33.04,137,Gasoline,3.4476\nHonda Odyssey,0H3011H,12.36,137,Gasoline,1.0598\nHonda Odyssey,0H3011H,13.42,137,Gasoline,1.1299\nHonda Odyssey,0H3011H,120.07,137,Gasoline,9.3884\nHonda Odyssey,0H3011H,13.65,137,Gasoline,1.633\nHonda Odyssey,0H3011H,3.49,137,Gasoline,0.5008\nHonda Odyssey,0H3011H,149.31,137,Gasoline,11.6478\nHonda Odyssey,0H3011H,26.58,137,Gasoline,2.4725\nHonda Odyssey,0H3011H,111.09,137,Gasoline,8.5174\nHonda Odyssey,0H3011H,21.29,137,Gasoline,1.9382\nHonda Odyssey,0H3011H,10.2,137,Gasoline,0.9747\nHonda Odyssey,0H3011H,2.64,137,Gasoline,0.3635\nHonda Odyssey,0H3011H,51.46,137,Gasoline,3.3993\nHonda Odyssey,0H3011H,49.39,137,Gasoline,4.1624\nHonda Odyssey,0H3011H,27.74,137,Gasoline,2.9126\nHonda Odyssey,0H3011H,4.35,137,Gasoline,0.4776\nHonda Odyssey,0H3011H,173.54,137,Gasoline,13.6937\nHonda Odyssey,0H3011H,54.65,137,Gasoline,3.9787\nHonda Odyssey,0H3011H,33.53,137,Gasoline,2.8825\nHonda Odyssey,0H3011H,15.05,137,Gasoline,1.1834\nHonda Odyssey,0H3011H,191.39,137,Gasoline,15.2125\nHonda Odyssey,0H3011H,85.39,137,Gasoline,6.5157\nHonda Odyssey,0H3011H,47.12,137,Gasoline,3.607\nHonda Odyssey,0H3011H,14.55,137,Gasoline,1.0843\nHonda Odyssey,0H3011H,11.44,137,Gasoline,1.1026\nHonda Odyssey,0H3011H,49.37,137,Gasoline,3.5217\nHonda Odyssey,0H3011H,12.66,137,Gasoline,1.5368\nHonda Odyssey,0H3011H,36.67,137,Gasoline,3.0588\nHonda Odyssey,0H3011H,21.93,137,Gasoline,2.0809\nHonda Odyssey,0H3011H,5.05,137,Gasoline,0.5618\nHonda Odyssey,0H3011H,1.57,137,Gasoline,0.231\nHonda Odyssey,0H3011H,9.86,137,Gasoline,1.1915\nHonda Odyssey,0H3011H,6.97,137,Gasoline,0.6739\nHonda Odyssey,0H3011H,23.06,137,Gasoline,1.9514\nHonda Odyssey,0H3011H,117.61,137,Gasoline,9.4727\nHonda Odyssey,0H3011H,7.91,137,Gasoline,0.6955\nHonda Odyssey,0H3011H,4.05,137,Gasoline,0.6278\nHonda Odyssey,0H3011H,3.89,137,Gasoline,0.4304\nHonda Odyssey,0H3011H,28.4,137,Gasoline,2.1984\nHonda Odyssey,0H3011H,8.33,137,Gasoline,0.974\nHonda Odyssey,0H3011H,22.64,137,Gasoline,2.0879\nHonda Odyssey,0H3011H,119.45,137,Gasoline,10.7545\nHonda Odyssey,0H3011H,47.52,137,Gasoline,4.5298\nHonda Odyssey,0H3011H,2.76,137,Gasoline,0.3895\nHonda Odyssey,0H3011H,45.31,137,Gasoline,3.3896\nHonda Odyssey,0H3011H,24.45,137,Gasoline,1.7332\nHonda Odyssey,0H3011H,1.98,137,Gasoline,0.2446\nHonda Odyssey,0H3011H,30.35,137,Gasoline,2.6903\nHonda Odyssey,0H3011H,4.93,137,Gasoline,0.593\nHonda Odyssey,0H3011H,86.72,137,Gasoline,10.2923\nHonda Odyssey,0H3011H,3.87,137,Gasoline,0.5439\nHonda Odyssey,0H3011H,34.1,137,Gasoline,4.0851\nHonda Odyssey,0H3011H,41.6,137,Gasoline,3.9027\nHonda Odyssey,0H3011H,10.66,137,Gasoline,0.8979\nHonda Odyssey,0H3011H,18.94,137,Gasoline,2.0351\nHonda Odyssey,0H3011H,30.31,137,Gasoline,3.1714\nHonda Odyssey,0H3011H,2.19,137,Gasoline,0.226\nHonda Odyssey,0H3011H,102.8,137,Gasoline,6.705\nHonda Odyssey,0H3011H,104.05,137,Gasoline,7.2764\nHonda Odyssey,0H3011H,36.55,137,Gasoline,3.8491\nHonda Odyssey,0H3011H,27.44,137,Gasoline,2.3275\nHonda Odyssey,0H3011H,76.83,137,Gasoline,5.7303\nHonda Odyssey,0H3011H,43.69,137,Gasoline,4.3312\nHonda Odyssey,0H3011H,26.86,137,Gasoline,2.4257\nHonda Odyssey,0H3011H,11.48,137,Gasoline,1.1832\nHonda Odyssey,0H3011H,52.64,137,Gasoline,3.9003\nHonda Odyssey,0H3011H,37.82,137,Gasoline,3.459\nHonda Odyssey,0H3011H,22.24,137,Gasoline,2.9785\nHonda Odyssey,0H3011H,10.4,137,Gasoline,1.8049\nHonda Odyssey,0H3011H,11.86,137,Gasoline,1.4988\nHonda Odyssey,0H3011H,210.84,137,Gasoline,16.3089\nHonda Odyssey,0H3011H,126.9,137,Gasoline,10.1625\nHonda Odyssey,0H3011H,84.14,137,Gasoline,6.9591\nHonda Odyssey,0H3011H,5.98,137,Gasoline,0.6969\nHonda Odyssey,0H3011H,49.45,137,Gasoline,4.1623\nHonda Odyssey,0H3011H,242.97,137,Gasoline,18.2999\nHonda Odyssey,0H3011H,96.29,137,Gasoline,6.6158\nHonda Odyssey,0H3011H,134.64,137,Gasoline,10.1986\nHonda Odyssey,0H3011H,1.9,137,Gasoline,0.3094\nHonda Odyssey,0H3011H,21.11,137,Gasoline,2.3247\nHonda Odyssey,0H3011H,272.26,137,Gasoline,18.7677\nHonda Odyssey,0H3011H,129.73,137,Gasoline,9.7322\nHonda Odyssey,0H3011H,10.35,137,Gasoline,1.1611\nHonda Odyssey,0H3011H,154.43,137,Gasoline,12.0948\nHonda Odyssey,0H3011H,264.51,137,Gasoline,23.5779\nHonda Odyssey,0H3011H,3.77,137,Gasoline,0.4683\nHonda Odyssey,0H3011H,299.61,137,Gasoline,24.5604\nHonda Odyssey,0H3011H,215.21,137,Gasoline,15.1431\nHonda Odyssey,0H3011H,167.73,137,Gasoline,13.8579\nHonda Odyssey,0H3011H,156.01,137,Gasoline,11.8038\nHonda Odyssey,0H3011H,34.06,137,Gasoline,2.7329\nHonda Odyssey,0H3011H,10.07,137,Gasoline,1.0578\nHonda Odyssey,0H3011H,199.58,137,Gasoline,13.8031\nHonda Odyssey,0H3011H,273.28,137,Gasoline,18.6922\nHonda Odyssey,0H3011H,121.82,137,Gasoline,12.1205\nHonda Odyssey,0H3011H,21.07,137,Gasoline,2.3706\nHonda Odyssey,0H3011H,140.94,137,Gasoline,12.8862\nHonda Odyssey,0H3011H,3.48,137,Gasoline,0.4011\nHonda Odyssey,0H3011H,126.46,137,Gasoline,9.4583\nHonda Odyssey,0H3011H,158.51,137,Gasoline,10.1705\nHonda Odyssey,0H3011H,158.6,137,Gasoline,10.9635\nHonda Odyssey,0H3011H,32.89,137,Gasoline,2.0226\nHonda Odyssey,0H3011H,158.83,137,Gasoline,9.9076\nHonda Odyssey,0H3011H,118.99,137,Gasoline,8.9546\nHonda Odyssey,0H3011H,2.33,137,Gasoline,0.3023\nHonda Odyssey,0H3011H,306.48,137,Gasoline,24.4943\nHonda Odyssey,0H3011H,242.32,137,Gasoline,19.4884\nHonda Odyssey,0H3011H,8.95,137,Gasoline,1.0297\nHonda Odyssey,0H3011H,17.79,137,Gasoline,1.4085\nHonda Odyssey,0H3011H,107.01,137,Gasoline,8.0376\nHonda Odyssey,0H3011H,70.24,137,Gasoline,4.3126\nHonda Odyssey,0H3011H,12.71,137,Gasoline,1.1836\nHonda Odyssey,0H3011H,107.6,137,Gasoline,7.5791\nHonda Odyssey,0H3011H,6.24,137,Gasoline,0.6612\nHonda Odyssey,0H3011H,17.68,137,Gasoline,1.1441\nHonda Odyssey,0H3011H,251.26,137,Gasoline,16.8258\nHonda Odyssey,0H3011H,55.75,137,Gasoline,5.1451\nHonda Odyssey,0H3011H,215.71,137,Gasoline,16.9099\nHonda Odyssey,0H3011H,44.17,137,Gasoline,3.3927\nHonda Odyssey,0H3011H,5.74,137,Gasoline,0.5998\nHonda Odyssey,0H3011H,14.25,137,Gasoline,1.3857\nHonda Odyssey,0H3011H,100.48,137,Gasoline,7.2079\nHonda Odyssey,0H3011H,53.17,137,Gasoline,4.2488\nHonda Odyssey,0H3011H,103.09,137,Gasoline,7.1061\nHonda Odyssey,0H3011H,91.6,137,Gasoline,7.2747\nHonda Odyssey,0H3011H,98.7,137,Gasoline,8.7812\nHonda Odyssey,0H3011H,39.61,137,Gasoline,3.531\nHonda Odyssey,0H3011H,52.48,137,Gasoline,5.0056\nHonda Odyssey,0H3011H,150.81,137,Gasoline,13.5356\nHonda Odyssey,0H3011H,15.8,137,Gasoline,1.3026\nHonda Odyssey,0H3011H,28.89,137,Gasoline,2.4753\nHonda Odyssey,0H3011H,42.35,137,Gasoline,4.0763\nHonda Odyssey,0H3011H,102.56,137,Gasoline,8.2297\nHonda Odyssey,0H3011H,160.6,137,Gasoline,13.5357\nHonda Odyssey,0H3011H,1.94,137,Gasoline,0.2732\nHonda Odyssey,0H3011H,55.91,137,Gasoline,5.6608\nHonda Odyssey,0H3011H,5.6,137,Gasoline,0.6303\nHonda Odyssey,0H3011H,3.14,137,Gasoline,0.325\nHonda Odyssey,0H3011H,46.78,137,Gasoline,3.7886\nHonda Odyssey,0H3011H,12.06,137,Gasoline,1.1354\nHonda Odyssey,0H3011H,146.18,137,Gasoline,11.8633\nHonda Odyssey,0H3011H,207.01,137,Gasoline,18.7627\nHonda Odyssey,0H3011H,22.69,137,Gasoline,1.634\nHonda Odyssey,0H3011H,19.41,137,Gasoline,2.6853\nHonda Odyssey,0H3011H,3.07,137,Gasoline,0.3688\nHonda Odyssey,0H3011H,104.39,137,Gasoline,9.4377\nHonda Odyssey,0H3011H,4.84,137,Gasoline,0.4288\nHonda Odyssey,0H3011H,481.71,137,Gasoline,35.7891\nHonda Odyssey,0H3011H,13.18,137,Gasoline,1.9973\nHonda Odyssey,0H3011H,156.22,137,Gasoline,14.4919\nHonda Odyssey,0H3011H,28.05,137,Gasoline,3.014\nHonda Odyssey,0H3011H,42.4,137,Gasoline,3.0409\nHonda Odyssey,0H3011H,115.41,137,Gasoline,11.8808\nHonda Odyssey,0H3011H,2.24,137,Gasoline,0.2845\nHonda Odyssey,0H3011H,34.19,137,Gasoline,2.4499\nHonda Odyssey,0H3011H,6.02,137,Gasoline,0.6598\nHonda Odyssey,0H3011H,3.96,137,Gasoline,0.4521\nHonda Odyssey,0H3011H,13.68,137,Gasoline,1.2637\nHonda Odyssey,0H3011H,52.15,137,Gasoline,3.8874\nHonda Odyssey,0H3011H,26.33,137,Gasoline,2.5501\nHonda Odyssey,0H3011H,12.52,137,Gasoline,1.3439\nHonda Odyssey,0H3011H,6.07,137,Gasoline,0.6256\nHonda Odyssey,0H3011H,9.52,137,Gasoline,0.7833\nHonda Odyssey,0H3011H,41.91,137,Gasoline,3.7007\nHonda Odyssey,0H3011H,1.55,137,Gasoline,0.2282\nHonda Odyssey,0H3011H,33.98,137,Gasoline,3.1849\nHonda Odyssey,0H3011H,27.34,137,Gasoline,2.3689\nHonda Odyssey,0H3011H,155.92,137,Gasoline,11.9988\nHonda Odyssey,0H3011H,12.26,137,Gasoline,1.2013\nHonda Odyssey,0H3011H,2.95,137,Gasoline,0.4136\nHonda Odyssey,0H3011H,56.34,137,Gasoline,3.7631\nHonda Odyssey,0H3011H,9.67,137,Gasoline,0.9156\nHonda Odyssey,0H3011H,38.35,137,Gasoline,3.2388\nHonda Odyssey,0H3011H,18.14,137,Gasoline,1.9629\nHonda Odyssey,0H3011H,141.75,137,Gasoline,11.431\nHonda Odyssey,0H3011H,10.06,137,Gasoline,1.038\nHonda Odyssey,0H3011H,53.04,137,Gasoline,5.2008\nHonda Odyssey,0H3011H,39.38,137,Gasoline,3.4259\nHonda Odyssey,0H3011H,24.96,137,Gasoline,1.7064\nHonda Odyssey,0H3011H,7.17,137,Gasoline,1.2257\nHonda Odyssey,0H3011H,16.65,137,Gasoline,2.2491\nHonda Odyssey,0H3011H,5.1,137,Gasoline,0.52\nHonda Odyssey,0H3011H,108.72,137,Gasoline,6.9055\nHonda Odyssey,0H3011H,3.94,137,Gasoline,0.6331\nHonda Odyssey,0H3011H,64.7,137,Gasoline,4.7271\nHonda Odyssey,0H3011H,36.19,137,Gasoline,3.228\nHonda Odyssey,0H3011H,19.42,137,Gasoline,2.5222\nHonda Odyssey,0H3011H,15.96,137,Gasoline,1.5195\nHonda Odyssey,0H3011H,79.43,137,Gasoline,5.1025\nHonda Odyssey,0H3011H,121.67,137,Gasoline,8.4466\nHonda Odyssey,0H3011H,17.35,137,Gasoline,1.7156\nHonda Odyssey,0H3011H,4.51,137,Gasoline,0.4933\nHonda Odyssey,0H3011H,7.28,137,Gasoline,0.6995\nHonda Odyssey,0H3011H,42.71,137,Gasoline,3.4944\nHonda Odyssey,0H3011H,36.33,137,Gasoline,2.7759\nHonda Odyssey,0H3011H,11.92,137,Gasoline,1.4441\nHonda Odyssey,0H3011H,45.88,137,Gasoline,4.3513\nHonda Odyssey,0H3011H,35.96,137,Gasoline,3.2257\nHonda Odyssey,0H3011H,3.31,137,Gasoline,0.3204\nHonda Odyssey,0H3011H,15.53,137,Gasoline,1.216\nHonda Odyssey,0H3011H,2.19,137,Gasoline,0.2831\nHonda Odyssey,0H3011H,14.41,137,Gasoline,1.2389\nHonda Odyssey,0H3011H,3.77,137,Gasoline,0.3727\nHonda Odyssey,0H3011H,45.86,137,Gasoline,4.3397\nHonda Odyssey,0H3011H,136.73,137,Gasoline,11.7197\nHonda Odyssey,0H3011H,20.77,137,Gasoline,2.2365\nHonda Odyssey,0H3011H,32.41,137,Gasoline,2.3511\nHonda Odyssey,0H3011H,130.15,137,Gasoline,10.2483\nHonda Odyssey,0H3011H,185.41,137,Gasoline,15.1965\nHonda Odyssey,0H3011H,48.62,137,Gasoline,4.1221\nHonda Odyssey,0H3011H,50.79,137,Gasoline,4.7496\nHonda Odyssey,0H3011H,241.17,137,Gasoline,18.1769\nHonda Odyssey,0H3011H,7.57,137,Gasoline,0.8472\nHonda Odyssey,0H3011H,10.6,137,Gasoline,0.9883\nHonda Odyssey,0H3011H,128.0,137,Gasoline,9.8464\nHonda Odyssey,0H3011H,188.68,137,Gasoline,13.4929\nHonda Odyssey,0H3011H,46.76,137,Gasoline,4.2942\nHonda Odyssey,0H3011H,2.63,137,Gasoline,0.355\nHonda Odyssey,0H3011H,12.15,137,Gasoline,1.1969\nHonda Odyssey,0H3011H,76.79,137,Gasoline,6.3892\nHonda Odyssey,0H3011H,191.35,137,Gasoline,14.1175\nHonda Odyssey,0H3011H,276.52,137,Gasoline,21.4603\nHonda Odyssey,0H3011H,118.69,137,Gasoline,10.3087\nHonda Odyssey,0H3011H,134.46,137,Gasoline,8.6845\nHonda Odyssey,0H3011H,3.05,137,Gasoline,0.2283\nHonda Odyssey,0H3011H,147.44,137,Gasoline,9.5454\nHonda Odyssey,0H3011H,64.31,137,Gasoline,4.9862\nHonda Odyssey,0H3011H,315.01,137,Gasoline,23.4081\nHonda Odyssey,0H3011H,5.86,137,Gasoline,0.8248\nHonda Odyssey,0H3011H,56.05,137,Gasoline,3.8189\nHonda Odyssey,0H3011H,6.96,137,Gasoline,0.7881\nHonda Odyssey,0H3011H,191.12,137,Gasoline,13.3944\nHonda Odyssey,0H3011H,38.53,137,Gasoline,2.9625\nHonda Odyssey,0H3011H,104.48,137,Gasoline,6.3841\nHonda Odyssey,0H3011H,120.79,137,Gasoline,9.6755\nHonda Odyssey,0H3011H,122.79,137,Gasoline,9.9457\nHonda Odyssey,0H3011H,143.1,137,Gasoline,10.3224\nHonda Odyssey,0H3011H,50.55,137,Gasoline,3.5101\nHonda Odyssey,0H3011H,17.88,137,Gasoline,1.7466\nHonda Odyssey,0H3011H,17.56,137,Gasoline,2.0358\nHonda Odyssey,0H3011H,1.98,137,Gasoline,0.2749\nHonda Odyssey,0H3011H,265.97,137,Gasoline,19.1147\nHonda Odyssey,0H3011H,24.9,137,Gasoline,1.9942\nHonda Odyssey,0H3011H,156.66,137,Gasoline,13.4022\nHonda Odyssey,0H3011H,29.22,137,Gasoline,2.2755\nHonda Odyssey,0H3011H,314.89,137,Gasoline,23.9525\nHonda Odyssey,0H3011H,160.6,137,Gasoline,15.839\nHonda Odyssey,0H3011H,10.71,137,Gasoline,1.0355\nHonda Odyssey,0H3011H,100.46,137,Gasoline,8.3094\nHonda Odyssey,0H3011H,330.18,137,Gasoline,24.5485\nHonda Odyssey,0H3011H,253.37,137,Gasoline,17.939\nHonda Odyssey,0H3011H,186.39,137,Gasoline,14.1958\nHonda Odyssey,0H3011H,186.61,137,Gasoline,12.829\nHonda Odyssey,0H3011H,60.24,137,Gasoline,4.5929\nHonda Odyssey,0H3011H,162.25,137,Gasoline,12.1997\nHonda Odyssey,0H3011H,33.98,137,Gasoline,3.1789\nHonda Odyssey,0H3011H,6.0,137,Gasoline,0.6421\nHonda Odyssey,0H3011H,111.9,137,Gasoline,9.0315\nHonda Odyssey,0H3011H,271.29,137,Gasoline,19.2051\nHonda Odyssey,0H3011H,16.4,137,Gasoline,1.1608\nHonda Odyssey,0H3011H,198.33,137,Gasoline,15.0602\nHonda Odyssey,0H3011H,7.26,137,Gasoline,0.6065\nHonda Odyssey,0H3011H,180.67,137,Gasoline,13.2684\nHonda Odyssey,0H3011H,181.02,137,Gasoline,11.9241\nHonda Odyssey,0H3011H,285.46,137,Gasoline,22.591\nHonda Odyssey,0H3011H,102.31,137,Gasoline,7.9065\nHonda Odyssey,0H3011H,10.87,137,Gasoline,1.2529\nHonda Odyssey,0H3011H,42.34,137,Gasoline,4.4434\nHonda Odyssey,0H3011H,232.49,137,Gasoline,17.7823\nHonda Odyssey,0H3011H,3.36,137,Gasoline,0.3868\nHonda Odyssey,0H3011H,15.6,137,Gasoline,1.2794\nHonda Odyssey,0H3011H,69.05,137,Gasoline,5.4155\nHonda Odyssey,0H3011H,33.09,137,Gasoline,3.2246\nHonda Odyssey,0H3011H,8.08,137,Gasoline,0.8424\nHonda Odyssey,0H3011H,21.07,137,Gasoline,2.0276\nHonda Odyssey,0H3011H,2.37,137,Gasoline,0.2741\nHonda Odyssey,0H3011H,21.02,137,Gasoline,1.7757\nHonda Odyssey,0H3011H,25.96,137,Gasoline,2.3485\nHonda Odyssey,0H3011H,38.77,137,Gasoline,4.2273\nHonda Odyssey,0H3011H,27.13,137,Gasoline,3.8131\nHonda Odyssey,0H3011H,306.69,137,Gasoline,23.5438\nHonda Odyssey,0H3011H,101.24,137,Gasoline,7.5283\nHonda Odyssey,0H3011H,254.41,137,Gasoline,18.189\nHonda Odyssey,0H3011H,185.94,137,Gasoline,13.5548\nHonda Odyssey,0H3011H,57.93,137,Gasoline,4.6843\nHonda Odyssey,0H3011H,82.33,137,Gasoline,5.8891\nHonda Odyssey,0H3011H,4.24,137,Gasoline,0.553\nHonda Odyssey,0H3011H,8.83,137,Gasoline,1.1329\nHonda Odyssey,0H3011H,11.73,137,Gasoline,1.2282\nHonda Odyssey,0H3011H,7.02,137,Gasoline,0.6867\nHonda Odyssey,0H3011H,23.27,137,Gasoline,1.8119\nHonda Odyssey,0H3011H,2.11,137,Gasoline,0.2099\nHonda Odyssey,0H3011H,15.81,137,Gasoline,1.3392\nHonda Odyssey,0H3011H,2.77,137,Gasoline,0.3789\nHonda Odyssey,0H3011H,50.82,137,Gasoline,4.0228\nHonda Odyssey,0H3011H,27.25,137,Gasoline,2.2165\nHonda Odyssey,0H3011H,8.78,137,Gasoline,1.1586\nHonda Odyssey,0H3011H,148.04,137,Gasoline,9.1034\nHonda Odyssey,0H3011H,10.34,137,Gasoline,1.221\nHonda Odyssey,0H3011H,15.83,137,Gasoline,1.5654\nHonda Odyssey,0H3011H,1.57,137,Gasoline,0.1948\nHonda Odyssey,0H3011H,20.73,137,Gasoline,2.3836\nHonda Odyssey,0H3011H,33.49,137,Gasoline,2.9789\nHonda Odyssey,0H3011H,121.21,137,Gasoline,8.7759\nHonda Odyssey,0H3011H,1.78,137,Gasoline,0.3241\nHonda Odyssey,0H3011H,255.74,137,Gasoline,19.9507\nHonda Odyssey,0H3011H,2.9,137,Gasoline,0.4028\nHonda Odyssey,0H3011H,33.78,137,Gasoline,3.3893\nHonda Odyssey,0H3011H,41.1,137,Gasoline,3.313\nHonda Odyssey,0H3011H,104.69,137,Gasoline,8.6343\nHonda Odyssey,0H3011H,17.95,137,Gasoline,1.6987\nHonda Odyssey,0H3011H,4.28,137,Gasoline,0.6236\nHonda Odyssey,0H3011H,9.55,137,Gasoline,1.1815\nHonda Odyssey,0H3011H,2.58,137,Gasoline,0.4181\nHonda Odyssey,0H3011H,141.03,137,Gasoline,11.16\nHonda Odyssey,0H3011H,10.1,137,Gasoline,0.9193\nHonda Odyssey,0H3011H,29.5,137,Gasoline,2.7666\nHonda Odyssey,0H3011H,61.02,137,Gasoline,6.3266\nHonda Odyssey,0H3011H,178.05,137,Gasoline,14.7664\nHonda Odyssey,0H3011H,2.55,137,Gasoline,0.3937\nHonda Odyssey,0H3011H,5.52,137,Gasoline,0.5656\nHonda Odyssey,0H3011H,11.54,137,Gasoline,1.7085\nHonda Odyssey,0H3011H,13.48,137,Gasoline,1.213\nHonda Odyssey,0H3011H,6.79,137,Gasoline,0.7328\nHonda Odyssey,0H3011H,3.76,137,Gasoline,0.3798\nHonda Odyssey,0H3011H,29.02,137,Gasoline,2.6763\nHonda Odyssey,0H3011H,31.97,137,Gasoline,2.7197\nHonda Odyssey,0H3011H,120.91,137,Gasoline,9.3592\nHonda Odyssey,0H3011H,21.82,137,Gasoline,2.3164\nHonda Odyssey,0H3011H,9.18,137,Gasoline,0.9946\nHonda Odyssey,0H3011H,1.59,137,Gasoline,0.2099\nHonda Odyssey,0H3011H,2.06,137,Gasoline,0.387\nHonda Odyssey,0H3011H,167.57,137,Gasoline,10.5295\nHonda Odyssey,0H3011H,141.42,137,Gasoline,11.0597\nHonda Odyssey,0H3011H,1.79,137,Gasoline,0.3713\nHonda Odyssey,0H3011H,4.09,137,Gasoline,0.6901\nHonda Odyssey,0H3011H,10.74,137,Gasoline,1.3025\nHonda Odyssey,0H3011H,2.75,137,Gasoline,0.3162\nHonda Odyssey,0H3011H,28.72,137,Gasoline,2.5226\nHonda Odyssey,0H3011H,235.19,137,Gasoline,18.6512\nHonda Odyssey,0H3011H,332.7,137,Gasoline,24.8299\nHonda Odyssey,0H3011H,61.39,137,Gasoline,5.7404\nHonda Odyssey,0H3011H,14.13,137,Gasoline,1.3588\nHonda Odyssey,0H3011H,4.24,137,Gasoline,0.62\nHonda Odyssey,0H3011H,2.29,137,Gasoline,0.2552\nHonda Odyssey,0H3011H,1.61,137,Gasoline,0.1965\nHonda Odyssey,0H3011H,36.19,137,Gasoline,4.1677\nHonda Odyssey,0H3011H,55.03,137,Gasoline,4.4259\nHonda Odyssey,0H3011H,103.09,137,Gasoline,9.7298\nHonda Odyssey,0H3011H,7.82,137,Gasoline,0.9367\nHonda Odyssey,0H3011H,56.4,137,Gasoline,5.2403\nHonda Odyssey,0H3011H,34.93,137,Gasoline,3.8579\nHonda Odyssey,0H3011H,10.48,137,Gasoline,1.148\nHonda Odyssey,0H3011H,158.07,137,Gasoline,16.445\nHonda Odyssey,0H3011H,30.85,137,Gasoline,2.5701\nHonda Odyssey,0H3011H,108.6,137,Gasoline,11.3489\nHonda Odyssey,0H3011H,6.14,137,Gasoline,0.9349\nHonda Odyssey,0H3011H,47.14,137,Gasoline,3.7276\nHonda Odyssey,0H3011H,22.36,137,Gasoline,2.5111\nHonda Odyssey,0H3011H,23.46,137,Gasoline,2.0023\nHonda Odyssey,0H3011H,54.96,137,Gasoline,3.9928\nHonda Odyssey,0H3011H,23.52,137,Gasoline,1.9083\nHonda Odyssey,0H3011H,13.9,137,Gasoline,1.3678\nHonda Odyssey,0H3011H,2.86,137,Gasoline,0.4319\nHonda Odyssey,0H3011H,1.81,137,Gasoline,0.2224\nHonda Odyssey,0H3011H,10.37,137,Gasoline,1.5674\nHonda Odyssey,0H3011H,42.36,137,Gasoline,3.9075\nHonda Odyssey,0H3011H,14.92,137,Gasoline,2.3143\nHonda Odyssey,0H3011H,3.5,137,Gasoline,0.3241\nHonda Odyssey,0H3011H,34.72,137,Gasoline,2.9565\nHonda Odyssey,0H3011H,6.6,137,Gasoline,0.927\nHonda Odyssey,0H3011H,12.9,137,Gasoline,1.5642\nHonda Odyssey,0H3011H,7.64,137,Gasoline,0.753\nHonda Odyssey,0H3011H,16.31,137,Gasoline,1.2783\nHonda Odyssey,0H3011H,17.86,137,Gasoline,1.7082\nHyundai ix35,3J1020O,22.31,121,Gasoline,2.2556\nHyundai ix35,3J1020O,6.61,121,Gasoline,0.7198\nHyundai ix35,3J1020O,51.82,121,Gasoline,5.0061\nHyundai ix35,3J1020O,47.98,121,Gasoline,4.7109\nHyundai ix35,3J1020O,4.3,121,Gasoline,0.7716\nHyundai ix35,3J1020O,2.71,121,Gasoline,0.4188\nHyundai ix35,3J1020O,3.02,121,Gasoline,0.5953\nHyundai ix35,3J1020O,4.72,121,Gasoline,0.5621\nHyundai ix35,3J1020O,15.61,121,Gasoline,1.8238\nHyundai ix35,3J1020O,11.94,121,Gasoline,1.4322\nHyundai ix35,3J1020O,18.05,121,Gasoline,1.7655\nHyundai ix35,3J1020O,18.77,121,Gasoline,1.4687\nHyundai ix35,3J1020O,4.17,121,Gasoline,0.6639\nHyundai ix35,3J1020O,12.21,121,Gasoline,1.3923\nHyundai ix35,3J1020O,12.25,121,Gasoline,1.2135\nHyundai ix35,3J1020O,27.71,121,Gasoline,3.2088\nHyundai ix35,3J1020O,1.76,121,Gasoline,0.2993\nHyundai ix35,3J1020O,2.76,121,Gasoline,0.318\nHyundai ix35,3J1020O,24.67,121,Gasoline,2.2686\nHyundai ix35,3J1020O,9.03,121,Gasoline,1.1173\nHyundai ix35,3J1020O,13.53,121,Gasoline,1.1405\nHyundai ix35,3J1020O,46.76,121,Gasoline,3.669\nHyundai ix35,3J1020O,7.41,121,Gasoline,0.9047\nHyundai ix35,3J1020O,8.72,121,Gasoline,0.8993\nHyundai ix35,3J1020O,5.6,121,Gasoline,0.5355\nHyundai ix35,3J1020O,68.9,121,Gasoline,6.1093\nHyundai ix35,3J1020O,15.04,121,Gasoline,1.4554\nHyundai ix35,3J1020O,59.13,121,Gasoline,4.625\nHyundai ix35,3J1020O,4.85,121,Gasoline,0.521\nHyundai ix35,3J1020O,13.23,121,Gasoline,1.701\nHyundai ix35,3J1020O,37.37,121,Gasoline,2.8855\nHyundai ix35,3J1020O,5.32,121,Gasoline,0.6846\nHyundai ix35,3J1020O,6.46,121,Gasoline,0.9825\nHyundai ix35,3J1020O,13.6,121,Gasoline,1.0798\nHyundai ix35,3J1020O,238.18,121,Gasoline,19.6107\nHyundai ix35,3J1020O,4.17,121,Gasoline,0.6429\nHyundai ix35,3J1020O,612.58,121,Gasoline,54.3232\nHyundai ix35,3J1020O,2.21,121,Gasoline,0.3797\nHyundai ix35,3J1020O,9.99,121,Gasoline,1.1228\nHyundai ix35,3J1020O,1.86,121,Gasoline,0.378\nHyundai ix35,3J1020O,94.46,121,Gasoline,9.3972\nHyundai ix35,3J1020O,39.28,121,Gasoline,3.5677\nHyundai ix35,3J1020O,17.08,121,Gasoline,1.5554\nHyundai ix35,3J1020O,3.0,121,Gasoline,0.3599\nHyundai ix35,3J1020O,34.05,121,Gasoline,2.816\nHyundai ix35,3J1020O,1.9,121,Gasoline,0.2403\nHyundai ix35,3J1020O,17.38,121,Gasoline,1.545\nHyundai ix35,3J1020O,63.14,121,Gasoline,4.839\nHyundai ix35,3J1020O,45.69,121,Gasoline,3.9351\nHyundai ix35,3J1020O,27.19,121,Gasoline,2.895\nHyundai ix35,3J1020O,46.73,121,Gasoline,4.7861\nHyundai ix35,3J1020O,227.83,121,Gasoline,21.4878\nHyundai ix35,3J1020O,151.65,121,Gasoline,15.1964\nHyundai ix35,3J1020O,124.24,121,Gasoline,11.8014\nHyundai ix35,3J1020O,57.33,121,Gasoline,5.8481\nHyundai ix35,3J1020O,114.7,121,Gasoline,10.945\nHyundai ix35,3J1020O,232.36,121,Gasoline,15.4063\nHyundai ix35,3J1020O,7.68,121,Gasoline,0.8368\nHyundai ix35,3J1020O,124.38,121,Gasoline,11.1704\nHyundai ix35,3J1020O,23.8,121,Gasoline,1.9506\nHyundai ix35,3J1020O,133.65,121,Gasoline,9.6868\nHyundai ix35,3J1020O,87.73,121,Gasoline,8.1607\nHyundai ix35,3J1020O,14.78,121,Gasoline,1.5056\nHyundai ix35,3J1020O,8.17,121,Gasoline,0.8678\nHyundai ix35,3J1020O,6.48,121,Gasoline,0.6593\nHyundai ix35,3J1020O,6.88,121,Gasoline,0.898\nHyundai ix35,3J1020O,612.65,121,Gasoline,52.1834\nHyundai ix35,3J1020O,156.66,121,Gasoline,13.9236\nHyundai ix35,3J1020O,106.69,121,Gasoline,9.2914\nHyundai ix35,3J1020O,39.58,121,Gasoline,2.8921\nHyundai ix35,3J1020O,11.04,121,Gasoline,1.4995\nHyundai ix35,3J1020O,12.01,121,Gasoline,1.3473\nHyundai ix35,3J1020O,9.57,121,Gasoline,1.3864\nHyundai ix35,3J1020O,13.17,121,Gasoline,1.5398\nHyundai ix35,3J1020O,359.63,121,Gasoline,34.2248\nHyundai ix35,3J1020O,231.07,121,Gasoline,20.2596\nHyundai ix35,3J1020O,14.8,121,Gasoline,1.3858\nHyundai ix35,3J1020O,31.87,121,Gasoline,2.5481\nHyundai ix35,3J1020O,47.22,121,Gasoline,4.1133\nHyundai ix35,3J1020O,5.0,121,Gasoline,0.568\nHyundai ix35,3J1020O,5.07,121,Gasoline,0.6657\nHyundai ix35,3J1020O,15.6,121,Gasoline,1.6908\nHyundai ix35,3J1020O,72.23,121,Gasoline,5.1381\nHyundai ix35,3J1020O,145.69,121,Gasoline,12.5644\nHyundai ix35,3J1020O,2.79,121,Gasoline,0.2914\nHyundai ix35,3J1020O,116.99,121,Gasoline,11.2903\nHyundai ix35,3J1020O,136.35,121,Gasoline,11.3946\nHyundai ix35,3J1020O,75.71,121,Gasoline,6.8158\nHyundai ix35,3J1020O,257.74,121,Gasoline,19.7827\nHyundai ix35,3J1020O,4.54,121,Gasoline,0.5051\nHyundai ix35,3J1020O,10.72,121,Gasoline,0.9803\nHyundai ix35,3J1020O,5.28,121,Gasoline,0.718\nHyundai ix35,3J1020O,24.03,121,Gasoline,2.0642\nHyundai ix35,3J1020O,3.5,121,Gasoline,0.3912\nHyundai ix35,3J1020O,107.0,121,Gasoline,9.2243\nHyundai ix35,3J1020O,25.81,121,Gasoline,2.2745\nHyundai ix35,3J1020O,57.22,121,Gasoline,4.46\nHyundai ix35,3J1020O,18.86,121,Gasoline,1.8849\nHyundai ix35,3J1020O,17.84,121,Gasoline,1.7054\nHyundai ix35,3J1020O,30.13,121,Gasoline,2.8755\nHyundai ix35,3J1020O,18.3,121,Gasoline,1.8643\nHyundai ix35,3J1020O,15.96,121,Gasoline,1.8144\nHyundai ix35,3J1020O,28.64,121,Gasoline,2.3849\nHyundai ix35,3J1020O,35.89,121,Gasoline,2.7521\nHyundai ix35,3J1020O,35.0,121,Gasoline,3.0176\nHyundai ix35,3J1020O,12.56,121,Gasoline,1.4766\nHyundai ix35,3J1020O,2.54,121,Gasoline,0.271\nHyundai ix35,3J1020O,3.47,121,Gasoline,0.4605\nHyundai ix35,3J1020O,11.11,121,Gasoline,1.15\nHyundai ix35,3J1020O,213.03,121,Gasoline,16.9291\nHyundai ix35,3J1020O,48.32,121,Gasoline,3.8201\nHyundai ix35,3J1020O,198.49,121,Gasoline,14.8457\nHyundai ix35,3J1020O,1.93,121,Gasoline,0.1995\nHyundai ix35,3J1020O,51.16,121,Gasoline,4.125\nHyundai ix35,3J1020O,43.37,121,Gasoline,3.4481\nHyundai ix35,3J1020O,38.08,121,Gasoline,3.4371\nHyundai ix35,3J1020O,38.52,121,Gasoline,3.5434\nHyundai ix35,3J1020O,77.12,121,Gasoline,6.1799\nHyundai ix35,3J1020O,48.63,121,Gasoline,3.5019\nHyundai ix35,3J1020O,3.38,121,Gasoline,0.3095\nHyundai ix35,3J1020O,15.54,121,Gasoline,1.3567\nHyundai ix35,3J1020O,117.52,121,Gasoline,10.6032\nHyundai ix35,3J1020O,3.08,121,Gasoline,0.4176\nHyundai ix35,3J1020O,113.28,121,Gasoline,9.7085\nHyundai ix35,3J1020O,36.04,121,Gasoline,3.0467\nHyundai ix35,3J1020O,4.41,121,Gasoline,0.4319\nHyundai ix35,3J1020O,2.52,121,Gasoline,0.255\nHyundai ix35,3J1020O,30.48,121,Gasoline,3.123\nHyundai ix35,3J1020O,14.97,121,Gasoline,1.4133\nHyundai ix35,3J1020O,51.07,121,Gasoline,4.1295\nHyundai ix35,3J1020O,18.51,121,Gasoline,1.5612\nHyundai ix35,3J1020O,2.91,121,Gasoline,0.461\nHyundai ix35,3J1020O,15.35,121,Gasoline,1.4228\nHyundai ix35,3J1020O,37.38,121,Gasoline,2.9752\nHyundai ix35,3J1020O,26.0,121,Gasoline,2.5063\nHyundai ix35,3J1020O,49.46,121,Gasoline,3.6811\nHyundai ix35,3J1020O,15.12,121,Gasoline,1.569\nHyundai ix35,3J1020O,38.05,121,Gasoline,3.7995\nHyundai ix35,3J1020O,3.76,121,Gasoline,0.4339\nHyundai ix35,3J1020O,5.01,121,Gasoline,0.5502\nHyundai ix35,3J1020O,2.54,121,Gasoline,0.3219\nHyundai ix35,3J1020O,27.56,121,Gasoline,2.7754\nHyundai ix35,3J1020O,17.15,121,Gasoline,2.1684\nHyundai ix35,3J1020O,44.51,121,Gasoline,4.5932\nHyundai ix35,3J1020O,39.22,121,Gasoline,2.8186\nHyundai ix35,3J1020O,45.6,121,Gasoline,3.4045\nHyundai ix35,3J1020O,5.2,121,Gasoline,0.5871\nHyundai ix35,3J1020O,152.16,121,Gasoline,10.7311\nHyundai ix35,3J1020O,22.81,121,Gasoline,2.8907\nHyundai ix35,3J1020O,183.3,121,Gasoline,15.1808\nHyundai ix35,3J1020O,35.44,121,Gasoline,2.8241\nHyundai ix35,3J1020O,2.23,121,Gasoline,0.2715\nHyundai ix35,3J1020O,195.25,121,Gasoline,15.1508\nHyundai ix35,3J1020O,49.2,121,Gasoline,4.0047\nHyundai ix35,3J1020O,286.0,121,Gasoline,20.5465\nHyundai ix35,3J1020O,34.22,121,Gasoline,2.7734\nHyundai ix35,3J1020O,40.83,121,Gasoline,3.4206\nHyundai ix35,3J1020O,78.94,121,Gasoline,6.4407\nHyundai ix35,3J1020O,7.58,121,Gasoline,0.6374\nHyundai ix35,3J1020O,26.93,121,Gasoline,1.702\nHyundai ix35,3J1020O,38.62,121,Gasoline,4.0429\nHyundai ix35,3J1020O,3.23,121,Gasoline,0.3683\nHyundai ix35,3J1020O,22.18,121,Gasoline,1.7887\nHyundai ix35,3J1020O,3.84,121,Gasoline,0.5292\nHyundai ix35,3J1020O,23.18,121,Gasoline,2.0351\nHyundai ix35,3J1020O,23.77,121,Gasoline,2.4778\nHyundai ix35,3J1020O,102.29,121,Gasoline,9.223\nHyundai ix35,3J1020O,27.05,121,Gasoline,2.5966\nHyundai ix35,3J1020O,31.05,121,Gasoline,2.7376\nHyundai ix35,3J1020O,15.26,121,Gasoline,1.664\nHyundai ix35,3J1020O,17.17,121,Gasoline,1.8654\nHyundai ix35,3J1020O,6.58,121,Gasoline,0.5945\nHyundai ix35,3J1020O,2.21,121,Gasoline,0.2664\nHyundai ix35,3J1020O,44.01,121,Gasoline,3.3115\nHyundai ix35,3J1020O,52.72,121,Gasoline,4.0483\nHyundai ix35,3J1020O,14.91,121,Gasoline,1.4957\nHyundai ix35,3J1020O,7.68,121,Gasoline,0.8306\nHyundai ix35,3J1020O,9.16,121,Gasoline,0.88\nHyundai ix35,3J1020O,14.79,121,Gasoline,1.2087\nHyundai ix35,3J1020O,15.73,121,Gasoline,1.2148\nHyundai ix35,3J1020O,4.7,121,Gasoline,0.5513\nHyundai ix35,3J1020O,19.08,121,Gasoline,2.2128\nHyundai ix35,3J1020O,6.26,121,Gasoline,0.8039\nHyundai ix35,3J1020O,24.16,121,Gasoline,1.9409\nHyundai ix35,3J1020O,15.35,121,Gasoline,1.4898\nHyundai ix35,3J1020O,5.28,121,Gasoline,0.7016\nHyundai ix35,3J1020O,30.74,121,Gasoline,2.9649\nHyundai ix35,3J1020O,13.82,121,Gasoline,1.235\nHyundai ix35,3J1020O,156.29,121,Gasoline,13.3169\nHyundai ix35,3J1020O,148.94,121,Gasoline,12.8479\nHyundai ix35,3J1020O,2.74,121,Gasoline,0.3975\nHyundai ix35,3J1020O,149.42,121,Gasoline,13.011\nHyundai ix35,3J1020O,32.01,121,Gasoline,2.7051\nHyundai ix35,3J1020O,103.3,121,Gasoline,8.911\nHyundai ix35,3J1020O,119.67,121,Gasoline,9.5458\nHyundai ix35,3J1020O,238.07,121,Gasoline,17.6533\nHyundai ix35,3J1020O,33.7,121,Gasoline,2.4813\nHyundai ix35,3J1020O,3.52,121,Gasoline,0.5116\nHyundai ix35,3J1020O,9.0,121,Gasoline,0.6844\nHyundai ix35,3J1020O,30.7,121,Gasoline,2.5958\nHyundai ix35,3J1020O,2.08,121,Gasoline,0.2578\nHyundai ix35,3J1020O,7.19,121,Gasoline,0.8348\nHyundai ix35,3J1020O,109.92,121,Gasoline,9.5915\nHyundai ix35,3J1020O,54.58,121,Gasoline,4.2214\nHyundai ix35,3J1020O,176.5,121,Gasoline,14.9044\nHyundai ix35,3J1020O,50.84,121,Gasoline,3.831\nHyundai ix35,3J1020O,38.38,121,Gasoline,3.175\nHyundai ix35,3J1020O,68.55,121,Gasoline,5.6283\nHyundai ix35,3J1020O,110.29,121,Gasoline,8.9859\nHyundai ix35,3J1020O,26.76,121,Gasoline,2.4281\nHyundai ix35,3J1020O,4.46,121,Gasoline,0.3864\nHyundai ix35,3J1020O,95.29,121,Gasoline,8.0622\nHyundai ix35,3J1020O,121.87,121,Gasoline,9.3858\nHyundai ix35,3J1020O,179.72,121,Gasoline,12.4657\nHyundai ix35,3J1020O,6.09,121,Gasoline,0.7912\nHyundai ix35,3J1020O,15.81,121,Gasoline,1.5232\nHyundai ix35,3J1020O,4.23,121,Gasoline,0.471\nHyundai ix35,3J1020O,7.97,121,Gasoline,0.813\nHyundai ix35,3J1020O,10.18,121,Gasoline,1.0575\nHyundai ix35,3J1020O,32.54,121,Gasoline,2.5309\nHyundai ix35,3J1020O,4.02,121,Gasoline,0.5799\nHyundai ix35,3J1020O,3.26,121,Gasoline,0.3893\nHyundai ix35,3J1020O,3.86,121,Gasoline,0.4745\nHyundai ix35,3J1020O,88.83,121,Gasoline,7.8857\nHyundai ix35,3J1020O,6.04,121,Gasoline,0.4933\nHyundai ix35,3J1020O,13.2,121,Gasoline,1.1717\nHyundai ix35,3J1020O,33.25,121,Gasoline,3.3127\nHyundai ix35,3J1020O,4.7,121,Gasoline,0.4919\nHyundai ix35,3J1020O,5.31,121,Gasoline,0.7054\nHyundai ix35,3J1020O,46.78,121,Gasoline,3.4326\nHyundai ix35,3J1020O,4.3,121,Gasoline,0.5659\nHyundai ix35,3J1020O,41.56,121,Gasoline,4.0622\nHyundai ix35,3J1020O,3.72,121,Gasoline,0.5347\nHyundai ix35,3J1020O,22.72,121,Gasoline,2.3646\nHyundai ix35,3J1020O,21.78,121,Gasoline,1.7506\nHyundai ix35,3J1020O,25.05,121,Gasoline,1.8563\nHyundai ix35,3J1020O,13.36,121,Gasoline,1.344\nHyundai ix35,3J1020O,15.38,121,Gasoline,1.0943\nHyundai ix35,3J1020O,4.82,121,Gasoline,0.5932\nHyundai ix35,3J1020O,4.82,121,Gasoline,0.6114\nHyundai ix35,3J1020O,59.73,121,Gasoline,5.12\nHyundai ix35,3J1020O,45.12,121,Gasoline,3.7177\nHyundai ix35,3J1020O,5.14,121,Gasoline,0.6629\nHyundai ix35,3J1020O,43.71,121,Gasoline,3.4704\nHyundai ix35,3J1020O,34.0,121,Gasoline,2.8047\nHyundai ix35,3J1020O,5.95,121,Gasoline,0.5968\nHyundai ix35,3J1020O,22.35,121,Gasoline,1.8098\nHyundai ix35,3J1020O,5.31,121,Gasoline,0.6328\nHyundai ix35,3J1020O,15.29,121,Gasoline,1.4877\nHyundai ix35,3J1020O,6.89,121,Gasoline,0.8708\nHyundai ix35,3J1020O,4.43,121,Gasoline,0.6457\nHyundai ix35,3J1020O,137.62,121,Gasoline,10.6713\nHyundai ix35,3J1020O,1.84,121,Gasoline,0.2372\nHyundai ix35,3J1020O,76.39,121,Gasoline,6.4598\nHyundai ix35,3J1020O,64.95,121,Gasoline,5.7835\nHyundai ix35,3J1020O,23.23,121,Gasoline,2.1942\nHyundai ix35,3J1020O,38.28,121,Gasoline,2.9922\nHyundai ix35,3J1020O,3.11,121,Gasoline,0.4502\nHyundai ix35,3J1020O,31.35,121,Gasoline,2.3879\nHyundai ix35,3J1020O,205.21,121,Gasoline,14.6456\nHyundai ix35,3J1020O,153.37,121,Gasoline,12.9457\nHyundai ix35,3J1020O,8.95,121,Gasoline,1.0556\nHyundai ix35,3J1020O,6.17,121,Gasoline,0.9104\nHyundai ix35,3J1020O,145.9,121,Gasoline,13.2126\nHyundai ix35,3J1020O,87.15,121,Gasoline,7.6769\nHyundai ix35,3J1020O,84.04,121,Gasoline,7.7693\nHyundai ix35,3J1020O,3.37,121,Gasoline,0.4413\nHyundai ix35,3J1020O,105.06,121,Gasoline,8.119\nHyundai ix35,3J1020O,104.77,121,Gasoline,7.825\nHyundai ix35,3J1020O,45.51,121,Gasoline,3.4847\nHyundai ix35,3J1020O,3.44,121,Gasoline,0.5901\nHyundai ix35,3J1020O,30.08,121,Gasoline,3.0565\nHyundai ix35,3J1020O,15.23,121,Gasoline,1.5324\nHyundai ix35,3J1020O,2.22,121,Gasoline,0.4007\nHyundai ix35,3J1020O,14.86,121,Gasoline,1.281\nHyundai ix35,3J1020O,50.98,121,Gasoline,4.1469\nHyundai ix35,3J1020O,49.26,121,Gasoline,4.1758\nHyundai ix35,3J1020O,16.2,121,Gasoline,1.852\nHyundai ix35,3J1020O,43.59,121,Gasoline,3.3277\nHyundai ix35,3J1020O,50.19,121,Gasoline,3.4828\nHyundai ix35,3J1020O,4.94,121,Gasoline,0.6084\nHyundai ix35,3J1020O,3.74,121,Gasoline,0.7383\nHyundai ix35,3J1020O,2.27,121,Gasoline,0.3003\nHyundai ix35,3J1020O,48.94,121,Gasoline,3.9783\nHyundai ix35,3J1020O,31.73,121,Gasoline,2.4366\nHyundai ix35,3J1020O,117.21,121,Gasoline,9.9568\nHyundai ix35,3J1020O,159.21,121,Gasoline,14.4043\nHyundai ix35,3J1020O,5.19,121,Gasoline,0.7525\nHyundai ix35,3J1020O,31.61,121,Gasoline,2.8845\nHyundai ix35,3J1020O,163.69,121,Gasoline,14.3976\nHyundai ix35,3J1020O,57.59,121,Gasoline,4.8175\nHyundai ix35,3J1020O,3.05,121,Gasoline,0.2991\nHyundai ix35,3J1020O,162.37,121,Gasoline,12.735\nHyundai ix35,3J1020O,249.12,121,Gasoline,19.2843\nHyundai ix35,3J1020O,193.46,121,Gasoline,15.5408\nHyundai ix35,3J1020O,5.8,121,Gasoline,0.7039\nHyundai ix35,3J1020O,231.95,121,Gasoline,19.688\nHyundai ix35,3J1020O,209.2,121,Gasoline,17.244\nHyundai ix35,3J1020O,280.97,121,Gasoline,22.7751\nHyundai ix35,3J1020O,46.19,121,Gasoline,3.564\nHyundai ix35,3J1020O,15.14,121,Gasoline,2.0072\nHyundai ix35,3J1020O,17.95,121,Gasoline,1.7235\nHyundai ix35,3J1020O,3.71,121,Gasoline,0.4627\nHyundai ix35,3J1020O,6.0,121,Gasoline,0.7967\nHyundai ix35,3J1020O,115.5,121,Gasoline,9.2385\nHyundai ix35,3J1020O,108.24,121,Gasoline,9.7288\nHyundai ix35,3J1020O,46.86,121,Gasoline,3.9632\nHyundai ix35,3J1020O,136.72,121,Gasoline,11.6225\nHyundai ix35,3J1020O,139.27,121,Gasoline,10.2304\nHyundai ix35,3J1020O,7.06,121,Gasoline,1.1\nHyundai ix35,3J1020O,120.66,121,Gasoline,10.4874\nHyundai ix35,3J1020O,146.7,121,Gasoline,12.7382\nHyundai ix35,3J1020O,10.28,121,Gasoline,1.3436\nHyundai ix35,3J1020O,9.74,121,Gasoline,1.204\nHyundai ix35,3J1020O,200.66,121,Gasoline,18.5976\nHyundai ix35,3J1020O,16.06,121,Gasoline,1.5371\nHyundai ix35,3J1020O,237.93,121,Gasoline,21.0145\nHyundai ix35,3J1020O,18.09,121,Gasoline,2.0143\nHyundai ix35,3J1020O,5.24,121,Gasoline,0.7567\nHyundai ix35,3J1020O,8.09,121,Gasoline,0.8352\nHyundai ix35,3J1020O,51.4,121,Gasoline,4.6692\nHyundai ix35,3J1020O,14.7,121,Gasoline,1.1086\nHyundai ix35,3J1020O,27.96,121,Gasoline,2.1923\nHyundai ix35,3J1020O,147.0,121,Gasoline,12.923\nHyundai ix35,3J1020O,154.97,121,Gasoline,13.4486\nHyundai ix35,3J1020O,6.02,121,Gasoline,0.8112\nHyundai ix35,3J1020O,282.43,121,Gasoline,22.5423\nHyundai ix35,3J1020O,49.28,121,Gasoline,4.1992\nHyundai ix35,3J1020O,5.39,121,Gasoline,0.629\nHyundai ix35,3J1020O,195.48,121,Gasoline,16.2912\nHyundai ix35,3J1020O,31.0,121,Gasoline,3.1265\nHyundai ix35,3J1020O,45.34,121,Gasoline,3.9639\nHyundai ix35,3J1020O,5.81,121,Gasoline,0.6315\nHyundai ix35,3J1020O,11.45,121,Gasoline,0.9134\nHyundai ix35,3J1020O,20.06,121,Gasoline,2.1814\nHyundai ix35,3J1020O,6.74,121,Gasoline,0.6544\nHyundai ix35,3J1020O,39.48,121,Gasoline,4.7622\nHyundai ix35,3J1020O,15.68,121,Gasoline,1.6177\nHyundai ix35,3J1020O,3.26,121,Gasoline,0.4809\nHyundai ix35,3J1020O,23.19,121,Gasoline,2.2966\nHyundai ix35,3J1020O,148.15,121,Gasoline,13.4098\nHyundai ix35,3J1020O,90.34,121,Gasoline,7.3386\nHyundai ix35,3J1020O,316.99,121,Gasoline,32.4718\nHyundai ix35,3J1020O,13.56,121,Gasoline,1.3377\nHyundai ix35,3J1020O,223.14,121,Gasoline,19.0603\nHyundai ix35,3J1020O,186.26,121,Gasoline,16.579\nHyundai ix35,3J1020O,37.77,121,Gasoline,2.7126\nHyundai ix35,3J1020O,4.32,121,Gasoline,0.7076\nHyundai ix35,3J1020O,270.96,121,Gasoline,21.6879\nHyundai ix35,3J1020O,49.9,121,Gasoline,4.4286\nHyundai ix35,3J1020O,190.53,121,Gasoline,17.005\nHyundai ix35,3J1020O,310.4,121,Gasoline,22.2454\nHyundai ix35,3J1020O,137.24,121,Gasoline,12.7824\nHyundai ix35,3J1020O,135.3,121,Gasoline,13.4387\nHyundai ix35,3J1020O,45.32,121,Gasoline,4.3654\nHyundai ix35,3J1020O,16.57,121,Gasoline,1.4544\nHyundai ix35,3J1020O,86.42,121,Gasoline,6.8607\nHyundai ix35,3J1020O,226.2,121,Gasoline,19.8782\nHyundai ix35,3J1020O,414.73,121,Gasoline,34.7721\nHyundai ix35,3J1020O,3.92,121,Gasoline,0.775\nHyundai ix35,3J1020O,15.65,121,Gasoline,1.3506\nHyundai ix35,3J1020O,46.06,121,Gasoline,4.1878\nHyundai ix35,3J1020O,10.74,121,Gasoline,1.5613\nHyundai ix35,3J1020O,18.2,121,Gasoline,1.5913\nHyundai ix35,3J1020O,15.56,121,Gasoline,1.4745\nHyundai ix35,3J1020O,31.41,121,Gasoline,2.9664\nHyundai ix35,3J1020O,28.44,121,Gasoline,2.8419\nHyundai ix35,3J1020O,5.72,121,Gasoline,0.8609\nHyundai ix35,3J1020O,5.3,121,Gasoline,0.6666\nHyundai ix35,3J1020O,4.38,121,Gasoline,0.6117\nHyundai ix35,3J1020O,34.75,121,Gasoline,3.4221\nHyundai ix35,3J1020O,2.64,121,Gasoline,0.3617\nHyundai ix35,3J1020O,54.43,121,Gasoline,4.5748\nHyundai ix35,3J1020O,8.89,121,Gasoline,0.7463\nHyundai ix35,3J1020O,15.24,121,Gasoline,1.4342\nHyundai ix35,3J1020O,273.74,121,Gasoline,24.2325\nHyundai ix35,3J1020O,121.39,121,Gasoline,12.758\nHyundai ix35,3J1020O,170.98,121,Gasoline,12.2685\nHyundai ix35,3J1020O,35.47,121,Gasoline,2.342\nHyundai ix35,3J1020O,148.56,121,Gasoline,11.359\nHyundai ix35,3J1020O,58.0,121,Gasoline,4.2336\nHyundai ix35,3J1020O,190.13,121,Gasoline,15.4043\nHyundai ix35,3J1020O,44.81,121,Gasoline,3.8231\nHyundai ix35,3J1020O,3.99,121,Gasoline,0.5071\nHyundai ix35,3J1020O,14.66,121,Gasoline,1.1488\nHyundai ix35,3J1020O,7.79,121,Gasoline,1.1271\nHyundai ix35,3J1020O,259.08,121,Gasoline,22.433\nHyundai ix35,3J1020O,19.82,121,Gasoline,1.7445\nHyundai ix35,3J1020O,115.68,121,Gasoline,8.8001\nHyundai ix35,3J1020O,38.89,121,Gasoline,3.0974\nHyundai ix35,3J1020O,116.72,121,Gasoline,9.4337\nHyundai ix35,3J1020O,110.49,121,Gasoline,8.1404\nHyundai ix35,3J1020O,126.39,121,Gasoline,10.6214\nHyundai ix35,3J1020O,23.65,121,Gasoline,2.4713\nHyundai ix35,3J1020O,42.28,121,Gasoline,4.4847\nHyundai ix35,3J1020O,8.05,121,Gasoline,1.1594\nHyundai ix35,3J1020O,35.32,121,Gasoline,2.4147\nHyundai ix35,3J1020O,195.05,121,Gasoline,14.6164\nHyundai ix35,3J1020O,5.57,121,Gasoline,0.823\nHyundai ix35,3J1020O,187.36,121,Gasoline,17.4302\nHyundai ix35,3J1020O,16.96,121,Gasoline,2.3881\nHyundai ix35,3J1020O,200.09,121,Gasoline,13.7864\nHyundai ix35,3J1020O,175.83,121,Gasoline,14.9879\nHyundai ix35,3J1020O,121.96,121,Gasoline,11.1336\nHyundai ix35,3J1020O,2.42,121,Gasoline,0.3619\nHyundai ix35,3J1020O,3.29,121,Gasoline,0.4878\nHyundai ix35,3J1020O,15.11,121,Gasoline,1.2548\nHyundai ix35,3J1020O,39.12,121,Gasoline,3.7156\nHyundai ix35,3J1020O,19.76,121,Gasoline,1.974\nHyundai ix35,3J1020O,183.08,121,Gasoline,15.6532\nHyundai ix35,3J1020O,136.44,121,Gasoline,12.2857\nHyundai ix35,3J1020O,4.86,121,Gasoline,0.54\nHyundai ix35,3J1020O,131.6,121,Gasoline,12.3489\nHyundai ix35,3J1020O,54.27,121,Gasoline,4.966\nHyundai ix35,3J1020O,236.97,121,Gasoline,20.2004\nHyundai ix35,3J1020O,138.44,121,Gasoline,11.9209\nHyundai ix35,3J1020O,12.54,121,Gasoline,1.7343\nHyundai ix35,3J1020O,31.54,121,Gasoline,5.3039\nHyundai ix35,3J1020O,212.88,121,Gasoline,17.9952\nHyundai ix35,3J1020O,111.24,121,Gasoline,8.5181\nHyundai ix35,3J1020O,16.37,121,Gasoline,1.7705\nHyundai ix35,3J1020O,32.4,121,Gasoline,2.7943\nHyundai ix35,3J1020O,17.31,121,Gasoline,1.4666\nHyundai ix35,3J1020O,183.49,121,Gasoline,15.6373\nHyundai ix35,3J1020O,83.17,121,Gasoline,7.211\nHyundai ix35,3J1020O,14.65,121,Gasoline,1.3008\nHyundai ix35,3J1020O,18.34,121,Gasoline,2.0355\nHyundai ix35,3J1020O,39.96,121,Gasoline,3.7856\nHyundai ix35,3J1020O,2.64,121,Gasoline,0.3079\nHyundai ix35,3J1020O,37.91,121,Gasoline,3.182\nHyundai ix35,3J1020O,78.78,121,Gasoline,5.9305\nHyundai ix35,3J1020O,18.86,121,Gasoline,1.7437\nHyundai ix35,3J1020O,31.88,121,Gasoline,2.0389\nHyundai ix35,3J1020O,163.68,121,Gasoline,12.083\nHyundai ix35,3J1020O,173.22,121,Gasoline,12.5281\nHyundai ix35,3J1020O,7.19,121,Gasoline,0.7734\nHyundai ix35,3J1020O,117.47,121,Gasoline,9.1085\nHyundai ix35,3J1020O,67.29,121,Gasoline,5.2171\nHyundai ix35,3J1020O,58.87,121,Gasoline,4.5091\nHyundai ix35,3J1020O,58.49,121,Gasoline,4.8472\nHyundai ix35,3J1020O,26.87,121,Gasoline,2.9313\nHyundai ix35,3J1020O,120.15,121,Gasoline,11.2729\nHyundai ix35,3J1020O,143.94,121,Gasoline,12.8986\nHyundai ix35,3J1020O,43.35,121,Gasoline,3.6228\nHyundai ix35,3J1020O,44.62,121,Gasoline,3.6474\nHyundai ix35,3J1020O,131.86,121,Gasoline,12.4374\nHyundai ix35,3J1020O,38.26,121,Gasoline,3.5667\nHyundai ix35,3J1020O,12.55,121,Gasoline,1.3915\nHyundai ix35,3J1020O,3.52,121,Gasoline,0.5615\nHyundai ix35,3J1020O,23.44,121,Gasoline,2.3314\nHyundai ix35,3J1020O,11.93,121,Gasoline,1.1237\nHyundai ix35,3J1020O,4.95,121,Gasoline,0.6868\nHyundai ix35,3J1020O,14.73,121,Gasoline,1.1157\nHyundai ix35,3J1020O,4.58,121,Gasoline,0.6331\nHyundai ix35,3J1020O,10.71,121,Gasoline,0.8496\nHyundai ix35,3J1020O,2.41,121,Gasoline,0.266\nHyundai ix35,3J1020O,179.19,121,Gasoline,15.3575\nHyundai ix35,3J1020O,165.33,121,Gasoline,14.3004\nHyundai ix35,3J1020O,20.4,121,Gasoline,2.2401\nHyundai ix35,3J1020O,16.69,121,Gasoline,1.7033\nHyundai ix35,3J1020O,118.85,121,Gasoline,8.6161\nHyundai ix35,3J1020O,69.68,121,Gasoline,5.8425\nHyundai ix35,3J1020O,2.08,121,Gasoline,0.2783\nHyundai ix35,3J1020O,33.26,121,Gasoline,2.4714\nHyundai ix35,3J1020O,22.77,121,Gasoline,1.5898\nHyundai ix35,3J1020O,108.14,121,Gasoline,8.9649\nHyundai ix35,3J1020O,124.44,121,Gasoline,10.6295\nHyundai ix35,3J1020O,181.51,121,Gasoline,14.8941\nHyundai ix35,3J1020O,118.48,121,Gasoline,8.6318\nHyundai ix35,3J1020O,18.04,121,Gasoline,1.9829\nHyundai ix35,3J1020O,16.29,121,Gasoline,1.3607\nHyundai ix35,3J1020O,30.06,121,Gasoline,2.5662\nHyundai ix35,3J1020O,1.62,121,Gasoline,0.2041\nHyundai ix35,3J1020O,116.47,121,Gasoline,10.3183\nHyundai ix35,3J1020O,17.99,121,Gasoline,1.7098\nHyundai ix35,3J1020O,14.84,121,Gasoline,1.3427\nHyundai ix35,3J1020O,129.1,121,Gasoline,9.9068\nHyundai ix35,3J1020O,68.23,121,Gasoline,5.3869\nHyundai ix35,3J1020O,10.7,121,Gasoline,1.0852\nHyundai ix35,3J1020O,237.38,121,Gasoline,19.6258\nHyundai ix35,3J1020O,80.0,121,Gasoline,6.2625\nHyundai ix35,3J1020O,17.22,121,Gasoline,1.4946\nHyundai ix35,3J1020O,111.13,121,Gasoline,8.4666\nHyundai ix35,3J1020O,241.82,121,Gasoline,19.7483\nHyundai ix35,3J1020O,45.6,121,Gasoline,3.8435\nHyundai ix35,3J1020O,190.47,121,Gasoline,14.5427\nHyundai ix35,3J1020O,143.57,121,Gasoline,10.6682\nHyundai ix35,3J1020O,2.55,121,Gasoline,0.4175\nHyundai ix35,3J1020O,491.39,121,Gasoline,38.0902\nHyundai ix35,3J1020O,151.15,121,Gasoline,13.1746\nHyundai ix35,3J1020O,121.29,121,Gasoline,9.7548\nHyundai ix35,3J1020O,182.19,121,Gasoline,16.7072\nHyundai ix35,3J1020O,121.41,121,Gasoline,9.2559\nHyundai ix35,3J1020O,164.65,121,Gasoline,14.1007\nHyundai ix35,3J1020O,224.59,121,Gasoline,19.4192\nHyundai ix35,3J1020O,59.08,121,Gasoline,4.9317\nHyundai ix35,3J1020O,102.47,121,Gasoline,7.8718\nHyundai ix35,3J1020O,169.95,121,Gasoline,13.8372\nHyundai ix35,3J1020O,8.09,121,Gasoline,0.7761\nHyundai ix35,3J1020O,102.69,121,Gasoline,8.6207\nHyundai ix35,3J1020O,132.1,121,Gasoline,10.0879\nHyundai ix35,3J1020O,141.81,121,Gasoline,10.2579\nHyundai ix35,3J1020O,56.54,121,Gasoline,4.871\nHyundai ix35,3J1020O,85.67,121,Gasoline,5.8636\nHyundai ix35,3J1020O,201.51,121,Gasoline,16.6553\nHyundai ix35,3J1020O,78.45,121,Gasoline,8.3855\nHyundai ix35,3J1020O,7.31,121,Gasoline,1.1394\nHyundai ix35,3J1020O,54.91,121,Gasoline,5.3218\nHyundai ix35,3J1020O,10.4,121,Gasoline,0.7821\nHyundai ix35,3J1020O,232.2,121,Gasoline,18.561\nHyundai ix35,3J1020O,285.15,121,Gasoline,21.454\nHyundai ix35,3J1020O,155.71,121,Gasoline,14.389\nHyundai ix35,3J1020O,145.74,121,Gasoline,12.4071\nHyundai ix35,3J1020O,75.76,121,Gasoline,6.9926\nHyundai ix35,3J1020O,33.92,121,Gasoline,2.9405\nHyundai ix35,3J1020O,36.07,121,Gasoline,2.9311\nHyundai ix35,3J1020O,30.8,121,Gasoline,2.3753\nHyundai ix35,3J1020O,2.98,121,Gasoline,0.4329\nHyundai ix35,3J1020O,30.32,121,Gasoline,2.3338\nHyundai ix35,3J1020O,18.82,121,Gasoline,1.4583\nHyundai ix35,3J1020O,121.74,121,Gasoline,8.7966\nHyundai ix35,3J1020O,35.44,121,Gasoline,3.0093\nHyundai ix35,3J1020O,5.45,121,Gasoline,0.5662\nHyundai ix35,3J1020O,11.36,121,Gasoline,0.9326\nHyundai ix35,3J1020O,15.55,121,Gasoline,1.2473\nHyundai ix35,3J1020O,188.46,121,Gasoline,13.822\nHyundai ix35,3J1020O,50.22,121,Gasoline,4.1892\nHyundai ix35,3J1020O,255.03,121,Gasoline,20.5121\nHyundai ix35,3J1020O,11.71,121,Gasoline,1.0047\nHyundai ix35,3J1020O,257.51,121,Gasoline,21.291\nHyundai ix35,3J1020O,30.87,121,Gasoline,3.2083\nHyundai ix35,3J1020O,3.7,121,Gasoline,0.3315\nHyundai ix35,3J1020O,2.51,121,Gasoline,0.3733\nHyundai ix35,3J1020O,128.96,121,Gasoline,12.4559\nHyundai ix35,3J1020O,132.89,121,Gasoline,10.3537\nHyundai ix35,3J1020O,32.45,121,Gasoline,4.0432\nHyundai ix35,3J1020O,4.82,121,Gasoline,0.7181\nHyundai ix35,3J1020O,5.54,121,Gasoline,0.8514\nHyundai ix35,3J1020O,11.2,121,Gasoline,1.6213\nHyundai ix35,3J1020O,42.7,121,Gasoline,3.625\nHyundai ix35,3J1020O,1.6,121,Gasoline,0.1568\nHyundai ix35,3J1020O,34.97,121,Gasoline,2.7206\nHyundai ix35,3J1020O,18.01,121,Gasoline,1.7756\nHyundai ix35,3J1020O,18.85,121,Gasoline,1.6335\nHyundai ix35,3J1020O,4.63,121,Gasoline,0.6959\nHyundai ix35,3J1020O,5.81,121,Gasoline,0.7596\nHyundai ix35,3J1020O,23.36,121,Gasoline,2.1208\nHyundai ix35,3J1020O,37.63,121,Gasoline,3.3883\nHyundai ix35,3J1020O,134.51,121,Gasoline,10.9143\nHyundai ix35,3J1020O,48.62,121,Gasoline,3.8541\nHyundai ix35,3J1020O,13.16,121,Gasoline,1.4056\nHyundai ix35,3J1020O,18.08,121,Gasoline,1.7952\nHyundai ix35,3J1020O,12.58,121,Gasoline,1.4233\nHyundai ix35,3J1020O,14.27,121,Gasoline,1.0502\nHyundai ix35,3J1020O,18.84,121,Gasoline,1.3738\nHyundai ix35,3J1020O,4.41,121,Gasoline,0.4864\nHyundai ix35,3J1020O,21.18,121,Gasoline,2.3697\nHyundai ix35,3J1020O,17.97,121,Gasoline,2.1488\nHyundai ix35,3J1020O,29.28,121,Gasoline,2.7534\nHyundai ix35,3J1020O,18.84,121,Gasoline,1.4769\nHyundai ix35,3J1020O,12.49,121,Gasoline,1.4723\nHyundai ix35,3J1020O,3.76,121,Gasoline,0.4053\nHyundai ix35,3J1020O,2.44,121,Gasoline,0.2939\nHyundai ix35,3J1020O,28.59,121,Gasoline,2.0751\nHyundai ix35,3J1020O,16.5,121,Gasoline,1.6124\nHyundai ix35,3J1020O,18.85,121,Gasoline,1.5873\nHyundai ix35,3J1020O,13.36,121,Gasoline,1.2359\nHyundai ix35,3J1020O,42.98,121,Gasoline,3.5448\nHyundai ix35,3J1020O,42.25,121,Gasoline,3.2812\nHyundai ix35,3J1020O,4.84,121,Gasoline,0.9411\nHyundai ix35,3J1020O,11.58,121,Gasoline,1.0156\nHyundai ix35,3J1020O,159.36,121,Gasoline,13.3218\nHyundai ix35,3J1020O,81.56,121,Gasoline,6.706\nHyundai ix35,3J1020O,265.22,121,Gasoline,21.9808\nHyundai ix35,3J1020O,23.71,121,Gasoline,2.5381\nHyundai ix35,3J1020O,5.28,121,Gasoline,0.6685\nHyundai ix35,3J1020O,4.59,121,Gasoline,0.6469\nHyundai ix35,3J1020O,15.14,121,Gasoline,1.4288\nHyundai ix35,3J1020O,75.92,121,Gasoline,7.5968\nHyundai ix35,3J1020O,38.48,121,Gasoline,3.3602\nHyundai ix35,3J1020O,106.97,121,Gasoline,7.5975\nHyundai ix35,3J1020O,14.83,121,Gasoline,1.6278\nHyundai ix35,3J1020O,5.27,121,Gasoline,0.6888\nHyundai ix35,3J1020O,133.16,121,Gasoline,12.3228\nHyundai ix35,3J1020O,51.86,121,Gasoline,4.8824\nHyundai ix35,3J1020O,19.61,121,Gasoline,1.7187\nHyundai ix35,3J1020O,144.57,121,Gasoline,11.672\nHyundai ix35,3J1020O,15.25,121,Gasoline,1.2337\nHyundai ix35,3J1020O,139.0,121,Gasoline,10.588\nHyundai ix35,3J1020O,13.68,121,Gasoline,1.3053\nHyundai ix35,3J1020O,12.57,121,Gasoline,1.0217\nHyundai ix35,3J1020O,44.79,121,Gasoline,4.0896\nHyundai ix35,3J1020O,16.58,121,Gasoline,1.8234\nHyundai ix35,3J1020O,4.84,121,Gasoline,0.5828\nHyundai ix35,3J1020O,11.44,121,Gasoline,0.9603\nHyundai ix35,3J1020O,7.23,121,Gasoline,0.9129\nHyundai ix35,3J1020O,1.9,121,Gasoline,0.3486\nHyundai ix35,3J1020O,2.7,121,Gasoline,0.5592\nHyundai ix35,3J1020O,5.84,121,Gasoline,0.8806\nHyundai ix35,3J1020O,4.42,121,Gasoline,0.488\nNissan Teana,2X10829,166.29,110,Gasoline,14.9333\nNissan Teana,2X1080Q,32.13,104,Gasoline,3.0415\nNissan Teana,2X10829,85.9,110,Gasoline,6.3947\nNissan Teana,2X1080Q,1.6,104,Gasoline,0.2118\nNissan Teana,2X10829,8.77,110,Gasoline,1.096\nNissan Teana,2X10829,10.14,110,Gasoline,1.131\nNissan Teana,2X10829,38.62,110,Gasoline,2.688\nNissan Teana,2X1080Q,7.92,104,Gasoline,0.7534\nNissan Teana,2X10829,114.99,110,Gasoline,10.9968\nNissan Teana,2X1080Q,14.18,104,Gasoline,1.374\nNissan Teana,2X10829,102.86,110,Gasoline,6.3384\nNissan Teana,2X10829,26.23,110,Gasoline,2.8089\nNissan Teana,2X10829,3.14,110,Gasoline,0.4034\nNissan Teana,2X10829,28.77,110,Gasoline,3.1768\nNissan Teana,2X10829,123.05,110,Gasoline,10.0071\nNissan Teana,2X10829,65.28,110,Gasoline,5.711\nNissan Teana,2X10829,5.55,110,Gasoline,0.4924\nNissan Teana,2X1080Q,6.25,104,Gasoline,0.7266\nNissan Teana,2X10829,154.66,110,Gasoline,10.4608\nNissan Teana,2X10829,17.89,110,Gasoline,1.597\nNissan Teana,2X10829,32.58,110,Gasoline,3.2636\nNissan Teana,2X10829,2.4,110,Gasoline,0.3347\nNissan Teana,2X10829,132.4,110,Gasoline,11.3636\nNissan Teana,2X1080Q,46.42,104,Gasoline,3.9477\nNissan Teana,2X10829,69.55,110,Gasoline,7.0511\nNissan Teana,2X1080Q,12.71,104,Gasoline,1.4462\nNissan Teana,2X1080Q,10.92,104,Gasoline,0.9886\nNissan Teana,2X1080Q,21.74,104,Gasoline,2.4432\nNissan Teana,2X10829,15.81,110,Gasoline,1.4129\nNissan Teana,2X10829,32.56,110,Gasoline,3.1103\nNissan Teana,2X10829,53.36,110,Gasoline,4.2886\nNissan Teana,2X10829,31.7,110,Gasoline,2.8122\nNissan Teana,2X10829,148.99,110,Gasoline,12.3223\nNissan Teana,2X1080Q,25.08,104,Gasoline,2.1868\nNissan Teana,2X10829,123.55,110,Gasoline,10.923\nNissan Teana,2X1080Q,74.99,104,Gasoline,7.0529\nNissan Teana,2X10829,55.18,110,Gasoline,6.1628\nNissan Teana,2X10829,102.74,110,Gasoline,11.0976\nNissan Teana,2X10829,225.14,110,Gasoline,19.9434\nNissan Teana,2X1080Q,141.39,104,Gasoline,12.7084\nNissan Teana,2X1080Q,7.63,104,Gasoline,0.7177\nNissan Teana,2X10829,107.49,110,Gasoline,9.3111\nNissan Teana,2X10829,12.46,110,Gasoline,1.1393\nNissan Teana,2X10829,3.92,110,Gasoline,0.4681\nNissan Teana,2X10829,37.09,110,Gasoline,2.7859\nNissan Teana,2X10829,90.98,110,Gasoline,8.3425\nNissan Teana,2X10829,52.29,110,Gasoline,4.7478\nNissan Teana,2X1080Q,73.05,104,Gasoline,9.0789\nNissan Teana,2X1080Q,10.04,104,Gasoline,1.293\nNissan Teana,2X10829,35.59,110,Gasoline,3.7098\nNissan Teana,2X1080Q,39.54,104,Gasoline,4.3517\nNissan Teana,2X1080Q,186.2,104,Gasoline,12.3769\nNissan Teana,2X10829,13.87,110,Gasoline,1.571\nNissan Teana,2X10829,24.56,110,Gasoline,1.8272\nNissan Teana,2X10829,117.71,110,Gasoline,9.9523\nNissan Teana,2X10829,145.49,110,Gasoline,8.7493\nNissan Teana,2X1080Q,17.03,104,Gasoline,1.3435\nNissan Teana,2X10829,124.2,110,Gasoline,9.0673\nNissan Teana,2X10829,151.5,110,Gasoline,8.5959\nNissan Teana,2X1080Q,33.47,104,Gasoline,3.1208\nNissan Teana,2X10829,35.71,110,Gasoline,4.4501\nNissan Teana,2X1080Q,12.14,104,Gasoline,1.0351\nNissan Teana,2X10829,240.77,110,Gasoline,15.1478\nNissan Teana,2X10829,18.57,110,Gasoline,1.7157\nNissan Teana,2X1080Q,31.88,104,Gasoline,3.1236\nNissan Teana,2X10829,155.51,110,Gasoline,9.6623\nNissan Teana,2X10829,3.32,110,Gasoline,0.3037\nNissan Teana,2X10829,16.65,110,Gasoline,1.2052\nNissan Teana,2X10829,11.28,110,Gasoline,0.7501\nNissan Teana,2X10829,24.96,110,Gasoline,2.3759\nNissan Teana,2X1080Q,112.52,104,Gasoline,8.3352\nNissan Teana,2X1080Q,297.0,104,Gasoline,19.3584\nNissan Teana,2X1080Q,6.27,104,Gasoline,0.6269\nNissan Teana,2X10829,21.63,110,Gasoline,2.3876\nNissan Teana,2X10829,136.82,110,Gasoline,11.0038\nNissan Teana,2X1080Q,246.95,104,Gasoline,17.2132\nNissan Teana,2X10829,129.75,110,Gasoline,10.1214\nNissan Teana,2X1080Q,27.61,104,Gasoline,2.1688\nNissan Teana,2X1080Q,114.28,104,Gasoline,8.5743\nNissan Teana,2X10829,57.53,110,Gasoline,4.5823\nNissan Teana,2X10829,96.2,110,Gasoline,7.1226\nNissan Teana,2X1080Q,57.3,104,Gasoline,3.267\nNissan Teana,2X10829,150.95,110,Gasoline,11.1283\nNissan Teana,2X10829,310.58,110,Gasoline,22.0068\nNissan Teana,2X1080Q,19.2,104,Gasoline,1.2566\nNissan Teana,2X10829,8.24,110,Gasoline,0.6086\nNissan Teana,2X1080Q,192.05,104,Gasoline,15.5864\nNissan Teana,2X1080Q,2.66,104,Gasoline,0.4809\nNissan Teana,2X1080Q,3.17,104,Gasoline,0.4269\nNissan Teana,2X10829,6.15,110,Gasoline,0.852\nNissan Teana,2X10829,87.3,110,Gasoline,5.4268\nNissan Teana,2X10829,7.29,110,Gasoline,0.5384\nNissan Teana,2X10829,10.47,110,Gasoline,0.938\nNissan Teana,2X10829,106.53,110,Gasoline,7.6999\nNissan Teana,2X1080Q,30.75,104,Gasoline,2.7213\nNissan Teana,2X10829,4.41,110,Gasoline,0.4072\nNissan Teana,2X10829,25.33,110,Gasoline,1.6954\nNissan Teana,2X1080Q,120.91,104,Gasoline,8.7987\nNissan Teana,2X1080Q,3.05,104,Gasoline,0.3582\nNissan Teana,2X10829,152.11,110,Gasoline,13.5452\nNissan Teana,2X10829,14.37,110,Gasoline,0.8741\nNissan Teana,2X10829,33.18,110,Gasoline,2.9211\nNissan Teana,2X10829,41.61,110,Gasoline,3.9825\nNissan Teana,2X1080Q,19.36,104,Gasoline,1.7688\nNissan Teana,2X10829,20.08,110,Gasoline,1.7681\nNissan Teana,2X1080Q,17.67,104,Gasoline,1.4264\nNissan Teana,2X10829,2.75,110,Gasoline,0.519\nNissan Teana,2X10829,110.83,110,Gasoline,9.8973\nNissan Teana,2X1080Q,9.53,104,Gasoline,1.0216\nNissan Teana,2X1080Q,21.74,104,Gasoline,1.796\nNissan Teana,2X10829,3.72,110,Gasoline,0.5689\nNissan Teana,2X10829,2.51,110,Gasoline,0.3487\nNissan Teana,2X10829,20.78,110,Gasoline,1.4878\nNissan Teana,2X1080Q,34.05,104,Gasoline,2.7173\nNissan Teana,2X1080Q,32.83,104,Gasoline,2.3324\nNissan Teana,2X1080Q,7.86,104,Gasoline,0.795\nNissan Teana,2X10829,12.25,110,Gasoline,1.2311\nNissan Teana,2X1080Q,18.82,104,Gasoline,1.2296\nNissan Teana,2X10829,39.57,110,Gasoline,3.3394\nNissan Teana,2X1080Q,13.7,104,Gasoline,1.0047\nNissan Teana,2X10829,10.27,110,Gasoline,1.3584\nNissan Teana,2X1080Q,86.26,104,Gasoline,8.6731\nNissan Teana,2X1080Q,47.66,104,Gasoline,5.7471\nNissan Teana,2X10829,49.17,110,Gasoline,5.9337\nNissan Teana,2X1080Q,10.38,104,Gasoline,1.0968\nNissan Teana,2X1080Q,221.46,104,Gasoline,15.2378\nNissan Teana,2X10829,8.75,110,Gasoline,1.0508\nNissan Teana,2X1080Q,39.07,104,Gasoline,2.8194\nNissan Teana,2X10829,5.33,110,Gasoline,0.6404\nNissan Teana,2X10829,13.41,110,Gasoline,1.2358\nNissan Teana,2X1080Q,41.39,104,Gasoline,4.2998\nNissan Teana,2X10829,19.02,110,Gasoline,1.3285\nNissan Teana,2X10829,24.15,110,Gasoline,1.9775\nNissan Teana,2X10829,26.15,110,Gasoline,2.303\nNissan Teana,2X1080Q,5.21,104,Gasoline,0.6094\nNissan Teana,2X10829,44.07,110,Gasoline,3.6284\nNissan Teana,2X1080Q,15.7,104,Gasoline,1.0195\nNissan Teana,2X10829,3.56,110,Gasoline,0.4984\nNissan Teana,2X1080Q,186.33,104,Gasoline,13.274\nNissan Teana,2X10829,13.06,110,Gasoline,1.7575\nNissan Teana,2X1080Q,29.31,104,Gasoline,2.4017\nNissan Teana,2X1080Q,90.18,104,Gasoline,7.8662\nNissan Teana,2X10829,14.76,110,Gasoline,1.3245\nNissan Teana,2X1080Q,16.68,104,Gasoline,2.0961\nNissan Teana,2X1080Q,16.76,104,Gasoline,1.5641\nNissan Teana,2X1080Q,20.27,104,Gasoline,1.8282\nNissan Teana,2X10829,17.14,110,Gasoline,1.4543\nNissan Teana,2X1080Q,48.36,104,Gasoline,5.1578\nNissan Teana,2X10829,102.74,110,Gasoline,6.5544\nNissan Teana,2X10829,1.97,110,Gasoline,0.1991\nNissan Teana,2X1080Q,40.63,104,Gasoline,3.7005\nNissan Teana,2X1080Q,14.25,104,Gasoline,1.5932\nNissan Teana,2X1080Q,4.95,104,Gasoline,0.9388\nNissan Teana,2X1080Q,259.16,104,Gasoline,18.646\nNissan Teana,2X1080Q,32.71,104,Gasoline,3.2312\nNissan Teana,2X10829,3.71,110,Gasoline,0.6022\nNissan Teana,2X1080Q,186.61,104,Gasoline,12.5666\nNissan Teana,2X1080Q,109.85,104,Gasoline,7.5035\nNissan Teana,2X1080Q,1.88,104,Gasoline,0.2086\nNissan Teana,2X10829,3.83,110,Gasoline,0.433\nNissan Teana,2X1080Q,2.37,104,Gasoline,0.2441\nNissan Teana,2X1080Q,2.67,104,Gasoline,0.3862\nNissan Teana,2X1080Q,14.65,104,Gasoline,1.8426\nNissan Teana,2X10829,177.52,110,Gasoline,14.1122\nNissan Teana,2X1080Q,7.66,104,Gasoline,1.0529\nNissan Teana,2X1080Q,1.79,104,Gasoline,0.2449\nNissan Teana,2X1080Q,204.17,104,Gasoline,15.3882\nNissan Teana,2X10829,28.81,110,Gasoline,2.8946\nNissan Teana,2X10829,1.62,110,Gasoline,0.1431\nNissan Teana,2X1080Q,10.1,104,Gasoline,0.7574\nNissan Teana,2X1080Q,138.08,104,Gasoline,9.7017\nNissan Teana,2X10829,193.83,110,Gasoline,10.8173\nNissan Teana,2X10829,144.28,110,Gasoline,8.3514\nNissan Teana,2X1080Q,5.73,104,Gasoline,0.6573\nNissan Teana,2X1080Q,12.07,104,Gasoline,1.2922\nNissan Teana,2X1080Q,11.59,104,Gasoline,1.163\nNissan Teana,2X10829,35.36,110,Gasoline,3.8481\nNissan Teana,2X10829,11.34,110,Gasoline,0.8197\nNissan Teana,2X10829,157.53,110,Gasoline,13.1028\nNissan Teana,2X10829,122.04,110,Gasoline,9.1805\nNissan Teana,2X10829,5.98,110,Gasoline,0.5557\nNissan Teana,2X10829,100.52,110,Gasoline,10.9928\nNissan Teana,2X1080Q,35.75,104,Gasoline,3.1489\nNissan Teana,2X1080Q,17.73,104,Gasoline,1.6636\nNissan Teana,2X10829,18.56,110,Gasoline,1.8535\nNissan Teana,2X10829,101.88,110,Gasoline,8.4718\nNissan Teana,2X10829,106.01,110,Gasoline,9.4993\nNissan Teana,2X1080Q,4.21,104,Gasoline,0.5356\nNissan Teana,2X10829,5.24,110,Gasoline,0.6411\nNissan Teana,2X10829,6.62,110,Gasoline,0.7075\nNissan Teana,2X1080Q,17.63,104,Gasoline,1.5174\nNissan Teana,2X1080Q,4.4,104,Gasoline,0.4158\nNissan Teana,2X1080Q,105.26,104,Gasoline,9.5942\nNissan Teana,2X10829,3.92,110,Gasoline,0.364\nNissan Teana,2X10829,45.89,110,Gasoline,4.442\nNissan Teana,2X10829,213.83,110,Gasoline,23.3468\nNissan Teana,2X1080Q,13.66,104,Gasoline,1.3407\nNissan Teana,2X10829,40.68,110,Gasoline,3.184\nNissan Teana,2X1080Q,7.23,104,Gasoline,0.7733\nNissan Teana,2X10829,143.75,110,Gasoline,14.4547\nNissan Teana,2X10829,43.58,110,Gasoline,3.1038\nNissan Teana,2X1080Q,146.03,104,Gasoline,9.7892\nNissan Teana,2X1080Q,2.31,104,Gasoline,0.2789\nNissan Teana,2X10829,3.31,110,Gasoline,0.3771\nNissan Teana,2X10829,191.94,110,Gasoline,12.6236\nNissan Teana,2X1080Q,72.32,104,Gasoline,6.2811\nNissan Teana,2X10829,67.61,110,Gasoline,5.4601\nNissan Teana,2X1080Q,21.56,104,Gasoline,1.6976\nNissan Teana,2X1080Q,127.29,104,Gasoline,7.4547\nNissan Teana,2X10829,94.92,110,Gasoline,9.0289\nNissan Teana,2X10829,5.07,110,Gasoline,0.4568\nNissan Teana,2X10829,252.02,110,Gasoline,23.3194\nNissan Teana,2X10829,4.31,110,Gasoline,0.3576\nNissan Teana,2X1080Q,4.56,104,Gasoline,0.6489\nNissan Teana,2X10829,3.88,110,Gasoline,0.4415\nNissan Teana,2X10829,128.05,110,Gasoline,8.3498\nNissan Teana,2X10829,123.7,110,Gasoline,8.2736\nNissan Teana,2X10829,56.84,110,Gasoline,5.4822\nNissan Teana,2X1080Q,127.74,104,Gasoline,7.9756\nNissan Teana,2X10829,12.62,110,Gasoline,1.24\nNissan Teana,2X10829,15.47,110,Gasoline,1.8967\nNissan Teana,2X1080Q,4.43,104,Gasoline,0.9374\nNissan Teana,2X10829,118.06,110,Gasoline,6.7525\nNissan Teana,2X1080Q,15.52,104,Gasoline,1.16\nNissan Teana,2X10829,64.95,110,Gasoline,4.3345\nNissan Teana,2X10829,34.3,110,Gasoline,3.3724\nNissan Teana,2X1080Q,185.33,104,Gasoline,12.1152\nNissan Teana,2X10829,64.99,110,Gasoline,5.8713\nNissan Teana,2X10829,63.27,110,Gasoline,9.7604\nNissan Teana,2X10829,17.51,110,Gasoline,2.1426\nNissan Teana,2X10829,158.03,110,Gasoline,10.186\nNissan Teana,2X10829,153.44,110,Gasoline,11.4864\nNissan Teana,2X10829,19.38,110,Gasoline,2.5023\nNissan Teana,2X10829,30.89,110,Gasoline,2.3173\nNissan Teana,2X1080Q,1.68,104,Gasoline,0.2216\nNissan Teana,2X10829,7.22,110,Gasoline,1.2097\nNissan Teana,2X10829,16.63,110,Gasoline,1.9522\nNissan Teana,2X1080Q,26.15,104,Gasoline,3.5037\nNissan Teana,2X1080Q,10.93,104,Gasoline,0.7638\nNissan Teana,2X1080Q,61.46,104,Gasoline,5.061\nNissan Teana,2X10829,20.01,110,Gasoline,2.2685\nNissan Teana,2X1080Q,34.95,104,Gasoline,2.5524\nNissan Teana,2X1080Q,10.28,104,Gasoline,1.245\nNissan Teana,2X10829,12.34,110,Gasoline,0.9155\nNissan Teana,2X10829,40.65,110,Gasoline,3.3253\nNissan Teana,2X10829,44.29,110,Gasoline,6.6199\nNissan Teana,2X1080Q,10.12,104,Gasoline,1.3769\nNissan Teana,2X1080Q,54.47,104,Gasoline,5.396\nNissan Teana,2X10829,155.23,110,Gasoline,12.0997\nNissan Teana,2X10829,13.03,110,Gasoline,0.9536\nNissan Teana,2X10829,12.98,110,Gasoline,2.1416\nNissan Teana,2X10829,175.1,110,Gasoline,14.468\nNissan Teana,2X1080Q,26.49,104,Gasoline,2.6171\nNissan Teana,2X1080Q,22.76,104,Gasoline,1.7091\nNissan Teana,2X10829,28.99,110,Gasoline,4.3001\nNissan Teana,2X10829,82.8,110,Gasoline,4.9172\nNissan Teana,2X10829,20.51,110,Gasoline,2.9584\nNissan Teana,2X10829,131.69,110,Gasoline,14.1397\nNissan Teana,2X1080Q,4.07,104,Gasoline,0.6744\nNissan Teana,2X10829,8.77,110,Gasoline,0.8926\nNissan Teana,2X10829,31.75,110,Gasoline,2.2737\nNissan Teana,2X10829,27.65,110,Gasoline,3.7665\nNissan Teana,2X1080Q,5.77,104,Gasoline,0.8647\nNissan Teana,2X10829,15.15,110,Gasoline,1.4712\nNissan Teana,2X1080Q,2.59,104,Gasoline,0.2632\nNissan Teana,2X10829,29.26,110,Gasoline,1.9491\nNissan Teana,2X1080Q,8.75,104,Gasoline,1.2199\nNissan Teana,2X10829,124.82,110,Gasoline,12.2966\nNissan Teana,2X10829,2.45,110,Gasoline,0.3461\nNissan Teana,2X1080Q,26.27,104,Gasoline,2.2675\nNissan Teana,2X1080Q,32.32,104,Gasoline,3.5055\nNissan Teana,2X1080Q,43.69,104,Gasoline,4.0948\nNissan Teana,2X10829,101.73,110,Gasoline,10.4624\nNissan Teana,2X1080Q,4.18,104,Gasoline,0.6086\nNissan Teana,2X1080Q,6.04,104,Gasoline,0.9562\nNissan Teana,2X1080Q,48.0,104,Gasoline,5.2945\nNissan Teana,2X1080Q,19.2,104,Gasoline,1.7305\nNissan Teana,2X10829,56.43,110,Gasoline,4.1893\nNissan Teana,2X10829,133.26,110,Gasoline,13.0842\nNissan Teana,2X10829,110.31,110,Gasoline,10.3963\nNissan Teana,2X10829,36.0,110,Gasoline,2.8642\nNissan Teana,2X10829,102.61,110,Gasoline,11.0323\nNissan Teana,2X10829,112.17,110,Gasoline,11.7821\nNissan Teana,2X10829,41.12,110,Gasoline,4.4519\nNissan Teana,2X1080Q,20.33,104,Gasoline,1.9623\nNissan Teana,2X1080Q,52.19,104,Gasoline,7.8897\nNissan Teana,2X10829,10.56,110,Gasoline,1.3744\nNissan Teana,2X1080Q,27.61,104,Gasoline,2.7943\nNissan Teana,2X1080Q,8.09,104,Gasoline,0.9142\nNissan Teana,2X1080Q,114.63,104,Gasoline,12.1346\nNissan Teana,2X10829,36.02,110,Gasoline,3.5905\nNissan Teana,2X10829,44.46,110,Gasoline,3.1149\nNissan Teana,2X10829,18.6,110,Gasoline,2.4007\nNissan Teana,2X1080Q,74.57,104,Gasoline,7.259\nNissan Teana,2X1080Q,32.98,104,Gasoline,2.3656\nNissan Teana,2X10829,11.15,110,Gasoline,0.8877\nNissan Teana,2X10829,114.08,110,Gasoline,7.5607\nNissan Teana,2X1080Q,2.65,104,Gasoline,0.5958\nNissan Teana,2X1080Q,61.17,104,Gasoline,3.9768\nNissan Teana,2X10829,82.74,110,Gasoline,8.1406\nNissan Teana,2X1080Q,10.67,104,Gasoline,1.6336\nNissan Teana,2X10829,179.7,110,Gasoline,12.5723\nNissan Teana,2X10829,15.78,110,Gasoline,1.5662\nNissan Teana,2X1080Q,34.57,104,Gasoline,4.7074\nNissan Teana,2X10829,684.17,110,Gasoline,44.3412\nNissan Teana,2X1080Q,3.42,104,Gasoline,0.4599\nNissan Teana,2X10829,10.36,110,Gasoline,1.5931\nNissan Teana,2X1080Q,10.53,104,Gasoline,0.7883\nNissan Teana,2X1080Q,35.29,104,Gasoline,3.3365\nNissan Teana,2X10829,36.79,110,Gasoline,2.3631\nNissan Teana,2X10829,17.68,110,Gasoline,3.2606\nNissan Teana,2X1080Q,155.31,104,Gasoline,10.5774\nNissan Teana,2X10829,112.89,110,Gasoline,7.7027\nNissan Teana,2X10829,113.78,110,Gasoline,11.7268\nNissan Teana,2X10829,103.72,110,Gasoline,8.2277\nNissan Teana,2X10829,26.64,110,Gasoline,2.5695\nNissan Teana,2X10829,26.2,110,Gasoline,2.6628\nNissan Teana,2X10829,172.26,110,Gasoline,19.7339\nNissan Teana,2X1080Q,15.03,104,Gasoline,1.4378\nNissan Teana,2X10829,39.47,110,Gasoline,2.6308\nNissan Teana,2X1080Q,258.11,104,Gasoline,17.3981\nNissan Teana,2X10829,36.84,110,Gasoline,3.9339\nNissan Teana,2X10829,25.64,110,Gasoline,2.2682\nNissan Teana,2X1080Q,37.14,104,Gasoline,2.7227\nNissan Teana,2X10829,41.87,110,Gasoline,4.3704\nNissan Teana,2X10829,111.44,110,Gasoline,7.8342\nNissan Teana,2X10829,63.26,110,Gasoline,8.518\nNissan Teana,2X1080Q,2.46,104,Gasoline,0.436\nNissan Teana,2X10829,49.05,110,Gasoline,6.1689\nNissan Teana,2X1080Q,24.92,104,Gasoline,3.8613\nNissan Teana,2X10829,108.18,110,Gasoline,8.0709\nNissan Teana,2X10829,5.01,110,Gasoline,0.6114\nNissan Teana,2X1080Q,5.75,104,Gasoline,0.84\nNissan Teana,2X10829,43.5,110,Gasoline,3.5639\nNissan Teana,2X10829,224.69,110,Gasoline,15.1228\nNissan Teana,2X10829,189.31,110,Gasoline,17.6471\nNissan Teana,2X10829,30.48,110,Gasoline,3.4908\nNissan Teana,2X10829,36.47,110,Gasoline,2.6988\nNissan Teana,2X10829,27.38,110,Gasoline,4.0156\nNissan Teana,2X1080Q,9.45,104,Gasoline,0.9827\nNissan Teana,2X1080Q,32.54,104,Gasoline,2.781\nNissan Teana,2X10829,36.58,110,Gasoline,2.7904\nNissan Teana,2X10829,41.01,110,Gasoline,4.4222\nNissan Teana,2X10829,23.81,110,Gasoline,2.3197\nNissan Teana,2X10829,36.35,110,Gasoline,4.6759\nNissan Teana,2X1080Q,228.56,104,Gasoline,15.9018\nNissan Teana,2X10829,182.21,110,Gasoline,16.6133\nNissan Teana,2X10829,169.12,110,Gasoline,18.8345\nNissan Teana,2X10829,19.25,110,Gasoline,1.8689\nNissan Teana,2X10829,62.28,110,Gasoline,10.6754\nNissan Teana,2X10829,53.1,110,Gasoline,6.0615\nNissan Teana,2X10829,125.69,110,Gasoline,11.2304\nNissan Teana,2X10829,108.11,110,Gasoline,7.6728\nNissan Teana,2X10829,39.67,110,Gasoline,2.8157\nNissan Teana,2X1080Q,7.87,104,Gasoline,1.3863\nNissan Teana,2X10829,86.3,110,Gasoline,11.5457\nNissan Teana,2X1080Q,4.62,104,Gasoline,0.5153\nNissan Teana,2X10829,113.68,110,Gasoline,11.871\nNissan Teana,2X1080Q,61.63,104,Gasoline,3.9448\nNissan Teana,2X1080Q,132.52,104,Gasoline,11.8354\nNissan Teana,2X1080Q,2.79,104,Gasoline,0.4856\nNissan Teana,2X10829,57.79,110,Gasoline,5.5192\nNissan Teana,2X10829,33.06,110,Gasoline,3.4191\nNissan Teana,2X10829,28.4,110,Gasoline,4.383\nNissan Teana,2X10829,53.76,110,Gasoline,6.3551\nNissan Teana,2X1080Q,22.18,104,Gasoline,1.9636\nNissan Teana,2X1080Q,31.03,104,Gasoline,3.1974\nNissan Teana,2X10829,52.74,110,Gasoline,3.7627\nNissan Teana,2X1080Q,112.35,104,Gasoline,10.906\nNissan Teana,2X1080Q,106.6,104,Gasoline,9.5931\nNissan Teana,2X10829,110.88,110,Gasoline,10.7378\nNissan Teana,2X10829,33.29,110,Gasoline,4.0668\nNissan Teana,2X10829,8.92,110,Gasoline,0.9245\nNissan Teana,2X1080Q,139.15,104,Gasoline,9.0177\nNissan Teana,2X10829,148.25,110,Gasoline,11.1764\nNissan Teana,2X10829,4.88,110,Gasoline,0.6032\nNissan Teana,2X1080Q,189.78,104,Gasoline,11.3128\nNissan Teana,2X10829,2.51,110,Gasoline,0.4828\nNissan Teana,2X10829,32.28,110,Gasoline,5.5093\nNissan Teana,2X10829,239.96,110,Gasoline,15.2088\nNissan Teana,2X1080Q,113.38,104,Gasoline,13.2862\nNissan Teana,2X1080Q,2.9,104,Gasoline,0.4004\nNissan Teana,2X10829,18.3,110,Gasoline,3.018\nNissan Teana,2X1080Q,105.97,104,Gasoline,11.7516\nNissan Teana,2X1080Q,21.54,104,Gasoline,1.9136\nNissan Teana,2X1080Q,35.52,104,Gasoline,6.0445\nNissan Teana,2X10829,4.94,110,Gasoline,0.6314\nNissan Teana,2X10829,24.97,110,Gasoline,2.5713\nNissan Teana,2X10829,147.3,110,Gasoline,13.7445\nNissan Teana,2X10829,56.58,110,Gasoline,4.5602\nNissan Teana,2X1080Q,2.48,104,Gasoline,0.413\nNissan Teana,2X10829,13.08,110,Gasoline,1.5132\nNissan Teana,2X10829,115.06,110,Gasoline,12.0877\nNissan Teana,2X10829,30.62,110,Gasoline,3.5181\nNissan Teana,2X1080Q,116.63,104,Gasoline,7.7302\nNissan Teana,2X10829,163.94,110,Gasoline,15.008\nNissan Teana,2X10829,26.68,110,Gasoline,1.9059\nNissan Teana,2X10829,299.87,110,Gasoline,19.6044\nNissan Teana,2X1080Q,2.03,104,Gasoline,0.3745\nNissan Teana,2X1080Q,113.26,104,Gasoline,10.0649\nNissan Teana,2X10829,50.49,110,Gasoline,7.4206\nNissan Teana,2X1080Q,1.77,104,Gasoline,0.19\nNissan Teana,2X10829,64.52,110,Gasoline,3.6289\nNissan Teana,2X10829,33.22,110,Gasoline,2.0136\nNissan Teana,2X1080Q,28.44,104,Gasoline,3.0959\nNissan Teana,2X1080Q,32.54,104,Gasoline,2.2818\nNissan Teana,2X1080Q,12.01,104,Gasoline,1.9723\nNissan Teana,2X10829,33.4,110,Gasoline,3.828\nNissan Teana,2X10829,5.14,110,Gasoline,0.8979\nNissan Teana,2X1080Q,2.23,104,Gasoline,0.2533\nNissan Teana,2X1080Q,3.36,104,Gasoline,0.4465\nNissan Teana,2X10829,104.26,110,Gasoline,6.2558\nNissan Teana,2X1080Q,37.03,104,Gasoline,3.6812\nNissan Teana,2X1080Q,10.02,104,Gasoline,1.4182\nNissan Teana,2X10829,136.2,110,Gasoline,12.3972\nNissan Teana,2X10829,37.7,110,Gasoline,4.4402\nNissan Teana,2X1080Q,8.44,104,Gasoline,1.3389\nNissan Teana,2X1080Q,15.51,104,Gasoline,1.4603\nNissan Teana,2X10829,4.29,110,Gasoline,0.3665\nNissan Teana,2X1080Q,17.31,104,Gasoline,1.6602\nNissan Teana,2X1080Q,15.12,104,Gasoline,2.1224\nNissan Teana,2X10829,160.93,110,Gasoline,11.1735\nNissan Teana,2X10829,3.38,110,Gasoline,0.3192\nNissan Teana,2X1080Q,10.48,104,Gasoline,0.6909\nNissan Teana,2X1080Q,7.7,104,Gasoline,1.3877\nNissan Teana,2X10829,20.24,110,Gasoline,2.0217\nNissan Teana,2X10829,17.1,110,Gasoline,2.3452\nNissan Teana,2X10829,3.66,110,Gasoline,0.4562\nNissan Teana,2X10829,9.53,110,Gasoline,0.8087\nNissan Teana,2X10829,27.02,110,Gasoline,2.9742\nNissan Teana,2X10829,9.82,110,Gasoline,0.8258\nNissan Teana,2X10829,15.44,110,Gasoline,1.3951\nNissan Teana,2X10829,192.23,110,Gasoline,16.0895\nNissan Teana,2X10829,9.6,110,Gasoline,0.7229\nNissan Teana,2X10829,25.05,110,Gasoline,2.385\nNissan Teana,2X10829,184.84,110,Gasoline,15.1696\nNissan Teana,2X10829,123.98,110,Gasoline,13.9413\nNissan Teana,2X1080Q,36.44,104,Gasoline,2.9802\nNissan Teana,2X1080Q,32.6,104,Gasoline,2.7529\nNissan Teana,2X1080Q,1.55,104,Gasoline,0.2113\nNissan Teana,2X1080Q,6.1,104,Gasoline,0.4898\nNissan Teana,2X10829,6.29,110,Gasoline,0.6284\nNissan Teana,2X10829,41.81,110,Gasoline,3.451\nNissan Teana,2X10829,8.51,110,Gasoline,0.8054\nNissan Teana,2X1080Q,100.17,104,Gasoline,7.9393\nNissan Teana,2X1080Q,6.28,104,Gasoline,0.868\nNissan Teana,2X1080Q,5.57,104,Gasoline,0.7951\nNissan Teana,2X10829,29.08,110,Gasoline,2.409\nNissan Teana,2X10829,4.89,110,Gasoline,0.5144\nNissan Teana,2X1080Q,192.72,104,Gasoline,13.94\nNissan Teana,2X1080Q,232.53,104,Gasoline,17.7389\nNissan Teana,2X10829,25.82,110,Gasoline,2.1992\nNissan Teana,2X10829,2.24,110,Gasoline,0.3023\nNissan Teana,2X10829,6.94,110,Gasoline,0.6346\nNissan Teana,2X1080Q,35.19,104,Gasoline,2.6239\nNissan Teana,2X10829,8.7,110,Gasoline,0.5699\nNissan Teana,2X10829,231.5,110,Gasoline,19.2424\nNissan Teana,2X10829,271.26,110,Gasoline,16.5205\nNissan Teana,2X1080Q,296.29,104,Gasoline,23.3245\nNissan Teana,2X1080Q,5.17,104,Gasoline,0.6932\nNissan Teana,2X1080Q,109.94,104,Gasoline,9.5769\nNissan Teana,2X10829,20.53,110,Gasoline,1.7003\nNissan Teana,2X10829,5.03,110,Gasoline,0.4715\nNissan Teana,2X10829,1.57,110,Gasoline,0.2279\nNissan Teana,2X1080Q,11.4,104,Gasoline,0.8909\nNissan Teana,2X1080Q,158.64,104,Gasoline,9.8191\nNissan Teana,2X1080Q,37.34,104,Gasoline,3.0691\nNissan Teana,2X1080Q,27.45,104,Gasoline,2.849\nNissan Teana,2X1080Q,135.47,104,Gasoline,9.2397\nNissan Teana,2X10829,8.26,110,Gasoline,0.8342\nNissan Teana,2X1080Q,22.85,104,Gasoline,2.6679\nNissan Teana,2X1080Q,2.13,104,Gasoline,0.2824\nNissan Teana,2X10829,30.9,110,Gasoline,1.948\nNissan Teana,2X1080Q,32.31,104,Gasoline,2.2814\nNissan Teana,2X10829,51.77,110,Gasoline,3.5436\nNissan Teana,2X10829,123.36,110,Gasoline,10.5223\nNissan Teana,2X1080Q,15.24,104,Gasoline,1.0204\nNissan Teana,2X1080Q,4.23,104,Gasoline,0.4944\nNissan Teana,2X10829,137.29,110,Gasoline,11.6188\nNissan Teana,2X1080Q,13.87,104,Gasoline,1.5218\nNissan Teana,2X10829,46.86,110,Gasoline,2.9015\nNissan Teana,2X1080Q,2.12,104,Gasoline,0.3165\nNissan Teana,2X10829,83.99,110,Gasoline,5.831\nNissan Teana,2X10829,63.29,110,Gasoline,5.0173\nNissan Teana,2X1080Q,51.49,104,Gasoline,4.6206\nNissan Teana,2X1080Q,13.02,104,Gasoline,0.9554\nNissan Teana,2X10829,13.7,110,Gasoline,1.0405\nNissan Teana,2X1080Q,63.82,104,Gasoline,4.9662\nNissan Teana,2X10829,26.93,110,Gasoline,3.2098\nNissan Teana,2X1080Q,129.7,104,Gasoline,9.2786\nNissan Teana,2X10829,1.52,110,Gasoline,0.2979\nNissan Teana,2X1080Q,329.0,104,Gasoline,22.9105\nNissan Teana,2X10829,57.04,110,Gasoline,3.8142\nNissan Teana,2X10829,114.71,110,Gasoline,8.8174\nNissan Teana,2X10829,108.56,110,Gasoline,7.6305\nNissan Teana,2X10829,4.15,110,Gasoline,0.3614\nNissan Teana,2X10829,100.77,110,Gasoline,10.7985\nNissan Teana,2X10829,190.22,110,Gasoline,14.6824\nNissan Teana,2X1080Q,189.01,104,Gasoline,11.8305\nNissan Teana,2X10829,2.43,110,Gasoline,0.264\nNissan Teana,2X10829,3.12,110,Gasoline,0.3797\nNissan Teana,2X1080Q,6.17,104,Gasoline,0.4717\nNissan Teana,2X1080Q,6.01,104,Gasoline,1.0535\nNissan Teana,2X10829,10.67,110,Gasoline,1.1223\nNissan Teana,2X10829,24.74,110,Gasoline,2.3057\nNissan Teana,2X10829,15.58,110,Gasoline,1.6584\nNissan Teana,2X1080Q,41.8,104,Gasoline,3.2583\nNissan Teana,2X1080Q,53.44,104,Gasoline,4.622\nNissan Teana,2X10829,57.26,110,Gasoline,4.2008\nNissan Teana,2X1080Q,137.69,104,Gasoline,9.263\nNissan Teana,2X10829,9.76,110,Gasoline,0.9352\nNissan Teana,2X10829,5.23,110,Gasoline,0.5523\nNissan Teana,2X10829,25.53,110,Gasoline,1.6242\nNissan Teana,2X10829,123.94,110,Gasoline,9.7791\nNissan Teana,2X1080Q,33.25,104,Gasoline,2.2223\nNissan Teana,2X1080Q,9.65,104,Gasoline,1.0525\nNissan Teana,2X10829,106.29,110,Gasoline,8.0278\nNissan Teana,2X10829,23.01,110,Gasoline,2.62\nNissan Teana,2X10829,39.78,110,Gasoline,3.6306\nNissan Teana,2X10829,28.64,110,Gasoline,1.9903\nNissan Teana,2X1080Q,14.62,104,Gasoline,1.6466\nNissan Teana,2X10829,5.89,110,Gasoline,0.542\nNissan Teana,2X10829,17.6,110,Gasoline,2.0578\nNissan Teana,2X1080Q,35.23,104,Gasoline,2.1788\nNissan Teana,2X1080Q,41.15,104,Gasoline,3.2656\nNissan Teana,2X1080Q,1.56,104,Gasoline,0.2573\nNissan Teana,2X10829,60.6,110,Gasoline,4.9026\nNissan Teana,2X10829,6.74,110,Gasoline,0.6232\nNissan Teana,2X1080Q,18.45,104,Gasoline,1.6443\nNissan Teana,2X10829,5.5,110,Gasoline,0.743\nNissan Teana,2X10829,5.84,110,Gasoline,0.6437\nNissan Teana,2X1080Q,12.28,104,Gasoline,1.2928\nNissan Teana,2X10829,111.4,110,Gasoline,10.7612\nNissan Teana,2X10829,2.1,110,Gasoline,0.2234\nNissan Teana,2X10829,5.65,110,Gasoline,0.5758\nNissan Teana,2X1080Q,10.9,104,Gasoline,1.2856\nNissan Teana,2X1080Q,1.54,104,Gasoline,0.2455\nNissan Teana,2X1080Q,6.36,104,Gasoline,0.978\nNissan Teana,2X1080Q,3.75,104,Gasoline,0.622\nNissan Teana,2X1080Q,9.18,104,Gasoline,0.7436\nNissan Teana,2X10829,42.78,110,Gasoline,3.4949\nNissan Teana,2X1080Q,6.02,104,Gasoline,1.1655\nNissan Teana,2X1080Q,10.96,104,Gasoline,0.8596\nNissan Teana,2X10829,100.8,110,Gasoline,7.8911\nNissan Teana,2X1080Q,2.38,104,Gasoline,0.3137\nNissan Teana,2X10829,109.52,110,Gasoline,7.0415\nNissan Teana,2X1080Q,17.61,104,Gasoline,1.8091\nNissan Teana,2X10829,6.23,110,Gasoline,0.556\nNissan Teana,2X1080Q,30.09,104,Gasoline,2.7952\nNissan Teana,2X10829,1.93,110,Gasoline,0.3237\nNissan Teana,2X10829,154.74,110,Gasoline,15.9634\nNissan Teana,2X1080Q,17.02,104,Gasoline,1.3565\nNissan Teana,2X10829,221.28,110,Gasoline,15.2072\nNissan Teana,2X10829,2.3,110,Gasoline,0.4492\nNissan Teana,2X1080Q,106.04,104,Gasoline,8.3727\nNissan Teana,2X1080Q,23.7,104,Gasoline,3.0222\nNissan Teana,2X10829,96.17,110,Gasoline,6.0218\nNissan Teana,2X10829,6.88,110,Gasoline,0.8763\nNissan Teana,2X1080Q,4.82,104,Gasoline,0.6541\nNissan Teana,2X1080Q,31.45,104,Gasoline,2.2827\nNissan Teana,2X1080Q,84.41,104,Gasoline,7.2107\nNissan Teana,2X10829,10.67,110,Gasoline,1.7457\nNissan Teana,2X10829,5.52,110,Gasoline,0.724\nNissan Teana,2X10829,6.78,110,Gasoline,0.8134\nNissan Teana,2X10829,115.19,110,Gasoline,7.5482\nNissan Teana,2X10829,37.86,110,Gasoline,3.6277\nNissan Teana,2X10829,107.06,110,Gasoline,9.5205\nNissan Teana,2X1080Q,4.68,104,Gasoline,0.6915\nNissan Teana,2X10829,1.8,110,Gasoline,0.2477\nNissan Teana,2X10829,2.11,110,Gasoline,0.2936\nNissan Teana,2X10829,6.8,110,Gasoline,0.5396\nNissan Teana,2X10829,13.32,110,Gasoline,1.3842\nNissan Teana,2X10829,61.51,110,Gasoline,5.2932\nNissan Teana,2X10829,1.83,110,Gasoline,0.3178\nNissan Teana,2X10829,80.44,110,Gasoline,7.9775\nNissan Teana,2X10829,177.83,110,Gasoline,11.3057\nNissan Teana,2X10829,103.48,110,Gasoline,6.9447\nNissan Teana,2X10829,41.56,110,Gasoline,3.7375\nNissan Teana,2X1080Q,101.82,104,Gasoline,7.7488\nNissan Teana,2X10829,55.9,110,Gasoline,5.398\nNissan Teana,2X10829,8.25,110,Gasoline,0.9144\nNissan Teana,2X10829,103.21,110,Gasoline,9.862\nNissan Teana,2X1080Q,8.36,104,Gasoline,0.8457\nNissan Teana,2X10829,49.7,110,Gasoline,5.4888\nNissan Teana,2X1080Q,91.48,104,Gasoline,6.2727\nNissan Teana,2X10829,19.91,110,Gasoline,2.0015\nNissan Teana,2X10829,152.3,110,Gasoline,14.4774\nNissan Teana,2X1080Q,3.68,104,Gasoline,0.569\nNissan Teana,2X1080Q,1.65,104,Gasoline,0.1844\nNissan Teana,2X10829,10.4,110,Gasoline,0.7474\nNissan Teana,2X10829,239.67,110,Gasoline,17.5372\nNissan Teana,2X1080Q,43.34,104,Gasoline,4.447\nNissan Teana,2X1080Q,3.1,104,Gasoline,0.3034\nNissan Teana,2X1080Q,1.82,104,Gasoline,0.267\nNissan Teana,2X1080Q,6.02,104,Gasoline,0.6544\nNissan Teana,2X10829,294.39,110,Gasoline,20.1759\nNissan Teana,2X10829,152.74,110,Gasoline,15.4014\nNissan Teana,2X10829,44.28,110,Gasoline,6.3042\nNissan Teana,2X1080Q,8.27,104,Gasoline,0.7798\nNissan Teana,2X1080Q,41.81,104,Gasoline,3.1037\nNissan Tiida,2X10608,4.79,140,Gasoline,0.5922\nNissan Tiida,2X10608,120.54,140,Gasoline,8.1449\nNissan Tiida,2X10608,34.22,140,Gasoline,3.2203\nNissan Tiida,2X10608,19.02,140,Gasoline,1.2144\nNissan Tiida,2X10608,6.6,140,Gasoline,0.8814\nNissan Tiida,2X10608,7.88,140,Gasoline,0.7857\nNissan Tiida,2X10608,33.96,140,Gasoline,2.4418\nNissan Tiida,2X10608,14.53,140,Gasoline,1.1783\nNissan Tiida,2X10608,156.35,140,Gasoline,10.6661\nNissan Tiida,2X10608,42.95,140,Gasoline,3.901\nNissan Tiida,2X10608,8.64,140,Gasoline,0.8609\nNissan Tiida,2X10608,9.37,140,Gasoline,0.7262\nNissan Tiida,2X10608,2.96,140,Gasoline,0.2862\nNissan Tiida,2X10608,63.6,140,Gasoline,5.1409\nNissan Tiida,2X10608,82.45,140,Gasoline,5.6821\nNissan Tiida,2X10608,26.28,140,Gasoline,1.9875\nNissan Tiida,2X10608,41.47,140,Gasoline,2.9412\nNissan Tiida,2X10608,7.54,140,Gasoline,0.8027\nNissan Tiida,2X10608,46.81,140,Gasoline,3.4927\nNissan Tiida,2X10608,19.6,140,Gasoline,1.2946\nNissan Tiida,2X10608,7.1,140,Gasoline,0.6852\nNissan Tiida,2X10608,3.49,140,Gasoline,0.3395\nNissan Tiida,2X10608,65.24,140,Gasoline,4.8832\nNissan Tiida,2X10608,198.64,140,Gasoline,12.6187\nNissan Tiida,2X10608,24.27,140,Gasoline,1.5582\nNissan Tiida,2X10608,15.74,140,Gasoline,1.0363\nNissan Tiida,2X10608,14.0,140,Gasoline,1.3241\nNissan Tiida,2X10608,108.71,140,Gasoline,7.5853\nNissan Tiida,2X10608,12.12,140,Gasoline,1.156\nNissan Tiida,2X10608,475.87,140,Gasoline,32.2584\nNissan Tiida,2X10608,2.18,140,Gasoline,0.201\nNissan Tiida,2X10608,152.24,140,Gasoline,9.9341\nNissan Tiida,2X10608,4.46,140,Gasoline,0.3993\nNissan Tiida,2X10608,8.29,140,Gasoline,0.4534\nNissan Tiida,2X10608,436.2,140,Gasoline,29.627\nNissan Tiida,2X10608,17.19,140,Gasoline,1.2549\nNissan Tiida,2X10608,3.94,140,Gasoline,0.2754\nNissan Tiida,2X10608,14.09,140,Gasoline,1.161\nNissan Tiida,2X10608,7.77,140,Gasoline,0.7415\nNissan Tiida,2X10608,10.59,140,Gasoline,0.6965\nNissan Tiida,2X10608,3.14,140,Gasoline,0.3095\nNissan Tiida,2X10608,2.17,140,Gasoline,0.1546\nNissan Tiida,2X10608,167.93,140,Gasoline,10.7118\nNissan Tiida,2X10608,24.57,140,Gasoline,1.5858\nNissan Tiida,2X10608,97.66,140,Gasoline,5.7692\nNissan Tiida,2X10608,109.13,140,Gasoline,5.8732\nNissan Tiida,2X10608,37.06,140,Gasoline,2.1611\nNissan Tiida,2X10608,147.65,140,Gasoline,10.3588\nNissan Tiida,2X10608,147.07,140,Gasoline,10.0156\nNissan Tiida,2X10608,128.99,140,Gasoline,8.5889\nNissan Tiida,2X10608,54.92,140,Gasoline,3.5599\nNissan Tiida,2X10608,17.34,140,Gasoline,1.2797\nNissan Tiida,2X10608,2.76,140,Gasoline,0.4064\nNissan Tiida,2X10608,3.9,140,Gasoline,0.2537\nNissan Tiida,2X10608,273.36,140,Gasoline,19.2918\nNissan Tiida,2X10608,16.97,140,Gasoline,1.2774\nNissan Tiida,2X10608,8.04,140,Gasoline,0.6482\nNissan Tiida,2X10608,131.18,140,Gasoline,9.9561\nNissan Tiida,2X10608,3.28,140,Gasoline,0.2567\nNissan Tiida,2X10608,14.32,140,Gasoline,1.1073\nNissan Tiida,2X10608,4.03,140,Gasoline,0.4631\nNissan Tiida,2X10608,104.68,140,Gasoline,6.4998\nNissan Tiida,2X10608,106.84,140,Gasoline,7.2493\nNissan Tiida,2X10608,3.39,140,Gasoline,0.466\nNissan Tiida,2X10608,34.06,140,Gasoline,2.3588\nNissan Tiida,2X10608,30.33,140,Gasoline,2.2953\nNissan Tiida,2X10608,42.75,140,Gasoline,2.1662\nNissan Tiida,2X10608,33.35,140,Gasoline,1.5088\nNissan Tiida,2X10608,67.82,140,Gasoline,3.4155\nNissan Tiida,2X10608,10.52,140,Gasoline,0.9178\nNissan Tiida,2X10608,120.96,140,Gasoline,8.3189\nNissan Tiida,2X10608,14.24,140,Gasoline,1.3965\nNissan Tiida,2X10608,11.06,140,Gasoline,1.0425\nNissan Tiida,2X10608,122.76,140,Gasoline,8.5897\nNissan Tiida,2X10608,62.61,140,Gasoline,3.7332\nNissan Tiida,2X10608,103.84,140,Gasoline,6.513\nNissan Tiida,2X10608,6.51,140,Gasoline,0.4699\nNissan Tiida,2X10608,13.94,140,Gasoline,0.9397\nNissan Tiida,2X10608,24.4,140,Gasoline,1.9054\nNissan Tiida,2X10608,13.1,140,Gasoline,0.8566\nNissan Tiida,2X10608,143.15,140,Gasoline,9.0319\nNissan Tiida,2X10608,3.19,140,Gasoline,0.3566\nNissan Tiida,2X10608,2.17,140,Gasoline,0.2609\nNissan Tiida,2X10608,7.57,140,Gasoline,0.4885\nNissan Tiida,2X10608,121.71,140,Gasoline,7.7896\nNissan Tiida,2X10608,5.22,140,Gasoline,0.4074\nNissan Tiida,2X10608,112.09,140,Gasoline,7.4927\nNissan Tiida,2X10608,9.21,140,Gasoline,0.6681\nNissan Tiida,2X10608,20.76,140,Gasoline,1.4296\nNissan Tiida,2X10608,1.54,140,Gasoline,0.2005\nNissan Tiida,2X10608,16.54,140,Gasoline,1.0269\nNissan Tiida,2X10608,4.75,140,Gasoline,0.4824\nNissan Tiida,2X10608,180.09,140,Gasoline,12.572\nNissan Tiida,2X10608,16.25,140,Gasoline,0.9823\nNissan Tiida,2X10608,148.96,140,Gasoline,8.8565\nNissan Tiida,2X10608,21.47,140,Gasoline,1.6496\nNissan Tiida,2X10608,44.16,140,Gasoline,2.7852\nNissan Tiida,2X10608,290.46,140,Gasoline,19.3477\nNissan Tiida,2X10608,24.94,140,Gasoline,1.6682\nNissan Tiida,2X10608,165.73,140,Gasoline,11.4698\nNissan Tiida,2X10608,128.25,140,Gasoline,8.2114\nNissan Tiida,2X10608,66.31,140,Gasoline,4.1512\nNissan Tiida,2X10608,2.99,140,Gasoline,0.3456\nNissan Tiida,2X10608,34.27,140,Gasoline,2.6777\nNissan Tiida,2X10608,47.22,140,Gasoline,2.6656\nNissan Tiida,2X10608,127.46,140,Gasoline,8.8123\nNissan Tiida,2X10608,2.16,140,Gasoline,0.3235\nNissan Tiida,2X10608,122.09,140,Gasoline,8.5268\nNissan Tiida,2X10608,18.53,140,Gasoline,1.2505\nNissan Tiida,2X10608,13.14,140,Gasoline,0.9251\nNissan Tiida,2X10608,24.76,140,Gasoline,1.7077\nNissan Tiida,2X10608,105.45,140,Gasoline,7.9955\nNissan Tiida,2X10608,32.97,140,Gasoline,2.4386\nNissan Tiida,2X10608,107.68,140,Gasoline,8.5672\nNissan Tiida,2X10608,48.5,140,Gasoline,3.0135\nNissan Tiida,2X10608,116.14,140,Gasoline,8.3276\nNissan Tiida,2X10608,105.58,140,Gasoline,7.4435\nNissan Tiida,2X10608,12.6,140,Gasoline,0.9272\nNissan Tiida,2X10608,24.7,140,Gasoline,1.6245\nNissan Tiida,2X10608,33.08,140,Gasoline,2.9958\nNissan Tiida,2X10608,3.73,140,Gasoline,0.2843\nNissan Tiida,2X10608,12.81,140,Gasoline,0.8286\nNissan Tiida,2X10608,2.4,140,Gasoline,0.3481\nNissan Tiida,2X10608,34.24,140,Gasoline,3.0555\nNissan Tiida,2X10608,13.06,140,Gasoline,0.8912\nNissan Tiida,2X10608,11.0,140,Gasoline,0.7354\nNissan Tiida,2X10608,34.05,140,Gasoline,2.3103\nNissan Tiida,2X10608,133.16,140,Gasoline,8.9931\nNissan Tiida,2X10608,85.41,140,Gasoline,5.3662\nNissan Tiida,2X10608,15.92,140,Gasoline,0.8991\nNissan Tiida,2X10608,57.89,140,Gasoline,3.4585\nNissan Tiida,2X10608,4.59,140,Gasoline,0.4376\nNissan Tiida,2X10608,50.88,140,Gasoline,3.4427\nNissan Tiida,2X10608,152.55,140,Gasoline,9.1935\nNissan Tiida,2X10608,109.9,140,Gasoline,5.7795\nNissan Tiida,2X10608,106.77,140,Gasoline,5.7926\nNissan Tiida,2X10608,48.24,140,Gasoline,3.2403\nNissan Tiida,2X10608,4.78,140,Gasoline,0.4984\nNissan Tiida,2X10608,34.15,140,Gasoline,2.5603\nNissan Tiida,2X10608,13.84,140,Gasoline,1.1142\nNissan Tiida,2X10608,22.56,140,Gasoline,1.425\nNissan Tiida,2X10608,30.68,140,Gasoline,2.8919\nNissan Tiida,2X10608,11.74,140,Gasoline,0.7344\nNissan Tiida,2X10608,1.7,140,Gasoline,0.1842\nNissan Tiida,2X10608,47.19,140,Gasoline,3.1989\nNissan Tiida,2X10608,9.25,140,Gasoline,0.6958\nNissan Tiida,2X10608,37.2,140,Gasoline,3.1314\nNissan Tiida,2X10608,10.83,140,Gasoline,0.7813\nNissan Tiida,2X10608,12.66,140,Gasoline,1.0877\nNissan Tiida,2X10608,11.28,140,Gasoline,0.9393\nNissan Tiida,2X10608,9.52,140,Gasoline,1.0022\nNissan Tiida,2X10608,31.49,140,Gasoline,2.0894\nNissan Tiida,2X10608,15.37,140,Gasoline,1.2865\nNissan Tiida,2X10608,72.52,140,Gasoline,3.8876\nNissan Tiida,2X10608,66.98,140,Gasoline,3.5855\nNissan Tiida,2X10608,31.78,140,Gasoline,1.9792\nNissan Tiida,2X10608,61.15,140,Gasoline,5.5357\nNissan Tiida,2X10608,5.63,140,Gasoline,0.5159\nNissan Tiida,2X10608,1.9,140,Gasoline,0.182\nNissan Tiida,2X10608,6.74,140,Gasoline,0.7528\nNissan Tiida,2X10608,11.17,140,Gasoline,0.8918\nNissan Tiida,2X10608,2.39,140,Gasoline,0.2527\nNissan Tiida,2X10608,104.45,140,Gasoline,7.1788\nNissan Tiida,2X10608,27.71,140,Gasoline,1.6384\nNissan Tiida,2X10608,24.31,140,Gasoline,1.9089\nNissan Tiida,2X10608,121.96,140,Gasoline,7.1375\nNissan Tiida,2X10608,13.2,140,Gasoline,1.3595\nNissan Tiida,2X10608,151.1,140,Gasoline,8.9495\nNissan Tiida,2X10608,109.88,140,Gasoline,5.9131\nNissan Tiida,2X10608,1.81,140,Gasoline,0.1916\nNissan Tiida,2X10608,185.01,140,Gasoline,10.1911\nNissan Tiida,2X10608,1.84,140,Gasoline,0.2096\nNissan Tiida,2X10608,117.08,140,Gasoline,8.4413\nNissan Tiida,2X10608,36.88,140,Gasoline,2.1989\nNissan Tiida,2X10608,31.84,140,Gasoline,2.0168\nNissan Tiida,2X10608,7.27,140,Gasoline,0.7293\nNissan Tiida,2X10608,1.56,140,Gasoline,0.2108\nNissan Tiida,2X10608,110.86,140,Gasoline,7.9841\nNissan Tiida,2X10608,120.12,140,Gasoline,7.8609\nNissan Tiida,2X10608,14.49,140,Gasoline,1.076\nNissan Tiida,2X10608,102.22,140,Gasoline,6.397\nNissan Tiida,2X10608,134.27,140,Gasoline,6.9157\nNissan Tiida,2X10608,105.51,140,Gasoline,6.08\nNissan Tiida,2X10608,14.11,140,Gasoline,0.8737\nNissan Tiida,2X10608,3.54,140,Gasoline,0.3086\nNissan Tiida,2X10608,10.34,140,Gasoline,0.7472\nNissan Tiida,2X10608,144.99,140,Gasoline,9.3501\nNissan Tiida,2X10608,1.53,140,Gasoline,0.1375\nNissan Tiida,2X10608,153.96,140,Gasoline,9.3588\nNissan Tiida,2X10608,11.18,140,Gasoline,0.7212\nNissan Tiida,2X10608,330.74,140,Gasoline,20.0879\nNissan Tiida,2X10608,33.64,140,Gasoline,2.0489\nNissan Tiida,2X10608,101.52,140,Gasoline,6.0068\nNissan Tiida,2X10608,53.85,140,Gasoline,3.031\nNissan Tiida,2X10608,1.8,140,Gasoline,0.1938\nNissan Tiida,2X10608,11.2,140,Gasoline,0.9011\nNissan Tiida,2X10608,170.37,140,Gasoline,9.5763\nNissan Tiida,2X10608,1.86,140,Gasoline,0.2481\nNissan Tiida,2X10608,121.64,140,Gasoline,9.029\nNissan Tiida,2X10608,4.86,140,Gasoline,0.4541\nNissan Tiida,2X10608,6.24,140,Gasoline,0.5506\nNissan Tiida,2X10608,129.67,140,Gasoline,7.7423\nNissan Tiida,2X10608,44.02,140,Gasoline,2.3549\nNissan Tiida,2X10608,10.66,140,Gasoline,0.766\nNissan Tiida,2X10608,2.5,140,Gasoline,0.2178\nNissan Tiida,2X10608,8.97,140,Gasoline,0.7961\nNissan Tiida,2X10608,2.05,140,Gasoline,0.3162\nNissan Tiida,2X10608,40.35,140,Gasoline,2.3312\nNissan Tiida,2X10608,16.76,140,Gasoline,1.2288\nNissan Tiida,2X10608,167.86,140,Gasoline,9.2028\nNissan Tiida,2X10608,2.34,140,Gasoline,0.2299\nNissan Tiida,2X10608,3.13,140,Gasoline,0.2723\nNissan Tiida,2X10608,6.46,140,Gasoline,0.6126\nNissan Tiida,2X10608,102.9,140,Gasoline,5.7174\nNissan Tiida,2X10608,6.6,140,Gasoline,0.6958\nNissan Tiida,2X10608,13.21,140,Gasoline,1.1139\nNissan Tiida,2X10608,27.68,140,Gasoline,2.4085\nNissan Tiida,2X10608,14.09,140,Gasoline,1.6579\nNissan Tiida,2X10608,33.78,140,Gasoline,3.8684\nNissan Tiida,2X10608,15.39,140,Gasoline,1.552\nNissan Tiida,2X10608,12.31,140,Gasoline,0.8678\nNissan Tiida,2X10608,174.73,140,Gasoline,9.9516\nNissan Tiida,2X10608,28.95,140,Gasoline,1.8183\nNissan Tiida,2X10608,1.83,140,Gasoline,0.2405\nNissan Tiida,2X10608,20.68,140,Gasoline,1.4064\nNissan Tiida,2X10608,109.48,140,Gasoline,7.5539\nNissan Tiida,2X10608,14.23,140,Gasoline,1.292\nNissan Tiida,2X10608,34.13,140,Gasoline,2.7448\nNissan Tiida,2X10608,102.77,140,Gasoline,8.6494\nNissan Tiida,2X10608,3.94,140,Gasoline,0.3008\nNissan Tiida,2X10608,3.27,140,Gasoline,0.2635\nNissan Tiida,2X10608,11.72,140,Gasoline,0.849\nNissan Tiida,2X10608,158.19,140,Gasoline,9.3205\nNissan Tiida,2X10608,7.99,140,Gasoline,0.5822\nNissan Tiida,2X10608,109.74,140,Gasoline,6.3542\nNissan Tiida,2X10608,123.43,140,Gasoline,7.5056\nNissan Tiida,2X10608,7.51,140,Gasoline,0.6291\nNissan Tiida,2X10608,5.33,140,Gasoline,0.4067\nNissan Tiida,2X10608,192.18,140,Gasoline,10.9794\nNissan Tiida,2X10608,154.46,140,Gasoline,9.4252\nNissan Tiida,2X10608,14.24,140,Gasoline,1.4243\nNissan Tiida,2X10608,122.89,140,Gasoline,7.2948\nNissan Tiida,2X10608,65.29,140,Gasoline,4.8265\nNissan Tiida,2X10608,121.55,140,Gasoline,6.7789\nNissan Tiida,2X10608,11.78,140,Gasoline,0.8503\nNissan Tiida,2X10608,118.56,140,Gasoline,7.3395\nNissan Tiida,2X10608,81.06,140,Gasoline,4.7082\nNissan Tiida,2X10608,246.77,140,Gasoline,15.0899\nNissan Tiida,2X10608,119.35,140,Gasoline,7.0773\nNissan Tiida,2X10608,2.73,140,Gasoline,0.2527\nNissan Tiida,2X10608,20.6,140,Gasoline,1.551\nNissan Tiida,2X10608,120.75,140,Gasoline,9.6317\nNissan Tiida,2X10608,140.5,140,Gasoline,8.3465\nNissan Tiida,2X10608,179.47,140,Gasoline,10.8604\nNissan Tiida,2X10608,160.27,140,Gasoline,9.556\nNissan Tiida,2X10608,4.65,140,Gasoline,0.3849\nNissan Tiida,2X10608,19.59,140,Gasoline,1.2589\nNissan Tiida,2X10608,68.22,140,Gasoline,3.9449\nNissan Tiida,2X10608,10.5,140,Gasoline,0.6428\nNissan Tiida,2X10608,103.48,140,Gasoline,6.7579\nNissan Tiida,2X10608,3.47,140,Gasoline,0.4313\nNissan Tiida,2X10608,51.22,140,Gasoline,5.0799\nNissan Tiida,2X10608,77.28,140,Gasoline,4.7466\nNissan Tiida,2X10608,7.38,140,Gasoline,0.5489\nNissan Tiida,2X10608,33.74,140,Gasoline,2.4923\nNissan Tiida,2X10608,25.15,140,Gasoline,1.9281\nNissan Tiida,2X10608,7.14,140,Gasoline,0.5905\nNissan Tiida,2X10608,10.87,140,Gasoline,0.9704\nNissan Tiida,2X10608,2.86,140,Gasoline,0.3404\nNissan Tiida,2X10608,12.11,140,Gasoline,1.417\nNissan Tiida,2X10608,10.99,140,Gasoline,1.0976\nNissan Tiida,2X10608,7.75,140,Gasoline,0.6231\nNissan Tiida,2X10608,56.73,140,Gasoline,3.834\nNissan Tiida,2X10608,4.6,140,Gasoline,0.3484\nNissan Tiida,2X10608,116.14,140,Gasoline,8.6941\nNissan Tiida,2X10608,107.34,140,Gasoline,7.8463\nNissan Tiida,2X10608,107.11,140,Gasoline,8.482\nNissan Tiida,2X10608,3.75,140,Gasoline,0.3043\nNissan Tiida,2X10608,40.96,140,Gasoline,3.441\nNissan Tiida,2X10608,120.69,140,Gasoline,7.608\nNissan Tiida,2X10608,3.65,140,Gasoline,0.3184\nNissan Tiida,2X10608,104.11,140,Gasoline,5.8661\nNissan Tiida,2X10608,108.79,140,Gasoline,6.1554\nNissan Tiida,2X10608,101.17,140,Gasoline,9.9627\nNissan Tiida,2X10608,55.35,140,Gasoline,2.8479\nNissan Tiida,2X10608,6.64,140,Gasoline,0.6013\nNissan Tiida,2X10608,105.63,140,Gasoline,7.4308\nNissan Tiida,2X10608,4.14,140,Gasoline,0.5421\nNissan Tiida,2X10608,47.49,140,Gasoline,2.8915\nNissan Tiida,2X10608,22.2,140,Gasoline,1.6728\nNissan Tiida,2X10608,74.34,140,Gasoline,5.7177\nNissan Tiida,2X10608,150.23,140,Gasoline,10.2121\nNissan Tiida,2X10608,304.67,140,Gasoline,19.7921\nNissan Tiida,2X10608,183.36,140,Gasoline,10.5895\nNissan Tiida,2X10608,36.26,140,Gasoline,4.8855\nNissan Tiida,2X10608,202.2,140,Gasoline,11.4791\nNissan Tiida,2X10608,117.69,140,Gasoline,9.9399\nNissan Tiida,2X10608,12.0,140,Gasoline,1.0518\nNissan Tiida,2X10608,32.56,140,Gasoline,2.2098\nNissan Tiida,2X10608,31.93,140,Gasoline,1.9281\nNissan Tiida,2X10608,103.24,140,Gasoline,7.185\nNissan Tiida,2X10608,30.93,140,Gasoline,2.1549\nNissan Tiida,2X10608,25.1,140,Gasoline,1.7379\nNissan Tiida,2X10608,10.59,140,Gasoline,0.7025\nNissan Tiida,2X10608,48.59,140,Gasoline,2.5605\nNissan Tiida,2X10608,34.34,140,Gasoline,3.4864\nNissan Tiida,2X10608,30.24,140,Gasoline,1.9204\nNissan Tiida,2X10608,48.4,140,Gasoline,3.3955\nNissan Tiida,2X10608,34.49,140,Gasoline,3.3778\nNissan Tiida,2X10608,19.7,140,Gasoline,1.5892\nNissan Tiida,2X10608,35.59,140,Gasoline,2.4338\nNissan Tiida,2X10608,125.35,140,Gasoline,9.0555\nNissan Tiida,2X10608,36.08,140,Gasoline,3.1682\nNissan Tiida,2X10608,129.53,140,Gasoline,7.2042\nNissan Tiida,2X10608,2.53,140,Gasoline,0.3076\nNissan Tiida,2X10608,130.79,140,Gasoline,7.9817\nNissan Tiida,2X10608,56.24,140,Gasoline,3.65\nNissan Tiida,2X10608,30.48,140,Gasoline,2.0611\nNissan Tiida,2X10608,2.47,140,Gasoline,0.2047\nNissan Tiida,2X10608,12.2,140,Gasoline,0.8771\nNissan Tiida,2X10608,19.05,140,Gasoline,1.5714\nNissan Tiida,2X10608,24.1,140,Gasoline,1.8404\nNissan Tiida,2X10608,4.22,140,Gasoline,0.4527\nNissan Tiida,2X10608,7.21,140,Gasoline,0.9174\nNissan Tiida,2X10608,10.59,140,Gasoline,0.9726\nNissan Tiida,2X10608,103.21,140,Gasoline,6.6077\nNissan Tiida,2X10608,6.89,140,Gasoline,0.6567\nNissan Tiida,2X10608,1.8,140,Gasoline,0.2992\nNissan Tiida,2X10608,40.05,140,Gasoline,4.2139\nNissan Tiida,2X10608,27.38,140,Gasoline,1.9751\nNissan Tiida,2X10608,13.22,140,Gasoline,1.3379\nNissan Tiida,2X10608,129.11,140,Gasoline,8.9933\nNissan Tiida,2X10608,2.04,140,Gasoline,0.2215\nNissan Tiida,2X10608,127.94,140,Gasoline,11.6902\nNissan Tiida,2X10608,17.08,140,Gasoline,1.6753\nNissan Tiida,2X10608,4.22,140,Gasoline,0.4349\nNissan Tiida,2X10608,120.26,140,Gasoline,11.7191\nNissan Tiida,2X10608,31.59,140,Gasoline,2.1896\nNissan Tiida,2X10608,14.23,140,Gasoline,1.33\nNissan Tiida,2X10608,100.55,140,Gasoline,9.934\nNissan Tiida,2X10608,3.59,140,Gasoline,0.4147\nNissan Tiida,2X10608,34.31,140,Gasoline,2.7327\nNissan Tiida,2X10608,24.71,140,Gasoline,2.3507\nNissan Tiida,2X10608,77.64,140,Gasoline,5.1235\nNissan Tiida,2X10608,4.49,140,Gasoline,0.4879\nNissan Tiida,2X10608,105.45,140,Gasoline,7.0658\nNissan Tiida,2X10608,213.06,140,Gasoline,13.4185\nNissan Tiida,2X10608,27.48,140,Gasoline,2.0506\nNissan Tiida,2X10608,10.97,140,Gasoline,0.8965\nNissan Tiida,2X10608,1.76,140,Gasoline,0.2344\nNissan Tiida,2X10608,2.73,140,Gasoline,0.2526\nNissan Tiida,2X10608,27.67,140,Gasoline,1.8105\nNissan Tiida,2X10608,4.23,140,Gasoline,0.5265\nNissan Tiida,2X10608,19.99,140,Gasoline,1.5436\nNissan Tiida,2X10608,2.09,140,Gasoline,0.2146\nNissan Tiida,2X10608,171.83,140,Gasoline,11.7163\nNissan Tiida,2X10608,37.79,140,Gasoline,3.4044\nNissan Tiida,2X10608,13.24,140,Gasoline,1.1236\nNissan Tiida,2X10608,12.55,140,Gasoline,0.9108\nNissan Tiida,2X10608,11.06,140,Gasoline,0.8053\nNissan Tiida,2X10608,7.16,140,Gasoline,0.9614\nNissan Tiida,2X10608,24.06,140,Gasoline,2.1229\nNissan Tiida,2X10608,9.8,140,Gasoline,0.6215\nNissan Tiida,2X10608,1.8,140,Gasoline,0.1398\nNissan Tiida,2X10608,26.3,140,Gasoline,2.2068\nNissan Tiida,2X10608,116.13,140,Gasoline,8.7795\nNissan Tiida,2X10608,19.11,140,Gasoline,1.2683\nNissan Tiida,2X10608,15.43,140,Gasoline,1.1577\nNissan Tiida,2X10608,3.34,140,Gasoline,0.4589\nNissan Tiida,2X10608,5.19,140,Gasoline,0.4043\nNissan Tiida,2X10608,7.06,140,Gasoline,0.684\nNissan Tiida,2X10608,56.34,140,Gasoline,3.4124\nNissan Tiida,2X10608,2.1,140,Gasoline,0.2277\nNissan Tiida,2X10608,41.34,140,Gasoline,3.2089\nNissan Tiida,2X10608,30.98,140,Gasoline,1.9242\nNissan Tiida,2X10608,9.0,140,Gasoline,0.5908\nNissan Tiida,2X10608,69.91,140,Gasoline,4.9517\nNissan Tiida,2X10608,18.62,140,Gasoline,1.6403\nNissan Tiida,2X10608,36.06,140,Gasoline,2.5416\nNissan Tiida,2X10608,39.67,140,Gasoline,2.7211\nNissan Tiida,2X10608,50.23,140,Gasoline,3.1616\nNissan Tiida,2X10608,9.33,140,Gasoline,0.8502\nNissan Tiida,2X10608,16.47,140,Gasoline,1.4984\nNissan Tiida,2X10608,7.31,140,Gasoline,0.5274\nNissan Tiida,2X10608,5.77,140,Gasoline,0.3716\nNissan Tiida,2X10608,102.37,140,Gasoline,9.2069\nNissan Tiida,2X10608,9.43,140,Gasoline,1.1074\nNissan Tiida,2X10608,42.79,140,Gasoline,2.1337\nNissan Tiida,2X10608,14.33,140,Gasoline,1.5055\nNissan Tiida,2X10608,109.82,140,Gasoline,8.913\nNissan Tiida,2X10608,55.09,140,Gasoline,3.2192\nNissan Tiida,2X10608,19.35,140,Gasoline,1.462\nNissan Tiida,2X10608,54.76,140,Gasoline,3.6869\nNissan Tiida,2X10608,18.4,140,Gasoline,1.4314\nNissan Tiida,2X10608,31.26,140,Gasoline,2.3279\nNissan Tiida,2X10608,2.14,140,Gasoline,0.3229\nNissan Tiida,2X10608,26.15,140,Gasoline,2.3335\nNissan Tiida,2X10608,21.35,140,Gasoline,1.5538\nNissan Tiida,2X10608,103.28,140,Gasoline,6.3527\nNissan Tiida,2X10608,7.89,140,Gasoline,0.9283\nNissan Tiida,2X10608,71.24,140,Gasoline,5.0744\nNissan Tiida,2X10608,37.24,140,Gasoline,2.3394\nNissan Tiida,2X10608,136.77,140,Gasoline,12.1609\nNissan Tiida,2X10608,132.88,140,Gasoline,9.4535\nNissan Tiida,2X10608,53.32,140,Gasoline,4.3642\nNissan Tiida,2X10608,109.76,140,Gasoline,7.8741\nNissan Tiida,2X10608,115.26,140,Gasoline,9.9743\nNissan Tiida,2X10608,171.58,140,Gasoline,12.0302\nNissan Tiida,2X10608,53.49,140,Gasoline,3.6221\nNissan Tiida,2X10608,63.26,140,Gasoline,5.3439\nNissan Tiida,2X10608,4.62,140,Gasoline,0.4815\nNissan Tiida,2X10608,58.8,140,Gasoline,3.1333\nNissan Tiida,2X10608,101.25,140,Gasoline,6.1031\nNissan Tiida,2X10608,7.36,140,Gasoline,0.6252\nNissan Tiida,2X10608,42.39,140,Gasoline,3.4233\nNissan Tiida,2X10608,58.02,140,Gasoline,5.19\nNissan Tiida,2X10608,12.78,140,Gasoline,0.9096\nNissan Tiida,2X10608,10.0,140,Gasoline,0.919\nNissan Tiida,2X10608,27.22,140,Gasoline,2.1921\nNissan Tiida,2X10608,7.03,140,Gasoline,0.6405\nNissan Tiida,2X10608,11.35,140,Gasoline,0.8766\nNissan Tiida,2X10608,4.02,140,Gasoline,0.3936\nNissan Tiida,2X10608,7.19,140,Gasoline,0.4837\nNissan Tiida,2X10608,23.79,140,Gasoline,1.411\nNissan Tiida,2X10608,65.02,140,Gasoline,5.8715\nNissan Tiida,2X10608,121.28,140,Gasoline,8.6287\nNissan Tiida,2X10608,4.82,140,Gasoline,0.5667\nNissan Tiida,2X10608,25.72,140,Gasoline,1.3705\nNissan Tiida,2X10608,8.54,140,Gasoline,0.5628\nNissan Tiida,2X10608,2.67,140,Gasoline,0.1991\nNissan Tiida,2X10608,31.41,140,Gasoline,1.8326\nNissan Tiida,2X10608,9.72,140,Gasoline,0.7079\nNissan Tiida,2X10608,76.22,140,Gasoline,4.7284\nNissan Tiida,2X10608,117.64,140,Gasoline,9.2401\nNissan Tiida,2X10608,98.49,140,Gasoline,8.1253\nNissan Tiida,2X10608,47.23,140,Gasoline,2.6383\nNissan Tiida,2X10608,46.13,140,Gasoline,3.09\nNissan Tiida,2X10608,27.42,140,Gasoline,1.9371\nNissan Tiida,2X10608,66.08,140,Gasoline,5.2703\nNissan Tiida,2X10608,20.97,140,Gasoline,1.7421\nNissan Tiida,2X10608,54.36,140,Gasoline,3.2003\nNissan Tiida,2X10608,45.66,140,Gasoline,2.7342\nNissan Tiida,2X10608,41.16,140,Gasoline,2.3225\nNissan Tiida,2X10608,97.27,140,Gasoline,7.803\nNissan Tiida,2X10608,109.31,140,Gasoline,6.1751\nNissan Tiida,2X10608,10.1,140,Gasoline,0.7134\nNissan Tiida,2X10608,148.69,140,Gasoline,10.2537\nNissan Tiida,2X10608,10.43,140,Gasoline,0.766\nNissan Tiida,2X10608,172.08,140,Gasoline,13.6812\nNissan Tiida,2X10608,34.48,140,Gasoline,1.893\nNissan Tiida,2X10608,38.35,140,Gasoline,2.8975\nNissan Tiida,2X10608,2.98,140,Gasoline,0.2677\nNissan Tiida,2X10608,178.47,140,Gasoline,9.3997\nNissan Tiida,2X10608,21.09,140,Gasoline,1.1823\nNissan Tiida,2X10608,110.49,140,Gasoline,8.5114\nNissan Tiida,2X10608,37.54,140,Gasoline,1.9673\nNissan Tiida,2X10608,30.14,140,Gasoline,1.9883\nNissan Tiida,2X10608,42.4,140,Gasoline,2.555\nNissan Tiida,2X10608,18.81,140,Gasoline,1.7856\nNissan Tiida,2X10608,33.8,140,Gasoline,1.8671\nNissan Tiida,2X10608,170.68,140,Gasoline,9.6885\nNissan Tiida,2X10608,71.48,140,Gasoline,4.0633\nNissan Tiida,2X10608,4.22,140,Gasoline,0.4026\nNissan Tiida,2X10608,21.49,140,Gasoline,1.2459\nNissan Tiida,2X10608,104.76,140,Gasoline,7.2551\nNissan Tiida,2X10608,37.26,140,Gasoline,3.7065\nNissan Tiida,2X10608,34.68,140,Gasoline,2.3471\nNissan Tiida,2X10608,36.07,140,Gasoline,1.4492\nNissan Tiida,2X10608,122.91,140,Gasoline,6.7281\nNissan Tiida,2X10608,47.51,140,Gasoline,3.8782\nNissan Tiida,2X10608,10.84,140,Gasoline,0.644\nNissan Tiida,2X10608,37.83,140,Gasoline,2.2891\nNissan Tiida,2X10608,39.92,140,Gasoline,2.9333\nNissan Tiida,2X10608,3.16,140,Gasoline,0.2497\nNissan Tiida,2X10608,111.64,140,Gasoline,8.7021\nNissan Tiida,2X10608,4.28,140,Gasoline,0.4646\nNissan Tiida,2X10608,10.01,140,Gasoline,0.9213\nNissan Tiida,2X10608,34.42,140,Gasoline,1.6975\nNissan Tiida,2X10608,11.32,140,Gasoline,0.7696\nNissan Tiida,2X10608,113.12,140,Gasoline,9.0655\nNissan Tiida,2X10608,4.64,140,Gasoline,0.5731\nNissan Tiida,2X10608,47.17,140,Gasoline,4.5886\nNissan Tiida,2X10608,28.28,140,Gasoline,1.5874\nNissan Tiida,2X10608,63.29,140,Gasoline,3.6864\nNissan Tiida,2X10608,40.61,140,Gasoline,2.4139\nNissan Tiida,2X10608,100.82,140,Gasoline,8.1221\nNissan Tiida,2X10608,38.52,140,Gasoline,2.5254\nNissan Tiida,2X10608,42.64,140,Gasoline,2.6811\nNissan Tiida,2X10608,14.46,140,Gasoline,1.1616\nNissan Tiida,2X10608,113.14,140,Gasoline,8.7948\nNissan Tiida,2X10608,21.34,140,Gasoline,1.4359\nNissan Tiida,2X10608,2.74,140,Gasoline,0.2082\nNissan Tiida,2X10608,101.89,140,Gasoline,7.7763\nNissan Tiida,2X10608,4.5,140,Gasoline,0.6627\nNissan Tiida,2X10608,12.14,140,Gasoline,0.9184\nNissan Tiida,2X10608,4.63,140,Gasoline,0.5485\nNissan Tiida,2X10608,10.72,140,Gasoline,0.9792\nNissan Tiida,2X10608,43.44,140,Gasoline,2.4302\nNissan Tiida,2X10608,5.86,140,Gasoline,0.5675\nNissan Tiida,2X10608,4.28,140,Gasoline,0.4534\nNissan Tiida,2X10608,103.28,140,Gasoline,8.7737\nNissan Tiida,2X10608,1.72,140,Gasoline,0.1762\nNissan Tiida,2X10608,3.87,140,Gasoline,0.3028\nNissan Tiida,2X10608,42.96,140,Gasoline,2.6489\nNissan Tiida,2X10608,59.61,140,Gasoline,3.6732\nNissan Tiida,2X10608,6.66,140,Gasoline,0.467\nNissan Tiida,2X10608,6.01,140,Gasoline,0.5462\nNissan Tiida,2X10608,10.46,140,Gasoline,0.8444\nNissan Tiida,2X10608,3.29,140,Gasoline,0.312\nNissan Tiida,2X10608,24.14,140,Gasoline,1.6699\nNissan Tiida,2X10608,35.24,140,Gasoline,2.2078\nNissan Tiida,2X10608,10.28,140,Gasoline,0.8167\nNissan Tiida,2X10608,20.24,140,Gasoline,1.4956\nNissan Tiida,2X10608,43.06,140,Gasoline,3.3923\nNissan Tiida,2X10608,101.77,140,Gasoline,6.2296\nNissan Tiida,2X10608,10.61,140,Gasoline,0.7541\nNissan Tiida,2X10608,63.78,140,Gasoline,4.4071\nNissan Tiida,2X10608,85.93,140,Gasoline,6.6714\nNissan Tiida,2X10608,166.82,140,Gasoline,10.9229\nNissan Tiida,2X10608,47.8,140,Gasoline,3.396\nNissan Tiida,2X10608,35.94,140,Gasoline,2.1283\nNissan Tiida,2X10608,35.9,140,Gasoline,2.3369\nNissan Tiida,2X10608,16.3,140,Gasoline,1.6812\nNissan Tiida,2X10608,149.51,140,Gasoline,10.1261\nNissan Tiida,2X10608,52.59,140,Gasoline,4.1861\nNissan Tiida,2X10608,4.19,140,Gasoline,0.5262\nNissan Tiida,2X10608,16.06,140,Gasoline,1.4397\nNissan Tiida,2X10608,10.66,140,Gasoline,0.8619\nNissan Tiida,2X10608,6.53,140,Gasoline,0.5375\nNissan Tiida,2X10608,6.57,140,Gasoline,0.7256\nNissan Tiida,2X10608,11.33,140,Gasoline,1.0521\nNissan Tiida,2X10608,14.06,140,Gasoline,1.5252\nNissan Tiida,2X10608,4.64,140,Gasoline,0.4297\nNissan Tiida,2X10608,4.61,140,Gasoline,0.4686\nNissan Tiida,2X10608,10.32,140,Gasoline,0.8089\nNissan Tiida,2X10608,3.53,140,Gasoline,0.5119\nNissan Tiida,2X10608,100.05,140,Gasoline,6.3144\nNissan Tiida,2X10608,187.35,140,Gasoline,11.9372\nNissan Tiida,2X10608,33.32,140,Gasoline,1.8405\nNissan Tiida,2X10608,35.24,140,Gasoline,2.8101\nNissan Tiida,2X10608,120.83,140,Gasoline,8.0068\nNissan Tiida,2X10608,159.81,140,Gasoline,11.389\nNissan Tiida,2X10608,15.36,140,Gasoline,1.2184\nNissan Tiida,2X10608,15.38,140,Gasoline,0.8323\nNissan Tiida,2X10608,13.33,140,Gasoline,1.0429\nNissan Tiida,2X10608,4.62,140,Gasoline,0.4556\nNissan Tiida,2X10608,21.46,140,Gasoline,1.8687\nNissan Tiida,2X10608,5.66,140,Gasoline,0.8188\nNissan Tiida,2X10608,40.49,140,Gasoline,2.7737\nNissan Tiida,2X10608,37.56,140,Gasoline,2.2929\nNissan Tiida,2X10608,4.56,140,Gasoline,0.5868\nNissan Tiida,2X10608,4.71,140,Gasoline,0.6816\nNissan Tiida,2X10608,16.1,140,Gasoline,1.2135\nNissan Tiida,2X10608,9.3,140,Gasoline,0.8403\nNissan Tiida,2X10608,38.49,140,Gasoline,2.2167\nNissan Tiida,2X10608,106.36,140,Gasoline,7.135\nNissan Tiida,2X10608,141.63,140,Gasoline,11.824\nNissan Tiida,2X10608,98.0,140,Gasoline,7.7187\nNissan Tiida,2X10608,34.9,140,Gasoline,3.0945\nNissan Tiida,2X10608,45.45,140,Gasoline,2.3745\nNissan Tiida,2X10608,10.59,140,Gasoline,0.9582\nNissan Tiida,2X10608,2.63,140,Gasoline,0.2789\nNissan Tiida,2X10608,129.7,140,Gasoline,10.4039\nNissan Tiida,2X10608,17.8,140,Gasoline,1.3533\nNissan Tiida,2X10608,124.36,140,Gasoline,9.8506\nNissan Tiida,2X10608,16.5,140,Gasoline,1.1468\nNissan Tiida,2X10608,34.29,140,Gasoline,2.5406\nNissan Tiida,2X10608,9.07,140,Gasoline,0.6718\nNissan Tiida,2X10608,132.78,140,Gasoline,9.365\nNissan Tiida,2X10608,14.32,140,Gasoline,0.9027\nNissan Tiida,2X10608,13.12,140,Gasoline,0.9581\nNissan Tiida,2X10608,10.1,140,Gasoline,0.8537\nNissan Tiida,2X10608,153.57,140,Gasoline,10.6553\nNissan Tiida,2X10608,202.56,140,Gasoline,14.1861\nNissan Tiida,2X10608,107.76,140,Gasoline,7.3226\nNissan Tiida,2X10608,4.28,140,Gasoline,0.426\nNissan Tiida,2X10608,160.86,140,Gasoline,13.2893\nNissan Tiida,2X10608,28.11,140,Gasoline,1.7096\nNissan Tiida,2X10608,19.92,140,Gasoline,2.4013\nNissan Tiida,2X10608,9.4,140,Gasoline,0.7488\nNissan Tiida,2X10608,147.58,140,Gasoline,11.8423\nNissan Tiida,2X10608,161.88,140,Gasoline,11.02\nNissan Tiida,2X10608,79.99,140,Gasoline,6.5045\nNissan Tiida,2X10608,119.1,140,Gasoline,8.6426\nNissan Tiida,2X10608,114.06,140,Gasoline,7.4664\nNissan Tiida,2X10608,20.39,140,Gasoline,1.4877\nNissan Tiida,2X10608,34.51,140,Gasoline,3.0047\nNissan Tiida,2X10608,58.85,140,Gasoline,3.9555\nNissan Tiida,2X10608,115.08,140,Gasoline,8.093\nNissan Tiida,2X10608,8.44,140,Gasoline,0.8757\nNissan Tiida,2X10608,9.9,140,Gasoline,1.0387\nNissan Tiida,2X10608,137.83,140,Gasoline,8.4758\nNissan Tiida,2X10608,38.76,140,Gasoline,2.0668\nNissan Tiida,2X10608,15.12,140,Gasoline,1.241\nNissan Tiida,2X10608,44.82,140,Gasoline,2.406\nNissan Tiida,2X10608,3.69,140,Gasoline,0.3578\nNissan Tiida,2X10608,113.3,140,Gasoline,7.9147\nNissan Tiida,2X10608,6.09,140,Gasoline,0.5011\nNissan Tiida,2X10608,3.1,140,Gasoline,0.3878\nNissan Tiida,2X10608,189.46,140,Gasoline,14.0215\nVolkswagen Magotan,0X3050M,188.12,118,Gasoline,15.9218\nVolkswagen Magotan,0X3050X,45.01,118,Gasoline,2.8705\nVolkswagen Magotan,0X3050X,66.3,118,Gasoline,5.3039\nVolkswagen Magotan,0X3050X,6.67,118,Gasoline,0.7953\nVolkswagen Magotan,0X3050M,1.9,118,Gasoline,0.33\nVolkswagen Magotan,0X3050M,34.16,118,Gasoline,2.5843\nVolkswagen Magotan,0X3050M,140.34,118,Gasoline,11.2399\nVolkswagen Magotan,0X3050X,65.44,118,Gasoline,5.1925\nVolkswagen Magotan,0X3050M,19.28,118,Gasoline,1.5226\nVolkswagen Magotan,0X3050X,10.26,118,Gasoline,0.969\nVolkswagen Magotan,0X3050X,107.38,118,Gasoline,8.6327\nVolkswagen Magotan,0X3050X,3.13,118,Gasoline,0.3917\nVolkswagen Magotan,0X3050M,3.83,118,Gasoline,0.6495\nVolkswagen Magotan,0X3050X,10.66,118,Gasoline,1.2885\nVolkswagen Magotan,0X3050M,78.12,118,Gasoline,7.1967\nVolkswagen Magotan,0X3050M,42.86,118,Gasoline,4.3047\nVolkswagen Magotan,0X3050X,3.54,118,Gasoline,0.4505\nVolkswagen Magotan,0X3050X,6.41,118,Gasoline,0.6947\nVolkswagen Magotan,0X3050X,8.74,118,Gasoline,0.974\nVolkswagen Magotan,0X3050X,39.52,118,Gasoline,3.2976\nVolkswagen Magotan,0X3050X,14.24,118,Gasoline,1.4972\nVolkswagen Magotan,0X3050X,2.82,118,Gasoline,0.3173\nVolkswagen Magotan,0X3050X,31.14,118,Gasoline,4.1615\nVolkswagen Magotan,0X3050M,157.84,118,Gasoline,13.6553\nVolkswagen Magotan,0X3050M,42.37,118,Gasoline,4.7366\nVolkswagen Magotan,0X3050X,3.43,118,Gasoline,0.3776\nVolkswagen Magotan,0X3050X,76.24,118,Gasoline,5.8501\nVolkswagen Magotan,0X3050X,5.82,118,Gasoline,0.8309\nVolkswagen Magotan,0X3050X,66.17,118,Gasoline,5.1114\nVolkswagen Magotan,0X3050X,10.72,118,Gasoline,1.1129\nVolkswagen Magotan,0X3050X,6.65,118,Gasoline,0.7803\nVolkswagen Magotan,0X3050M,13.47,118,Gasoline,1.3135\nVolkswagen Magotan,0X3050X,7.84,118,Gasoline,0.8995\nVolkswagen Magotan,0X3050M,30.57,118,Gasoline,2.8776\nVolkswagen Magotan,0X3050M,32.26,118,Gasoline,2.3543\nVolkswagen Magotan,0X3050X,38.48,118,Gasoline,3.511\nVolkswagen Magotan,0X3050M,19.15,118,Gasoline,2.664\nVolkswagen Magotan,0X3050X,66.35,118,Gasoline,5.0148\nVolkswagen Magotan,0X3050M,115.62,118,Gasoline,8.3181\nVolkswagen Magotan,0X3050X,100.41,118,Gasoline,6.8885\nVolkswagen Magotan,0X3050M,114.29,118,Gasoline,8.1606\nVolkswagen Magotan,0X3050X,49.21,118,Gasoline,3.5336\nVolkswagen Magotan,0X3050X,103.3,118,Gasoline,7.0515\nVolkswagen Magotan,0X3050X,15.12,118,Gasoline,1.5887\nVolkswagen Magotan,0X3050X,151.31,118,Gasoline,11.1565\nVolkswagen Magotan,0X3050M,30.01,118,Gasoline,3.5412\nVolkswagen Magotan,0X3050M,14.66,118,Gasoline,1.5545\nVolkswagen Magotan,0X3050X,183.38,118,Gasoline,16.4662\nVolkswagen Magotan,0X3050X,78.93,118,Gasoline,5.0881\nVolkswagen Magotan,0X3050M,13.41,118,Gasoline,1.3121\nVolkswagen Magotan,0X3050X,370.35,118,Gasoline,29.5737\nVolkswagen Magotan,0X3050X,2.5,118,Gasoline,0.3342\nVolkswagen Magotan,0X3050M,25.1,118,Gasoline,2.1037\nVolkswagen Magotan,0X3050X,145.38,118,Gasoline,11.0587\nVolkswagen Magotan,0X3050X,4.39,118,Gasoline,0.4576\nVolkswagen Magotan,0X3050X,12.82,118,Gasoline,1.3285\nVolkswagen Magotan,0X3050X,142.04,118,Gasoline,10.5084\nVolkswagen Magotan,0X3050M,9.13,118,Gasoline,1.2028\nVolkswagen Magotan,0X3050M,23.08,118,Gasoline,1.8624\nVolkswagen Magotan,0X3050X,36.65,118,Gasoline,2.7885\nVolkswagen Magotan,0X3050M,2.06,118,Gasoline,0.3064\nVolkswagen Magotan,0X3050M,4.13,118,Gasoline,0.4838\nVolkswagen Magotan,0X3050X,114.2,118,Gasoline,9.3128\nVolkswagen Magotan,0X3050X,138.91,118,Gasoline,13.1504\nVolkswagen Magotan,0X3050X,142.37,118,Gasoline,11.6573\nVolkswagen Magotan,0X3050X,182.65,118,Gasoline,14.0711\nVolkswagen Magotan,0X3050X,135.93,118,Gasoline,10.4006\nVolkswagen Magotan,0X3050X,227.69,118,Gasoline,17.9207\nVolkswagen Magotan,0X3050M,14.12,118,Gasoline,1.185\nVolkswagen Magotan,0X3050X,121.1,118,Gasoline,9.3661\nVolkswagen Magotan,0X3050X,148.68,118,Gasoline,11.2461\nVolkswagen Magotan,0X3050X,104.75,118,Gasoline,8.5792\nVolkswagen Magotan,0X3050X,129.37,118,Gasoline,9.5347\nVolkswagen Magotan,0X3050X,181.38,118,Gasoline,12.6805\nVolkswagen Magotan,0X3050X,242.44,118,Gasoline,21.0012\nVolkswagen Magotan,0X3050X,16.8,118,Gasoline,1.4258\nVolkswagen Magotan,0X3050M,22.62,118,Gasoline,1.7979\nVolkswagen Magotan,0X3050X,1.84,118,Gasoline,0.3078\nVolkswagen Magotan,0X3050X,60.61,118,Gasoline,3.9318\nVolkswagen Magotan,0X3050M,15.25,118,Gasoline,1.9393\nVolkswagen Magotan,0X3050M,2.89,118,Gasoline,0.5137\nVolkswagen Magotan,0X3050X,13.81,118,Gasoline,1.3604\nVolkswagen Magotan,0X3050X,7.9,118,Gasoline,0.835\nVolkswagen Magotan,0X3050X,180.38,118,Gasoline,17.9587\nVolkswagen Magotan,0X3050X,40.41,118,Gasoline,3.2518\nVolkswagen Magotan,0X3050X,241.08,118,Gasoline,22.8804\nVolkswagen Magotan,0X3050X,137.89,118,Gasoline,9.952\nVolkswagen Magotan,0X3050M,7.86,118,Gasoline,0.7175\nVolkswagen Magotan,0X3050M,101.06,118,Gasoline,6.5657\nVolkswagen Magotan,0X3050M,104.81,118,Gasoline,6.8989\nVolkswagen Magotan,0X3050M,89.35,118,Gasoline,6.454\nVolkswagen Magotan,0X3050X,61.7,118,Gasoline,4.0392\nVolkswagen Magotan,0X3050M,30.9,118,Gasoline,3.072\nVolkswagen Magotan,0X3050X,12.57,118,Gasoline,1.1399\nVolkswagen Magotan,0X3050X,15.8,118,Gasoline,1.2786\nVolkswagen Magotan,0X3050X,5.87,118,Gasoline,0.7256\nVolkswagen Magotan,0X3050X,12.45,118,Gasoline,1.3547\nVolkswagen Magotan,0X3050X,104.17,118,Gasoline,5.8973\nVolkswagen Magotan,0X3050M,40.71,118,Gasoline,3.032\nVolkswagen Magotan,0X3050M,207.06,118,Gasoline,16.1903\nVolkswagen Magotan,0X3050X,5.18,118,Gasoline,0.6648\nVolkswagen Magotan,0X3050M,239.54,118,Gasoline,18.3215\nVolkswagen Magotan,0X3050X,3.96,118,Gasoline,0.5031\nVolkswagen Magotan,0X3050X,89.76,118,Gasoline,7.5883\nVolkswagen Magotan,0X3050X,117.54,118,Gasoline,10.5547\nVolkswagen Magotan,0X3050M,17.75,118,Gasoline,1.1866\nVolkswagen Magotan,0X3050X,2.19,118,Gasoline,0.4389\nVolkswagen Magotan,0X3050X,106.72,118,Gasoline,7.6472\nVolkswagen Magotan,0X3050M,7.26,118,Gasoline,0.9285\nVolkswagen Magotan,0X3050M,15.24,118,Gasoline,1.7269\nVolkswagen Magotan,0X3050X,37.8,118,Gasoline,2.8645\nVolkswagen Magotan,0X3050M,13.71,118,Gasoline,1.598\nVolkswagen Magotan,0X3050M,63.93,118,Gasoline,4.7857\nVolkswagen Magotan,0X3050M,105.49,118,Gasoline,7.6609\nVolkswagen Magotan,0X3050M,13.35,118,Gasoline,1.2781\nVolkswagen Magotan,0X3050M,69.96,118,Gasoline,5.2352\nVolkswagen Magotan,0X3050X,15.11,118,Gasoline,1.24\nVolkswagen Magotan,0X3050M,18.02,118,Gasoline,1.9426\nVolkswagen Magotan,0X3050X,100.54,118,Gasoline,8.0215\nVolkswagen Magotan,0X3050X,103.42,118,Gasoline,6.9501\nVolkswagen Magotan,0X3050X,103.57,118,Gasoline,7.8951\nVolkswagen Magotan,0X3050M,5.82,118,Gasoline,0.6071\nVolkswagen Magotan,0X3050M,25.53,118,Gasoline,2.3582\nVolkswagen Magotan,0X3050X,27.92,118,Gasoline,2.4909\nVolkswagen Magotan,0X3050M,3.31,118,Gasoline,0.39\nVolkswagen Magotan,0X3050X,190.21,118,Gasoline,13.8131\nVolkswagen Magotan,0X3050M,17.28,118,Gasoline,1.8338\nVolkswagen Magotan,0X3050M,3.72,118,Gasoline,0.3666\nVolkswagen Magotan,0X3050X,14.43,118,Gasoline,1.3379\nVolkswagen Magotan,0X3050M,9.9,118,Gasoline,1.4054\nVolkswagen Magotan,0X3050X,1.75,118,Gasoline,0.3141\nVolkswagen Magotan,0X3050X,1.94,118,Gasoline,0.2847\nVolkswagen Magotan,0X3050X,4.14,118,Gasoline,0.57\nVolkswagen Magotan,0X3050M,13.33,118,Gasoline,1.2493\nVolkswagen Magotan,0X3050X,6.48,118,Gasoline,0.5297\nVolkswagen Magotan,0X3050M,29.26,118,Gasoline,2.4754\nVolkswagen Magotan,0X3050M,15.53,118,Gasoline,1.1893\nVolkswagen Magotan,0X3050M,2.2,118,Gasoline,0.2184\nVolkswagen Magotan,0X3050M,8.48,118,Gasoline,0.8346\nVolkswagen Magotan,0X3050X,121.93,118,Gasoline,7.7285\nVolkswagen Magotan,0X3050X,119.94,118,Gasoline,7.8107\nVolkswagen Magotan,0X3050X,103.49,118,Gasoline,8.3008\nVolkswagen Magotan,0X3050X,4.68,118,Gasoline,0.5105\nVolkswagen Magotan,0X3050X,14.1,118,Gasoline,1.0903\nVolkswagen Magotan,0X3050M,5.98,118,Gasoline,0.6706\nVolkswagen Magotan,0X3050X,66.78,118,Gasoline,5.5576\nVolkswagen Magotan,0X3050X,101.17,118,Gasoline,5.796\nVolkswagen Magotan,0X3050X,20.22,118,Gasoline,2.7866\nVolkswagen Magotan,0X3050X,32.68,118,Gasoline,2.9427\nVolkswagen Magotan,0X3050X,14.44,118,Gasoline,1.3111\nVolkswagen Magotan,0X3050X,102.04,118,Gasoline,7.2979\nVolkswagen Magotan,0X3050M,27.27,118,Gasoline,2.5356\nVolkswagen Magotan,0X3050X,4.0,118,Gasoline,0.3675\nVolkswagen Magotan,0X3050M,10.91,118,Gasoline,1.4369\nVolkswagen Magotan,0X3050M,152.14,118,Gasoline,10.9475\nVolkswagen Magotan,0X3050X,9.52,118,Gasoline,0.9372\nVolkswagen Magotan,0X3050X,97.17,118,Gasoline,7.8722\nVolkswagen Magotan,0X3050X,81.13,118,Gasoline,5.571\nVolkswagen Magotan,0X3050M,153.12,118,Gasoline,11.8545\nVolkswagen Magotan,0X3050X,30.78,118,Gasoline,2.3202\nVolkswagen Magotan,0X3050X,79.11,118,Gasoline,5.1087\nVolkswagen Magotan,0X3050X,14.61,118,Gasoline,1.46\nVolkswagen Magotan,0X3050M,177.89,118,Gasoline,13.5424\nVolkswagen Magotan,0X3050X,7.46,118,Gasoline,1.0408\nVolkswagen Magotan,0X3050X,13.02,118,Gasoline,1.4944\nVolkswagen Magotan,0X3050X,12.55,118,Gasoline,1.0442\nVolkswagen Magotan,0X3050M,60.26,118,Gasoline,4.948\nVolkswagen Magotan,0X3050M,109.31,118,Gasoline,8.4614\nVolkswagen Magotan,0X3050M,14.93,118,Gasoline,1.3062\nVolkswagen Magotan,0X3050M,113.34,118,Gasoline,7.3947\nVolkswagen Magotan,0X3050X,2.94,118,Gasoline,0.4197\nVolkswagen Magotan,0X3050M,14.13,118,Gasoline,1.8665\nVolkswagen Magotan,0X3050X,36.57,118,Gasoline,2.6371\nVolkswagen Magotan,0X3050M,31.81,118,Gasoline,4.0688\nVolkswagen Magotan,0X3050M,24.7,118,Gasoline,2.3464\nVolkswagen Magotan,0X3050M,2.42,118,Gasoline,0.3379\nVolkswagen Magotan,0X3050M,59.24,118,Gasoline,4.41\nVolkswagen Magotan,0X3050X,31.56,118,Gasoline,2.2539\nVolkswagen Magotan,0X3050X,6.72,118,Gasoline,0.7399\nVolkswagen Magotan,0X3050X,143.97,118,Gasoline,10.2673\nVolkswagen Magotan,0X3050M,7.31,118,Gasoline,0.6552\nVolkswagen Magotan,0X3050M,15.29,118,Gasoline,1.3369\nVolkswagen Magotan,0X3050M,15.33,118,Gasoline,2.0697\nVolkswagen Magotan,0X3050X,33.58,118,Gasoline,2.762\nVolkswagen Magotan,0X3050M,162.8,118,Gasoline,11.7408\nVolkswagen Magotan,0X3050X,2.92,118,Gasoline,0.3148\nVolkswagen Magotan,0X3050M,124.32,118,Gasoline,10.3356\nVolkswagen Magotan,0X3050M,114.98,118,Gasoline,9.873\nVolkswagen Magotan,0X3050X,88.22,118,Gasoline,6.2653\nVolkswagen Magotan,0X3050X,22.76,118,Gasoline,1.7032\nVolkswagen Magotan,0X3050M,68.84,118,Gasoline,4.6009\nVolkswagen Magotan,0X3050M,25.77,118,Gasoline,2.2397\nVolkswagen Magotan,0X3050X,123.79,118,Gasoline,8.1046\nVolkswagen Magotan,0X3050M,55.19,118,Gasoline,4.5849\nVolkswagen Magotan,0X3050M,60.08,118,Gasoline,4.8839\nVolkswagen Magotan,0X3050M,3.24,118,Gasoline,0.4192\nVolkswagen Magotan,0X3050M,47.85,118,Gasoline,4.9392\nVolkswagen Magotan,0X3050X,24.12,118,Gasoline,2.2771\nVolkswagen Magotan,0X3050X,5.84,118,Gasoline,0.754\nVolkswagen Magotan,0X3050X,197.6,118,Gasoline,13.7958\nVolkswagen Magotan,0X3050M,2.2,118,Gasoline,0.2229\nVolkswagen Magotan,0X3050X,33.06,118,Gasoline,3.254\nVolkswagen Magotan,0X3050M,11.68,118,Gasoline,1.3615\nVolkswagen Magotan,0X3050X,23.0,118,Gasoline,2.5145\nVolkswagen Magotan,0X3050M,123.12,118,Gasoline,10.5942\nVolkswagen Magotan,0X3050M,21.21,118,Gasoline,1.5969\nVolkswagen Magotan,0X3050M,1.74,118,Gasoline,0.2534\nVolkswagen Magotan,0X3050X,120.36,118,Gasoline,9.775\nVolkswagen Magotan,0X3050X,118.62,118,Gasoline,9.4905\nVolkswagen Magotan,0X3050X,89.76,118,Gasoline,6.3297\nVolkswagen Magotan,0X3050X,60.07,118,Gasoline,5.3088\nVolkswagen Magotan,0X3050X,157.67,118,Gasoline,10.568\nVolkswagen Magotan,0X3050X,81.67,118,Gasoline,5.4392\nVolkswagen Magotan,0X3050X,76.64,118,Gasoline,5.545\nVolkswagen Magotan,0X3050X,226.12,118,Gasoline,18.6235\nVolkswagen Magotan,0X3050X,92.41,118,Gasoline,6.0929\nVolkswagen Magotan,0X3050M,93.52,118,Gasoline,6.9522\nVolkswagen Magotan,0X3050M,3.16,118,Gasoline,0.2648\nVolkswagen Magotan,0X3050X,157.16,118,Gasoline,10.6957\nVolkswagen Magotan,0X3050X,28.36,118,Gasoline,2.2454\nVolkswagen Magotan,0X3050X,39.75,118,Gasoline,3.0594\nVolkswagen Magotan,0X3050X,106.56,118,Gasoline,9.5008\nVolkswagen Magotan,0X3050X,2.74,118,Gasoline,0.7425\nVolkswagen Magotan,0X3050X,56.04,118,Gasoline,5.5437\nVolkswagen Magotan,0X3050M,141.06,118,Gasoline,9.4278\nVolkswagen Magotan,0X3050X,121.3,118,Gasoline,8.8237\nVolkswagen Magotan,0X3050M,49.88,118,Gasoline,2.9444\nVolkswagen Magotan,0X3050X,2.2,118,Gasoline,0.247\nVolkswagen Magotan,0X3050X,18.53,118,Gasoline,1.7763\nVolkswagen Magotan,0X3050M,15.26,118,Gasoline,1.5503\nVolkswagen Magotan,0X3050M,104.47,118,Gasoline,9.4581\nVolkswagen Magotan,0X3050M,37.42,118,Gasoline,4.8288\nVolkswagen Magotan,0X3050X,101.01,118,Gasoline,7.3048\nVolkswagen Magotan,0X3050M,116.59,118,Gasoline,8.5499\nVolkswagen Magotan,0X3050X,36.19,118,Gasoline,2.6312\nVolkswagen Magotan,0X3050M,9.4,118,Gasoline,1.293\nVolkswagen Magotan,0X3050M,33.81,118,Gasoline,4.0109\nVolkswagen Magotan,0X3050M,71.33,118,Gasoline,6.0818\nVolkswagen Magotan,0X3050M,23.46,118,Gasoline,2.0659\nVolkswagen Magotan,0X3050X,2.59,118,Gasoline,0.2699\nVolkswagen Magotan,0X3050X,5.0,118,Gasoline,0.676\nVolkswagen Magotan,0X3050X,5.95,118,Gasoline,0.5807\nVolkswagen Magotan,0X3050X,12.25,118,Gasoline,1.0623\nVolkswagen Magotan,0X3050X,11.97,118,Gasoline,0.9492\nVolkswagen Magotan,0X3050M,15.7,118,Gasoline,1.6371\nVolkswagen Magotan,0X3050X,38.83,118,Gasoline,2.7729\nVolkswagen Magotan,0X3050X,10.7,118,Gasoline,1.3662\nVolkswagen Magotan,0X3050M,38.35,118,Gasoline,4.2364\nVolkswagen Magotan,0X3050M,30.82,118,Gasoline,3.478\nVolkswagen Magotan,0X3050M,5.11,118,Gasoline,0.8647\nVolkswagen Magotan,0X3050X,48.87,118,Gasoline,4.0183\nVolkswagen Magotan,0X3050M,54.63,118,Gasoline,4.252\nVolkswagen Magotan,0X3050X,5.16,118,Gasoline,0.5507\nVolkswagen Magotan,0X3050M,6.06,118,Gasoline,0.5289\nVolkswagen Magotan,0X3050X,3.91,118,Gasoline,0.3556\nVolkswagen Magotan,0X3050X,42.56,118,Gasoline,3.2841\nVolkswagen Magotan,0X3050M,125.98,118,Gasoline,9.3095\nVolkswagen Magotan,0X3050M,258.74,118,Gasoline,19.5664\nVolkswagen Magotan,0X3050X,153.05,118,Gasoline,12.0291\nVolkswagen Magotan,0X3050M,8.37,118,Gasoline,0.9824\nVolkswagen Magotan,0X3050X,2.8,118,Gasoline,0.3562\nVolkswagen Magotan,0X3050M,32.61,118,Gasoline,3.6736\nVolkswagen Magotan,0X3050M,10.6,118,Gasoline,0.9274\nVolkswagen Magotan,0X3050M,28.2,118,Gasoline,2.6548\nVolkswagen Magotan,0X3050M,24.09,118,Gasoline,2.5503\nVolkswagen Magotan,0X3050M,16.24,118,Gasoline,2.3762\nVolkswagen Magotan,0X3050X,31.81,118,Gasoline,2.9072\nVolkswagen Magotan,0X3050M,55.72,118,Gasoline,4.7956\nVolkswagen Magotan,0X3050X,15.11,118,Gasoline,1.1035\nVolkswagen Magotan,0X3050X,18.03,118,Gasoline,1.4064\nVolkswagen Magotan,0X3050X,9.74,118,Gasoline,0.8285\nVolkswagen Magotan,0X3050X,41.03,118,Gasoline,3.5504\nVolkswagen Magotan,0X3050X,10.1,118,Gasoline,1.1173\nVolkswagen Magotan,0X3050X,7.2,118,Gasoline,0.8191\nVolkswagen Magotan,0X3050M,2.89,118,Gasoline,0.2653\nVolkswagen Magotan,0X3050X,25.99,118,Gasoline,2.4852\nVolkswagen Magotan,0X3050M,3.74,118,Gasoline,0.6349\nVolkswagen Magotan,0X3050X,17.87,118,Gasoline,1.9122\nVolkswagen Magotan,0X3050X,29.34,118,Gasoline,2.9858\nVolkswagen Magotan,0X3050M,36.33,118,Gasoline,3.6293\nVolkswagen Magotan,0X3050M,261.32,118,Gasoline,19.7331\nVolkswagen Magotan,0X3050X,191.95,118,Gasoline,11.8912\nVolkswagen Magotan,0X3050M,30.48,118,Gasoline,2.68\nVolkswagen Magotan,0X3050M,360.35,118,Gasoline,27.546\nVolkswagen Magotan,0X3050X,165.12,118,Gasoline,11.6037\nVolkswagen Magotan,0X3050M,177.16,118,Gasoline,14.7186\nVolkswagen Magotan,0X3050X,34.96,118,Gasoline,3.5279\nVolkswagen Magotan,0X3050X,13.9,118,Gasoline,1.4022\nVolkswagen Magotan,0X3050M,29.31,118,Gasoline,3.3911\nVolkswagen Magotan,0X3050X,14.29,118,Gasoline,1.3091\nVolkswagen Magotan,0X3050X,13.82,118,Gasoline,1.4418\nVolkswagen Magotan,0X3050X,10.0,118,Gasoline,1.0281\nVolkswagen Magotan,0X3050M,1.5,118,Gasoline,0.2213\nVolkswagen Magotan,0X3050M,15.88,118,Gasoline,1.7224\nVolkswagen Magotan,0X3050M,4.47,118,Gasoline,0.3732\nVolkswagen Magotan,0X3050M,133.73,118,Gasoline,11.1993\nVolkswagen Magotan,0X3050M,30.82,118,Gasoline,2.2649\nVolkswagen Magotan,0X3050M,7.71,118,Gasoline,0.722\nVolkswagen Magotan,0X3050X,89.49,118,Gasoline,9.4148\nVolkswagen Magotan,0X3050X,32.37,118,Gasoline,3.5845\nVolkswagen Magotan,0X3050M,106.25,118,Gasoline,8.3703\nVolkswagen Magotan,0X3050X,17.74,118,Gasoline,2.0955\nVolkswagen Magotan,0X3050M,53.44,118,Gasoline,4.5015\nVolkswagen Magotan,0X3050X,38.23,118,Gasoline,3.0478\nVolkswagen Magotan,0X3050X,39.49,118,Gasoline,3.5426\nVolkswagen Magotan,0X3050X,13.92,118,Gasoline,1.1542\nVolkswagen Magotan,0X3050M,58.97,118,Gasoline,7.0264\nVolkswagen Magotan,0X3050M,21.89,118,Gasoline,1.6622\nVolkswagen Magotan,0X3050M,84.05,118,Gasoline,7.4909\nVolkswagen Magotan,0X3050M,30.28,118,Gasoline,3.3567\nVolkswagen Magotan,0X3050M,36.1,118,Gasoline,4.8757\nVolkswagen Magotan,0X3050M,59.11,118,Gasoline,7.2288\nVolkswagen Magotan,0X3050X,2.25,118,Gasoline,0.2709\nVolkswagen Magotan,0X3050X,49.51,118,Gasoline,7.1717\nVolkswagen Magotan,0X3050M,36.45,118,Gasoline,2.8326\nVolkswagen Magotan,0X3050X,2.57,118,Gasoline,0.3078\nVolkswagen Magotan,0X3050M,14.42,118,Gasoline,2.3748\nVolkswagen Magotan,0X3050M,138.35,118,Gasoline,13.8723\nVolkswagen Magotan,0X3050X,137.32,118,Gasoline,9.2188\nVolkswagen Magotan,0X3050X,6.75,118,Gasoline,0.6383\nVolkswagen Magotan,0X3050M,7.08,118,Gasoline,1.0683\nVolkswagen Magotan,0X3050X,14.98,118,Gasoline,1.3221\nVolkswagen Magotan,0X3050X,137.26,118,Gasoline,9.8381\nVolkswagen Magotan,0X3050M,13.61,118,Gasoline,1.2824\nVolkswagen Magotan,0X3050M,50.77,118,Gasoline,3.7888\nVolkswagen Magotan,0X3050X,8.66,118,Gasoline,0.7723\nVolkswagen Magotan,0X3050X,48.15,118,Gasoline,3.8988\nVolkswagen Magotan,0X3050M,14.64,118,Gasoline,1.53\nVolkswagen Magotan,0X3050M,6.22,118,Gasoline,0.693\nVolkswagen Magotan,0X3050X,17.89,118,Gasoline,1.9936\nVolkswagen Magotan,0X3050X,103.64,118,Gasoline,6.9079\nVolkswagen Magotan,0X3050M,41.33,118,Gasoline,6.0109\nVolkswagen Magotan,0X3050M,13.86,118,Gasoline,1.4224\nVolkswagen Magotan,0X3050M,7.79,118,Gasoline,0.9689\nVolkswagen Magotan,0X3050X,2.28,118,Gasoline,0.3089\nVolkswagen Magotan,0X3050X,105.1,118,Gasoline,6.9162\nVolkswagen Magotan,0X3050M,35.53,118,Gasoline,3.5854\nVolkswagen Magotan,0X3050X,34.35,118,Gasoline,2.4568\nVolkswagen Magotan,0X3050M,6.3,118,Gasoline,0.6921\nVolkswagen Magotan,0X3050M,2.7,118,Gasoline,0.5238\nVolkswagen Magotan,0X3050M,28.43,118,Gasoline,3.7952\nVolkswagen Magotan,0X3050X,6.23,118,Gasoline,0.7645\nVolkswagen Magotan,0X3050M,58.98,118,Gasoline,8.1572\nVolkswagen Magotan,0X3050X,347.24,118,Gasoline,26.1172\nVolkswagen Magotan,0X3050M,7.97,118,Gasoline,1.3587\nVolkswagen Magotan,0X3050M,60.29,118,Gasoline,4.8992\nVolkswagen Magotan,0X3050M,134.76,118,Gasoline,12.5057\nVolkswagen Magotan,0X3050M,45.32,118,Gasoline,3.8848\nVolkswagen Magotan,0X3050M,55.82,118,Gasoline,4.7061\nVolkswagen Magotan,0X3050X,190.84,118,Gasoline,13.6161\nVolkswagen Magotan,0X3050M,21.65,118,Gasoline,3.019\nVolkswagen Magotan,0X3050M,2.78,118,Gasoline,0.4749\nVolkswagen Magotan,0X3050M,194.59,118,Gasoline,18.1185\nVolkswagen Magotan,0X3050M,11.68,118,Gasoline,1.2216\nVolkswagen Magotan,0X3050M,32.75,118,Gasoline,2.1282\nVolkswagen Magotan,0X3050X,185.21,118,Gasoline,12.8335\nVolkswagen Magotan,0X3050M,147.38,118,Gasoline,11.9143\nVolkswagen Magotan,0X3050M,129.4,118,Gasoline,10.279\nVolkswagen Magotan,0X3050M,163.98,118,Gasoline,13.1125\nVolkswagen Magotan,0X3050X,2.18,118,Gasoline,0.3023\nVolkswagen Magotan,0X3050X,2.18,118,Gasoline,0.3573\nVolkswagen Magotan,0X3050M,108.4,118,Gasoline,9.4048\nVolkswagen Magotan,0X3050M,132.43,118,Gasoline,14.1866\nVolkswagen Magotan,0X3050M,7.9,118,Gasoline,0.6507\nVolkswagen Magotan,0X3050X,136.09,118,Gasoline,9.9195\nVolkswagen Magotan,0X3050M,18.48,118,Gasoline,2.8182\nVolkswagen Magotan,0X3050M,35.39,118,Gasoline,4.4258\nVolkswagen Magotan,0X3050X,14.6,118,Gasoline,1.5846\nVolkswagen Magotan,0X3050M,204.84,118,Gasoline,16.1392\nVolkswagen Magotan,0X3050X,92.75,118,Gasoline,7.9247\nVolkswagen Magotan,0X3050M,32.84,118,Gasoline,3.1975\nVolkswagen Magotan,0X3050M,38.06,118,Gasoline,4.1334\nVolkswagen Magotan,0X3050X,15.22,118,Gasoline,1.2094\nVolkswagen Magotan,0X3050M,103.28,118,Gasoline,8.1152\nVolkswagen Magotan,0X3050M,61.68,118,Gasoline,6.5475\nVolkswagen Magotan,0X3050M,53.44,118,Gasoline,4.4073\nVolkswagen Magotan,0X3050X,2.77,118,Gasoline,0.4319\nVolkswagen Magotan,0X3050M,105.22,118,Gasoline,9.3554\nVolkswagen Magotan,0X3050M,13.27,118,Gasoline,1.497\nVolkswagen Magotan,0X3050X,105.44,118,Gasoline,6.9574\nVolkswagen Magotan,0X3050M,1.67,118,Gasoline,0.2694\nVolkswagen Magotan,0X3050X,6.47,118,Gasoline,0.569\nVolkswagen Magotan,0X3050M,44.12,118,Gasoline,3.8738\nVolkswagen Magotan,0X3050X,8.21,118,Gasoline,0.5502\nVolkswagen Magotan,0X3050M,189.94,118,Gasoline,16.5156\nVolkswagen Magotan,0X3050X,234.43,118,Gasoline,20.1598\nVolkswagen Magotan,0X3050X,128.89,118,Gasoline,7.1705\nVolkswagen Magotan,0X3050X,5.57,118,Gasoline,0.545\nVolkswagen Magotan,0X3050X,105.0,118,Gasoline,6.6088\nVolkswagen Magotan,0X3050X,150.9,118,Gasoline,8.9398\nVolkswagen Magotan,0X3050X,6.3,118,Gasoline,0.7576\nVolkswagen Magotan,0X3050X,239.67,118,Gasoline,16.3596\nVolkswagen Magotan,0X3050X,142.0,118,Gasoline,10.3291\nVolkswagen Magotan,0X3050X,3.45,118,Gasoline,0.3553\nVolkswagen Magotan,0X3050X,136.41,118,Gasoline,11.4779\nVolkswagen Magotan,0X3050M,10.69,118,Gasoline,1.2001\nVolkswagen Magotan,0X3050M,110.81,118,Gasoline,8.1848\nVolkswagen Magotan,0X3050M,81.31,118,Gasoline,6.7277\nVolkswagen Magotan,0X3050M,19.79,118,Gasoline,2.4664\nVolkswagen Magotan,0X3050M,3.41,118,Gasoline,0.3999\nVolkswagen Magotan,0X3050X,13.89,118,Gasoline,1.3406\nVolkswagen Magotan,0X3050M,1.63,118,Gasoline,0.2954\nVolkswagen Magotan,0X3050M,3.02,118,Gasoline,0.2612\nVolkswagen Magotan,0X3050M,102.14,118,Gasoline,8.3351\nVolkswagen Magotan,0X3050X,14.59,118,Gasoline,1.5351\nVolkswagen Magotan,0X3050M,34.83,118,Gasoline,3.9518\nVolkswagen Magotan,0X3050X,17.18,118,Gasoline,1.6469\nVolkswagen Magotan,0X3050X,34.76,118,Gasoline,3.0308\nVolkswagen Magotan,0X3050X,113.17,118,Gasoline,6.7526\nVolkswagen Magotan,0X3050X,3.42,118,Gasoline,0.3375\nVolkswagen Magotan,0X3050M,132.93,118,Gasoline,12.2665\nVolkswagen Magotan,0X3050M,164.17,118,Gasoline,14.0208\nVolkswagen Magotan,0X3050X,43.95,118,Gasoline,2.8723\nVolkswagen Magotan,0X3050M,3.51,118,Gasoline,0.3767\nVolkswagen Magotan,0X3050X,4.56,118,Gasoline,0.5241\nVolkswagen Magotan,0X3050X,3.64,118,Gasoline,0.342\nVolkswagen Magotan,0X3050X,14.48,118,Gasoline,1.2468\nVolkswagen Magotan,0X3050X,8.03,118,Gasoline,0.8524\nVolkswagen Magotan,0X3050X,13.86,118,Gasoline,1.3302\nVolkswagen Magotan,0X3050X,9.4,118,Gasoline,0.7974\nVolkswagen Magotan,0X3050M,163.95,118,Gasoline,11.562\nVolkswagen Magotan,0X3050X,5.16,118,Gasoline,0.66\nVolkswagen Magotan,0X3050M,7.29,118,Gasoline,0.5841\nVolkswagen Magotan,0X3050M,29.6,118,Gasoline,2.9199\nVolkswagen Magotan,0X3050M,117.53,118,Gasoline,9.2039\nVolkswagen Magotan,0X3050X,7.0,118,Gasoline,0.7212\nVolkswagen Magotan,0X3050M,175.75,118,Gasoline,11.8422\nVolkswagen Magotan,0X3050X,6.64,118,Gasoline,0.689\nVolkswagen Magotan,0X3050M,1.94,118,Gasoline,0.3107\nVolkswagen Magotan,0X3050M,124.68,118,Gasoline,8.4759\nVolkswagen Magotan,0X3050M,7.64,118,Gasoline,0.6787\nVolkswagen Magotan,0X3050X,121.88,118,Gasoline,9.7089\nVolkswagen Magotan,0X3050M,103.78,118,Gasoline,7.4453\nVolkswagen Magotan,0X3050M,8.53,118,Gasoline,0.7968\nVolkswagen Magotan,0X3050X,40.8,118,Gasoline,3.2306\nVolkswagen Magotan,0X3050X,32.12,118,Gasoline,2.7547\nVolkswagen Magotan,0X3050X,18.67,118,Gasoline,1.3043\nVolkswagen Magotan,0X3050X,4.04,118,Gasoline,0.3919\nVolkswagen Magotan,0X3050X,3.08,118,Gasoline,0.451\nVolkswagen Magotan,0X3050M,22.26,118,Gasoline,1.9676\nVolkswagen Magotan,0X3050X,44.68,118,Gasoline,4.0699\nVolkswagen Magotan,0X3050M,43.36,118,Gasoline,3.6723\nVolkswagen Magotan,0X3050M,11.56,118,Gasoline,0.9967\nVolkswagen Magotan,0X3050X,18.64,118,Gasoline,1.4356\nVolkswagen Magotan,0X3050M,12.7,118,Gasoline,1.1848\nVolkswagen Magotan,0X3050X,13.12,118,Gasoline,1.2463\nVolkswagen Magotan,0X3050M,11.9,118,Gasoline,1.2164\nVolkswagen Magotan,0X3050M,52.62,118,Gasoline,3.6949\nVolkswagen Magotan,0X3050M,52.08,118,Gasoline,3.3057\nVolkswagen Magotan,0X3050X,4.89,118,Gasoline,0.4043\nVolkswagen Magotan,0X3050X,115.34,118,Gasoline,7.8856\nVolkswagen Magotan,0X3050X,115.24,118,Gasoline,7.2319\nVolkswagen Magotan,0X3050M,190.46,118,Gasoline,14.0522\nVolkswagen Magotan,0X3050M,261.7,118,Gasoline,17.8483\nVolkswagen Magotan,0X3050X,173.68,118,Gasoline,12.15\nVolkswagen Magotan,0X3050M,3.4,118,Gasoline,0.6881\nVolkswagen Magotan,0X3050M,42.38,118,Gasoline,3.3438\nVolkswagen Magotan,0X3050X,145.17,118,Gasoline,10.3022\nVolkswagen Magotan,0X3050M,360.8,118,Gasoline,29.5901\nVolkswagen Magotan,0X3050X,112.06,118,Gasoline,7.872\nVolkswagen Magotan,0X3050M,124.8,118,Gasoline,9.0711\nVolkswagen Magotan,0X3050M,105.02,118,Gasoline,7.1204\nVolkswagen Magotan,0X3050M,30.1,118,Gasoline,2.0766\nVolkswagen Magotan,0X3050M,140.43,118,Gasoline,9.875\nVolkswagen Magotan,0X3050M,3.81,118,Gasoline,0.3265\nVolkswagen Magotan,0X3050M,35.33,118,Gasoline,3.8089\nVolkswagen Magotan,0X3050X,7.26,118,Gasoline,0.7148\nVolkswagen Magotan,0X3050X,2.41,118,Gasoline,0.3009\nVolkswagen Magotan,0X3050M,17.27,118,Gasoline,1.7782\nVolkswagen Magotan,0X3050X,46.33,118,Gasoline,3.4526\nVolkswagen Magotan,0X3050X,8.19,118,Gasoline,0.6873\nVolkswagen Magotan,0X3050M,6.55,118,Gasoline,0.8379\nVolkswagen Magotan,0X3050M,47.19,118,Gasoline,3.5243\nVolkswagen Magotan,0X3050X,3.87,118,Gasoline,0.6464\nVolkswagen Magotan,0X3050M,31.66,118,Gasoline,4.0555\nVolkswagen Magotan,0X3050X,5.85,118,Gasoline,0.7042\nVolkswagen Magotan,0X3050M,51.59,118,Gasoline,5.4614\nVolkswagen Magotan,0X3050M,4.3,118,Gasoline,0.8228\nVolkswagen Magotan,0X3050M,55.2,118,Gasoline,4.6815\nVolkswagen Magotan,0X3050X,6.67,118,Gasoline,0.6481\nVolkswagen Magotan,0X3050M,146.55,118,Gasoline,11.5362\nVolkswagen Magotan,0X3050M,59.39,118,Gasoline,4.714\nVolkswagen Magotan,0X3050M,63.26,118,Gasoline,4.6685\nVolkswagen Magotan,0X3050M,7.48,118,Gasoline,0.6548\nVolkswagen Magotan,0X3050M,14.58,118,Gasoline,1.5676\nVolkswagen Magotan,0X3050X,28.42,118,Gasoline,3.1214\nVolkswagen Magotan,0X3050M,74.05,118,Gasoline,5.216\nVolkswagen Magotan,0X3050M,13.04,118,Gasoline,1.1762\nVolkswagen Magotan,0X3050M,30.93,118,Gasoline,3.0801\nVolkswagen Magotan,0X3050M,13.25,118,Gasoline,1.0878\nVolkswagen Magotan,0X3050X,15.7,118,Gasoline,1.8586\nVolkswagen Magotan,0X3050M,109.28,118,Gasoline,8.464\nVolkswagen Magotan,0X3050X,16.09,118,Gasoline,1.3348\nVolkswagen Magotan,0X3050X,4.22,118,Gasoline,0.493\nVolkswagen Magotan,0X3050M,11.75,118,Gasoline,1.3376\nVolkswagen Magotan,0X3050M,7.22,118,Gasoline,1.0226\nVolkswagen Magotan,0X3050X,14.25,118,Gasoline,1.4576\nVolkswagen Magotan,0X3050M,144.54,118,Gasoline,10.1016\nVolkswagen Magotan,0X3050M,17.6,118,Gasoline,1.5453\nVolkswagen Magotan,0X3050X,15.78,118,Gasoline,1.2674\nVolkswagen Magotan,0X3050M,63.27,118,Gasoline,4.2733\nVolkswagen Magotan,0X3050M,8.59,118,Gasoline,0.888\nVolkswagen Magotan,0X3050X,3.86,118,Gasoline,0.5169\nVolkswagen Magotan,0X3050M,44.24,118,Gasoline,5.2027\nVolkswagen Magotan,0X3050M,109.48,118,Gasoline,8.7564\nVolkswagen Magotan,0X3050M,22.85,118,Gasoline,1.5476\nVolkswagen Magotan,0X3050X,26.05,118,Gasoline,2.9619\nVolkswagen Magotan,0X3050X,2.98,118,Gasoline,0.4095\nVolkswagen Magotan,0X3050X,24.91,118,Gasoline,1.9332\nVolkswagen Magotan,0X3050M,111.24,118,Gasoline,9.2859\nVolkswagen Magotan,0X3050X,9.8,118,Gasoline,0.8939\nVolkswagen Magotan,0X3050M,120.94,118,Gasoline,10.2537\nVolkswagen Magotan,0X3050M,11.72,118,Gasoline,1.3966\nVolkswagen Magotan,0X3050X,14.45,118,Gasoline,1.392\nVolkswagen Magotan,0X3050X,6.59,118,Gasoline,0.7983\nVolkswagen Magotan,0X3050X,24.02,118,Gasoline,1.6345\nVolkswagen Magotan,0X3050M,62.88,118,Gasoline,4.9576\nVolkswagen Magotan,0X3050X,5.29,118,Gasoline,0.5673\nVolkswagen Magotan,0X3050M,59.93,118,Gasoline,3.7398\nVolkswagen Magotan,0X3050X,7.12,118,Gasoline,0.9039\nVolkswagen Magotan,0X3050X,3.92,118,Gasoline,0.4973\nVolkswagen Magotan,0X3050M,2.69,118,Gasoline,0.3265\nVolkswagen Magotan,0X3050M,25.98,118,Gasoline,2.1309\nVolkswagen Magotan,0X3050X,17.37,118,Gasoline,1.2393\nVolkswagen Magotan,0X3050M,64.85,118,Gasoline,6.3494\nVolkswagen Magotan,0X3050M,13.25,118,Gasoline,1.2721\nVolkswagen Magotan,0X3050M,150.0,118,Gasoline,12.3641\nVolkswagen Magotan,0X3050M,164.04,118,Gasoline,14.9306\nVolkswagen Magotan,0X3050M,18.49,118,Gasoline,2.5754\nVolkswagen Magotan,0X3050M,22.67,118,Gasoline,2.0999\nVolkswagen Magotan,0X3050X,14.5,118,Gasoline,1.4301\nVolkswagen Magotan,0X3050M,59.39,118,Gasoline,5.0414\nVolkswagen Magotan,0X3050X,14.79,118,Gasoline,1.267\nVolkswagen Magotan,0X3050X,1.59,118,Gasoline,0.2278\nVolkswagen Magotan,0X3050M,18.85,118,Gasoline,2.4251\nVolkswagen Magotan,0X3050X,14.24,118,Gasoline,1.4878\nVolkswagen Magotan,0X3050X,5.33,118,Gasoline,0.5925\nVolkswagen Magotan,0X3050X,6.56,118,Gasoline,0.8226\nVolkswagen Magotan,0X3050X,2.77,118,Gasoline,0.4251\nVolkswagen Magotan,0X3050M,37.74,118,Gasoline,2.8482\nVolkswagen Magotan,0X3050M,8.47,118,Gasoline,1.1416\nVolkswagen Magotan,0X3050M,115.07,118,Gasoline,12.4735\nVolkswagen Magotan,0X3050M,74.0,118,Gasoline,5.1975\nVolkswagen Magotan,0X3050M,76.34,118,Gasoline,6.4937\nVolkswagen Magotan,0X3050X,38.24,118,Gasoline,2.598\nVolkswagen Magotan,0X3050M,114.64,118,Gasoline,10.7074\nVolkswagen Magotan,0X3050M,180.95,118,Gasoline,17.6402\nVolkswagen Magotan,0X3050M,34.4,118,Gasoline,2.911\nVolkswagen Magotan,0X3050X,65.22,118,Gasoline,3.7045\nVolkswagen Magotan,0X3050X,10.5,118,Gasoline,1.0489\nVolkswagen Magotan,0X3050M,87.4,118,Gasoline,7.4564\nVolkswagen Magotan,0X3050M,15.85,118,Gasoline,1.6267\nVolkswagen Magotan,0X3050M,121.24,118,Gasoline,12.2854\nVolkswagen Magotan,0X3050M,33.94,118,Gasoline,3.5854\nVolkswagen Magotan,0X3050X,4.84,118,Gasoline,0.6932\nVolkswagen Magotan,0X3050M,24.87,118,Gasoline,2.4802\nVolkswagen Magotan,0X3050X,91.8,118,Gasoline,6.8156\nVolkswagen Magotan,0X3050M,2.31,118,Gasoline,0.3372\nVolkswagen Magotan,0X3050X,15.81,118,Gasoline,1.3834\nVolkswagen Magotan,0X3050X,26.14,118,Gasoline,2.939\nVolkswagen Magotan,0X3050M,14.61,118,Gasoline,2.2972\nVolkswagen Magotan,0X3050M,20.88,118,Gasoline,2.5418\nVolkswagen Magotan,0X3050M,53.72,118,Gasoline,3.7823\nVolkswagen Magotan,0X3050M,41.51,118,Gasoline,3.3817\nVolkswagen Magotan,0X3050M,130.02,118,Gasoline,10.6861\nVolkswagen Magotan,0X3050X,16.76,118,Gasoline,1.2893\nVolkswagen Magotan,0X3050M,103.84,118,Gasoline,8.7559\nVolkswagen Magotan,0X3050M,11.0,118,Gasoline,1.1694\nVolkswagen Magotan,0X3050M,11.6,118,Gasoline,0.9591\nVolkswagen Magotan,0X3050M,102.78,118,Gasoline,9.604\nVolkswagen Magotan,0X3050X,10.36,118,Gasoline,1.1169\nVolkswagen Magotan,0X3050X,9.7,118,Gasoline,0.9364\nVolkswagen Magotan,0X3050M,64.25,118,Gasoline,4.4857\nVolkswagen Magotan,0X3050M,110.96,118,Gasoline,11.677\nVolkswagen Magotan,0X3050X,18.94,118,Gasoline,1.7009\nVolkswagen Magotan,0X3050M,73.4,118,Gasoline,6.0954\nVolkswagen Magotan,0X3050X,18.53,118,Gasoline,1.335\nVolkswagen Magotan,0X3050X,16.3,118,Gasoline,1.5867\nVolkswagen Magotan,0X3050M,116.99,118,Gasoline,11.0964\nVolkswagen Magotan,0X3050M,4.05,118,Gasoline,0.5804\nVolkswagen Magotan,0X3050M,69.48,118,Gasoline,5.5949\nVolkswagen Magotan,0X3050M,49.14,118,Gasoline,4.619\nVolkswagen Magotan,0X3050M,150.67,118,Gasoline,12.0852\nVolkswagen Magotan,0X3050M,2.91,118,Gasoline,0.4119\nVolkswagen Magotan,0X3050M,14.63,118,Gasoline,1.4838\nVolkswagen Magotan,0X3050M,123.91,118,Gasoline,11.3385\nVolkswagen Magotan,0X3050X,37.11,118,Gasoline,3.1228\nVolkswagen Magotan,0X3050M,137.39,118,Gasoline,12.8837\nVolkswagen Magotan,0X3050X,2.13,118,Gasoline,0.251\nVolkswagen Magotan,0X3050M,14.6,118,Gasoline,1.8152\nVolkswagen Magotan,0X3050X,6.73,118,Gasoline,0.7995\nVolkswagen Magotan,0X3050X,9.77,118,Gasoline,0.8681\nVolkswagen Magotan,0X3050X,6.54,118,Gasoline,0.7988\nVolkswagen Magotan,0X3050M,106.47,118,Gasoline,8.5679\nVolkswagen Magotan,0X3050M,45.02,118,Gasoline,3.7581\nVolkswagen Magotan,0X3050M,49.1,118,Gasoline,4.7464\nVolkswagen Magotan,0X3050M,116.6,118,Gasoline,11.6503\nVolkswagen Magotan,0X3050M,10.11,118,Gasoline,0.7063\nVolkswagen Magotan,0X3050M,33.5,118,Gasoline,2.5784\nVolkswagen Magotan,0X3050X,5.79,118,Gasoline,0.6321\nVolkswagen Passat,0X2072T,117.86,118,Gasoline,8.9624\nVolkswagen Passat,0X2072T,34.41,118,Gasoline,2.4417\nVolkswagen Passat,0X2072T,3.22,118,Gasoline,0.4127\nVolkswagen Passat,0X2072R,44.38,118,Gasoline,5.4209\nVolkswagen Passat,0X2072S,3.06,118,Gasoline,0.5472\nVolkswagen Passat,0X2072T,3.89,118,Gasoline,0.4001\nVolkswagen Passat,0X2072R,46.16,118,Gasoline,4.436\nVolkswagen Passat,0X2072S,209.12,118,Gasoline,14.8715\nVolkswagen Passat,0X2072R,134.57,118,Gasoline,10.6775\nVolkswagen Passat,0X2072T,37.66,118,Gasoline,3.439\nVolkswagen Passat,0X2072T,19.36,118,Gasoline,1.5899\nVolkswagen Passat,0X2072S,22.65,118,Gasoline,2.6283\nVolkswagen Passat,0X2072R,11.12,118,Gasoline,1.1923\nVolkswagen Passat,0X2072T,43.16,118,Gasoline,2.8738\nVolkswagen Passat,0X2072T,79.86,118,Gasoline,5.9211\nVolkswagen Passat,0X2072S,85.73,118,Gasoline,6.1538\nVolkswagen Passat,0X2072T,20.95,118,Gasoline,2.3463\nVolkswagen Passat,0X2072T,2.44,118,Gasoline,0.2139\nVolkswagen Passat,0X2072S,30.26,118,Gasoline,2.2797\nVolkswagen Passat,0X2072S,106.88,118,Gasoline,9.0285\nVolkswagen Passat,0X2072T,5.34,118,Gasoline,0.6716\nVolkswagen Passat,0X2072S,126.96,118,Gasoline,9.8811\nVolkswagen Passat,0X2072T,13.4,118,Gasoline,1.3766\nVolkswagen Passat,0X2072S,37.31,118,Gasoline,2.7131\nVolkswagen Passat,0X2072S,104.62,118,Gasoline,10.9274\nVolkswagen Passat,0X2072R,17.46,118,Gasoline,1.6938\nVolkswagen Passat,0X2072T,47.5,118,Gasoline,4.4997\nVolkswagen Passat,0X2072T,10.18,118,Gasoline,0.9045\nVolkswagen Passat,0X2072S,36.65,118,Gasoline,2.6149\nVolkswagen Passat,0X2072R,126.15,118,Gasoline,9.6986\nVolkswagen Passat,0X2072S,2.6,118,Gasoline,0.3189\nVolkswagen Passat,0X2072S,2.62,118,Gasoline,0.3567\nVolkswagen Passat,0X2072R,31.63,118,Gasoline,3.0002\nVolkswagen Passat,0X2072T,16.85,118,Gasoline,1.5432\nVolkswagen Passat,0X2072S,5.98,118,Gasoline,0.6855\nVolkswagen Passat,0X2072S,1.93,118,Gasoline,0.1733\nVolkswagen Passat,0X2072S,147.31,118,Gasoline,11.7163\nVolkswagen Passat,0X2072T,32.14,118,Gasoline,2.7235\nVolkswagen Passat,0X2072S,30.93,118,Gasoline,3.5376\nVolkswagen Passat,0X2072T,199.95,118,Gasoline,12.8973\nVolkswagen Passat,0X2072T,11.09,118,Gasoline,1.0311\nVolkswagen Passat,0X2072R,16.39,118,Gasoline,1.2929\nVolkswagen Passat,0X2072S,29.03,118,Gasoline,2.468\nVolkswagen Passat,0X2072S,138.96,118,Gasoline,12.0541\nVolkswagen Passat,0X2072R,412.54,118,Gasoline,35.4871\nVolkswagen Passat,0X2072S,110.99,118,Gasoline,9.7038\nVolkswagen Passat,0X2072T,11.82,118,Gasoline,1.1047\nVolkswagen Passat,0X2072T,25.39,118,Gasoline,2.0438\nVolkswagen Passat,0X2072R,1.96,118,Gasoline,0.2587\nVolkswagen Passat,0X2072T,2.39,118,Gasoline,0.3192\nVolkswagen Passat,0X2072T,105.3,118,Gasoline,10.2148\nVolkswagen Passat,0X2072R,60.34,118,Gasoline,5.4359\nVolkswagen Passat,0X2072T,1.84,118,Gasoline,0.2905\nVolkswagen Passat,0X2072T,193.36,118,Gasoline,15.7093\nVolkswagen Passat,0X2072T,2.25,118,Gasoline,0.3111\nVolkswagen Passat,0X2072S,37.34,118,Gasoline,3.3422\nVolkswagen Passat,0X2072S,9.0,118,Gasoline,0.7888\nVolkswagen Passat,0X2072S,152.6,118,Gasoline,11.2772\nVolkswagen Passat,0X2072S,689.99,118,Gasoline,54.3992\nVolkswagen Passat,0X2072S,214.7,118,Gasoline,16.9352\nVolkswagen Passat,0X2072T,335.71,118,Gasoline,23.7141\nVolkswagen Passat,0X2072S,107.59,118,Gasoline,6.6269\nVolkswagen Passat,0X2072T,100.55,118,Gasoline,8.4341\nVolkswagen Passat,0X2072S,6.72,118,Gasoline,0.5769\nVolkswagen Passat,0X2072S,140.98,118,Gasoline,11.5561\nVolkswagen Passat,0X2072R,208.88,118,Gasoline,13.6914\nVolkswagen Passat,0X2072S,115.12,118,Gasoline,7.9669\nVolkswagen Passat,0X2072T,22.28,118,Gasoline,1.9588\nVolkswagen Passat,0X2072T,207.16,118,Gasoline,14.5586\nVolkswagen Passat,0X2072T,21.54,118,Gasoline,1.7043\nVolkswagen Passat,0X2072R,42.48,118,Gasoline,4.2533\nVolkswagen Passat,0X2072S,1.56,118,Gasoline,0.1494\nVolkswagen Passat,0X2072T,169.57,118,Gasoline,14.4071\nVolkswagen Passat,0X2072R,3.34,118,Gasoline,0.3573\nVolkswagen Passat,0X2072R,110.83,118,Gasoline,7.8306\nVolkswagen Passat,0X2072S,44.85,118,Gasoline,3.417\nVolkswagen Passat,0X2072S,64.74,118,Gasoline,4.4716\nVolkswagen Passat,0X2072T,44.5,118,Gasoline,2.8394\nVolkswagen Passat,0X2072T,12.34,118,Gasoline,0.9755\nVolkswagen Passat,0X2072T,49.46,118,Gasoline,3.7551\nVolkswagen Passat,0X2072T,3.9,118,Gasoline,0.3782\nVolkswagen Passat,0X2072R,18.87,118,Gasoline,1.7924\nVolkswagen Passat,0X2072T,12.72,118,Gasoline,1.3277\nVolkswagen Passat,0X2072T,42.68,118,Gasoline,3.1501\nVolkswagen Passat,0X2072R,25.58,118,Gasoline,3.0432\nVolkswagen Passat,0X2072R,196.97,118,Gasoline,12.0557\nVolkswagen Passat,0X2072S,13.17,118,Gasoline,1.1992\nVolkswagen Passat,0X2072R,3.62,118,Gasoline,0.4127\nVolkswagen Passat,0X2072T,40.93,118,Gasoline,3.0324\nVolkswagen Passat,0X2072S,5.03,118,Gasoline,0.4218\nVolkswagen Passat,0X2072T,2.33,118,Gasoline,0.4036\nVolkswagen Passat,0X2072S,12.41,118,Gasoline,1.1296\nVolkswagen Passat,0X2072S,7.23,118,Gasoline,0.693\nVolkswagen Passat,0X2072T,21.29,118,Gasoline,1.7722\nVolkswagen Passat,0X2072S,19.32,118,Gasoline,1.7825\nVolkswagen Passat,0X2072R,6.99,118,Gasoline,0.8086\nVolkswagen Passat,0X2072S,2.84,118,Gasoline,0.2931\nVolkswagen Passat,0X2072T,259.12,118,Gasoline,17.4159\nVolkswagen Passat,0X2072R,3.09,118,Gasoline,0.2705\nVolkswagen Passat,0X2072S,37.17,118,Gasoline,2.6909\nVolkswagen Passat,0X2072T,8.32,118,Gasoline,0.8773\nVolkswagen Passat,0X2072R,3.84,118,Gasoline,0.4568\nVolkswagen Passat,0X2072R,182.9,118,Gasoline,13.1056\nVolkswagen Passat,0X2072S,138.76,118,Gasoline,12.973\nVolkswagen Passat,0X2072R,24.96,118,Gasoline,3.4232\nVolkswagen Passat,0X2072R,37.27,118,Gasoline,3.2365\nVolkswagen Passat,0X2072S,80.75,118,Gasoline,5.3678\nVolkswagen Passat,0X2072R,35.79,118,Gasoline,3.8562\nVolkswagen Passat,0X2072S,6.15,118,Gasoline,0.7689\nVolkswagen Passat,0X2072S,30.76,118,Gasoline,2.0032\nVolkswagen Passat,0X2072S,39.04,118,Gasoline,4.6926\nVolkswagen Passat,0X2072R,5.13,118,Gasoline,0.5268\nVolkswagen Passat,0X2072S,2.86,118,Gasoline,0.2267\nVolkswagen Passat,0X2072T,43.58,118,Gasoline,3.6078\nVolkswagen Passat,0X2072R,6.43,118,Gasoline,0.5915\nVolkswagen Passat,0X2072S,26.08,118,Gasoline,2.2851\nVolkswagen Passat,0X2072T,16.27,118,Gasoline,1.2322\nVolkswagen Passat,0X2072T,214.84,118,Gasoline,14.9066\nVolkswagen Passat,0X2072S,45.64,118,Gasoline,3.5263\nVolkswagen Passat,0X2072R,81.23,118,Gasoline,6.6154\nVolkswagen Passat,0X2072T,4.1,118,Gasoline,0.5191\nVolkswagen Passat,0X2072S,36.51,118,Gasoline,3.6357\nVolkswagen Passat,0X2072S,19.0,118,Gasoline,1.5663\nVolkswagen Passat,0X2072S,71.39,118,Gasoline,6.8774\nVolkswagen Passat,0X2072S,2.91,118,Gasoline,0.4039\nVolkswagen Passat,0X2072S,1.64,118,Gasoline,0.2178\nVolkswagen Passat,0X2072T,33.55,118,Gasoline,3.1028\nVolkswagen Passat,0X2072T,13.54,118,Gasoline,1.2665\nVolkswagen Passat,0X2072T,52.57,118,Gasoline,4.1762\nVolkswagen Passat,0X2072R,12.06,118,Gasoline,1.2738\nVolkswagen Passat,0X2072R,16.25,118,Gasoline,1.889\nVolkswagen Passat,0X2072T,10.5,118,Gasoline,1.0714\nVolkswagen Passat,0X2072T,39.22,118,Gasoline,5.1346\nVolkswagen Passat,0X2072S,32.6,118,Gasoline,2.3633\nVolkswagen Passat,0X2072R,5.9,118,Gasoline,0.6741\nVolkswagen Passat,0X2072S,16.25,118,Gasoline,1.136\nVolkswagen Passat,0X2072R,16.3,118,Gasoline,1.0881\nVolkswagen Passat,0X2072R,28.94,118,Gasoline,2.3503\nVolkswagen Passat,0X2072S,6.07,118,Gasoline,0.7811\nVolkswagen Passat,0X2072R,23.38,118,Gasoline,1.7501\nVolkswagen Passat,0X2072R,23.0,118,Gasoline,2.5688\nVolkswagen Passat,0X2072S,32.3,118,Gasoline,2.6104\nVolkswagen Passat,0X2072R,4.31,118,Gasoline,0.5688\nVolkswagen Passat,0X2072S,16.84,118,Gasoline,1.2272\nVolkswagen Passat,0X2072T,27.24,118,Gasoline,1.9678\nVolkswagen Passat,0X2072T,23.85,118,Gasoline,1.8241\nVolkswagen Passat,0X2072T,129.96,118,Gasoline,10.3185\nVolkswagen Passat,0X2072S,129.9,118,Gasoline,11.4208\nVolkswagen Passat,0X2072T,14.01,118,Gasoline,1.8078\nVolkswagen Passat,0X2072T,7.35,118,Gasoline,0.8184\nVolkswagen Passat,0X2072R,15.3,118,Gasoline,2.2257\nVolkswagen Passat,0X2072T,133.43,118,Gasoline,9.9063\nVolkswagen Passat,0X2072T,154.53,118,Gasoline,9.5601\nVolkswagen Passat,0X2072R,27.29,118,Gasoline,2.0156\nVolkswagen Passat,0X2072R,22.66,118,Gasoline,2.0701\nVolkswagen Passat,0X2072T,20.98,118,Gasoline,1.883\nVolkswagen Passat,0X2072T,55.66,118,Gasoline,3.7516\nVolkswagen Passat,0X2072R,189.32,118,Gasoline,14.1355\nVolkswagen Passat,0X2072T,23.4,118,Gasoline,1.5182\nVolkswagen Passat,0X2072T,32.3,118,Gasoline,2.0633\nVolkswagen Passat,0X2072T,109.86,118,Gasoline,7.5381\nVolkswagen Passat,0X2072S,55.89,118,Gasoline,5.8977\nVolkswagen Passat,0X2072S,3.55,118,Gasoline,0.4448\nVolkswagen Passat,0X2072T,3.97,118,Gasoline,0.3611\nVolkswagen Passat,0X2072S,39.19,118,Gasoline,3.5035\nVolkswagen Passat,0X2072S,6.46,118,Gasoline,0.6201\nVolkswagen Passat,0X2072S,114.76,118,Gasoline,9.6976\nVolkswagen Passat,0X2072S,4.26,118,Gasoline,0.4838\nVolkswagen Passat,0X2072S,65.77,118,Gasoline,6.3996\nVolkswagen Passat,0X2072S,2.14,118,Gasoline,0.1875\nVolkswagen Passat,0X2072S,10.07,118,Gasoline,0.9204\nVolkswagen Passat,0X2072S,13.67,118,Gasoline,1.701\nVolkswagen Passat,0X2072S,52.53,118,Gasoline,5.7766\nVolkswagen Passat,0X2072R,79.98,118,Gasoline,4.9231\nVolkswagen Passat,0X2072S,17.89,118,Gasoline,1.7123\nVolkswagen Passat,0X2072S,36.02,118,Gasoline,3.4298\nVolkswagen Passat,0X2072T,21.23,118,Gasoline,2.0539\nVolkswagen Passat,0X2072S,121.79,118,Gasoline,12.1123\nVolkswagen Passat,0X2072R,123.83,118,Gasoline,9.2186\nVolkswagen Passat,0X2072R,6.32,118,Gasoline,0.9183\nVolkswagen Passat,0X2072R,103.99,118,Gasoline,7.8252\nVolkswagen Passat,0X2072S,152.19,118,Gasoline,9.8839\nVolkswagen Passat,0X2072R,113.26,118,Gasoline,9.1502\nVolkswagen Passat,0X2072R,5.67,118,Gasoline,0.8873\nVolkswagen Passat,0X2072T,120.75,118,Gasoline,7.6442\nVolkswagen Passat,0X2072S,45.67,118,Gasoline,3.3458\nVolkswagen Passat,0X2072T,152.6,118,Gasoline,10.1678\nVolkswagen Passat,0X2072S,34.08,118,Gasoline,2.4664\nVolkswagen Passat,0X2072R,137.08,118,Gasoline,8.3428\nVolkswagen Passat,0X2072T,13.54,118,Gasoline,1.4257\nVolkswagen Passat,0X2072T,105.13,118,Gasoline,7.2027\nVolkswagen Passat,0X2072R,100.89,118,Gasoline,7.9764\nVolkswagen Passat,0X2072T,8.23,118,Gasoline,0.7745\nVolkswagen Passat,0X2072S,45.65,118,Gasoline,3.1414\nVolkswagen Passat,0X2072R,149.52,118,Gasoline,12.3709\nVolkswagen Passat,0X2072T,10.87,118,Gasoline,1.289\nVolkswagen Passat,0X2072T,2.41,118,Gasoline,0.2876\nVolkswagen Passat,0X2072S,14.01,118,Gasoline,1.3775\nVolkswagen Passat,0X2072S,35.92,118,Gasoline,2.6957\nVolkswagen Passat,0X2072T,33.35,118,Gasoline,2.0519\nVolkswagen Passat,0X2072R,2.41,118,Gasoline,0.3361\nVolkswagen Passat,0X2072T,271.1,118,Gasoline,20.6155\nVolkswagen Passat,0X2072S,4.9,118,Gasoline,0.62\nVolkswagen Passat,0X2072S,36.94,118,Gasoline,2.3476\nVolkswagen Passat,0X2072R,6.8,118,Gasoline,0.8969\nVolkswagen Passat,0X2072T,6.87,118,Gasoline,0.7149\nVolkswagen Passat,0X2072R,158.02,118,Gasoline,13.0705\nVolkswagen Passat,0X2072S,55.74,118,Gasoline,3.7464\nVolkswagen Passat,0X2072T,8.89,118,Gasoline,0.7764\nVolkswagen Passat,0X2072S,102.6,118,Gasoline,10.2051\nVolkswagen Passat,0X2072T,6.34,118,Gasoline,0.6457\nVolkswagen Passat,0X2072R,1.97,118,Gasoline,0.2988\nVolkswagen Passat,0X2072R,63.41,118,Gasoline,4.7118\nVolkswagen Passat,0X2072T,23.19,118,Gasoline,2.3137\nVolkswagen Passat,0X2072R,15.03,118,Gasoline,1.4798\nVolkswagen Passat,0X2072R,9.69,118,Gasoline,1.3299\nVolkswagen Passat,0X2072S,137.0,118,Gasoline,12.6086\nVolkswagen Passat,0X2072T,17.54,118,Gasoline,1.6006\nVolkswagen Passat,0X2072T,10.78,118,Gasoline,0.8957\nVolkswagen Passat,0X2072T,292.93,118,Gasoline,20.9489\nVolkswagen Passat,0X2072T,153.13,118,Gasoline,14.2016\nVolkswagen Passat,0X2072T,16.32,118,Gasoline,3.124\nVolkswagen Passat,0X2072R,258.29,118,Gasoline,17.3402\nVolkswagen Passat,0X2072S,14.52,118,Gasoline,1.2073\nVolkswagen Passat,0X2072S,16.22,118,Gasoline,1.0469\nVolkswagen Passat,0X2072T,16.1,118,Gasoline,1.332\nVolkswagen Passat,0X2072S,46.52,118,Gasoline,5.4687\nVolkswagen Passat,0X2072T,120.06,118,Gasoline,8.5563\nVolkswagen Passat,0X2072R,19.72,118,Gasoline,1.5605\nVolkswagen Passat,0X2072S,24.06,118,Gasoline,2.5773\nVolkswagen Passat,0X2072R,8.8,118,Gasoline,0.8878\nVolkswagen Passat,0X2072T,12.1,118,Gasoline,1.3674\nVolkswagen Passat,0X2072T,25.53,118,Gasoline,1.712\nVolkswagen Passat,0X2072S,108.67,118,Gasoline,11.2886\nVolkswagen Passat,0X2072S,4.06,118,Gasoline,0.5372\nVolkswagen Passat,0X2072S,82.38,118,Gasoline,6.9603\nVolkswagen Passat,0X2072S,34.29,118,Gasoline,2.3279\nVolkswagen Passat,0X2072R,12.06,118,Gasoline,1.4064\nVolkswagen Passat,0X2072T,20.29,118,Gasoline,1.4843\nVolkswagen Passat,0X2072T,18.53,118,Gasoline,1.7952\nVolkswagen Passat,0X2072S,119.16,118,Gasoline,11.1266\nVolkswagen Passat,0X2072T,5.11,118,Gasoline,0.6075\nVolkswagen Passat,0X2072R,30.23,118,Gasoline,2.9608\nVolkswagen Passat,0X2072T,164.4,118,Gasoline,11.3456\nVolkswagen Passat,0X2072T,84.44,118,Gasoline,8.4707\nVolkswagen Passat,0X2072T,15.09,118,Gasoline,1.4681\nVolkswagen Passat,0X2072T,1.66,118,Gasoline,0.2266\nVolkswagen Passat,0X2072S,11.96,118,Gasoline,2.1423\nVolkswagen Passat,0X2072S,257.14,118,Gasoline,23.6811\nVolkswagen Passat,0X2072S,273.36,118,Gasoline,17.6986\nVolkswagen Passat,0X2072T,80.25,118,Gasoline,5.9939\nVolkswagen Passat,0X2072S,75.56,118,Gasoline,9.1931\nVolkswagen Passat,0X2072S,3.89,118,Gasoline,0.4917\nVolkswagen Passat,0X2072T,104.06,118,Gasoline,12.3057\nVolkswagen Passat,0X2072T,103.98,118,Gasoline,11.8248\nVolkswagen Passat,0X2072R,10.46,118,Gasoline,0.8036\nVolkswagen Passat,0X2072S,166.92,118,Gasoline,11.6535\nVolkswagen Passat,0X2072S,125.15,118,Gasoline,9.0521\nVolkswagen Passat,0X2072T,37.8,118,Gasoline,3.3037\nVolkswagen Passat,0X2072S,62.8,118,Gasoline,5.6534\nVolkswagen Passat,0X2072T,4.88,118,Gasoline,0.4808\nVolkswagen Passat,0X2072S,60.58,118,Gasoline,5.0114\nVolkswagen Passat,0X2072R,4.98,118,Gasoline,0.4856\nVolkswagen Passat,0X2072R,2.79,118,Gasoline,0.2723\nVolkswagen Passat,0X2072S,1.7,118,Gasoline,0.2673\nVolkswagen Passat,0X2072R,2.79,118,Gasoline,0.2906\nVolkswagen Passat,0X2072T,40.8,118,Gasoline,5.4397\nVolkswagen Passat,0X2072R,123.41,118,Gasoline,9.9286\nVolkswagen Passat,0X2072T,21.12,118,Gasoline,2.8673\nVolkswagen Passat,0X2072T,27.79,118,Gasoline,2.5959\nVolkswagen Passat,0X2072T,31.73,118,Gasoline,3.8128\nVolkswagen Passat,0X2072R,5.57,118,Gasoline,0.7143\nVolkswagen Passat,0X2072S,3.99,118,Gasoline,0.4414\nVolkswagen Passat,0X2072S,1.76,118,Gasoline,0.172\nVolkswagen Passat,0X2072S,10.66,118,Gasoline,0.8773\nVolkswagen Passat,0X2072S,37.47,118,Gasoline,3.3474\nVolkswagen Passat,0X2072T,19.44,118,Gasoline,1.6743\nVolkswagen Passat,0X2072R,39.75,118,Gasoline,4.5446\nVolkswagen Passat,0X2072T,3.47,118,Gasoline,0.4379\nVolkswagen Passat,0X2072R,3.18,118,Gasoline,0.6272\nVolkswagen Passat,0X2072R,4.5,118,Gasoline,0.3386\nVolkswagen Passat,0X2072S,238.61,118,Gasoline,15.1552\nVolkswagen Passat,0X2072T,13.54,118,Gasoline,1.0961\nVolkswagen Passat,0X2072R,7.78,118,Gasoline,0.709\nVolkswagen Passat,0X2072T,8.46,118,Gasoline,1.1784\nVolkswagen Passat,0X2072R,4.47,118,Gasoline,0.5861\nVolkswagen Passat,0X2072T,48.88,118,Gasoline,3.7779\nVolkswagen Passat,0X2072S,102.39,118,Gasoline,7.6732\nVolkswagen Passat,0X2072R,25.28,118,Gasoline,2.4862\nVolkswagen Passat,0X2072T,4.0,118,Gasoline,0.5344\nVolkswagen Passat,0X2072S,167.79,118,Gasoline,16.2989\nVolkswagen Passat,0X2072S,51.08,118,Gasoline,3.8605\nVolkswagen Passat,0X2072S,12.95,118,Gasoline,2.3022\nVolkswagen Passat,0X2072T,18.48,118,Gasoline,2.099\nVolkswagen Passat,0X2072S,119.85,118,Gasoline,10.3736\nVolkswagen Passat,0X2072R,27.01,118,Gasoline,5.914\nVolkswagen Passat,0X2072R,209.24,118,Gasoline,14.5146\nVolkswagen Passat,0X2072T,37.63,118,Gasoline,4.3479\nVolkswagen Passat,0X2072S,17.05,118,Gasoline,2.1521\nVolkswagen Passat,0X2072T,108.09,118,Gasoline,7.5497\nVolkswagen Passat,0X2072T,38.17,118,Gasoline,4.1364\nVolkswagen Passat,0X2072R,3.77,118,Gasoline,0.5358\nVolkswagen Passat,0X2072R,63.01,118,Gasoline,4.7326\nVolkswagen Passat,0X2072S,136.21,118,Gasoline,8.1377\nVolkswagen Passat,0X2072S,153.73,118,Gasoline,9.8338\nVolkswagen Passat,0X2072S,95.38,118,Gasoline,11.2222\nVolkswagen Passat,0X2072T,34.56,118,Gasoline,2.7673\nVolkswagen Passat,0X2072S,104.03,118,Gasoline,8.147\nVolkswagen Passat,0X2072S,8.01,118,Gasoline,0.8733\nVolkswagen Passat,0X2072S,11.37,118,Gasoline,0.6376\nVolkswagen Passat,0X2072T,16.51,118,Gasoline,1.5442\nVolkswagen Passat,0X2072R,5.12,118,Gasoline,0.5972\nVolkswagen Passat,0X2072T,18.05,118,Gasoline,1.3538\nVolkswagen Passat,0X2072T,16.96,118,Gasoline,1.8003\nVolkswagen Passat,0X2072S,364.66,118,Gasoline,25.0547\nVolkswagen Passat,0X2072T,108.84,118,Gasoline,7.7649\nVolkswagen Passat,0X2072S,237.04,118,Gasoline,18.2991\nVolkswagen Passat,0X2072S,70.07,118,Gasoline,4.0896\nVolkswagen Passat,0X2072T,1.74,118,Gasoline,0.2307\nVolkswagen Passat,0X2072R,39.74,118,Gasoline,3.273\nVolkswagen Passat,0X2072R,30.85,118,Gasoline,3.1764\nVolkswagen Passat,0X2072S,102.66,118,Gasoline,7.5983\nVolkswagen Passat,0X2072S,61.21,118,Gasoline,5.9935\nVolkswagen Passat,0X2072R,197.74,118,Gasoline,13.3316\nVolkswagen Passat,0X2072T,13.02,118,Gasoline,1.2149\nVolkswagen Passat,0X2072R,3.66,118,Gasoline,0.4528\nVolkswagen Passat,0X2072T,25.71,118,Gasoline,2.4764\nVolkswagen Passat,0X2072T,133.17,118,Gasoline,13.5086\nVolkswagen Passat,0X2072S,13.37,118,Gasoline,2.3227\nVolkswagen Passat,0X2072T,43.62,118,Gasoline,4.1807\nVolkswagen Passat,0X2072S,22.98,118,Gasoline,2.0675\nVolkswagen Passat,0X2072S,3.37,118,Gasoline,0.4645\nVolkswagen Passat,0X2072R,29.62,118,Gasoline,2.6973\nVolkswagen Passat,0X2072R,8.1,118,Gasoline,0.9236\nVolkswagen Passat,0X2072T,44.56,118,Gasoline,4.3483\nVolkswagen Passat,0X2072T,10.12,118,Gasoline,1.1\nVolkswagen Passat,0X2072S,103.58,118,Gasoline,9.6398\nVolkswagen Passat,0X2072T,8.63,118,Gasoline,1.6098\nVolkswagen Passat,0X2072R,124.96,118,Gasoline,7.6279\nVolkswagen Passat,0X2072S,45.96,118,Gasoline,3.7409\nVolkswagen Passat,0X2072R,145.44,118,Gasoline,9.093\nVolkswagen Passat,0X2072R,4.72,118,Gasoline,0.6269\nVolkswagen Passat,0X2072R,134.88,118,Gasoline,13.8572\nVolkswagen Passat,0X2072T,10.4,118,Gasoline,1.3477\nVolkswagen Passat,0X2072R,46.07,118,Gasoline,3.0879\nVolkswagen Passat,0X2072T,104.33,118,Gasoline,12.494\nVolkswagen Passat,0X2072S,72.4,118,Gasoline,8.4018\nVolkswagen Passat,0X2072T,34.95,118,Gasoline,3.6648\nVolkswagen Passat,0X2072S,22.16,118,Gasoline,3.25\nVolkswagen Passat,0X2072S,21.65,118,Gasoline,1.9768\nVolkswagen Passat,0X2072R,1.59,118,Gasoline,0.2289\nVolkswagen Passat,0X2072R,327.42,118,Gasoline,21.2782\nVolkswagen Passat,0X2072S,27.43,118,Gasoline,2.3088\nVolkswagen Passat,0X2072R,16.16,118,Gasoline,1.4748\nVolkswagen Passat,0X2072R,177.95,118,Gasoline,13.3453\nVolkswagen Passat,0X2072S,70.51,118,Gasoline,6.662\nVolkswagen Passat,0X2072S,32.76,118,Gasoline,3.0387\nVolkswagen Passat,0X2072R,87.41,118,Gasoline,10.9554\nVolkswagen Passat,0X2072S,11.97,118,Gasoline,1.6555\nVolkswagen Passat,0X2072R,5.62,118,Gasoline,0.7033\nVolkswagen Passat,0X2072R,46.96,118,Gasoline,5.815\nVolkswagen Passat,0X2072S,122.05,118,Gasoline,12.879\nVolkswagen Passat,0X2072R,3.84,118,Gasoline,0.5027\nVolkswagen Passat,0X2072R,2.33,118,Gasoline,0.2431\nVolkswagen Passat,0X2072T,107.35,118,Gasoline,11.5603\nVolkswagen Passat,0X2072R,113.25,118,Gasoline,7.1154\nVolkswagen Passat,0X2072R,129.54,118,Gasoline,8.5281\nVolkswagen Passat,0X2072R,9.38,118,Gasoline,1.0104\nVolkswagen Passat,0X2072T,12.03,118,Gasoline,0.9737\nVolkswagen Passat,0X2072S,4.7,118,Gasoline,0.5503\nVolkswagen Passat,0X2072T,3.97,118,Gasoline,0.3224\nVolkswagen Passat,0X2072S,120.5,118,Gasoline,13.4684\nVolkswagen Passat,0X2072T,221.11,118,Gasoline,16.5594\nVolkswagen Passat,0X2072S,178.15,118,Gasoline,11.5255\nVolkswagen Passat,0X2072R,13.17,118,Gasoline,1.3024\nVolkswagen Passat,0X2072S,240.8,118,Gasoline,20.6385\nVolkswagen Passat,0X2072S,126.72,118,Gasoline,12.1778\nVolkswagen Passat,0X2072S,60.0,118,Gasoline,5.0991\nVolkswagen Passat,0X2072S,50.0,118,Gasoline,3.8615\nVolkswagen Passat,0X2072S,306.21,118,Gasoline,19.7999\nVolkswagen Passat,0X2072T,127.69,118,Gasoline,12.0868\nVolkswagen Passat,0X2072R,117.16,118,Gasoline,10.4308\nVolkswagen Passat,0X2072R,3.22,118,Gasoline,0.239\nVolkswagen Passat,0X2072T,52.89,118,Gasoline,5.5609\nVolkswagen Passat,0X2072S,27.72,118,Gasoline,1.7142\nVolkswagen Passat,0X2072S,23.1,118,Gasoline,1.7541\nVolkswagen Passat,0X2072R,171.27,118,Gasoline,13.9281\nVolkswagen Passat,0X2072R,238.73,118,Gasoline,16.4847\nVolkswagen Passat,0X2072R,22.81,118,Gasoline,1.9475\nVolkswagen Passat,0X2072R,9.7,118,Gasoline,1.3529\nVolkswagen Passat,0X2072T,101.81,118,Gasoline,8.2239\nVolkswagen Passat,0X2072R,5.22,118,Gasoline,0.6945\nVolkswagen Passat,0X2072T,103.46,118,Gasoline,10.9085\nVolkswagen Passat,0X2072R,70.64,118,Gasoline,6.1372\nVolkswagen Passat,0X2072T,8.53,118,Gasoline,1.0793\nVolkswagen Passat,0X2072S,15.8,118,Gasoline,1.9899\nVolkswagen Passat,0X2072S,103.29,118,Gasoline,11.8013\nVolkswagen Passat,0X2072R,20.96,118,Gasoline,1.7842\nVolkswagen Passat,0X2072T,30.66,118,Gasoline,2.3141\nVolkswagen Passat,0X2072R,90.08,118,Gasoline,7.6207\nVolkswagen Passat,0X2072T,61.64,118,Gasoline,7.0661\nVolkswagen Passat,0X2072T,22.06,118,Gasoline,1.5966\nVolkswagen Passat,0X2072R,103.59,118,Gasoline,10.0803\nVolkswagen Passat,0X2072R,1.57,118,Gasoline,0.2032\nVolkswagen Passat,0X2072S,37.8,118,Gasoline,3.6473\nVolkswagen Passat,0X2072T,76.74,118,Gasoline,5.518\nVolkswagen Passat,0X2072S,40.58,118,Gasoline,4.6565\nVolkswagen Passat,0X2072R,11.06,118,Gasoline,0.8886\nVolkswagen Passat,0X2072S,8.32,118,Gasoline,0.575\nVolkswagen Passat,0X2072T,23.12,118,Gasoline,2.6857\nVolkswagen Passat,0X2072T,4.83,118,Gasoline,0.6393\nVolkswagen Passat,0X2072R,24.17,118,Gasoline,1.7202\nVolkswagen Passat,0X2072R,16.8,118,Gasoline,1.5216\nVolkswagen Passat,0X2072R,103.83,118,Gasoline,12.2127\nVolkswagen Passat,0X2072S,23.17,118,Gasoline,1.879\nVolkswagen Passat,0X2072T,91.9,118,Gasoline,8.0854\nVolkswagen Passat,0X2072R,5.06,118,Gasoline,0.6388\nVolkswagen Passat,0X2072S,51.53,118,Gasoline,6.0058\nVolkswagen Passat,0X2072S,3.06,118,Gasoline,0.3665\nVolkswagen Passat,0X2072T,3.9,118,Gasoline,0.5265\nVolkswagen Passat,0X2072S,33.75,118,Gasoline,4.1647\nVolkswagen Passat,0X2072R,150.95,118,Gasoline,11.8726\nVolkswagen Passat,0X2072T,79.26,118,Gasoline,6.8209\nVolkswagen Passat,0X2072S,78.35,118,Gasoline,5.0538\nVolkswagen Passat,0X2072T,102.77,118,Gasoline,7.6547\nVolkswagen Passat,0X2072S,101.38,118,Gasoline,6.1882\nVolkswagen Passat,0X2072S,39.86,118,Gasoline,3.5863\nVolkswagen Passat,0X2072R,44.1,118,Gasoline,3.6031\nVolkswagen Passat,0X2072T,31.02,118,Gasoline,3.1968\nVolkswagen Passat,0X2072S,87.84,118,Gasoline,7.4176\nVolkswagen Passat,0X2072T,4.0,118,Gasoline,0.5627\nVolkswagen Passat,0X2072T,25.1,118,Gasoline,2.5064\nVolkswagen Passat,0X2072S,85.25,118,Gasoline,9.667\nVolkswagen Passat,0X2072T,32.1,118,Gasoline,2.8196\nVolkswagen Passat,0X2072S,31.69,118,Gasoline,3.4746\nVolkswagen Passat,0X2072R,24.96,118,Gasoline,2.285\nVolkswagen Passat,0X2072R,117.72,118,Gasoline,8.9545\nVolkswagen Passat,0X2072S,85.35,118,Gasoline,6.1502\nVolkswagen Passat,0X2072S,109.88,118,Gasoline,7.4798\nVolkswagen Passat,0X2072S,31.88,118,Gasoline,3.6235\nVolkswagen Passat,0X2072S,68.43,118,Gasoline,6.1394\nVolkswagen Passat,0X2072R,42.66,118,Gasoline,3.3318\nVolkswagen Passat,0X2072R,192.12,118,Gasoline,13.8259\nVolkswagen Passat,0X2072S,21.81,118,Gasoline,1.9359\nVolkswagen Passat,0X2072S,41.1,118,Gasoline,4.7486\nVolkswagen Passat,0X2072S,18.14,118,Gasoline,1.825\nVolkswagen Passat,0X2072T,2.36,118,Gasoline,0.3304\nVolkswagen Passat,0X2072R,3.39,118,Gasoline,0.223\nVolkswagen Passat,0X2072R,144.63,118,Gasoline,10.2558\nVolkswagen Passat,0X2072S,111.24,118,Gasoline,7.1283\nVolkswagen Passat,0X2072S,271.21,118,Gasoline,19.1249\nVolkswagen Passat,0X2072R,6.26,118,Gasoline,0.7311\nVolkswagen Passat,0X2072R,5.01,118,Gasoline,0.5667\nVolkswagen Passat,0X2072R,210.11,118,Gasoline,14.1765\nVolkswagen Passat,0X2072T,9.16,118,Gasoline,0.7972\nVolkswagen Passat,0X2072T,7.26,118,Gasoline,0.7847\nVolkswagen Passat,0X2072R,175.41,118,Gasoline,12.4947\nVolkswagen Passat,0X2072T,17.7,118,Gasoline,1.2334\nVolkswagen Passat,0X2072R,19.21,118,Gasoline,1.581\nVolkswagen Passat,0X2072S,151.18,118,Gasoline,10.9588\nVolkswagen Passat,0X2072R,32.38,118,Gasoline,2.5342\nVolkswagen Passat,0X2072R,224.36,118,Gasoline,14.7112\nVolkswagen Passat,0X2072R,224.61,118,Gasoline,14.7485\nVolkswagen Passat,0X2072T,42.2,118,Gasoline,2.9711\nVolkswagen Passat,0X2072R,5.52,118,Gasoline,0.6544\nVolkswagen Passat,0X2072S,31.47,118,Gasoline,1.8727\nVolkswagen Passat,0X2072T,17.23,118,Gasoline,1.3248\nVolkswagen Passat,0X2072S,1.79,118,Gasoline,0.225\nVolkswagen Passat,0X2072S,77.07,118,Gasoline,5.7299\nVolkswagen Passat,0X2072R,34.73,118,Gasoline,2.5898\nVolkswagen Passat,0X2072R,50.76,118,Gasoline,5.3701\nVolkswagen Passat,0X2072R,5.15,118,Gasoline,0.4882\nVolkswagen Passat,0X2072R,57.62,118,Gasoline,5.6231\nVolkswagen Passat,0X2072R,120.66,118,Gasoline,10.455\nVolkswagen Passat,0X2072S,30.94,118,Gasoline,2.0725\nVolkswagen Passat,0X2072T,54.65,118,Gasoline,4.6813\nVolkswagen Passat,0X2072R,5.78,118,Gasoline,0.6491\nVolkswagen Passat,0X2072T,36.95,118,Gasoline,2.4516\nVolkswagen Passat,0X2072S,1.77,118,Gasoline,0.2181\nVolkswagen Passat,0X2072T,4.68,118,Gasoline,0.4212\nVolkswagen Passat,0X2072S,2.53,118,Gasoline,0.292\nVolkswagen Passat,0X2072T,15.58,118,Gasoline,1.2978\nVolkswagen Passat,0X2072S,20.93,118,Gasoline,2.4605\nVolkswagen Passat,0X2072T,27.58,118,Gasoline,2.9986\nVolkswagen Passat,0X2072S,33.68,118,Gasoline,3.5491\nVolkswagen Passat,0X2072S,3.83,118,Gasoline,0.5065\nVolkswagen Passat,0X2072S,23.6,118,Gasoline,2.3583\nVolkswagen Passat,0X2072T,106.25,118,Gasoline,7.776\nVolkswagen Passat,0X2072R,124.66,118,Gasoline,11.089\nVolkswagen Passat,0X2072S,121.77,118,Gasoline,8.6275\nVolkswagen Passat,0X2072T,35.9,118,Gasoline,3.21\nVolkswagen Passat,0X2072R,8.04,118,Gasoline,0.7625\nVolkswagen Passat,0X2072S,121.67,118,Gasoline,8.9023\nVolkswagen Passat,0X2072S,2.48,118,Gasoline,0.3023\nVolkswagen Passat,0X2072S,17.56,118,Gasoline,2.0055\nVolkswagen Passat,0X2072R,21.62,118,Gasoline,2.4773\nVolkswagen Passat,0X2072T,13.69,118,Gasoline,1.2196\nVolkswagen Passat,0X2072T,42.43,118,Gasoline,3.1286\nVolkswagen Passat,0X2072R,3.66,118,Gasoline,0.3018\nVolkswagen Passat,0X2072S,2.52,118,Gasoline,0.3209\nVolkswagen Passat,0X2072S,15.66,118,Gasoline,1.4696\nVolkswagen Passat,0X2072R,16.31,118,Gasoline,1.4071\nVolkswagen Passat,0X2072R,101.99,118,Gasoline,9.9864\nVolkswagen Passat,0X2072S,2.11,118,Gasoline,0.2255\nVolkswagen Passat,0X2072S,46.24,118,Gasoline,4.3712\nVolkswagen Passat,0X2072R,29.16,118,Gasoline,3.0528\nVolkswagen Passat,0X2072T,35.27,118,Gasoline,2.8967\nVolkswagen Passat,0X2072R,25.18,118,Gasoline,3.757\nVolkswagen Passat,0X2072S,5.54,118,Gasoline,0.6832\nVolkswagen Passat,0X2072S,18.13,118,Gasoline,1.9549\nVolkswagen Passat,0X2072S,11.6,118,Gasoline,1.1532\nVolkswagen Passat,0X2072T,40.27,118,Gasoline,3.5429\nVolkswagen Passat,0X2072S,48.6,118,Gasoline,2.9685\nVolkswagen Passat,0X2072R,124.89,118,Gasoline,9.8022\nVolkswagen Passat,0X2072T,102.26,118,Gasoline,9.0359\nVolkswagen Passat,0X2072R,151.13,118,Gasoline,12.6978\nVolkswagen Passat,0X2072S,2.13,118,Gasoline,0.2689\nVolkswagen Passat,0X2072R,48.41,118,Gasoline,4.3469\nVolkswagen Passat,0X2072R,2.2,118,Gasoline,0.2955\nVolkswagen Passat,0X2072T,71.6,118,Gasoline,5.8994\nVolkswagen Passat,0X2072T,6.08,118,Gasoline,0.7597\nVolkswagen Passat,0X2072S,239.2,118,Gasoline,17.1487\nVolkswagen Passat,0X2072S,39.3,118,Gasoline,3.013\nVolkswagen Passat,0X2072S,24.01,118,Gasoline,1.6751\nVolkswagen Passat,0X2072T,115.33,118,Gasoline,9.259\nVolkswagen Passat,0X2072T,106.47,118,Gasoline,9.4473\nVolkswagen Passat,0X2072T,6.01,118,Gasoline,0.7236\nVolkswagen Passat,0X2072T,25.49,118,Gasoline,2.1181\nVolkswagen Passat,0X2072S,70.76,118,Gasoline,6.6219\nVolkswagen Passat,0X2072R,81.76,118,Gasoline,8.6124\nVolkswagen Passat,0X2072S,127.1,118,Gasoline,10.4388\nVolkswagen Passat,0X2072S,14.8,118,Gasoline,1.5607\nVolkswagen Passat,0X2072S,124.18,118,Gasoline,12.061\nVolkswagen Passat,0X2072R,21.95,118,Gasoline,1.7457\nVolkswagen Passat,0X2072S,10.26,118,Gasoline,1.0961\nVolkswagen Passat,0X2072T,24.65,118,Gasoline,2.0215\nVolkswagen Passat,0X2072R,3.25,118,Gasoline,0.3598\nVolkswagen Passat,0X2072T,10.62,118,Gasoline,1.1327\nVolkswagen Passat,0X2072S,17.24,118,Gasoline,1.1702\nVolkswagen Passat,0X2072T,100.78,118,Gasoline,10.0388\nVolkswagen Passat,0X2072R,23.76,118,Gasoline,1.7294\nVolkswagen Passat,0X2072R,109.36,118,Gasoline,7.4319\nVolkswagen Passat,0X2072S,3.72,118,Gasoline,0.3745\nVolkswagen Passat,0X2072R,2.45,118,Gasoline,0.286\nVolkswagen Passat,0X2072S,18.16,118,Gasoline,2.2879\nVolkswagen Passat,0X2072S,10.34,118,Gasoline,0.7393\nVolkswagen Passat,0X2072S,105.48,118,Gasoline,9.1797\nVolkswagen Passat,0X2072R,147.36,118,Gasoline,11.9156\nVolkswagen Passat,0X2072S,12.43,118,Gasoline,1.3353\nVolkswagen Passat,0X2072T,5.12,118,Gasoline,0.5692\nVolkswagen Passat,0X2072R,4.77,118,Gasoline,0.6158\nVolkswagen Passat,0X2072R,104.47,118,Gasoline,8.4174\nVolkswagen Passat,0X2072R,2.24,118,Gasoline,0.3962\nVolkswagen Passat,0X2072R,3.1,118,Gasoline,0.2856\nVolkswagen Passat,0X2072T,21.94,118,Gasoline,2.0229\nVolkswagen Passat,0X2072R,17.9,118,Gasoline,2.0853\nVolkswagen Passat,0X2072R,1.53,118,Gasoline,0.1435\nVolkswagen Passat,0X2072T,59.52,118,Gasoline,4.1186\nVolkswagen Passat,0X2072S,212.08,118,Gasoline,14.3049\nVolkswagen Passat,0X2072R,42.52,118,Gasoline,3.1342\nVolkswagen Passat,0X2072R,4.1,118,Gasoline,0.4013\nVolkswagen Passat,0X2072S,86.4,118,Gasoline,6.2197\nVolkswagen Passat,0X2072R,104.26,118,Gasoline,8.901\nVolkswagen Passat,0X2072S,3.78,118,Gasoline,0.6431\nVolkswagen Passat,0X2072T,20.89,118,Gasoline,1.8083\nVolkswagen Passat,0X2072S,2.3,118,Gasoline,0.2217\nVolkswagen Passat,0X2072T,17.82,118,Gasoline,1.4668\nVolkswagen Passat,0X2072T,15.07,118,Gasoline,1.516\nVolkswagen Passat,0X2072T,9.98,118,Gasoline,1.6219\nVolkswagen Passat,0X2072S,17.78,118,Gasoline,1.1397\nVolkswagen Passat,0X2072T,3.45,118,Gasoline,0.396\nVolkswagen Passat,0X2072R,4.95,118,Gasoline,0.6034\nVolkswagen Passat,0X2072S,23.39,118,Gasoline,1.7062\nVolkswagen Passat,0X2072T,37.78,118,Gasoline,3.4416\nVolkswagen Passat,0X2072R,2.34,118,Gasoline,0.1816\nVolkswagen Passat,0X2072T,28.1,118,Gasoline,2.3589\nVolkswagen Passat,0X2072T,8.89,118,Gasoline,0.6757\nVolkswagen Passat,0X2072R,5.91,118,Gasoline,0.6186\nVolkswagen Passat,0X2072S,5.67,118,Gasoline,0.7052\nVolkswagen Passat,0X2072S,106.98,118,Gasoline,11.3806\nVolkswagen Passat,0X2072T,8.0,118,Gasoline,0.8803\nVolkswagen Passat,0X2072S,8.05,118,Gasoline,1.1607\nVolkswagen Passat,0X2072S,149.28,118,Gasoline,14.4109\nVolkswagen Passat,0X2072T,40.48,118,Gasoline,7.0457\nVolkswagen Passat,0X2072R,2.3,118,Gasoline,0.2595\nVolkswagen Passat,0X2072R,78.98,118,Gasoline,4.972\nVolkswagen Passat,0X2072S,147.06,118,Gasoline,11.8055\nVolkswagen Passat,0X2072T,120.05,118,Gasoline,12.6301\nVolkswagen Passat,0X2072R,148.46,118,Gasoline,10.2424\nVolkswagen Passat,0X2072T,216.55,118,Gasoline,16.9845\nVolkswagen Passat,0X2072S,32.19,118,Gasoline,2.7864\nVolkswagen Passat,0X2072T,4.93,118,Gasoline,0.5914\nVolkswagen Passat,0X2072R,116.05,118,Gasoline,9.6063\nVolkswagen Passat,0X2072S,16.18,118,Gasoline,1.6079\nVolkswagen Passat,0X2072T,2.62,118,Gasoline,0.2729\nVolkswagen Passat,0X2072S,30.62,118,Gasoline,2.4453\nVolkswagen Passat,0X2072R,67.31,118,Gasoline,5.9978\nVolkswagen Passat,0X2072R,38.38,118,Gasoline,2.9474\nVolkswagen Passat,0X2072S,2.52,118,Gasoline,0.4429\nVolkswagen Passat,0X2072R,87.04,118,Gasoline,7.5522\n"
for path, text in ((TRIPS_PATH, TRIPS_CSV), (VEHICLES_PATH, VEHICLES_CSV), (SUPPLEMENTARY_PATH, SUPPLEMENTARY_CSV)):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
print("Wrote", TRIPS_PATH, VEHICLES_PATH, SUPPLEMENTARY_PATH)

## 2. Preprocessing and diagnostics code (the web application's `model/preprocess.py` and `model/diagnostics.py`)
The two files are written next to the notebook and imported, so the saved model can be loaded by the web application.

In [ ]:
os.makedirs("model", exist_ok=True)
open(os.path.join("model", "__init__.py"), "a").close()

In [ ]:
%%writefile model/preprocess.py
"""
Shared preprocessing for the fuel-consumption Linear Regression model
(Chapter III, How the Project Will Work). Imported by BOTH train_model.py and
app.py, so the exact same features and encoding are used when the model is
trained and when it serves a live prediction.

Predictors (Scope): distance traveled (km), engine power (kW) and fuel type;
the predicted value is the fuel consumed per trip in liters. The model is

    y^ = b0 + b1(Distance) + b2(Engine Power) + b3(Fuel Type) + b4(Distance x Engine Power)

with fuel type encoded as 1 = Gasoline, 0 = Diesel.
"""
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

FUEL_TYPES = ("Diesel", "Gasoline")
FUEL_TYPE_CODES = {"Diesel": 0, "Gasoline": 1}  # Encoding: 1 represents gasoline and 0 represents diesel

# The three inputs a prediction needs (from the trip and the selected vehicle's record).
FEATURE_COLUMNS = ["distance_km", "engine_power_kw", "fuel_type"]
# The four terms of the regression equation, after encoding and the interaction term.
MODEL_TERMS = ["distance_km", "engine_power_kw", "fuel_type_code", "distance_x_power"]
TERM_LABELS = {
    "distance_km": "Distance (km)",
    "engine_power_kw": "Engine Power (kW)",
    "fuel_type_code": "Fuel Type (1 = Gasoline)",
    "distance_x_power": "Distance × Engine Power",
    "actual_fuel_consumption": "Fuel Consumed (L)",
}
TARGET_COLUMN = "actual_fuel_consumption"

# Kept for code that still asks for them: every model term is numeric now.
NUMERIC_FEATURES = MODEL_TERMS
CATEGORICAL_FEATURES = []

# The local Monthly Report of Official Travel export (one row per trip) and the
# vehicle records it is merged with on the plate number.
TRIP_COLUMNS = ["trip_date", "plate_number", "distance_km", TARGET_COLUMN]
VEHICLE_COLUMNS = ["plate_number", "brand_model", "fuel_type", "engine_power_kw"]


def encode_features(frame, centers=None):
    """Encoding and interaction term: fuel type as 1 (Gasoline) / 0 (Diesel),
    plus Distance x Engine Power for the higher fuel use per kilometre of
    larger engines. Returns the four model terms in a fixed order.

    With `centers` (mean distance, mean engine power of the training set),
    distance and power are mean-centered before the interaction is formed.
    Centering removes the collinearity between a predictor and an interaction
    built from it (high VIF) without changing any prediction."""
    frame = pd.DataFrame(frame)
    distance = pd.to_numeric(frame["distance_km"], errors="coerce").astype(float)
    power = pd.to_numeric(frame["engine_power_kw"], errors="coerce").astype(float)
    fuel = frame["fuel_type"].map(lambda v: FUEL_TYPE_CODES.get(str(v).strip().title(), float("nan"))).astype(float)
    if centers is not None:
        distance, power = distance - centers[0], power - centers[1]
    return pd.DataFrame({
        "distance_km": distance,
        "engine_power_kw": power,
        "fuel_type_code": fuel,
        "distance_x_power": distance * power,
    }, index=frame.index)


class TermEncoder(BaseEstimator, TransformerMixin):
    """Pipeline step: learns the mean distance and mean engine power of the
    training data (fit) and builds the four centered terms (transform)."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.distance_mean_ = float(pd.to_numeric(X["distance_km"], errors="coerce").mean())
        self.power_mean_ = float(pd.to_numeric(X["engine_power_kw"], errors="coerce").mean())
        return self

    def transform(self, X):
        return encode_features(X, centers=(self.distance_mean_, self.power_mean_))


def build_feature_encoder():
    """The encoding step bundled into the saved joblib pipeline in front of the
    regression, so a prediction is always encoded exactly like the training data."""
    return TermEncoder()


def raw_equation(intercept, coefficients, centers):
    """The fitted centered equation written back in the original units:
    y = b0 + b1(D - d) + b2(P - p) + b3 F + b4(D - d)(P - p)
      = [b0 - b1 d - b2 p + b4 d p] + (b1 - b4 p) D + (b2 - b4 d) P + b3 F + b4 D P."""
    b1, b2, b3, b4 = (coefficients[t] for t in MODEL_TERMS)
    d, p = centers
    return (intercept - b1 * d - b2 * p + b4 * d * p,
            {"distance_km": b1 - b4 * p, "engine_power_kw": b2 - b4 * d, "fuel_type_code": b3, "distance_x_power": b4})


def merge_trips_and_vehicles(trips, vehicles):
    """The trip records and the vehicle records are stored separately, so they are
    merged using the plate number (each trip gets its vehicle's engine power and
    fuel type)."""
    vehicles = vehicles[["plate_number", "fuel_type", "engine_power_kw"]].drop_duplicates("plate_number")
    trips = trips.drop(columns=[c for c in ("fuel_type", "engine_power_kw") if c in trips.columns])
    return trips.merge(vehicles, on="plate_number", how="left")


def supplementary_info(supplementary_path):
    """Describe the supplementary dataset: online trip records with the same
    variables (distance, engine power, fuel type, fuel consumed)."""
    columns = list(pd.read_csv(supplementary_path, nrows=0).columns)
    missing = [c for c in FEATURE_COLUMNS + [TARGET_COLUMN] if c not in columns]
    return {"kind": "trips", "dropped_features": [], "missing_columns": missing, "columns": columns}


def load_and_merge_datasets(historical_path, supplementary_path, extra_df=None):
    """Combine the local trips (the historical CSV plus the trip records in the
    system, already merged with their vehicles) and the supplementary dataset on
    their shared variables. Each record is labeled with its source
    (`data_source`), so the local trips can be identified after merging."""
    frames = []
    historical = pd.read_csv(historical_path)
    if not historical.empty:
        frames.append(historical.assign(data_source="historical"))
    if extra_df is not None and not extra_df.empty:
        frames.append(extra_df.assign(data_source="system"))
    supplementary = pd.read_csv(supplementary_path)
    frames.append(supplementary.assign(data_source="supplementary"))
    df = pd.concat(frames, ignore_index=True)
    for col in FEATURE_COLUMNS + [TARGET_COLUMN, "trip_date", "plate_number"]:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def clean_data(df):
    """Remove records with missing values or zero distance, then extreme
    outliers with the IQR rule on the fuel used per 100 km, within each group
    of vehicles with the same engine power (large vehicles are compared only
    with vehicles of similar size)."""
    df = df.copy()
    for col in ("distance_km", "engine_power_kw", TARGET_COLUMN):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["fuel_type"] = df["fuel_type"].astype("string").str.strip().str.title()
    df = df.dropna(subset=["distance_km", "engine_power_kw", "fuel_type", TARGET_COLUMN])
    df = df[df["fuel_type"].isin(FUEL_TYPES)]
    df = df[(df["distance_km"] > 0) & (df["engine_power_kw"] > 0) & (df[TARGET_COLUMN] > 0)]

    # IQR = Q3 - Q1; lower bound = Q1 - 1.5(IQR); upper bound = Q3 + 1.5(IQR).
    # The spread never counts as less than 25% of the group's median: a group
    # whose records all share one value (e.g. fuel written down as distance / 10)
    # would otherwise have an IQR of 0 and reject every real trip added later.
    def within_iqr(per_100km):
        q1, q3 = per_100km.quantile(0.25), per_100km.quantile(0.75)
        iqr = max(q3 - q1, 0.25 * per_100km.median())
        return per_100km.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

    per_100km = df[TARGET_COLUMN] / df["distance_km"] * 100
    keep = per_100km.groupby(df["engine_power_kw"], group_keys=False).apply(within_iqr)
    df = df[keep.reindex(df.index).fillna(False).astype(bool)]
    return df.reset_index(drop=True)


def build_prediction_frame(values):
    """A single-row frame from the trip distance and the selected vehicle's
    engine power and fuel type, in the schema the pipeline expects."""
    return pd.DataFrame([{
        "distance_km": float(values["distance_km"]),
        "engine_power_kw": float(values["engine_power_kw"]),
        "fuel_type": str(values["fuel_type"]),
    }])


In [ ]:
%%writefile model/diagnostics.py
"""
Regression diagnostics (Chapter II, Theoretical Background; Chapter III, How
the Project Will Work), computed with statsmodels:

* Variance Inflation Factor (VIF) of each predictor - a value above 5
  indicates high collinearity (reported; the equation's four terms are kept).
* LINE conditions on the residuals of an OLS fit of the same equation:
    Linearity        - Ramsey RESET test
    Independence     - Durbin-Watson statistic on the residuals of the LOCAL
                       trips arranged by the date of travel
    Normality        - Jarque-Bera test (with the residual histogram)
    Equal variance   - Breusch-Pagan test
* Coefficient standard errors and p-values use heteroskedasticity-robust
  (HC3) standard errors, which stay valid when the equal-variance condition
  is not met (larger errors on longer trips).
"""
import math
import warnings

import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson, jarque_bera

VIF_THRESHOLD = 5.0
ALPHA = 0.05
VIF_CAP = 1000.0  # perfect collinearity gives an infinite VIF; stored as ">1000"


def compute_vif(frame):
    """VIF_j = 1 / (1 - R_j^2) for every column of `frame`, with an intercept.
    A column with the same value in every record has no VIF (None)."""
    usable = [c for c in frame.columns if frame[c].astype(float).std() > 0]
    out = {c: None for c in frame.columns if c not in usable}
    if len(usable) < 2:
        out.update({c: 1.0 for c in usable})
        return out
    X = sm.add_constant(frame[usable].astype(float), has_constant="add")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # divide-by-zero for perfectly collinear columns
        for i, col in enumerate(X.columns):
            if col == "const":
                continue
            value = variance_inflation_factor(X.values, i)
            out[col] = VIF_CAP if (not math.isfinite(value) or value > VIF_CAP) else round(float(value), 2)
    return out


def _p(value):
    return None if value is None or not math.isfinite(float(value)) else round(float(value), 4)


def regression_diagnostics(terms_train, y_train, vif, local_residuals_by_date=None):
    """Fit OLS on the training split with the model's terms and test the LINE
    conditions. `local_residuals_by_date` are the residuals of the local trips
    in date order (for the Durbin-Watson test); without local trips the test
    falls back to the training residuals."""
    usable = [c for c in terms_train.columns if terms_train[c].astype(float).std() > 0]
    design = sm.add_constant(terms_train[usable].astype(float), has_constant="add")
    y = np.asarray(y_train, dtype=float)
    result = sm.OLS(y, design).fit()
    resid = np.asarray(result.resid)

    reset = linear_reset(result, power=2, use_f=True)
    if local_residuals_by_date is not None and len(local_residuals_by_date) >= 3:
        dw = float(durbin_watson(np.asarray(local_residuals_by_date, dtype=float)))
        dw_note = f"Residuals of the {len(local_residuals_by_date)} local trips arranged by date of travel."
    else:
        dw = float(durbin_watson(resid))
        dw_note = "No dated local trips yet: computed on the training residuals."
    jb_stat, jb_p, skew, kurtosis = jarque_bera(resid)
    bp_lm, bp_p, _, _ = het_breuschpagan(resid, design)
    vif_values = [v for v in vif.values() if v is not None]
    max_vif = max(vif_values) if vif_values else 0.0

    tests = [
        {"key": "linearity", "assumption": "Linearity", "test": "Ramsey RESET test",
         "statistic": round(float(reset.fvalue), 3), "p_value": _p(reset.pvalue),
         "passed": bool(reset.pvalue > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": "Fuel consumed is a straight-line function of the predictors."},
        {"key": "independence", "assumption": "Independence of errors", "test": "Durbin-Watson statistic",
         "statistic": round(dw, 3), "p_value": None,
         "passed": bool(1.5 <= dw <= 2.5), "rule": "1.5 to 2.5 (close to 2 = independent)",
         "meaning": dw_note},
        {"key": "normality", "assumption": "Normality of errors", "test": "Jarque-Bera test",
         "statistic": round(float(jb_stat), 3), "p_value": _p(jb_p),
         "passed": bool(jb_p > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": f"Residuals follow a normal distribution (skew {float(skew):.2f}, kurtosis {float(kurtosis):.2f})."},
        {"key": "equal_variance", "assumption": "Equal variance of errors", "test": "Breusch-Pagan test",
         "statistic": round(float(bp_lm), 3), "p_value": _p(bp_p),
         "passed": bool(bp_p > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": "The spread of errors is the same for short and long trips."},
        {"key": "multicollinearity", "assumption": "No multicollinearity", "test": "Variance Inflation Factor",
         "statistic": round(float(max_vif), 2), "p_value": None,
         "passed": bool(max_vif <= VIF_THRESHOLD), "rule": f"every VIF ≤ {VIF_THRESHOLD:g}",
         "meaning": "Predictors do not carry the same information (an interaction term usually shares "
                    "information with the predictors it is built from)."},
    ]
    # Same coefficients; HC3 robust standard errors for the significance tests.
    robust = sm.OLS(y, design).fit(cov_type="HC3")
    coefficients = []
    for name in ["const"] + usable:
        coefficients.append({
            "feature": "Intercept" if name == "const" else name,
            "coef": float(f"{float(robust.params[name]):.4g}"),
            "std_err": float(f"{float(robust.bse[name]):.4g}"),
            "p_value": _p(robust.pvalues[name]),
            "significant": bool(robust.pvalues[name] < ALPHA),
        })
    return {
        "tests": tests,
        "coefficients": coefficients,
        "constant_terms": [c for c in terms_train.columns if c not in usable],
        "ols_r2": round(float(result.rsquared), 4),
        "ols_adj_r2": round(float(result.rsquared_adj), 4),
        "alpha": ALPHA,
        "std_err_type": "HC3 (heteroskedasticity-robust)",
    }


In [ ]:
from model.preprocess import (FEATURE_COLUMNS, MODEL_TERMS, TARGET_COLUMN, TERM_LABELS, build_feature_encoder,
                              clean_data, merge_trips_and_vehicles, raw_equation)
from model.diagnostics import VIF_THRESHOLD, compute_vif, regression_diagnostics

## 3. Merge and clean the data
The trips are merged with the vehicle records using the plate number; each record is labeled with its source.

In [ ]:
trips = pd.read_csv(TRIPS_PATH)
vehicles = pd.read_csv(VEHICLES_PATH)
local = merge_trips_and_vehicles(trips, vehicles).assign(data_source="system")
supplementary = pd.read_csv(SUPPLEMENTARY_PATH).assign(data_source="supplementary")
raw = pd.concat([f for f in (local, supplementary) if not f.empty], ignore_index=True)
df = clean_data(raw)
print(f"Records before cleaning: {len(raw)}  after cleaning: {len(df)}")
print(df["data_source"].value_counts().to_string())
df.head()

## 4. Train-test split (80/20), encoding, interaction term and multicollinearity (VIF)
Fuel type is encoded 1 = Gasoline, 0 = Diesel. Distance and engine power are mean-centered on the training set before the interaction term is formed (this removes the collinearity of the interaction without changing any prediction). A VIF above 5 indicates high collinearity.

In [ ]:
X, y = df[FEATURE_COLUMNS], df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Training set: {len(X_train)}  Testing set: {len(X_test)}")
encoder = build_feature_encoder().fit(X_train)
centers = (encoder.distance_mean_, encoder.power_mean_)
print(f"Mean distance {centers[0]:.4f} km, mean engine power {centers[1]:.4f} kW")
terms = encoder.transform(X)
vif = compute_vif(terms)
display(pd.DataFrame({"Term": [TERM_LABELS[t] for t in vif], "VIF": list(vif.values()),
                      "High collinearity (VIF > 5)": [v is not None and v > VIF_THRESHOLD for v in vif.values()]}))
plt.figure(figsize=(8, 6))
cols = [c for c in MODEL_TERMS if terms[c].nunique() > 1]
corr = terms[cols].assign(**{TARGET_COLUMN: y.values}).corr().rename(index=TERM_LABELS, columns=TERM_LABELS)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap of the Terms"); plt.show()

## 5. Train the Linear Regression model

In [ ]:
pipeline = Pipeline(steps=[("encoder", build_feature_encoder()), ("regressor", LinearRegression())])
pipeline.fit(X_train, y_train)
reg = pipeline.named_steps["regressor"]
symbols = {"distance_km": "Distance", "engine_power_kw": "Power", "fuel_type_code": "FuelType", "distance_x_power": "Distance×Power"}
coef = dict(zip(MODEL_TERMS, map(float, reg.coef_)))
D, P = f"(D - {centers[0]:.4f})", f"(P - {centers[1]:.4f})"
centered = {"distance_km": D, "engine_power_kw": P, "fuel_type_code": "(FuelType)", "distance_x_power": D + P}
print("Centered:       ŷ = " + f"{reg.intercept_:.6g}" + "".join(f" {'-' if c < 0 else '+'} {abs(c):.6g}{centered[t]}" for t, c in coef.items()))
raw_b0, raw_coef = raw_equation(float(reg.intercept_), coef, centers)
print("Original units: ŷ = " + f"{raw_b0:.6g}" + "".join(f" {'-' if c < 0 else '+'} {abs(c):.6g}({symbols[t]})" for t, c in raw_coef.items()))

## 6. Evaluate — R², MAE, RMSE and 5-fold cross-validation

In [ ]:
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
kfold = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv = cross_validate(pipeline, X, y, cv=kfold, scoring=("r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"))
cv_scores, cv_mae, cv_rmse = cv["test_r2"], -cv["test_neg_mean_absolute_error"], -cv["test_neg_root_mean_squared_error"]
display(pd.DataFrame({"Test set": [r2, mae, rmse],
                      "CV mean": [cv_scores.mean(), cv_mae.mean(), cv_rmse.mean()],
                      "CV std": [cv_scores.std(), cv_mae.std(), cv_rmse.std()]},
                     index=["R²", "MAE (L)", "RMSE (L)"]).round(4))

## 7. Regression assumption checks (LINE conditions)
Durbin-Watson is computed on the residuals of the local trips arranged by date of travel. Coefficient standard errors and p-values are heteroskedasticity-robust (HC3).

In [ ]:
local_rows = df[df["data_source"] == "system"]
local_resid = None
if not local_rows.empty:
    local_rows = local_rows.assign(_date=pd.to_datetime(local_rows["trip_date"], errors="coerce")).sort_values(["_date"], kind="stable")
    local_resid = (local_rows[TARGET_COLUMN] - pipeline.predict(local_rows[FEATURE_COLUMNS])).to_numpy()
diagnostics = regression_diagnostics(encoder.transform(X_train), y_train, vif, local_resid)
display(pd.DataFrame(diagnostics["tests"])[["assumption", "test", "statistic", "p_value", "rule", "passed"]])
pd.DataFrame(diagnostics["coefficients"])

## 8. Evaluation plots — actual vs predicted, residual plot and residual histogram

In [ ]:
residuals = y_test.values - y_pred
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(y_test, y_pred, alpha=0.4); lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax[0].plot(lim, lim, "--", color="teal"); ax[0].set(xlabel="Actual Fuel Consumed (L)", ylabel="Predicted (L)", title="Actual vs Predicted")
ax[1].scatter(y_pred, residuals, alpha=0.4); ax[1].axhline(0, ls="--", color="teal")
ax[1].set(xlabel="Predicted (L)", ylabel="Residual (Actual − Predicted)", title="Residual Plot")
sns.histplot(residuals, kde=True, ax=ax[2]); ax[2].axvline(0, ls="--", color="teal")
ax[2].set(xlabel="Residual (L)", ylabel="Number of Test Trips", title="Histogram of Residuals")
plt.tight_layout(); plt.show()

## 9. Save the trained model (Joblib) for the web application

In [ ]:
joblib.dump(pipeline, "linear_regression_pipeline.joblib")
metrics = {"r2_score": round(r2, 4), "mae": round(mae, 4), "rmse": round(rmse, 4), "cv_folds": CV_FOLDS,
           "cv_mean_r2": round(float(cv_scores.mean()), 4), "cv_std_r2": round(float(cv_scores.std()), 4),
           "cv_mean_mae": round(float(cv_mae.mean()), 4), "cv_std_mae": round(float(cv_mae.std()), 4),
           "cv_mean_rmse": round(float(cv_rmse.mean()), 4), "cv_std_rmse": round(float(cv_rmse.std()), 4),
           "train_size": len(X_train), "test_size": len(X_test), "total_records": len(df),
           "equation_terms": MODEL_TERMS, "intercept": float(reg.intercept_),
           "coefficients": coef, "centering": dict(zip(["distance_km", "engine_power_kw"], centers)),
           "raw_intercept": raw_b0, "raw_coefficients": raw_coef, "vif": vif, "diagnostics": diagnostics}
with open("metrics_colab.json", "w") as f:
    json.dump(metrics, f, indent=2)
try:
    from google.colab import files
    files.download("linear_regression_pipeline.joblib")
except ImportError:
    print("Saved linear_regression_pipeline.joblib and metrics_colab.json")

> The web application retrains this same model automatically whenever trip records or vehicle records change, so
> copying the `.joblib` file into the `model/` folder is optional. The notebook documents and reproduces the process.